# Home Credit – GCI – ver5 (bản gọn)

Viết lại từ `gci-ver4` dựa trên **kết quả đã chạy thật** của ver4. Mục tiêu: ít feature hơn, ít thí nghiệm thừa hơn, nhưng ghép lại các cải tiến đã được CV xác nhận mà ver4 chưa bao giờ ghép chung.

**Các kết quả của ver4 được dùng để quyết định (AUC trung bình 5 fold)**

| Thí nghiệm ver4 | AUC | Kết luận được dùng |
|---|---|---|
| B. 157 feature | 0.7589 | mốc so sánh |
| Bỏ feature gain = 0 (129 feature) | 0.7586 | bỏ chúng không mất gì |
| Bỏ feature tương quan > 0.95 (110 feature) | 0.7576 | trùng thông tin nhiều, bỏ được |
| PCA, interaction tự sinh, kNN target, k-means, OpenFE, meta-feature, stacking | ≤ 0.7589 | **bỏ hết** (không hơn nhiễu) |
| scale_pos_weight, SMOTE, oversample, EasyEnsemble | giảm 0.005 đến 0.042 | **bỏ hết** |
| D4. ext_source điền bằng ML | +0.0009 so với B | **giữ** |
| D9. target encoding OOF (occupation, organization) | +0.0011 so với D4 | **giữ**, nhưng tính lại trong từng fold |
| T2. lr 0.02, depth 4, 15 lá, colsample 0.3, lambda 10 | 0.7631 (+0.0033 so với lr 0.05) | **giữ** làm tham số chính |
| Optuna 60 trial (308 phút) | 0.7630 | không hơn T2, **bỏ** |

**Điểm mấu chốt:** tuning (T2) chạy trên bộ D4, còn target encoding (D9) chạy với tham số cũ. Hai cải tiến này **chưa từng được ghép chung**. Notebook này ghép chúng, thêm vài feature mới (đánh dấu là giả thuyết, có bước kiểm tra), rồi báo kết quả cùng cách đọc `delta ± delta_se` như ver4.

**Cấu trúc**
1. Nạp dữ liệu → 2. Feature (64 feature gốc, bỏ ~90 feature thừa) → 3. Khối feature mới (ML-impute, target encoding, count) → 4. Hàm CV → 5. Kiểm tra từng khối mới → 6. Mô hình cuối (LightGBM nhiều seed, tùy chọn CatBoost) → 7. Accuracy và ngưỡng → 8. File nộp bài.

> `DEBUG_FAST = True` ở ô đầu để chạy thử toàn bộ notebook trong vài phút (20 nghìn dòng, ít cây) và kiểm tra code trước khi chạy thật.

### Notebook chốt: blend tốt nhất, 5 seed, bản nộp

Chế độ `REPLAY = True`: không tìm kiếm, không thử nghiệm. Chỉ dựng feature, rồi **huấn luyện đúng một lần** blend cuối `LGB(6i) 0.6 + CatTune 0.4` với **5 seed** mỗi mô hình (trước là 3), sau đó 6s huấn luyện lại 100% dữ liệu cho `test_pred`, rồi mục 7 và 8 xuất `submission.csv` (xác suất, dùng cho AUC) và `submission_binary.csv`.

Kết quả huấn luyện lưu ở `replay5_lgb6i.npz` và `replay5_cat_tuned.npz` (chạy lại sẽ nạp, xóa file để huấn luyện lại).

### Chế độ `REPLAY` (mặc định bật)

`REPLAY = True` ở ô thiết lập: notebook **dùng thẳng cấu hình đã xác định** ở các lần chạy trước, không lặp lại các bước đã có kết quả.

- **Vẫn chạy:** nạp dữ liệu, dựng feature, ba khối feature, và dựng các khối cần để tạo ra cấu hình LightGBM tốt nhất.
- **Bỏ qua:** ablation mục 5, huấn luyện LightGBM/CatBoost mục 6, tìm kiếm từng bước của 6c, sàng lọc + xác nhận + huấn luyện lại của 6f và 6i, 6d, 6e, 6h, huấn luyện lại 100% ở 6s.
- **Cấu hình cố định dùng:** `+term` và `reg_alpha 5` (6c), khối `te_term` (6f), khối `ext_term_ctx` (6i). Đây là cấu hình LightGBM(6i) trong blend cuối.
- Sau đó chạy thẳng ô **6z**, ô dùng cấu hình này làm mô hình.
- Vì không huấn luyện mô hình cuối nên mục 7 và 8 (ngưỡng, file nộp) bị bỏ qua. Muốn có file nộp, đặt `REPLAY = False`.


## 1. Thiết lập & nạp dữ liệu

In [1]:
import os
import time
import warnings

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

SEED = 42
N_FOLDS = 5
REPLAY = True             # True: dùng thẳng cấu hình đã xác định ở các lần chạy trước. KHÔNG tìm kiếm, KHÔNG sàng lọc, KHÔNG huấn luyện lại bản cuối ở 6c/6f/6i/6h/6s; chỉ dựng feature rồi chạy thẳng ô 6z. False: chạy toàn bộ như cũ (mất vài giờ)
DEBUG_FAST = False        # True: chạy thử nhanh để kiểm tra code (20k dòng, ít cây)
RUN_ABLATION = not REPLAY       # False: bỏ qua mục 5 và dùng ["ml", "te_core"] (hai khối đã được ver4 xác nhận)
USE_CATBOOST = True       # thêm CatBoost vào mục 6 nếu tốt hơn LightGBM theo quy tắc delta_se
FINAL_SEEDS = (42, 7, 2024, 11, 99)        # 5 seed (trước đây 3)

KAGGLE_PATH = "/kaggle/input/datasets/kiethe/homecredit-gci"
DATA_PATH = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else "."
OUT_PATH = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
ID_COL, TARGET = "customer_loan_id", "target"
RENAME_MAP = {"sk_id_curr": "customer_loan_id", "amt_income_total": "income",
              "amt_credit": "credit_amount", "amt_annuity": "annuity"}


def load(name):
    df = pd.read_csv(f"{DATA_PATH}/{name}.csv")
    df.columns = df.columns.str.lower()
    return df.rename(columns=RENAME_MAP)


train, test = load("train"), load("test")
if DEBUG_FAST:
    train = train.sample(20_000, random_state=SEED).reset_index(drop=True)
    test = test.head(5_000).reset_index(drop=True)
y = train[TARGET].astype(int).to_numpy()
print("train:", train.shape, "| test:", test.shape, f"| target=1: {y.mean():.2%}")

def _replay_cache_load(path, sig):
    try:
        z = np.load(path, allow_pickle=False)
        if str(z["sig"]) != sig:
            return None
        r = {"oof": z["oof"], "test_pred": z["test_pred"], "scores": [float(v) for v in z["scores"]], "name": str(z["name"])}
        if "iters" in z.files:
            r["iters"] = [int(v) for v in z["iters"]]
        return r
    except Exception:
        return None

def _replay_cache_save(path, r, sig):
    try:
        np.savez(path, oof=r["oof"], test_pred=r["test_pred"], scores=np.array(r["scores"]), sig=np.array(sig), name=np.array(r["name"]),
                 iters=np.array(r.get("iters", []), dtype=int))
    except Exception as e:
        print("Không lưu được cache", path, ":", e)


train: (171202, 34) | test: (61500, 33) | target=1: 8.07%


## 2. Feature engineering (bản gọn)

**Nguyên tắc chọn.** Mọi feature đều tính theo từng dòng (không dùng `target`), nên tính chung trên train + test là hợp lệ.

**Đã bỏ so với ver4 (khoảng 90 cột)** và lý do, đều lấy từ kết quả ver4:
- **Trùng hoàn toàn với feature khác** (|Spearman| = 1.000 trong bảng 9.3 và 10.1; cây không phân biệt được hai biến đổi đơn điệu của nhau): `log_income`, `log_credit`, `payment_rate`, `annuity_credit_ratio` (≡ `credit_term`), `dscr` (≡ `annuity_income_ratio`), `ltv_proxy`, `down_payment_ratio` (≡ `goods_credit_ratio`), `income_credit_ratio` (≡ `credit_income_ratio`), `down_payment_proxy` (≡ `credit_goods_diff`), `disposable_income`, `income_annuity_diff`, `job_start_age`, `age_employed_diff`, `age_at_first_job_proxy`, `employment_length_gap`, `job_stability` (≡ `employed_age_ratio`), `phone_emp_work_sum`, `flag_phones_sum`, `car_age_owned`, `flag_emp_phone`, `days_employed_anom` (cả ba cùng nhận diện nhóm "không đi làm").
- **Gain = 0 trong cả 5 fold** (mô hình không bao giờ chia theo chúng): các cờ `ext1_missing`, `ext3_missing`, `credit_bureau_missing`, `bureau_hour_flag`, `negative_disposable`, `high_ltv`, `long_term_loan`, `dscr_lt_1`, `dscr_lt_1_5`, `ext_score_missing_any`, `ext2_only`, `senior_borrower`, `young_borrower`, `flag_own_car`, `phone_changed_today`, `ext_count`, các cột one-hot hiếm; hai cột `*_log_bin`; `grey_zone_x_dscr_risk`, `senior_grey_zone`, `ext2_grey_zone`.
- **Permutation ≤ 0 và không nằm trong top 40 theo đồng thuận**: các tích với `own_car_age`, `bureau_velocity_ratio`, `bureau_recent_ratio`, `region_income_ratio`, `children_ratio`, `credit_per_child`, `pensioner_*`, `young_stable_job`, `missing_count_row`, `ext_mul`, `ext_prod_12`, `ext_var`, `ext_range`, `ext_mean_vs_ext2`.
- **One-hot** của giới tính, loại thu nhập, tình trạng gia đình, nhà ở: thay bằng label encoding cộng target encoding (mục 3), vì nhiều cột one-hot có gain = 0.

**Giữ lại** 64 feature: toàn bộ top 30 theo đồng thuận của mục 9, cộng nhóm `ext_source` và các tỷ số tài chính còn lại.

In [2]:

EXT = ["ext_source_1", "ext_source_2", "ext_source_3"]

BUREAU = [
    "amt_req_credit_bureau_hour",
    "amt_req_credit_bureau_mon",
    "amt_req_credit_bureau_qrt",
    "amt_req_credit_bureau_year",
]

EDU_ORDER = [
    "Lower secondary",
    "Secondary / secondary special",
    "Incomplete higher",
    "Higher education",
    "Academic degree",
]

CAT_LABEL = [
    "name_contract_type",
    "code_gender",
    "name_income_type",
    "name_family_status",
    "name_housing_type",
    "occupation_type",
    "organization_type",
]

RAW_NUM = [
    "cnt_children", "income", "credit_amount", "annuity",
    "amt_goods_price", "region_population_relative", "own_car_age",
    "flag_work_phone", "cnt_fam_members", "region_rating_client",
] + BUREAU

TIME = [
    "age_years", "years_employed", "years_registration",
    "years_id_publish", "years_last_phone_change",
]

CATS = CAT_LABEL + ["name_education_type"]

# Tất cả feature sử dụng EXT_SOURCE
FE_EXT = EXT + [
    "ext_mean", "ext_min", "ext_max", "ext_std",
    "ext_weighted", "ext2_anchored_mean",
    "ext_prod_13", "ext_prod_23",
    "ext_mean_x_housing_rent", "ext_mean_x_gender_M",
    "ext_mean_x_education_low",
    "prod_ext_mean_log_income", "prod_ext_mean_credit_term",
    "income_ext_mean_ratio", "annuity_ext3_ratio",
    "ltv_x_ext_mean", "composite_risk",
    "age_x_employed_x_ext",

    # EXT aggregate features mới
    "ext_geometric_mean",
    "ext_harmonic_mean",
    "ext_log_mean",
    "ext_logit_mean",
    "ext_range",
    "ext_cv",
]

FE_OTHER = [
    "credit_term", "goods_credit_ratio", "annuity_goods_ratio",
    "credit_income_ratio", "annuity_income_ratio",
    "credit_goods_diff", "income_per_person",
    "employed_age_ratio", "id_issue_age", "id_publish_age",
    "registration_age", "details_change_sum",
    "credit_term_x_age", "car_age_ratio",
    "region_rating_income", "total_bureau_req",
]

BASE_FEATURES = RAW_NUM + TIME + CATS + FE_EXT + FE_OTHER

# Cột mã category để tạo target encoding.
# Cần xử lý encoding theo từng fold, không fit trên validation labels.
TE_CORE = ["c_occupation", "c_organization"]

TE_EXTRA = [
    "c_income_type", "c_family", "c_housing", "c_education",
    "c_occ_gender", "c_edu_gender", "c_edu_income", "c_org_gender",
]

CNT_COLS = ["cnt_goods_price", "cnt_credit", "cnt_annuity"]


def sdiv(a, b):
    """Chia an toàn: mẫu số bằng 0 -> NaN."""
    return a / b.replace(0, np.nan)


def make_features(df):
    d = df.copy()

    # --------------------------------------------------
    # 1. Thời gian
    # --------------------------------------------------
    # 365243 là sentinel của DAYS_EMPLOYED.
    d["days_employed"] = d["days_employed"].replace(365243, np.nan)

    d["age_years"] = -d["days_birth"] / 365.0
    d["years_employed"] = -d["days_employed"] / 365.0
    d["years_registration"] = -d["days_registration"] / 365.0
    d["years_id_publish"] = -d["days_id_publish"] / 365.0
    d["years_last_phone_change"] = -d["days_last_phone_change"] / 365.0

    # --------------------------------------------------
    # 2. Categorical và mã tổ hợp
    # --------------------------------------------------
    is_male = (d["code_gender"] == "M").astype(float)
    is_rent = (d["name_housing_type"] == "Rented apartment").astype(float)

    edu_ord = d["name_education_type"].map(
        {v: i for i, v in enumerate(EDU_ORDER)}
    )

    # Factorize trên cùng dataframe đầu vào để mã nhất quán
    # khi train và test được concat trước khi tạo feature.
    codes = {
        c: pd.factorize(d[c].fillna("Unknown"), sort=True)[0]
        for c in CAT_LABEL
    }

    edu_code = pd.factorize(
        d["name_education_type"].fillna("Unknown"),
        sort=True,
    )[0]

    def pair(a, b):
        values = (
            pd.Series(a, index=d.index).astype(str)
            + "_"
            + pd.Series(b, index=d.index).astype(str)
        )
        return pd.factorize(values, sort=True)[0]

    d["c_occupation"] = codes["occupation_type"]
    d["c_organization"] = codes["organization_type"]
    d["c_income_type"] = codes["name_income_type"]
    d["c_family"] = codes["name_family_status"]
    d["c_housing"] = codes["name_housing_type"]
    d["c_education"] = edu_code

    d["c_occ_gender"] = pair(
        codes["occupation_type"], codes["code_gender"]
    )
    d["c_edu_gender"] = pair(edu_code, codes["code_gender"])
    d["c_edu_income"] = pair(edu_code, codes["name_income_type"])
    d["c_org_gender"] = pair(
        codes["organization_type"], codes["code_gender"]
    )

    for c in CAT_LABEL:
        d[c] = codes[c]

    # Giáo dục được giữ theo thứ tự bậc đã khai báo.
    d["name_education_type"] = edu_ord

    # --------------------------------------------------
    # 3. EXT_SOURCE cơ bản
    # --------------------------------------------------
    e1, e2, e3 = (d[c] for c in EXT)
    ext = d[EXT]

    d["ext_mean"] = ext.mean(axis=1)
    d["ext_min"] = ext.min(axis=1)
    d["ext_max"] = ext.max(axis=1)
    d["ext_std"] = ext.std(axis=1)

    # Trọng số tương đối: EXT1 = 1, EXT2 = 2, EXT3 = 1.
    # Chuẩn hóa theo tổng trọng số của các nguồn thực sự có dữ liệu.
    weights = pd.Series([1.0, 2.0, 1.0], index=EXT)
    weighted_sum = ext.mul(weights).sum(axis=1, min_count=1)
    available_weight = ext.notna().mul(weights).sum(axis=1)

    d["ext_weighted"] = (
        weighted_sum / available_weight.replace(0, np.nan)
    )

    # Lấy EXT2 làm điểm neo; nếu thiếu thì dùng ext_mean.
    e2_anchor = e2.fillna(d["ext_mean"])
    d["ext2_anchored_mean"] = (
        2.0 * e2_anchor
        + e1.fillna(e2_anchor)
        + e3.fillna(e2_anchor)
    ) / 4.0

    d["ext_prod_13"] = e1 * e3
    d["ext_prod_23"] = e2 * e3

    # --------------------------------------------------
    # 4. EXT aggregate features mới
    # --------------------------------------------------
    # Các phép tổng hợp này yêu cầu đủ cả 3 nguồn.
    # Nhờ đó không nhầm việc thiếu nguồn với điểm thấp.
    all_ext = ext.notna().all(axis=1)
    ext_complete = ext.where(all_ext, axis=0)

    # 4.1 Trung bình nhân.
    # Nếu bất kỳ điểm nào bằng 0, geometric mean bằng 0.
    # min_count=3 ngăn trường hợp toàn NaN cho kết quả tích bằng 1.
    ext_product = ext_complete.clip(lower=0).prod(
        axis=1, min_count=3
    )
    d["ext_geometric_mean"] = ext_product.pow(1.0 / 3.0)

    # 4.2 Trung bình điều hòa.
    # Điểm bằng 0 được xử lý riêng để kết quả bằng 0.
    ext_zero = ext_complete.eq(0).any(axis=1)
    ext_safe = ext_complete.where(ext_complete != 0)
    reciprocal_sum = (1.0 / ext_safe).sum(axis=1, min_count=3)

    d["ext_harmonic_mean"] = (
        3.0 / reciprocal_sum.replace(0, np.nan)
    )
    d.loc[all_ext & ext_zero, "ext_harmonic_mean"] = 0.0

    # 4.3 Trung bình log1p.
    # Nén khác biệt giữa các giá trị lớn hơn.
    d["ext_log_mean"] = np.log1p(
        ext_complete.clip(lower=0)
    ).mean(axis=1)

    # 4.4 Trung bình logit.
    # Clip để tránh log(0) và chia cho 0.
    eps = 1e-6
    ext_clipped = ext_complete.clip(lower=eps, upper=1.0 - eps)
    ext_logit = np.log(ext_clipped / (1.0 - ext_clipped))
    d["ext_logit_mean"] = ext_logit.mean(axis=1)

    # 4.5 Range: chênh lệch giữa nguồn lớn nhất và nhỏ nhất.
    d["ext_range"] = (
        ext_complete.max(axis=1) - ext_complete.min(axis=1)
    )

    # 4.6 Coefficient of variation: độ phân tán tương đối.
    # Nếu mean gần 0, CV không ổn định -> NaN.
    d["ext_cv"] = (
        d["ext_std"]
        / d["ext_mean"].where(d["ext_mean"] > 1e-6)
    )

    # --------------------------------------------------
    # 5. Interaction giữa EXT và nhóm khách hàng
    # --------------------------------------------------
    d["ext_mean_x_housing_rent"] = d["ext_mean"] * is_rent
    d["ext_mean_x_gender_M"] = d["ext_mean"] * is_male

    edu_low = (edu_ord <= 1).astype(float).where(edu_ord.notna())
    d["ext_mean_x_education_low"] = d["ext_mean"] * edu_low

    # --------------------------------------------------
    # 6. Tỷ số tài chính
    # --------------------------------------------------
    d["credit_term"] = sdiv(d["credit_amount"], d["annuity"])
    d["goods_credit_ratio"] = sdiv(
        d["amt_goods_price"], d["credit_amount"]
    )
    d["annuity_goods_ratio"] = sdiv(
        d["annuity"], d["amt_goods_price"]
    )
    d["credit_income_ratio"] = sdiv(
        d["credit_amount"], d["income"]
    )
    d["annuity_income_ratio"] = sdiv(
        d["annuity"], d["income"]
    )
    d["credit_goods_diff"] = (
        d["credit_amount"] - d["amt_goods_price"]
    )
    d["income_per_person"] = sdiv(
        d["income"], d["cnt_fam_members"]
    )

    # --------------------------------------------------
    # 7. Interaction giữa EXT và tài chính
    # --------------------------------------------------
    log_income = np.log1p(d["income"].where(d["income"] >= 0))

    d["prod_ext_mean_log_income"] = d["ext_mean"] * log_income
    d["prod_ext_mean_credit_term"] = (
        d["ext_mean"] * d["credit_term"]
    )
    d["income_ext_mean_ratio"] = sdiv(
        d["income"], d["ext_mean"]
    )
    d["annuity_ext3_ratio"] = sdiv(d["annuity"], e3)

    d["ltv_x_ext_mean"] = (
        sdiv(d["credit_amount"], d["amt_goods_price"])
        * d["ext_mean"]
    )

    dscr = sdiv(d["income"], d["annuity"])
    ltv = sdiv(d["credit_amount"], d["amt_goods_price"])

    d["composite_risk"] = (
        1.0 / dscr.clip(lower=0.1)
    ) * (1.0 - d["ext_mean"].fillna(0.5)) * ltv.fillna(1.0)

    # --------------------------------------------------
    # 8. Tuổi, việc làm và giấy tờ
    # --------------------------------------------------
    d["employed_age_ratio"] = sdiv(
        d["years_employed"], d["age_years"]
    )

    # Tuổi tại thời điểm cấp giấy tờ.
    d["id_issue_age"] = (
        d["age_years"] - d["years_id_publish"]
    )

    # Giữ nguyên tên cũ, nhưng tránh tạo công thức cộng sai.
    # Biến này biểu diễn số năm từ lúc cấp giấy tờ đến hiện tại.
    d["id_publish_age"] = d["years_id_publish"]

    # Tuổi tại thời điểm đăng ký.
    d["registration_age"] = (
        d["age_years"] - d["years_registration"]
    )

    d["details_change_sum"] = d[
        [
            "years_last_phone_change",
            "years_registration",
            "years_id_publish",
        ]
    ].sum(axis=1, min_count=1)

    d["credit_term_x_age"] = (
        d["credit_term"] * d["age_years"]
    )

    # Không thay missing employment bằng 0.
    d["age_x_employed_x_ext"] = (
        d["age_years"] * d["years_employed"] * d["ext_mean"]
    )

    d["car_age_ratio"] = sdiv(
        d["own_car_age"], d["age_years"]
    )
    d["region_rating_income"] = (
        d["region_rating_client"] * log_income
    )

    d["total_bureau_req"] = d[BUREAU].sum(
        axis=1, min_count=1
    )

    # --------------------------------------------------
    # 9. Tần suất xuất hiện của giá trị
    # --------------------------------------------------
    d["cnt_goods_price"] = d["amt_goods_price"].map(
        d["amt_goods_price"].value_counts()
    )
    d["cnt_credit"] = d["credit_amount"].map(
        d["credit_amount"].value_counts()
    )
    d["cnt_annuity"] = d["annuity"].map(
        d["annuity"].value_counts()
    )

    # --------------------------------------------------
    # 10. Trả về toàn bộ feature
    # --------------------------------------------------
    keep = BASE_FEATURES + TE_CORE + TE_EXTRA + CNT_COLS

    return (
        d[keep]
        .replace([np.inf, -np.inf], np.nan)
        .astype("float32")
    )


# --------------------------------------------------
# Tạo feature trên train + test
# --------------------------------------------------
n_tr = len(train)

FULL = make_features(
    pd.concat(
        [train.drop(columns=TARGET), test],
        ignore_index=True,
    )
)

print(
    f"BASE_FEATURES: {len(BASE_FEATURES)} "
    f"| Tổng feature trả về: {FULL.shape[1]} "
    f"| Dữ liệu: {FULL.shape}"
)

print("\nThiếu nhiều nhất (%):")
print(
    (FULL[BASE_FEATURES].isna().mean() * 100)
    .sort_values(ascending=False)
    .head(10)
    .round(1)
    .to_string()
)

print("\nKiểm tra các EXT aggregate:")
print(
    FULL[
        [
            "ext_mean",
            "ext_weighted",
            "ext_geometric_mean",
            "ext_harmonic_mean",
            "ext_log_mean",
            "ext_logit_mean",
            "ext_range",
            "ext_cv",
        ]
    ].describe().round(4)
)


BASE_FEATURES: 70 | Tổng feature trả về: 83 | Dữ liệu: (232702, 83)

Thiếu nhiều nhất (%):
ext_geometric_mean    78.9
ext_log_mean          78.9
ext_harmonic_mean     78.9
ext_logit_mean        78.9
ext_range             78.9
ext_prod_13           78.9
ext_source_1          69.5
own_car_age           66.1
car_age_ratio         66.1
ext_prod_23           32.0

Kiểm tra các EXT aggregate:
          ext_mean  ext_weighted  ext_geometric_mean  ext_harmonic_mean  \
count  232542.0000   232542.0000          49000.0000         49000.0000   
mean        0.5071        0.5083              0.4802             0.4545   
std         0.1580        0.1606              0.1494             0.1648   
min         0.0000        0.0000              0.0062             0.0001   
25%         0.4073        0.4083              0.3783             0.3402   
50%         0.5265        0.5362              0.4905             0.4671   
75%         0.6282        0.6307              0.5929             0.5807   
max       

## 3. Ba khối feature mới

| Khối | Nội dung | Căn cứ | Mức chắc chắn |
|---|---|---|---|
| `ml` | `ext_source_1/2/3_ml`, `ext_mean_ml`: điền `ext_source` thiếu bằng LightGBM hồi quy từ các cột còn lại | D4 của ver4: +0.0009 (nested +0.0013) | đã đo |
| `te_core` | target encoding của `occupation_type`, `organization_type` | D9 của ver4: +0.0011 | đã đo, **nhưng ở ver4 tính OOF trên toàn train rồi mới CV** (nhãn của fold validation gián tiếp ảnh hưởng giá trị của hàng train). Ở đây tính **trong từng fold** |
| `te_extra` | target encoding của 4 biến phân loại còn lại và 4 cặp (nghề × giới, học vấn × giới, học vấn × loại thu nhập, tổ chức × giới) | giả thuyết mới | chưa đo, mục 5 quyết định |
| `cnt` | số lần xuất hiện của `amt_goods_price`, `credit_amount`, `annuity` trên train + test | giả thuyết mới | chưa đo, mục 5 quyết định |

Điền ML không dùng `target` nên fit trên train + test là hợp lệ. Target encoding dùng nhãn nên **bắt buộc** tính trong từng fold: hàng train của fold nhận giá trị từ CV lồng bên trong, hàng validation và test nhận giá trị từ toàn bộ train của fold, có làm trơn Laplace (`alpha = 20`).

In [3]:

# ============================================================
# LIGHTGBM IMPUTATION FOR EXT_SOURCE + FEATURE CONFIGURATION
# ============================================================

import numpy as np
import pandas as pd
import lightgbm as lgb
import time

# ------------------------------------------------------------
# 1. Khai báo feature dùng để dự đoán EXT_SOURCE bị thiếu
# ------------------------------------------------------------

EXT = [
    "ext_source_1",
    "ext_source_2",
    "ext_source_3",
]

BASE_PRED = list(dict.fromkeys(
    RAW_NUM + TIME + CATS + FE_OTHER
))

# Kiểm tra các cột cần thiết
required_cols = list(dict.fromkeys(BASE_PRED + EXT))
missing_cols = [c for c in required_cols if c not in FULL.columns]

if missing_cols:
    raise ValueError(
        f"FULL đang thiếu các cột cần thiết: {missing_cols}"
    )

# ------------------------------------------------------------
# 2. Hàm điền missing bằng LightGBM
# ------------------------------------------------------------

def impute_ext_ml(F, pred_cols, seed=SEED):
    """
    Điền missing cho EXT_SOURCE bằng LightGBM Regressor.

    - Chỉ huấn luyện trên các dòng có giá trị EXT_SOURCE quan sát được.
    - Giữ nguyên giá trị gốc không missing.
    - Chỉ dự đoán các dòng missing.
    - Giới hạn dự đoán trong [0, 1].
    - Trả về 3 cột EXT_SOURCE đã impute và ext_mean_ml.
    """

    out = {}

    for c in EXT:
        # Đầu vào gồm feature cơ sở và hai EXT_SOURCE còn lại
        cols = list(dict.fromkeys(
            pred_cols + [other for other in EXT if other != c]
        ))

        observed = F[c].notna().to_numpy()
        missing = ~observed

        # Nếu không có giá trị quan sát thì không thể train model
        if observed.sum() == 0:
            raise ValueError(
                f"{c} không có giá trị quan sát để huấn luyện imputer."
            )

        # Lấy dữ liệu train của imputer
        X_obs = F.loc[observed, cols]
        y_obs = F.loc[observed, c]

        # Huấn luyện LightGBM
        model = lgb.LGBMRegressor(
            objective="regression",
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=100,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        )

        model.fit(X_obs, y_obs)

        # Giữ nguyên các giá trị đã có
        values = F[c].to_numpy(dtype=float, copy=True)

        # Chỉ dự đoán các giá trị missing
        if missing.any():
            X_missing = F.loc[missing, cols]
            pred = model.predict(X_missing)
            values[missing] = np.clip(pred, 0, 1)

        out[f"{c}_ml"] = values

        print(
            f"{c}: observed={observed.sum():,}, "
            f"imputed={missing.sum():,}"
        )

    # Trung bình ba cột EXT_SOURCE sau khi impute
    out["ext_mean_ml"] = np.mean(
        [out[f"{c}_ml"] for c in EXT],
        axis=0
    )

    return pd.DataFrame(out, index=F.index).astype("float32")


# ------------------------------------------------------------
# 3. Chạy imputation
# ------------------------------------------------------------

t0 = time.time()

ML_DF = impute_ext_ml(FULL, BASE_PRED)

ML_COLS = list(ML_DF.columns)

# Tránh nối trùng cột nếu chạy lại cell
FULL = FULL.drop(
    columns=[c for c in ML_COLS if c in FULL.columns],
    errors="ignore"
)

FULL = pd.concat([FULL, ML_DF], axis=1)

print(f"\nImputation hoàn tất sau {time.time() - t0:.1f} giây")
print("Feature ML:", ML_COLS)
print("FULL shape:", FULL.shape)


# ------------------------------------------------------------
# 4. Chia lại train và test
# ------------------------------------------------------------

n_tr = len(train)

X_all = FULL.iloc[:n_tr].reset_index(drop=True)
X_test_all = FULL.iloc[n_tr:].reset_index(drop=True)

print("\nX_all shape:", X_all.shape)
print("X_test_all shape:", X_test_all.shape)


# ------------------------------------------------------------
# 5. Cấu hình feature blocks
# ------------------------------------------------------------

FEAT_BLOCKS = {
    "ml": ML_COLS,
    "cnt": CNT_COLS,
}

TE_BLOCKS = {
    "te_core": TE_CORE,
    "te_extra": TE_EXTRA,
}


def make_cfg(selected):
    feats = BASE_FEATURES + [
        c
        for block in selected
        if block in FEAT_BLOCKS
        for c in FEAT_BLOCKS[block]
    ]

    te = [
        c
        for block in selected
        if block in TE_BLOCKS
        for c in TE_BLOCKS[block]
    ]

    return list(dict.fromkeys(feats)), list(dict.fromkeys(te))


# ------------------------------------------------------------
# 6. Kiểm tra các cấu hình feature
# ------------------------------------------------------------

for sel in (
    [],
    ["ml", "te_core"],
    ["ml", "te_core", "te_extra", "cnt"],
):
    features, te_features = make_cfg(sel)

    print(
        f"{str(sel):<42} "
        f"{len(features) + len(te_features):>3} features"
    )


ext_source_1: observed=70,862, imputed=161,840
ext_source_2: observed=232,203, imputed=499
ext_source_3: observed=158,426, imputed=74,276

Imputation hoàn tất sau 19.9 giây
Feature ML: ['ext_source_1_ml', 'ext_source_2_ml', 'ext_source_3_ml', 'ext_mean_ml']
FULL shape: (232702, 87)

X_all shape: (171202, 87)
X_test_all shape: (61500, 87)
[]                                          70 features
['ml', 'te_core']                           76 features
['ml', 'te_core', 'te_extra', 'cnt']        87 features


## 4. Hàm CV

- Cùng 5 fold (`SPLITS`) cho mọi cấu hình, nên so sánh được theo cặp từng fold, với `delta ± delta_se` như ver4.
- Target encoding tính **một lần cho mỗi fold** rồi dùng chung cho mọi seed.
- Nhiều seed: AUC theo fold tính trên dự đoán OOF **đã trung bình qua các seed** (seed chỉ đổi cách lấy mẫu của mô hình, không đổi fold).
- Quy tắc kết luận: "tốt hơn" khi `delta > delta_se` **và** dương ở ít nhất 3 trong 5 fold (chặt hơn ver4 một chút, vì ver4 có vài kết luận chỉ dựa vào delta > se).

In [4]:
SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))


def fold_data(feats, te_cols, tr, va):
    Xtr, Xva = X_all.iloc[tr].reset_index(drop=True), X_all.iloc[va].reset_index(drop=True)
    Xte = X_test_all.reset_index(drop=True)
    f = list(feats)
    if te_cols:
        t_tr, (t_va, t_te) = target_encode(Xtr[te_cols], y[tr], [Xva[te_cols], Xte[te_cols]], te_cols)
        Xtr, Xva, Xte = (pd.concat([a, b], axis=1) for a, b in ((Xtr, t_tr), (Xva, t_va), (Xte, t_te)))
        f += list(t_tr.columns)
    return Xtr[f], Xva[f], Xte[f], f


def run_lgb_cv(feats, te_cols, params, seeds=(SEED,), es_rounds=200, name=""):
    t0 = time.time()
    oof, te_pred, imp, iters = np.zeros(len(y)), np.zeros(len(X_test_all)), None, []
    for tr, va in SPLITS:
        Xtr, Xva, Xte, f = fold_data(feats, te_cols, tr, va)
        imp = pd.Series(0.0, index=f) if imp is None else imp
        for s in seeds:
            m = lgb.LGBMClassifier(**{**params, "random_state": s})
            m.fit(Xtr, y[tr], eval_set=[(Xva, y[va])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(es_rounds, verbose=False), lgb.log_evaluation(0)])
            oof[va] += m.predict_proba(Xva)[:, 1] / len(seeds)
            te_pred += m.predict_proba(Xte)[:, 1] / (len(seeds) * N_FOLDS)
            imp += m.booster_.feature_importance("gain") / (len(seeds) * N_FOLDS)
            iters.append(m.best_iteration_)
    scores = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
    print(f"[{name}] AUC = {np.mean(scores):.4f} ± {np.std(scores):.4f} | OOF AUC = {roc_auc_score(y, oof):.4f} "
          f"| log-loss = {log_loss(y, np.clip(oof, 1e-6, 1 - 1e-6)):.5f} | {len(f)} feature "
          f"| cây trung bình = {np.mean(iters):.0f} | {time.time() - t0:.0f}s")
    return {"oof": oof, "test_pred": te_pred, "scores": scores, "imp": imp.sort_values(ascending=False), "name": name}


def compare(new, ref):
    d = np.asarray(new["scores"]) - np.asarray(ref["scores"])
    se = d.std(ddof=1) / np.sqrt(len(d))
    pos = int((d > 0).sum())
    verdict = "tốt hơn" if (d.mean() > se and pos >= 3) else ("kém hơn" if d.mean() < -se else "≈ như")
    return {"delta": d.mean(), "delta_se": se, "fold_duong": f"{pos}/{len(d)}", "verdict": verdict}

## 5. Tham số và kiểm tra từng khối mới

**Bộ tham số chính** (dựa trên lưới 10.11 của ver4, đã chạy thật):

| Tham số | Giá trị | Vì sao |
|---|---|---|
| `learning_rate` | 0.02 | riêng việc hạ từ 0.05 xuống 0.02 đã cho +0.0020 (T0) |
| `num_leaves` / `max_depth` | 15 / 4 | cây nông: T1, T2, T3 đều ≈ 0.7630; cây sâu hơn không giúp |
| `colsample_bytree` | 0.4 | vùng 0.3 đến 0.5 phẳng (T1 0.7630, T4 0.7627). ver4 có 161 cột, bản này khoảng 75, nên nâng nhẹ từ 0.3 để mỗi cây vẫn thấy cỡ 30 cột |
| `min_child_samples` | 100 | như T2 |
| `reg_lambda` / `reg_alpha` | 10 / 0.1 | như T2 |
| `subsample` | 0.8 (freq 1) | như T2 |
| `n_estimators` | 8000 + early stopping 200 | cây dừng ở cỡ 1000 đến 2500 |

**Không chạy Optuna lại:** 60 trial mất 308 phút và chỉ cho 0.76300 so với 0.76306 của T2 (hơn kém nhau trong nhiễu). Vùng tham số này phẳng.

**Kiểm tra khối mới:** chọn tiến theo từng khối (`ml` → `te_core` → `te_extra` → `cnt`), mỗi bước so cấu hình có khối với cấu hình đã chọn, bằng `lr = 0.05` cho nhanh. Khối nào "tốt hơn" theo quy tắc ở mục 4 thì được giữ. Lưu ý: chọn 4 khối trên cùng các fold dùng để báo kết quả nên hơi lạc quan, nhưng chỉ có 4 quyết định nhị phân nên mức lạc quan nhỏ.

In [5]:

# ============================================================
# TARGET ENCODING + ABLATION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# ------------------------------------------------------------
# 1. Target encoding với smoothing
# ------------------------------------------------------------

def _smooth_map(keys, yv, gm, alpha=20.0):
    g = pd.DataFrame({
        "k": np.asarray(keys),
        "y": np.asarray(yv)
    })

    # Missing category được giữ như một nhóm riêng
    g["k"] = g["k"].astype("object").where(
        g["k"].notna(), "__MISSING__"
    )

    stats = g.groupby("k", dropna=False)["y"].agg(
        ["sum", "count"]
    )

    return (stats["sum"] + alpha * gm) / (
        stats["count"] + alpha
    )


def target_encode(
    tr_X,
    ytr,
    others,
    cols,
    alpha=20.0,
    inner=5,
    seed=SEED
):
    """
    Train:
        Dùng inner-fold OOF target encoding để tránh mỗi dòng
        tự sử dụng target của chính nó.

    Validation/test:
        Dùng mapping học từ toàn bộ training fold.
    """

    ytr = np.asarray(ytr)
    gm = float(ytr.mean())
    names = [f"{c}_te" for c in cols]

    te_tr = np.full(
        (len(tr_X), len(cols)),
        gm,
        dtype=np.float64
    )

    inner_cv = StratifiedKFold(
        n_splits=inner,
        shuffle=True,
        random_state=seed
    )

    # OOF encoding cho training fold
    for a, b in inner_cv.split(tr_X, ytr):
        gm_inner = float(ytr[a].mean())

        for j, c in enumerate(cols):
            mapping = _smooth_map(
                tr_X.iloc[a][c].to_numpy(),
                ytr[a],
                gm_inner,
                alpha
            )

            keys_b = tr_X.iloc[b][c].astype(
                "object"
            ).where(
                tr_X.iloc[b][c].notna(),
                "__MISSING__"
            )

            te_tr[b, j] = (
                keys_b.map(mapping)
                .fillna(gm_inner)
                .to_numpy()
            )

    # Mapping fit trên toàn bộ training fold
    maps = {
        c: _smooth_map(
            tr_X[c].to_numpy(),
            ytr,
            gm,
            alpha
        )
        for c in cols
    }

    encoded_others = []

    for other in others:
        data = {}

        for c in cols:
            keys = other[c].astype("object").where(
                other[c].notna(),
                "__MISSING__"
            )

            data[f"{c}_te"] = (
                keys.map(maps[c])
                .fillna(gm)
                .to_numpy()
            )

        encoded_others.append(
            pd.DataFrame(data, index=other.index)
        )

    te_train = pd.DataFrame(
        te_tr,
        columns=names,
        index=tr_X.index
    )

    return te_train, encoded_others


# ------------------------------------------------------------
# 2. Kiểm tra các hàm và dữ liệu cần thiết
# ------------------------------------------------------------

assert callable(run_lgb_cv), (
    "Chưa định nghĩa run_lgb_cv(). Hãy chạy cell định nghĩa hàm này."
)

assert callable(fold_data), (
    "Chưa định nghĩa fold_data(). Hãy chạy cell định nghĩa hàm này."
)

assert "y" in globals(), "Chưa có biến y."
assert "X_all" in globals(), "Chưa có biến X_all."
assert "X_test_all" in globals(), "Chưa có biến X_test_all."

# ------------------------------------------------------------
# 3. Khai báo tham số
# ------------------------------------------------------------

PARAMS = dict(
    objective="binary",
    n_estimators=8000,
    learning_rate=0.02,
    num_leaves=15,
    max_depth=4,
    min_child_samples=100,
    colsample_bytree=0.4,
    subsample=0.8,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=10.0,
    n_jobs=-1,
    verbose=-1
)

FAST_PARAMS = {
    **PARAMS,
    "learning_rate": 0.05,
    "n_estimators": 3000
}

ES_FAST, ES_FINAL = 100, 200

if DEBUG_FAST:
    PARAMS.update(
        n_estimators=300,
        learning_rate=0.1
    )

    FAST_PARAMS.update(
        n_estimators=200,
        learning_rate=0.1
    )

    ES_FAST, ES_FINAL, FINAL_SEEDS = (
        30, 30, (SEED,)
    )


# ------------------------------------------------------------
# 4. Ablation: thử thêm từng block
# ------------------------------------------------------------

if RUN_ABLATION:
    selected = []
    rows = []

    # Baseline
    f0, t0_ = make_cfg(selected)

    cur = run_lgb_cv(
        f0,
        t0_,
        FAST_PARAMS,
        es_rounds=ES_FAST,
        name="base"
    )

    rows.append({
        "khối": "base",
        "n_feature": len(f0) + len(t0_),
        "AUC": np.mean(cur["scores"]),
        "giữ": "-"
    })

    # Thử từng block theo thứ tự
    for blk in ["ml", "te_core", "te_extra", "cnt"]:
        candidate = selected + [blk]
        f1, t1 = make_cfg(candidate)

        new = run_lgb_cv(
            f1,
            t1,
            FAST_PARAMS,
            es_rounds=ES_FAST,
            name=f"+{blk}"
        )

        c = compare(new, cur)
        keep = c["verdict"] == "tốt hơn"

        rows.append({
            "khối": f"+{blk}",
            "n_feature": len(f1) + len(t1),
            "AUC": np.mean(new["scores"]),
            **{
                k: c[k]
                for k in (
                    "delta",
                    "delta_se",
                    "fold_duong",
                    "verdict"
                )
            },
            "giữ": "có" if keep else "không"
        })

        if keep:
            selected = candidate
            cur = new

    display(
        pd.DataFrame(rows)
        .set_index("khối")
        .round(4)
    )

else:
    selected = ["ml", "te_core"]


# ------------------------------------------------------------
# 5. Cấu hình feature cuối
# ------------------------------------------------------------

FEATS, TE_USED = make_cfg(selected)

print(
    "Khối được giữ:", selected,
    "->", len(FEATS) + len(TE_USED),
    "feature"
)


Khối được giữ: ['ml', 'te_core'] -> 76 feature


## 6. Mô hình cuối

LightGBM với bộ tham số ở mục 5, trung bình `FINAL_SEEDS` seed (giảm phương sai, không tốn thêm nhãn). Sau đó thử CatBoost trên cùng feature và cùng fold, rồi trộn 50/50 trên thang logit. **Chỉ dùng bản trộn nếu "tốt hơn" LightGBM theo quy tắc `delta_se`**; nếu không thì giữ LightGBM đơn (ver4 cho thấy ensemble không thắng được mô hình đơn đã tuning).

In [6]:
if REPLAY:
    res_lgb = {"name": "LightGBM mục 6 (REPLAY: không huấn luyện lại)", "scores": [np.nan] * N_FOLDS, "oof": None, "test_pred": None, "imp": pd.Series(dtype=float)}
    FINAL = res_lgb
    print("REPLAY: bỏ qua huấn luyện LightGBM/CatBoost mục 6; dùng cấu hình cố định ở các ô sau.")
else:
    res_lgb = run_lgb_cv(FEATS, TE_USED, PARAMS, seeds=FINAL_SEEDS, es_rounds=ES_FINAL, name="LightGBM cuối")
    print("Tham chiếu ver4 (T2, 161 feature, 1 seed): AUC = 0.7631, log-loss = 0.24477")

    FINAL = res_lgb
    if USE_CATBOOST:
        try:
            from catboost import CatBoostClassifier

            def run_cat_cv(feats, te_cols):
                t0 = time.time()
                oof, te_pred = np.zeros(len(y)), np.zeros(len(X_test_all))
                for tr, va in SPLITS:
                    Xtr, Xva, Xte, _ = fold_data(feats, te_cols, tr, va)
                    m = CatBoostClassifier(iterations=200 if DEBUG_FAST else 4000, learning_rate=0.1 if DEBUG_FAST else 0.04,
                                           depth=6, l2_leaf_reg=10, loss_function="Logloss", eval_metric="AUC",
                                           od_type="Iter", od_wait=30 if DEBUG_FAST else 200, random_seed=SEED,
                                           thread_count=-1, verbose=False, allow_writing_files=False)
                    m.fit(Xtr, y[tr], eval_set=(Xva, y[va]), use_best_model=True)
                    oof[va] = m.predict_proba(Xva)[:, 1]
                    te_pred += m.predict_proba(Xte)[:, 1] / N_FOLDS
                scores = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
                print(f"[CatBoost] AUC = {np.mean(scores):.4f} ± {np.std(scores):.4f} | OOF AUC = {roc_auc_score(y, oof):.4f} | {time.time() - t0:.0f}s")
                return {"oof": oof, "test_pred": te_pred, "scores": scores, "name": "CatBoost"}

            res_cat = run_cat_cv(FEATS, TE_USED)
            lg = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
            oof_b = expit((lg(res_lgb["oof"]) + lg(res_cat["oof"])) / 2)
            res_blend = {"oof": oof_b, "test_pred": expit((lg(res_lgb["test_pred"]) + lg(res_cat["test_pred"])) / 2),
                         "scores": [roc_auc_score(y[va], oof_b[va]) for _, va in SPLITS], "name": "Blend LGB+Cat"}
            c = compare(res_blend, res_lgb)
            ll = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
            print(f"[Blend] AUC = {np.mean(res_blend['scores']):.4f} | log-loss = {ll(res_blend):.5f} | "
                  f"so với LightGBM: delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}")
            if c["verdict"] == "tốt hơn" and ll(res_blend) <= ll(res_lgb) + 1e-5:
                FINAL = res_blend
        except ImportError:
            print("Không có catboost, bỏ qua.")
    print("Mô hình dùng cho mục 7 và 8:", FINAL["name"])

    top = res_lgb["imp"].head(20)[::-1]
    new_cols = set(ML_COLS) | set(CNT_COLS) | {f"{c}_te" for c in TE_CORE + TE_EXTRA}
    plt.figure(figsize=(7, 6))
    plt.barh(top.index, top.values, color=["#c0504d" if c in new_cols else "#4f81bd" for c in top.index])
    plt.title("Top 20 gain, LightGBM cuối (đỏ: feature mới của ver5)")
    plt.tight_layout()
    plt.show()

REPLAY: bỏ qua huấn luyện LightGBM/CatBoost mục 6; dùng cấu hình cố định ở các ô sau.


## 6c. Thử các cách tăng AUC cho mô hình này

Kết quả thật ở mục 6: LightGBM cuối 0.7639, CatBoost 0.7617, trộn hai mô hình **không** hơn (delta -0.0000). Nên cách tăng không nằm ở việc thêm một mô hình cùng loại. Mục này thử các hướng **chưa từng thử** ở ver4 và ver5, mỗi hướng một bước, chọn tiến theo đúng quy tắc `compare` ở mục 4 (delta > delta_se và dương ở ít nhất 3/5 fold), tất cả ở `lr = 0.05` cho nhanh:

| Bước | Ý tưởng | Vì sao có thể giúp | Mức chắc chắn |
|---|---|---|---|
| `+extadv` | tổ hợp nâng cao của ba điểm ext đã điền: trung bình hình học, điều hòa, bậc 2, bậc 1/2, trung vị, khoảng cách giữa các điểm đã xếp, tích đôi và tích ba, tỷ số, bình phương, và tích của `ext_mean_ml` với tỷ số tài chính, tuổi, vùng | ext_mean chiếm 17.5% SHAP nên mọi tổ hợp phi tuyến của ba điểm đều đáng đo; **căn và log của một cột thì vô ích với cây** (cây bất biến với biến đổi đơn điệu), nên chỉ thử tổ hợp nhiều cột | giả thuyết, ver4 đã đo các tích tương tự và không thắng |
| `+resid` | phần dư `ext_source_i` = giá trị thật − dự đoán ngoài fold từ các cột còn lại (không dùng nhãn) | tách phần thông tin riêng của điểm ext, mà mô hình điền ML đã chứng minh có giá trị (+0.0010) | giả thuyết |
| `+extml` | min, max, std, hiệu từng cặp của ba điểm ext đã điền | độ bất đồng giữa ba nguồn | giả thuyết |
| `+term` | số kỳ ước lượng (`credit/annuity` làm tròn), phần lẻ, lãi ngầm | sản phẩm vay khác nhau có lãi và kỳ hạn khác nhau | giả thuyết |
| `extra_trees` | ngưỡng chia ngẫu nhiên | chống overfit trên nhãn nhiễu (8% dương) | giả thuyết |
| `reg_alpha 5` | phạt L1 mạnh | Optuna #48 của ver4 chọn alpha 8.4, lambda ≈ 0: gợi ý vùng này chưa được lưới thử | có gợi ý từ dữ liệu |
| `max_bin 63` | gộp giá trị thành ít bin hơn | làm mượt, chống overfit | giả thuyết |
| `sâu` (3 mức) | 31 lá depth 6, 63 lá depth 8, 127 lá không giới hạn | trả lời câu hỏi "tăng độ sâu có giúp không" bằng số, ở lr 0.05 | ver4 đã cho thấy 15, 31 lá và Optuna (tối đa 63 lá) đều không hơn; khả năng thấp |
| `monotone ext` | ép rủi ro chỉ giảm khi điểm ext tăng | SHAP ở ver4 cho thấy chiều tác động đơn điệu; ràng buộc loại bớt nhiễu | giả thuyết |
| CatBoost native | CatBoost nhận trực tiếp các biến phân loại (nghề, tổ chức, các cặp) thay vì target encoding | CatBoost mạnh nhất ở biến phân loại; bản CatBoost ở mục 6 chưa dùng điều này; đa dạng hơn LightGBM | giả thuyết |

**Đã loại trước khi thử** (vì ver4 đã đo, hoặc có bằng chứng sẵn): Optuna, thêm kNN/meta-feature/OpenFE, mất cân bằng lớp (giảm AUC), `cnt` (đã đo ở mục 5: kém hơn).

**Kỳ vọng thực tế:** mỗi bước cùng lắm +0.0003 đến +0.001, vì bộ một bảng này đã gần trần. Nếu không bước nào thắng, kết luận trung thực là LightGBM ở mục 6 đã là mô hình tốt nhất có thể trên dữ liệu này.

Mục này sửa `X_all`, `X_test_all` (thêm cột mới) và có thể gán lại `FINAL` để mục 7, 8 dùng mô hình tốt hơn. Đặt `RUN_IMPROVE = False` để bỏ qua.

In [7]:
RUN_IMPROVE = True
RUN_CAT_NATIVE = False       # đã có CatBoost tune ở 6h (thay thế bản native này), nên bỏ để tiết kiệm ~30 phút

if RUN_IMPROVE:
    from sklearn.model_selection import KFold

    # ---------- 1) Khối feature mới (không dùng nhãn, tính chung train + test) ----------
    def ext_resid(F, pred_cols, seed=SEED):
        res = {}
        for c in EXT:
            cols = pred_cols + [o for o in EXT if o != c]
            obs = np.where(F[c].notna().to_numpy())[0]
            pred = np.zeros(len(obs))
            for a, b in KFold(5, shuffle=True, random_state=seed).split(obs):
                m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed,
                                      n_jobs=-1, verbose=-1).fit(F.iloc[obs[a]][cols], F.iloc[obs[a]][c])
                pred[b] = m.predict(F.iloc[obs[b]][cols])
            r = np.full(len(F), np.nan)
            r[obs] = F[c].to_numpy(dtype=float)[obs] - pred
            res[f"{c}_resid"] = r
        out = pd.DataFrame(res, index=F.index)
        out["ext_resid_mean"] = out.mean(axis=1)
        return out

    t0 = time.time()
    B_resid = ext_resid(FULL, BASE_PRED)
    E = FULL[[f"{c}_ml" for c in EXT]]
    B_extml = pd.DataFrame({"ext_ml_min": E.min(axis=1), "ext_ml_max": E.max(axis=1), "ext_ml_std": E.std(axis=1),
                            "ext_ml_d12": E.iloc[:, 0] - E.iloc[:, 1], "ext_ml_d13": E.iloc[:, 0] - E.iloc[:, 2],
                            "ext_ml_d23": E.iloc[:, 1] - E.iloc[:, 2]})
    tn = FULL["credit_term"].round()
    B_term = pd.DataFrame({"term_n": tn, "term_frac": FULL["credit_term"] - tn})
    B_term["interest_total"] = (FULL["annuity"] * tn / FULL["credit_amount"] - 1).replace([np.inf, -np.inf], np.nan)
    B_term["interest_annual"] = (B_term["interest_total"] / (tn / 12).replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    # Tổ hợp nâng cao của ba điểm ext (dùng bản đã điền nên không NaN). Biến đổi một cột (căn, log) vô ích với cây,
    # nên chỉ có tổ hợp NHIỀU cột mới thêm thông tin: các loại trung bình, tích, tỷ số, khoảng cách giữa các điểm, tích với biến tài chính.
    a1, a2, a3 = (FULL[f"{c}_ml"].clip(1e-3, 1) for c in EXT)
    srt = np.sort(np.c_[a1, a2, a3], axis=1)
    B_adv = pd.DataFrame({
        "ext_geo": (a1 * a2 * a3) ** (1 / 3),
        "ext_harm": 3 / (1 / a1 + 1 / a2 + 1 / a3),
        "ext_pm2": np.sqrt((a1 ** 2 + a2 ** 2 + a3 ** 2) / 3),
        "ext_pm_half": ((np.sqrt(a1) + np.sqrt(a2) + np.sqrt(a3)) / 3) ** 2,
        "ext_med": srt[:, 1], "ext_gap_hi": srt[:, 2] - srt[:, 1], "ext_gap_lo": srt[:, 1] - srt[:, 0],
        "ext_p12": a1 * a2, "ext_p13": a1 * a3, "ext_p23": a2 * a3, "ext_p123": a1 * a2 * a3,
        "ext_r13": a1 / a3, "ext_r23": a2 / a3, "ext_r12": a1 / a2,
        "ext_sq_mean": ((a1 + a2 + a3) / 3) ** 2,
        "ext_x_ann_inc": FULL["ext_mean_ml"] * FULL["annuity_income_ratio"],
        "ext_x_cred_inc": FULL["ext_mean_ml"] * FULL["credit_income_ratio"],
        "ext_x_age": FULL["ext_mean_ml"] * FULL["age_years"],
        "ext_x_goods": FULL["ext_mean_ml"] * FULL["goods_credit_ratio"],
        "ext_x_region": FULL["ext_mean_ml"] * FULL["region_rating_client"],
        "ext_geo_x_term": ((a1 * a2 * a3) ** (1 / 3)) * FULL["credit_term"],
    }, index=FULL.index).replace([np.inf, -np.inf], np.nan)
    NEWB = {"resid": B_resid, "extml": B_extml, "term": B_term, "extadv": B_adv}
    NB = pd.concat(NEWB.values(), axis=1).astype("float32")
    X_all = pd.concat([X_all.drop(columns=[c for c in NB.columns if c in X_all.columns]),
                       NB.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in NB.columns if c in X_test_all.columns]),
                            NB.iloc[n_tr:].reset_index(drop=True)], axis=1)
    BLK = {k: list(v.columns) for k, v in NEWB.items()}
    print(f"Dựng khối mới xong ({time.time() - t0:.0f}s):", {k: len(v) for k, v in BLK.items()})

    # ---------- 2) Ràng buộc đơn điệu: chỉ áp cho cột mà tương quan thật sự âm với nhãn ----------
    MONO_CAND = [c for c in EXT + ["ext_mean", "ext_min", "ext_max", "ext_weighted", "ext2_anchored_mean"] + ML_COLS
                 if c in X_all.columns]
    MONO = set()
    for c in MONO_CAND:
        ok = X_all[c].notna().to_numpy()
        if np.corrcoef(X_all.loc[ok, c], y[ok])[0, 1] < 0:
            MONO.add(c)
    print(f"Đơn điệu giảm cho {len(MONO)}/{len(MONO_CAND)} cột ext (cột có tương quan dương bị bỏ ra)")

    # ---------- 3) CV có cấu hình ----------
    def run_cfg(cfg, base, seeds=(SEED,), es=100, name=""):
        t0 = time.time()
        oof, te_pred, iters, imp = np.zeros(len(y)), np.zeros(len(X_test_all)), [], None
        for tr, va in SPLITS:
            Xtr, Xva, Xte, f = fold_data(cfg["feats"], cfg["te"], tr, va)
            imp = pd.Series(0.0, index=f) if imp is None else imp
            prm = {**base, **cfg["params"]}
            if cfg["mono"]:
                prm["monotone_constraints"] = [-1 if c in MONO else 0 for c in f]
            for s in seeds:
                m = lgb.LGBMClassifier(**{**prm, "random_state": s})
                m.fit(Xtr, y[tr], eval_set=[(Xva, y[va])], eval_metric="auc",
                      callbacks=[lgb.early_stopping(es, verbose=False), lgb.log_evaluation(0)])
                oof[va] += m.predict_proba(Xva)[:, 1] / len(seeds)
                te_pred += m.predict_proba(Xte)[:, 1] / (len(seeds) * N_FOLDS)
                imp += m.booster_.feature_importance("gain") / (len(seeds) * N_FOLDS)
                iters.append(m.best_iteration_)
        scores = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
        print(f"[{name}] AUC = {np.mean(scores):.4f} ± {np.std(scores):.4f} | log-loss = {log_loss(y, np.clip(oof, 1e-6, 1 - 1e-6)):.5f} "
              f"| {len(f)} feature | cây TB = {np.mean(iters):.0f} | {time.time() - t0:.0f}s")
        return {"oof": oof, "test_pred": te_pred, "scores": scores, "imp": imp.sort_values(ascending=False), "name": name}

    cfg0 = dict(feats=list(FEATS), te=list(TE_USED), params={}, mono=False)
    add_feats = lambda k: (lambda c: {**c, "feats": c["feats"] + BLK[k]})
    add_param = lambda **kw: (lambda c: {**c, "params": {**c["params"], **kw}})
    STEPS = [("+extadv", add_feats("extadv")), ("+resid", add_feats("resid")), ("+extml", add_feats("extml")), ("+term", add_feats("term")),
             ("extra_trees", add_param(extra_trees=True)), ("reg_alpha 5", add_param(reg_alpha=5.0)),
             ("max_bin 63", add_param(max_bin=63)), ("monotone ext", lambda c: {**c, "mono": True}),
             ("sâu: 31 lá, depth 6", add_param(num_leaves=31, max_depth=6)),
             ("sâu: 63 lá, depth 8", add_param(num_leaves=63, max_depth=8, min_child_samples=200)),
             ("sâu: 127 lá, không giới hạn depth", add_param(num_leaves=127, max_depth=-1, min_child_samples=300))]

    # Kết quả tìm kiếm trước đó: chỉ "+term" và "reg_alpha 5" qua quy tắc delta_se. Dùng lại, không chạy lại 11 bước.
    IMPROVE_PRESET = ["+term", "reg_alpha 5"]   # None = chạy lại tìm kiếm từng bước (mất khoảng 15-20 phút)
    if IMPROVE_PRESET is not None:
        cfg = cfg0
        for nm, mut in STEPS:
            if nm in IMPROVE_PRESET:
                cfg = mut(cfg)
        kept = [nm for nm, _ in STEPS if nm in IMPROVE_PRESET]
        print("Dùng lại kết quả tìm kiếm có sẵn, bỏ qua các bước thử:", kept)
    else:
        cfg, cur = cfg0, run_cfg(cfg0, FAST_PARAMS, es=ES_FAST, name="mốc (cấu hình mục 6, lr 0.05)")
        rows = [{"bước": "mốc", "AUC": np.mean(cur["scores"]), "giữ": "-"}]
        for nm, mut in STEPS:
            cand_cfg = mut(cfg)
            new = run_cfg(cand_cfg, FAST_PARAMS, es=ES_FAST, name=nm)
            c = compare(new, cur)
            keep = c["verdict"] == "tốt hơn"
            rows.append({"bước": nm, "AUC": np.mean(new["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")},
                         "giữ": "có" if keep else "không"})
            if keep:
                cfg, cur = cand_cfg, new
        display(pd.DataFrame(rows).set_index("bước").round(4))
        kept = [r["bước"] for r in rows if r["giữ"] == "có"]
        print("Bước được giữ:", kept or "không có")


    # ---------- 4) Chạy lại cấu hình thắng với tham số cuối + nhiều seed ----------
    LGB_BEST = res_lgb
    if kept and REPLAY:
        FEATS, TE_USED = cfg["feats"], cfg["te"]
        print("REPLAY 6c: cấu hình cố định = +term, reg_alpha 5 (không huấn luyện lại)")
    elif kept:
        res_imp = run_cfg(cfg, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM cải tiến (cuối)")
        c = compare(res_imp, res_lgb)
        ll = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
        print(f"So với LightGBM mục 6: delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']} "
              f"| log-loss {ll(res_imp):.5f} vs {ll(res_lgb):.5f}")
        if c["verdict"] == "tốt hơn" and ll(res_imp) <= ll(res_lgb) + 1e-5:
            LGB_BEST = res_imp
            FINAL = res_imp
            FEATS, TE_USED = cfg["feats"], cfg["te"]
    print("Mô hình LightGBM tốt nhất tới đây:", LGB_BEST["name"])

    # ---------- 5) CatBoost với biến phân loại gốc, rồi trộn với LightGBM tốt nhất ----------
    if RUN_CAT_NATIVE:
        try:
            from catboost import CatBoostClassifier
            CAT_SRC = TE_CORE + TE_EXTRA
            feats_c = [c for c in cfg["feats"] if c not in CAT_SRC] + CAT_SRC
            t0 = time.time()
            oof_c, te_c = np.zeros(len(y)), np.zeros(len(X_test_all))
            Xtest_c = X_test_all[feats_c].copy()
            Xtest_c[CAT_SRC] = Xtest_c[CAT_SRC].astype("int32")
            for tr, va in SPLITS:
                Xtr, Xva = X_all.iloc[tr][feats_c].copy(), X_all.iloc[va][feats_c].copy()
                for d_ in (Xtr, Xva):
                    d_[CAT_SRC] = d_[CAT_SRC].astype("int32")
                m = CatBoostClassifier(iterations=200 if DEBUG_FAST else 4000, learning_rate=0.1 if DEBUG_FAST else 0.04,
                                       depth=6, l2_leaf_reg=10, loss_function="Logloss", eval_metric="AUC",
                                       od_type="Iter", od_wait=30 if DEBUG_FAST else 200, random_seed=SEED,
                                       thread_count=-1, verbose=False, allow_writing_files=False, cat_features=CAT_SRC)
                m.fit(Xtr, y[tr], eval_set=(Xva, y[va]), use_best_model=True)
                oof_c[va] = m.predict_proba(Xva)[:, 1]
                te_c += m.predict_proba(Xtest_c)[:, 1] / N_FOLDS
            sc = [roc_auc_score(y[va], oof_c[va]) for _, va in SPLITS]
            res_catn = {"oof": oof_c, "test_pred": te_c, "scores": sc, "name": "CatBoost native"}
            print(f"[CatBoost native] AUC = {np.mean(sc):.4f} ± {np.std(sc):.4f} | {time.time() - t0:.0f}s "
                  f"(CatBoost mục 6 dùng target encoding: tham chiếu thật 0.7617)")
            lg = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
            ll = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
            best_b, best_c = None, None
            for w in (0.5, 0.7):
                ob = expit(w * lg(LGB_BEST["oof"]) + (1 - w) * lg(res_catn["oof"]))
                rb = {"oof": ob, "test_pred": expit(w * lg(LGB_BEST["test_pred"]) + (1 - w) * lg(res_catn["test_pred"])),
                      "scores": [roc_auc_score(y[va], ob[va]) for _, va in SPLITS], "name": f"Blend LGB {w:.1f} + CatNative {1 - w:.1f}"}
                c = compare(rb, LGB_BEST)
                print(f"[{rb['name']}] AUC = {np.mean(rb['scores']):.4f} | log-loss = {ll(rb):.5f} | so với LightGBM: delta = {c['delta']:+.4f} "
                      f"(se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}")
                if c["verdict"] == "tốt hơn" and ll(rb) <= ll(LGB_BEST) + 1e-5 and (best_b is None or np.mean(rb["scores"]) > np.mean(best_b["scores"])):
                    best_b = rb
            if best_b is not None:
                FINAL = best_b
        except ImportError:
            print("Không có catboost, bỏ qua.")
    print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")
else:
    print("Bỏ qua mục 6c (RUN_IMPROVE = False).")

Dựng khối mới xong (76s): {'resid': 4, 'extml': 6, 'term': 4, 'extadv': 21}
Đơn điệu giảm cho 12/12 cột ext (cột có tương quan dương bị bỏ ra)
Dùng lại kết quả tìm kiếm có sẵn, bỏ qua các bước thử: ['+term', 'reg_alpha 5']
REPLAY 6c: cấu hình cố định = +term, reg_alpha 5 (không huấn luyện lại)
Mô hình LightGBM tốt nhất tới đây: LightGBM mục 6 (REPLAY: không huấn luyện lại)

>>> Mô hình dùng cho mục 7 và 8: LightGBM mục 6 (REPLAY: không huấn luyện lại) | AUC CV = nan


## 6am. Khối `price`: lãi suất ngầm định theo kỳ hạn chuẩn (vào pipeline chính)

Ở `gci-ver5-6am` khối này qua cổng hai tầng (+0.0011 tầng 1 và tầng 2, 5/5 fold; LGB cuối +0.0009, blend +0.0006, đều 5/5 fold). `credit/annuity` luôn nhỏ hơn kỳ hạn thật vì có lãi. Dò kỳ hạn chuẩn gần nhất (6, 12, 18, 24, 30, 36, 48, 60...) rồi suy ra lãi suất ngầm định: một biến giá cần hai cột cùng lúc, cây khó tự suy ra.

Ô dưới dựng 6 cột (`p_n1`, `p_rate_n1`, `p_rate_n2`, `p_gap_n1`, `p_fee_gap`, `p_fee_ratio`), thêm vào `cfg['feats']`. Các ô sau (6h CatBoost, 6i LightGBM, 6s refit, mục 7 và 8) tự dùng cấu hình này, nên **cả LGB và CatBoost đều được huấn luyện lại với `price`**, và `submission.csv` ở mục 8 là bản có `price`. Cache dùng tên `*_price.npz` để không đè cache cũ.

In [8]:
PRICE_ON = True
PRICE_COLS = ["p_n1", "p_rate_n1", "p_rate_n2", "p_gap_n1", "p_fee_gap", "p_fee_ratio"]
if PRICE_ON:
    F_ = FULL
    NA_ = pd.Series(np.nan, index=F_.index)
    RAWF = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
    assert len(RAWF) == len(F_), "thứ tự/chiều dài RAWF không khớp FULL"
    def _g(c):
        if c in F_.columns:
            return F_[c].astype(float)
        if c in RAWF.columns:
            return pd.Series(RAWF[c].astype(float).to_numpy(), index=F_.index)
        print("CẢNH BÁO: không có cột", c); return NA_
    _cr, _an, _gp = _g("credit_amount"), _g("annuity"), _g("amt_goods_price")
    _dv = lambda a, b: a / b.replace(0, np.nan)
    PR_GRID = np.array([6, 12, 18, 24, 30, 36, 48, 60, 72, 84, 96, 120], float)
    _term = _dv(_cr, _an).to_numpy(float)
    _idx = np.searchsorted(PR_GRID, np.where(np.isnan(_term), 1e9, _term), side="left")        # kỳ hạn chuẩn nhỏ nhất >= credit/annuity
    PR_N1 = np.where(_idx < len(PR_GRID), PR_GRID[np.minimum(_idx, len(PR_GRID) - 1)], np.nan)
    PR_N2 = np.where(_idx + 1 < len(PR_GRID), PR_GRID[np.minimum(_idx + 1, len(PR_GRID) - 1)], np.nan)
    PR_C, PR_A, PR_TERM = _cr.to_numpy(float), _an.to_numpy(float), _term
    def PR_rate_for(n):
        with np.errstate(all="ignore"):
            lo, hi = np.full(len(PR_C), 1e-7), np.full(len(PR_C), 0.1)
            for _ in range(40):
                mid = (lo + hi) / 2
                big = PR_C * mid / (1 - (1 + mid) ** (-n)) > PR_A
                hi, lo = np.where(big, mid, hi), np.where(big, lo, mid)
            return np.where((PR_A * n >= PR_C) & np.isfinite(n), 12 * (lo + hi) / 2, np.nan)         # lãi suất năm quy đổi
    PR_R1, PR_R2 = PR_rate_for(PR_N1), PR_rate_for(PR_N2)
    PRICE_DF = pd.DataFrame({"p_n1": PR_N1, "p_rate_n1": PR_R1, "p_rate_n2": PR_R2, "p_gap_n1": PR_N1 - PR_TERM,
                             "p_fee_gap": _dv(_cr - _gp, _an).to_numpy(float), "p_fee_ratio": _dv(_cr - _gp, _gp).to_numpy(float)},
                            index=F_.index).replace([np.inf, -np.inf], np.nan).astype("float32")
    X_all = pd.concat([X_all.drop(columns=[c for c in PRICE_COLS if c in X_all.columns]), PRICE_DF.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in PRICE_COLS if c in X_test_all.columns]), PRICE_DF.iloc[n_tr:].reset_index(drop=True)], axis=1)
    cfg = dict(cfg)
    cfg["feats"] = list(cfg["feats"]) + [c for c in PRICE_COLS if c not in cfg["feats"]]
    print("price: thêm", len(PRICE_COLS), "cột vào cfg ->", len(cfg["feats"]), "feature gốc | thiếu (%):", (PRICE_DF.iloc[:n_tr].isna().mean() * 100).round(1).to_dict())
    print("Lãi suất ngầm định (năm) 25/50/75%:", PRICE_DF["p_rate_n1"].describe()[["25%", "50%", "75%"]].round(3).to_dict(), "| kỳ hạn chuẩn phổ biến:", pd.Series(PR_N1).value_counts().head(4).to_dict())


price: thêm 6 cột vào cfg -> 84 feature gốc | thiếu (%): {'p_n1': 0.0, 'p_rate_n1': 0.0, 'p_rate_n2': 0.0, 'p_gap_n1': 0.0, 'p_fee_gap': 0.0, 'p_fee_ratio': 0.0}
Lãi suất ngầm định (năm) 25/50/75%: {'25%': 0.08, '50%': 0.147, '75%': 0.247} | kỳ hạn chuẩn phổ biến: {24.0: 83532, 18.0: 46515, 36.0: 39179, 30.0: 27130}


## 6d. Thử các nhóm feature mới (so với nhóm tương đồng, nhất quán hồ sơ, thiếu dữ liệu, mô hình phụ, cụm)

Mốc để so sánh là cấu hình tốt nhất sau 6c (đã gồm `+term` và `reg_alpha 5`). Mỗi khối được thêm lần lượt, giữ nếu "tốt hơn" theo quy tắc `compare` (delta > delta_se và dương ở ít nhất 3/5 fold), ở `lr = 0.05`.

**Những gì bộ dữ liệu này cho phép.** File chỉ có 34 cột nên một số mục trong đề xuất không làm được đúng nghĩa:
- không có `FLAG_DOCUMENT_*`, `REG_REGION_NOT_*`, `LIVE_*`: không có `living_region_mismatch`, và "số trường giấy tờ thiếu" gần như hằng số. Mình thay bằng các cờ **không nhất quán thời gian** (đăng ký trước khi sinh, cấp giấy tờ ở tuổi quá nhỏ, tuổi bắt đầu đi làm quá thấp hoặc quá cao...) với ngưỡng lấy từ phân vị của dữ liệu, không đặt tay.
- `flag_emp_phone` đồng nhất với nhóm "không đi làm", nên `contact_document_mismatch` suy biến; mình dùng tương tác `flag_work_phone` với thay đổi điện thoại và giấy tờ gần đây.
- `ext_model_residual` đã đo ở 6c (`+resid`: ≈ như, delta -0.0002): không lặp lại.
- `peer_default_rate_oof` (kNN dùng nhãn): ver4 đã đo (D6/D7: delta 0.0000), rủi ro rò rỉ cao, **không làm**.

**Nhóm có label-free** (nhóm, trung vị, cụm, khoảng cách, PCA) tính trên train + test là hợp lệ, vì không dùng `target`. **Nhóm dùng nhãn** (mô hình phụ, target encoding của các mã mới) tính **trong từng fold**.

| Khối | Nội dung |
|---|---|
| `peer` | thu nhập, khoản vay, annuity so với trung vị nhóm (nghề × học vấn, vùng, gia đình × học vấn); phân vị tỷ lệ vay/thu nhập và thu nhập theo nhóm tuổi. Nhóm dưới 100 mẫu dùng nhóm thô hơn (chỉ nghề) |
| `incons` | tuổi bắt đầu đi làm bất thường, số năm làm so với nhóm tuổi, thu nhập so với nhóm nghề × thâm niên, vay/thu nhập và annuity/vay so với nhóm, số cờ không nhất quán, tương tác điện thoại công việc với thay đổi gần đây |
| `miss` | mẫu thiếu của ba nguồn ext (0 đến 7), số cột thiếu theo nhóm (tài chính, nhân khẩu, bureau, giấy tờ), tổng thiếu, cụm theo vector thiếu, thiếu so với trung bình nhóm; tương tác mẫu thiếu × nghề và × loại thu nhập đưa qua target encoding trong fold |
| `mresid` | phần dư của thu nhập, khoản vay, annuity so với dự đoán từ hồ sơ (hồi quy LightGBM, dự đoán ngoài fold, không dùng nhãn) |
| `cluster` | KMeans 12 cụm trên biến hồ sơ đã chuẩn hóa: khoảng cách tới tâm, mật độ cụm, mã cụm (qua target encoding) |
| `peerdist` | khoảng cách trung bình và nhỏ nhất tới 10 hồ sơ gần nhất, mật độ lân cận (không dùng nhãn hàng xóm) |
| `pca` | 6 thành phần PCA của các feature gốc |
| `submodel` | 5 mô hình phụ trên lát cắt biến (nhân khẩu, tài chính, ext, giấy tờ, thiếu dữ liệu); dự đoán **OOF lồng trong từng fold** cùng độ bất đồng, max, min, entropy |

**Kỳ vọng thực tế, dựa trên những gì đã đo.** Thống kê theo nhóm (F ở ver4) không tăng; meta-feature OOF của mô hình phụ (M1 đến M3) delta 0.0000; PCA và kNN không tăng; các cờ thiếu dữ liệu có gain = 0 vì cây tự học hướng đi của NaN. Điểm sáng duy nhất cùng họ là `+term` (cấu trúc kỳ hạn, +0.0009), nên `mresid` của annuity có thể đáng đo. Nếu không khối nào thắng thì đó là kết quả bình thường.

In [9]:
RUN_6D = not REPLAY
RUN_SUBMODEL = True      # mô hình phụ OOF lồng trong fold: chậm nhất, đặt False để bỏ

if RUN_6D:
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.decomposition import PCA
    from sklearn.model_selection import KFold
    from sklearn.neighbors import NearestNeighbors
    from sklearn.preprocessing import StandardScaler

    t0 = time.time()
    RAW = pd.concat([train.drop(columns=TARGET), test], ignore_index=True)
    F_ = FULL

    def key_codes(*arrs):
        s = pd.Series(np.asarray(arrs[0])).astype(str)
        for a in arrs[1:]:
            s = s + "_" + pd.Series(np.asarray(a)).astype(str)
        return pd.factorize(s)[0]

    def med_by(v, key, fb_key, min_n=100):
        d = pd.DataFrame({"v": np.asarray(v, dtype=float), "k": key, "f": fb_key})
        cnt = d.groupby("k")["v"].transform("count")
        return np.where(cnt >= min_n, d.groupby("k")["v"].transform("median"), d.groupby("f")["v"].transform("median"))

    def pct_in(v, key):
        return pd.Series(np.asarray(v, dtype=float)).groupby(key).rank(pct=True).to_numpy()

    qbin = lambda s, q: pd.qcut(pd.Series(np.asarray(s, dtype=float)), q, labels=False, duplicates="drop").to_numpy()

    occ, edu = F_["c_occupation"].to_numpy(), F_["name_education_type"].to_numpy()
    inc, cr, an = F_["income"], F_["credit_amount"], F_["annuity"]
    age, ye = F_["age_years"], F_["years_employed"]
    g_oe, g_occ = key_codes(occ, edu), key_codes(occ)
    g_fe = key_codes(F_["c_family"], edu)
    g_reg = key_codes(F_["region_rating_client"], qbin(F_["region_population_relative"], 10))
    g_rating = key_codes(F_["region_rating_client"])
    g_age = key_codes(qbin(age, 10))
    B, TE6 = {}, {}

    # ---- peer: so với nhóm tương đồng (không dùng nhãn)
    ci = cr / inc.replace(0, np.nan)
    B["peer"] = pd.DataFrame({
        "income_occ_edu_resid": inc - med_by(inc, g_oe, g_occ),
        "income_occ_edu_ratio": inc / med_by(inc, g_oe, g_occ),
        "credit_occ_edu_ratio": cr / med_by(cr, g_oe, g_occ),
        "annuity_occ_edu_ratio": an / med_by(an, g_oe, g_occ),
        "income_region_resid": inc - med_by(inc, g_reg, g_rating),
        "income_family_edu_ratio": inc / med_by(inc, g_fe, g_occ),
        "credit_income_group_pct": pct_in(ci, g_oe),
        "age_income_pct": pct_in(inc, g_age),
    })

    # ---- incons: nhất quán hồ sơ, ngưỡng lấy từ phân vị dữ liệu
    start = age - ye
    lo, hi = np.nanquantile(start, [0.01, 0.99])
    q_id = np.nanquantile(F_["id_issue_age"], 0.01)
    q_ph = np.nanquantile(F_["years_last_phone_change"], 0.10)
    q_idy = np.nanquantile(F_["years_id_publish"], 0.10)
    ye_bin = qbin(ye, 5)
    ye_bin = np.where(np.isnan(ye_bin), -1, ye_bin)
    ac = an / cr.replace(0, np.nan)
    g_cc = key_codes(F_["name_contract_type"], qbin(cr, 10))
    g_oy = key_codes(occ, ye_bin)
    fl = pd.DataFrame({
        "start": ((start < lo) | (start > hi)).astype(float).where(start.notna()),
        "reg": (F_["years_registration"] > age).astype(float).where(F_["years_registration"].notna()),
        "id": (F_["id_issue_age"] < q_id).astype(float).where(F_["id_issue_age"].notna()),
        "car": (F_["own_car_age"] > (age - 16)).astype(float).where(F_["own_car_age"].notna()),
    })
    B["incons"] = pd.DataFrame({
        "start_age_implausible": fl["start"],
        "employment_age_residual": ye - med_by(ye, g_age, g_age),
        "income_employment_residual": np.log1p(inc) - med_by(np.log1p(inc), g_oy, g_occ),
        "credit_income_residual": ci - med_by(ci, g_oe, g_occ),
        "annuity_credit_residual": ac - med_by(ac, g_cc, key_codes(F_["name_contract_type"])),
        "inconsistency_count": fl.sum(axis=1, min_count=1),
        "workphone_x_recent_phone": F_["flag_work_phone"] * (F_["years_last_phone_change"] < q_ph).astype(float),
        "workphone_x_recent_id": F_["flag_work_phone"] * (F_["years_id_publish"] < q_idy).astype(float),
    })

    # ---- miss: mẫu thiếu dữ liệu
    na = RAW.isna()
    e_na = na[EXT].to_numpy().astype(int)
    pattern = e_na[:, 0] + 2 * e_na[:, 1] + 4 * e_na[:, 2]
    cnt_na = lambda cols: na[[c for c in cols if c in na.columns]].sum(axis=1).astype(float)
    miss_total = na.sum(axis=1).astype(float)
    any_na_cols = [c for c in RAW.columns if c != ID_COL and na[c].any()]
    miss_cl = MiniBatchKMeans(8, random_state=SEED, n_init=3, batch_size=4096).fit_predict(na[any_na_cols].to_numpy().astype(float))
    B_miss = pd.DataFrame({
        "ext_missing_pattern": pattern.astype(float),
        "financial_missing_count": cnt_na(["income", "credit_amount", "annuity", "amt_goods_price"]),
        "demographic_missing_count": cnt_na(["occupation_type", "name_family_status", "cnt_children", "cnt_fam_members", "code_gender"]),
        "bureau_missing_count": cnt_na(BUREAU),
        "document_missing_count": cnt_na(["days_registration", "days_id_publish", "days_last_phone_change", "flag_work_phone"]),
        "missing_total": miss_total,
        "missingness_vs_group": miss_total - pd.Series(miss_total).groupby(g_oe).transform("mean").to_numpy(),
        "missingness_cluster": miss_cl.astype(float),
        "ext_missing_x_occupation": key_codes(pattern, occ).astype(float),
        "ext_missing_x_income_type": key_codes(pattern, F_["c_income_type"]).astype(float),
    }, index=F_.index)
    B["miss"] = B_miss
    TE6["miss"] = ["missingness_cluster", "ext_missing_x_occupation", "ext_missing_x_income_type"]

    # ---- mresid: phần dư so với dự đoán từ hồ sơ (hồi quy, dự đoán ngoài fold, không dùng nhãn)
    def oof_resid(target, cols, seed=SEED):
        tgt = np.asarray(target, dtype=float)
        idx = np.where(np.isfinite(tgt))[0]
        mis = np.where(~np.isfinite(tgt))[0]
        pred = np.full(len(tgt), np.nan)
        for a, b in KFold(5, shuffle=True, random_state=seed).split(idx):
            m = lgb.LGBMRegressor(n_estimators=250, learning_rate=0.05, num_leaves=31, min_child_samples=100, subsample=0.8,
                                  subsample_freq=1, colsample_bytree=0.8, random_state=seed, n_jobs=-1, verbose=-1
                                  ).fit(F_.iloc[idx[a]][cols], tgt[idx[a]])
            pred[idx[b]] = m.predict(F_.iloc[idx[b]][cols])
        return tgt - pred

    PROFILE = ["age_years", "years_employed", "c_occupation", "c_organization", "c_income_type", "c_family", "c_housing",
               "name_education_type", "code_gender", "cnt_children", "cnt_fam_members", "region_rating_client",
               "region_population_relative", "own_car_age"]
    B["mresid"] = pd.DataFrame({
        "income_model_residual": oof_resid(np.log1p(inc), PROFILE),
        "credit_model_residual": oof_resid(np.log1p(cr), PROFILE + ["income"]),
        "annuity_model_residual": oof_resid(np.log1p(an), ["credit_amount", "amt_goods_price", "name_contract_type", "age_years"]),
    })

    # ---- cluster / peerdist / pca: cấu trúc của tập khách hàng (không dùng nhãn)
    M = pd.DataFrame({
        "e1": F_["ext_source_1_ml"], "e2": F_["ext_source_2_ml"], "e3": F_["ext_source_3_ml"],
        "li": np.log1p(inc), "lc": np.log1p(cr), "la": np.log1p(an), "age": age, "ye": ye,
        "term": F_["credit_term"].clip(0, 80), "rr": F_["region_rating_client"], "edu": F_["name_education_type"],
    })
    Z = StandardScaler().fit_transform(M.fillna(M.median()).to_numpy())
    km = MiniBatchKMeans(12, random_state=SEED, n_init=3, batch_size=4096).fit(Z)
    lab = km.labels_
    d_all = km.transform(Z)
    B["cluster"] = pd.DataFrame({
        "customer_cluster": lab.astype(float),
        "distance_to_cluster_center": d_all[np.arange(len(Z)), lab],
        "cluster_density": pd.Series(lab).map(pd.Series(lab).value_counts(normalize=True)).to_numpy(),
    }, index=F_.index)
    TE6["cluster"] = ["customer_cluster"]
    dist, _ = NearestNeighbors(n_neighbors=11, algorithm="kd_tree", n_jobs=-1).fit(Z).kneighbors(Z)
    B["peerdist"] = pd.DataFrame({"peer_distance_mean": dist[:, 1:].mean(axis=1), "peer_distance_min": dist[:, 1],
                                  "peer_density": 1.0 / (dist[:, 1:].mean(axis=1) + 1e-6)}, index=F_.index)
    Xp = F_[BASE_FEATURES]
    Zp = np.clip(StandardScaler().fit_transform(Xp.fillna(Xp.median()).to_numpy()), -5, 5)
    B["pca"] = pd.DataFrame(PCA(6, random_state=SEED).fit_transform(Zp), columns=[f"latent_{i + 1}" for i in range(6)], index=F_.index)

    # ---- đưa vào X_all / X_test_all (làm lại được nhiều lần)
    for k in B:
        B[k].index = F_.index
    ALL6 = pd.concat(B.values(), axis=1).replace([np.inf, -np.inf], np.nan).astype("float32")
    X_all = pd.concat([X_all.drop(columns=[c for c in ALL6.columns if c in X_all.columns]), ALL6.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in ALL6.columns if c in X_test_all.columns]),
                            ALL6.iloc[n_tr:].reset_index(drop=True)], axis=1)
    BL6 = {k: dict(te=TE6.get(k, []), feats=[c for c in B[k].columns if c not in TE6.get(k, [])]) for k in B}
    print(f"Dựng khối xong ({time.time() - t0:.0f}s):", {k: len(v["feats"]) + len(v["te"]) for k, v in BL6.items()})
    chk = ALL6.iloc[:n_tr]
    print("Cột gần như hằng số (std = 0):", [c for c in chk.columns if chk[c].std() == 0] or "không có")
    print("Thiếu nhiều nhất (%):", (chk.isna().mean() * 100).sort_values(ascending=False).head(5).round(1).to_dict())

## 6e. Tổ hợp ext_source nâng cao, mũ và nhân, rồi kẹp với từng nhóm

**Một điều cần nói trước.** Mũ hay căn của **một** cột (ví dụ `ext_source_3 ** 2`) không thêm gì cho LightGBM: cây chỉ so sánh thứ hạng, nên mọi biến đổi đơn điệu của một cột cho đúng cùng một cây. Chỉ có hai kiểu "khuếch đại" là thật sự khác:
1. **Đổi hướng kết hợp ba điểm**: tích có mũ khác nhau (`e1 · e2² · e3`), trung bình lũy thừa (p = −2 nhấn mạnh điểm thấp nhất, p = 3 nhấn mạnh điểm cao nhất), tổng logit có trọng số. Cây chia theo từng trục nên không tự dựng được những hướng chéo này.
2. **Đặt điểm ext vào ngữ cảnh nhóm**: cùng một điểm ext có ý nghĩa khác nhau tùy khách thuộc nhóm nào.

**Ba khối làm ở mục này**
- `extpow`: 7 tích có mũ, 5 trung bình lũy thừa, 4 tổng logit, trung bình odds. Bản `+extadv` ở 6c (trung bình hình học, điều hòa, tích, tỷ số...) cho −0.0008 (1/5 fold), nên kỳ vọng ở đây cũng thấp.
- **Bảng AUC đơn biến** (chỉ để xem, không dùng để chọn): AUC của từng tổ hợp đứng một mình so với `ext_mean_ml`. Cho biết tổ hợp nào mang nhiều thông tin nhất.
- **Kẹp với từng nhóm** (12 nhóm: giới, học vấn, loại thu nhập, gia đình, nhà ở, nghề, tổ chức, hạng vùng, tuổi, loại hợp đồng, có xe, mẫu thiếu ext). Với mỗi nhóm và mỗi trong 4 tổ hợp (`ext_mean_ml`, trung bình hình học, `e2·e3`, trung bình lũy thừa p = −2) tạo: **hiệu so với trung vị nhóm** và **phân vị trong nhóm** (label-free), cộng một mã `nhóm × 10 bin ext_mean_ml` đưa qua target encoding trong fold. Mỗi nhóm được thử **riêng** so với mốc, nên bảng kết quả cho biết nhóm nào kẹp có ích; sau đó chỉ các nhóm "tốt hơn" được gộp dần, mỗi lần kiểm lại.

Mốc là cấu hình tốt nhất sau 6d (hoặc 6c nếu bỏ 6d). Quy tắc kết luận như các mục trước.

In [10]:
RUN_6E = not REPLAY
if RUN_6E:
    from sklearn.metrics import roc_auc_score as _auc
    RUNNER = globals().get("run_cfg2") or globals().get("run_cfg")
    if "cfg6" in globals():
        base_cfg = dict(cfg6)
    elif "cfg" in globals():
        base_cfg = dict(cfg)
    else:
        base_cfg = dict(feats=list(FEATS), te=list(TE_USED), params={}, mono=False)
    t0 = time.time()

    # ---------- 1) Khối extpow (không dùng nhãn) ----------
    E_ = np.clip(FULL[[f"{c}_ml" for c in EXT]].to_numpy(dtype=float), 1e-3, 1 - 1e-3)
    E1, E2, E3 = E_[:, 0], E_[:, 1], E_[:, 2]
    tag = lambda w: "_".join(str(x).replace(".", "p").replace("-", "m") for x in w)
    P = {}
    for w in [(1, 2, 1), (1, 1, 2), (2, 1, 1), (1, 2, 2), (0.5, 1, 1), (1, 1, 0.5), (1, 3, 2)]:
        P[f"ext_pw_{tag(w)}"] = E1 ** w[0] * E2 ** w[1] * E3 ** w[2]
    for p in (-2, -1, 0.5, 2, 3):
        P[f"ext_pmean_{tag([p])}"] = np.mean(E_ ** p, axis=1) ** (1.0 / p)
    L_ = np.log(E_ / (1 - E_))
    for w in [(1, 1, 1), (1, 2, 1), (1, 2, 2), (0.5, 1, 1.5)]:
        P[f"ext_lgs_{tag(w)}"] = L_ @ np.array(w, dtype=float)
    P["ext_odds_mean"] = np.mean(E_ / (1 - E_), axis=1)
    B_pow = pd.DataFrame(P, index=FULL.index)

    # ---------- 2) AUC đơn biến (chỉ để xem): tổ hợp nào mang nhiều thông tin nhất ----------
    uni = {"ext_mean_ml": FULL["ext_mean_ml"].to_numpy(dtype=float), "ext_source_2_ml": E2, "ext_source_3_ml": E3}
    uni.update({k: v.to_numpy() for k, v in B_pow.items()})
    uni_auc = pd.Series({k: _auc(y, v[:n_tr]) for k, v in uni.items()})
    uni_auc = pd.DataFrame({"AUC": np.maximum(uni_auc, 1 - uni_auc)}).sort_values("AUC", ascending=False)
    uni_auc["hơn ext_mean_ml"] = uni_auc["AUC"] - uni_auc.loc["ext_mean_ml", "AUC"]
    display(uni_auc.head(12).round(4))
    print("AUC đơn biến của ext_mean_ml là mốc; tổ hợp nào chỉ hơn vài phần nghìn thì thông tin gần như trùng.")

    # ---------- 3) Kẹp tổ hợp với nhóm ----------
    def key_codes(*arrs):
        s = pd.Series(np.asarray(arrs[0])).astype(str)
        for a in arrs[1:]:
            s = s + "_" + pd.Series(np.asarray(a)).astype(str)
        return pd.factorize(s)[0]

    def med_by(v, key, min_n=100):
        d = pd.DataFrame({"v": np.asarray(v, dtype=float), "k": key})
        cnt = d.groupby("k")["v"].transform("count")
        return np.where(cnt >= min_n, d.groupby("k")["v"].transform("median"), np.nanmedian(d["v"]))

    def pct_in(v, key):
        return pd.Series(np.asarray(v, dtype=float)).groupby(key).rank(pct=True).to_numpy()

    qbin = lambda s, q: pd.qcut(pd.Series(np.asarray(s, dtype=float)), q, labels=False, duplicates="drop").to_numpy()
    RAW_ = pd.concat([train.drop(columns=TARGET), test], ignore_index=True)
    e_na = RAW_[EXT].isna().to_numpy().astype(int)
    GROUPS = {
        "gioi": FULL["code_gender"].to_numpy(), "hocvan": FULL["name_education_type"].to_numpy(),
        "loai_thu_nhap": FULL["c_income_type"].to_numpy(), "gia_dinh": FULL["c_family"].to_numpy(),
        "nha_o": FULL["c_housing"].to_numpy(), "nghe": FULL["c_occupation"].to_numpy(),
        "to_chuc": FULL["c_organization"].to_numpy(), "hang_vung": FULL["region_rating_client"].to_numpy(),
        "tuoi": qbin(FULL["age_years"], 10), "hop_dong": FULL["name_contract_type"].to_numpy(),
        "co_xe": FULL["own_car_age"].notna().to_numpy().astype(int), "mau_thieu_ext": e_na[:, 0] + 2 * e_na[:, 1] + 4 * e_na[:, 2],
    }
    COMBOS = {"mean": FULL["ext_mean_ml"].to_numpy(dtype=float), "geo": (E1 * E2 * E3) ** (1 / 3),
              "p23": E2 * E3, "pm2": P["ext_pmean_m2"]}
    bin_mean = qbin(FULL["ext_mean_ml"], 10)
    XG, GFEAT, GTE = {}, {}, {}
    for g, gk in GROUPS.items():
        key = key_codes(gk)
        cols = {}
        for cn, cv in COMBOS.items():
            cols[f"x_{cn}_rel_{g}"] = cv - med_by(cv, key)
            cols[f"x_{cn}_pct_{g}"] = pct_in(cv, key)
        cols[f"x_{g}_bin"] = key_codes(gk, bin_mean).astype(float)
        XG[g] = pd.DataFrame(cols, index=FULL.index)
        GTE[g] = [f"x_{g}_bin"]
        GFEAT[g] = [c for c in cols if c != f"x_{g}_bin"]

    ALLE = pd.concat([B_pow] + list(XG.values()), axis=1).replace([np.inf, -np.inf], np.nan).astype("float32")
    X_all = pd.concat([X_all.drop(columns=[c for c in ALLE.columns if c in X_all.columns]), ALLE.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in ALLE.columns if c in X_test_all.columns]),
                            ALLE.iloc[n_tr:].reset_index(drop=True)], axis=1)
    print(f"Dựng khối xong ({time.time() - t0:.0f}s): extpow = {B_pow.shape[1]} cột, mỗi nhóm = {len(GFEAT['gioi'])} số + 1 mã TE, {len(GROUPS)} nhóm")

    # ---------- 4) Kiểm định ----------
    cur_e = RUNNER(base_cfg, FAST_PARAMS, es=ES_FAST, name="mốc 6e (cấu hình tốt nhất trước đó, lr 0.05)")
    rows = [{"bước": "mốc", "AUC": np.mean(cur_e["scores"]), "giữ": "-"}]

    def add(cfgx, feats, te=()):
        return {**cfgx, "feats": cfgx["feats"] + list(feats), "te": cfgx["te"] + list(te)}

    new = RUNNER(add(base_cfg, B_pow.columns), FAST_PARAMS, es=ES_FAST, name="+extpow")
    c = compare(new, cur_e)
    rows.append({"bước": "+extpow", "AUC": np.mean(new["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if c["verdict"] == "tốt hơn" else "không"})
    cfg_e, cur_e2 = base_cfg, cur_e
    if c["verdict"] == "tốt hơn":
        cfg_e, cur_e2 = add(base_cfg, B_pow.columns), new

    # 4a) mỗi nhóm thử riêng so với mốc (bảng "nhóm nào kẹp có ích")
    single = {}
    for g in GROUPS:
        r = RUNNER(add(cfg_e, GFEAT[g], GTE[g]), FAST_PARAMS, es=ES_FAST, name=f"+kẹp {g}")
        c = compare(r, cur_e2)
        single[g] = (c["delta"], c["verdict"], r)
        rows.append({"bước": f"kẹp riêng: {g}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
    winners = sorted([g for g, v in single.items() if v[1] == "tốt hơn"], key=lambda g: -single[g][0])

    # 4b) gộp dần các nhóm thắng, mỗi lần kiểm lại với cấu hình đang có (tránh chọn theo may rủi)
    kept_g = []
    for g in winners:
        cand = add(cfg_e, GFEAT[g], GTE[g])
        r = RUNNER(cand, FAST_PARAMS, es=ES_FAST, name=f"gộp +{g}")
        c = compare(r, cur_e2)
        keep = c["verdict"] == "tốt hơn"
        rows.append({"bước": f"gộp dần: {g}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if keep else "không"})
        if keep:
            cfg_e, cur_e2, kept_g = cand, r, kept_g + [g]
    display(pd.DataFrame(rows).set_index("bước").round(4))
    print("extpow:", "giữ" if cfg_e is not base_cfg and any(r["bước"] == "+extpow" and r["giữ"] == "có" for r in rows) else "không giữ",
          "| nhóm được giữ:", kept_g or "không có")

    # ---------- 5) Chạy lại cấu hình thắng với tham số cuối, rồi so với mô hình đang dùng ----------
    if cfg_e is not base_cfg:
        ref = globals().get("LGB_BEST", res_lgb)
        res_6e = RUNNER(cfg_e, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối 6e (cuối)")
        ll = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
        c = compare(res_6e, ref)
        print(f"So với LightGBM tốt nhất trước 6e ({ref['name']}): delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) "
              f"-> {c['verdict']} | log-loss {ll(res_6e):.5f} vs {ll(ref):.5f}")
        if c["verdict"] == "tốt hơn" and ll(res_6e) <= ll(ref) + 1e-5:
            LGB_BEST = res_6e
            cand_final = res_6e
            if "res_catn" in globals():
                lg = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
                ob = expit(0.7 * lg(res_6e["oof"]) + 0.3 * lg(res_catn["oof"]))
                cand_final = {"oof": ob, "test_pred": expit(0.7 * lg(res_6e["test_pred"]) + 0.3 * lg(res_catn["test_pred"])),
                              "scores": [roc_auc_score(y[va], ob[va]) for _, va in SPLITS], "name": "Blend LGB(6e) 0.7 + CatNative 0.3"}
            c2 = compare(cand_final, FINAL)
            print(f"So với mô hình đang dùng ({FINAL['name']}, AUC {np.mean(FINAL['scores']):.4f}): delta = {c2['delta']:+.4f} "
                  f"(se {c2['delta_se']:.4f}, fold dương {c2['fold_duong']}) -> {c2['verdict']}")
            if c2["verdict"] == "tốt hơn" and ll(cand_final) <= ll(FINAL) + 1e-5:
                FINAL = cand_final
    else:
        print("Không khối nào của 6e thắng mốc: giữ nguyên mô hình đang dùng.")
    print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")
else:
    print("Bỏ qua mục 6e (RUN_6E = False).")

Bỏ qua mục 6e (RUN_6E = False).


## 6f. Thử thật nhiều feature engineering, có kiểm soát chọn nhầm

Khi thử 12 khối trên cùng các fold, thế nào cũng có khối "thắng" chỉ vì may. Vì vậy mục này dùng **hai tầng**:
1. **Sàng lọc** từng khối riêng so với mốc, rồi gộp dần các khối thắng (như 6c đến 6e).
2. **Xác nhận trên bộ fold khác** (một lần chia 5 fold mới, seed khác): cấu hình cuối so với mốc phải **vẫn "tốt hơn"**, nếu không thì bỏ hết. Đây là bước ngăn việc giữ khối nhờ may.

Mốc là cấu hình tốt nhất sau các mục trước (6e, hoặc 6d, hoặc 6c, lấy cái mới nhất có).

**Nhóm target encoding mới (làm trong fold, có làm mượt)**

| Khối | Nội dung |
|---|---|
| `te_resid` | TE của **phần dư** `y − p_ext`: trong mỗi lần mã hóa, `p_ext` là tỷ lệ vỡ nợ theo 10 bin `ext_mean_ml` tính chỉ trên các hàng dùng để fit. Đo phần rủi ro của nhóm mà ext chưa giải thích, nên ít trùng thông tin với ext |
| `te_hier` | TE **phân cấp**: ô con (nghề × học vấn × loại thu nhập, tổ chức × tuổi, hạng vùng × nhà ở × gia đình) co về TE của nhóm cha thay vì về trung bình toàn cục |
| `te_term` | TE của số kỳ làm tròn `term_n`, và `term_n` × loại hợp đồng, × loại thu nhập (`+term` là khối duy nhất thắng rõ ở 6c) |
| `te_region` | TE của vùng (hạng × decile dân số), vùng × nhà ở, hạng vùng × loại thu nhập |
| `te_bins2d` | TE của ô 10 × 10 giữa hai biến: ext × kỳ hạn, ext × tuổi, ext × thu nhập, ext × annuity/thu nhập, ext × thâm niên, ext2 × ext3 |

**Nhóm số (không dùng nhãn)**

| Khối | Nội dung |
|---|---|
| `zgrp` | z-score của ext_mean_ml, ext2, ext3 trong từng nhóm (nghề, tổ chức, học vấn, loại thu nhập, tuổi, hạng vùng) |
| `cluster_gap` | ext của khách trừ trung vị ext của cụm KMeans hồ sơ |
| `round` | cờ và phần dư của số "tròn": thu nhập, khoản vay, giá hàng, annuity (số tròn thường là khai ước lượng) |
| `timediff` | hiệu và tỷ số giữa các mốc thời gian: cấp giấy tờ, đăng ký, đổi điện thoại, đi làm, tuổi |
| `bureau2` | cấu trúc truy vấn bureau: tỷ trọng gần đây, theo quý, nhân với ext |
| `family` | tính theo đầu người: vay, annuity, hàng; số người lớn; tỷ lệ trẻ em |
| `flags` | các cờ rủi ro với ngưỡng theo phân vị dữ liệu (ext thấp, vay/thu nhập cao, trẻ, thâm niên ngắn, đổi giấy tờ gần đây, hạng vùng 3...) và **số cờ**: cây khó dựng được đặc trưng cộng dồn như vậy |

**Đã loại vì ver4/ver5 đã đo, không thắng:** `cnt`, `extadv`, `resid`, `extml`, PCA, kNN và kNN theo nhãn, meta-feature OOF, OpenFE, tích và tỷ số với ext điền ML.

**Kỳ vọng thật:** có thể vài khối cho +0.0003 đến +0.001; cũng có thể không khối nào qua bước xác nhận, và đó là kết quả hợp lệ.

In [11]:
RUN_6F = True
if RUN_6F:
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.preprocessing import StandardScaler

    t0 = time.time()
    base_cfg = next((dict(globals()[n]) for n in ("cfg_e", "cfg6", "cfg") if n in globals() and isinstance(globals()[n], dict)),
                    dict(feats=list(FEATS), te=list(TE_USED), params={}, mono=False))
    base_cfg["te"] = list(dict.fromkeys(base_cfg.get("te", [])))
    for k in ("te_resid", "te_hier"):
        base_cfg.setdefault(k, [])
    MONO = globals().get("MONO", set())
    F_ = FULL

    def key_codes(*arrs):
        s = pd.Series(np.asarray(arrs[0])).astype(str)
        for a in arrs[1:]:
            s = s + "_" + pd.Series(np.asarray(a)).astype(str)
        return pd.factorize(s)[0]

    qbin = lambda s, q: pd.qcut(pd.Series(np.asarray(s, dtype=float)), q, labels=False, duplicates="drop").to_numpy()

    # ================== 1) KHỐI SỐ (không dùng nhãn) ==================
    NUM = {}
    occ, org = F_["c_occupation"].to_numpy(), F_["c_organization"].to_numpy()
    inc, cr, an, gd = F_["income"], F_["credit_amount"], F_["annuity"], F_["amt_goods_price"]
    age, ye = F_["age_years"], F_["years_employed"]
    e_m, e2, e3 = F_["ext_mean_ml"], F_["ext_source_2_ml"], F_["ext_source_3_ml"]
    age_bin = qbin(age, 10)

    def zin(v, key, min_n=100):
        d = pd.DataFrame({"v": np.asarray(v, dtype=float), "k": key})
        g = d.groupby("k")["v"]
        cnt, mu, sd = g.transform("count"), g.transform("mean"), g.transform("std")
        ok = (cnt >= min_n) & (sd > 0)
        return np.where(ok, (d["v"] - mu) / sd, (d["v"] - d["v"].mean()) / d["v"].std())

    zk = {"nghe": key_codes(occ), "to_chuc": key_codes(org), "hocvan": key_codes(F_["name_education_type"]),
          "thu_nhap": key_codes(F_["c_income_type"]), "tuoi": key_codes(age_bin), "vung": key_codes(F_["region_rating_client"])}
    NUM["zgrp"] = pd.DataFrame({f"z_{vn}_{kn}": zin(v, k) for vn, v in (("e", e_m), ("e2", e2), ("e3", e3)) for kn, k in zk.items()})

    M = pd.DataFrame({"e1": F_["ext_source_1_ml"], "e2": e2, "e3": e3, "li": np.log1p(inc), "lc": np.log1p(cr), "la": np.log1p(an),
                      "age": age, "ye": ye, "term": F_["credit_term"].clip(0, 80), "rr": F_["region_rating_client"],
                      "edu": F_["name_education_type"]})
    Z = StandardScaler().fit_transform(M.fillna(M.median()).to_numpy())
    lab = MiniBatchKMeans(12, random_state=SEED, n_init=3, batch_size=4096).fit_predict(Z)
    NUM["cluster_gap"] = pd.DataFrame({f"{n}_gap_cluster": np.asarray(v, dtype=float) - pd.Series(np.asarray(v, dtype=float)).groupby(lab).transform("median").to_numpy()
                                       for n, v in (("ext_mean", e_m), ("ext2", e2), ("ext3", e3))})

    rflag = pd.DataFrame({
        "inc_r10k": (inc % 10000 == 0), "inc_r5k": (inc % 5000 == 0), "inc_r1k": (inc % 1000 == 0),
        "cr_r5k": (cr % 5000 == 0), "cr_r1k": (cr % 1000 == 0), "gd_r5k": (gd % 5000 == 0), "gd_r1k": (gd % 1000 == 0),
        "an_int": (an % 1 == 0), "cr_eq_gd": (cr == gd)}).astype(float)
    NUM["round"] = pd.concat([rflag.add_prefix("rf_"), pd.DataFrame({
        "round_count": rflag.sum(axis=1), "inc_mod10k": inc % 10000, "cr_mod5k": cr % 5000, "gd_mod5k": gd % 5000})], axis=1)

    yi, yr, yp, ya = F_["years_id_publish"], F_["years_registration"], F_["years_last_phone_change"], F_["age_years"]
    NUM["timediff"] = pd.DataFrame({
        "id_minus_reg": yi - yr, "phone_minus_id": yp - yi, "phone_minus_reg": yp - yr,
        "reg_over_age": yr / ya, "id_over_age": yi / ya, "phone_over_age": yp / ya,
        "emp_over_reg": ye / yr.replace(0, np.nan), "emp_minus_reg": ye - yr, "id_reg_absgap": (yi - yr).abs()})

    bh, bm_, bq, by_ = (F_[f"amt_req_credit_bureau_{k}"] for k in ("hour", "mon", "qrt", "year"))
    tot = F_["total_bureau_req"]
    NUM["bureau2"] = pd.DataFrame({
        "bur_year_per_qrt": by_ / (bq + 1), "bur_mon_share": bm_ / (by_ + 1), "bur_recent_share": (bh + bm_ + bq) / (tot + 1),
        "bur_total_x_ext": tot * e_m, "bur_qrt_x_ext": bq * e_m})

    ch, fm = F_["cnt_children"], F_["cnt_fam_members"]
    NUM["family"] = pd.DataFrame({
        "credit_per_person": cr / fm.replace(0, np.nan), "annuity_per_person": an / fm.replace(0, np.nan),
        "goods_per_person": gd / fm.replace(0, np.nan), "income_per_child": inc / (ch + 1),
        "adults": fm - ch, "children_share": ch / fm.replace(0, np.nan)})

    q = lambda s, p: np.nanquantile(np.asarray(s, dtype=float), p)
    ci_, ai_, ipp = cr / inc.replace(0, np.nan), an / inc.replace(0, np.nan), inc / fm.replace(0, np.nan)
    fl = pd.DataFrame({
        "fl_low_ext": (e_m < q(e_m, 0.2)), "fl_high_credit_income": (ci_ > q(ci_, 0.8)), "fl_high_annuity_income": (ai_ > q(ai_, 0.8)),
        "fl_young": (age < q(age, 0.1)), "fl_short_job": (ye < q(ye, 0.1)), "fl_rating3": (F_["region_rating_client"] == 3),
        "fl_recent_phone": (yp < q(yp, 0.1)), "fl_recent_id": (yi < q(yi, 0.1)), "fl_low_income_pc": (ipp < q(ipp, 0.1))}).astype(float)
    NUM["flags"] = pd.concat([fl, pd.DataFrame({"risk_flag_count": fl.sum(axis=1)})], axis=1)

    # ================== 2) NGUỒN MÃ CHO TARGET ENCODING ==================
    tn = F_["credit_term"].round().clip(0, 120)
    reg_pop = qbin(F_["region_population_relative"], 10)
    ex10 = qbin(e_m, 10)
    SRC = pd.DataFrame({
        "c_term": key_codes(tn), "c_term_ctype": key_codes(tn, F_["name_contract_type"]), "c_term_inc": key_codes(tn, F_["c_income_type"]),
        "c_region": key_codes(F_["region_rating_client"], reg_pop),
        "c_region_house": key_codes(F_["region_rating_client"], reg_pop, F_["c_housing"]),
        "c_rating_inc": key_codes(F_["region_rating_client"], F_["c_income_type"]),
        "c_b_ext_term": key_codes(ex10, qbin(F_["credit_term"], 10)), "c_b_ext_age": key_codes(ex10, age_bin),
        "c_b_ext_inc": key_codes(ex10, qbin(inc, 10)), "c_b_ext_ai": key_codes(ex10, qbin(ai_, 10)),
        "c_b_ext_job": key_codes(ex10, np.where(np.isnan(qbin(ye, 5)), -1, qbin(ye, 5))), "c_b_e2_e3": key_codes(qbin(e2, 10), qbin(e3, 10)),
        "h_occ_edu_inc": key_codes(occ, F_["name_education_type"], F_["c_income_type"]), "h_org_age": key_codes(org, age_bin),
        "h_rh": key_codes(F_["region_rating_client"], F_["c_housing"]),
        "h_rhf": key_codes(F_["region_rating_client"], F_["c_housing"], F_["c_family"]),
    }, index=F_.index)

    ALL6F = pd.concat([v for v in NUM.values()] + [SRC], axis=1).replace([np.inf, -np.inf], np.nan).astype("float32")
    ALL6F.index = F_.index
    X_all = pd.concat([X_all.drop(columns=[c for c in ALL6F.columns if c in X_all.columns]), ALL6F.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in ALL6F.columns if c in X_test_all.columns]),
                            ALL6F.iloc[n_tr:].reset_index(drop=True)], axis=1)

    BLK6F = {k: dict(feats=list(v.columns)) for k, v in NUM.items()}
    BLK6F["te_resid"] = dict(te_resid=["c_occupation", "c_organization", "c_income_type", "c_family", "c_housing", "c_education",
                                       "c_occ_gender", "c_edu_income", "h_occ_edu_inc", "h_org_age"])
    BLK6F["te_hier"] = dict(te_hier=[("h_occ_edu_inc", "c_occupation"), ("h_org_age", "c_organization"), ("h_rhf", "h_rh")])
    BLK6F["te_term"] = dict(te=["c_term", "c_term_ctype", "c_term_inc"])
    BLK6F["te_region"] = dict(te=["c_region", "c_region_house", "c_rating_inc"])
    BLK6F["te_bins2d"] = dict(te=["c_b_ext_term", "c_b_ext_age", "c_b_ext_inc", "c_b_ext_ai", "c_b_ext_job", "c_b_e2_e3"])
    chk = ALL6F.iloc[:n_tr]
    print(f"Dựng khối xong ({time.time() - t0:.0f}s): " + ", ".join(f"{k}={sum(len(x) for x in v.values())}" for k, v in BLK6F.items()))
    print("Cột hằng số:", [c for c in chk.columns if chk[c].std() == 0] or "không có")

    # ================== 3) MÃ HÓA TRONG FOLD ==================
    def te_resid_enc(A, yA, others, cols, ext_col="ext_mean_ml", m=20.0, inner=5, seed=SEED):
        ext = A[ext_col].to_numpy(dtype=float)

        def stats(idx):
            e, yy = ext[idx], yA[idx]
            edges = np.unique(np.nanquantile(e, np.linspace(0.1, 0.9, 9)))
            b = np.digitize(e, edges)
            r = yy - pd.Series(b).map(pd.Series(yy).groupby(b).mean()).to_numpy()
            st = {c: pd.DataFrame({"k": A[c].to_numpy()[idx], "r": r}).groupby("k")["r"].agg(["sum", "count"]) for c in cols}
            return {c: s["sum"] / (s["count"] + m) for c, s in st.items()}

        tr_out = {f"{c}_rte": np.zeros(len(A)) for c in cols}
        for a, b in StratifiedKFold(inner, shuffle=True, random_state=seed).split(A, yA):
            st = stats(a)
            for c in cols:
                tr_out[f"{c}_rte"][b] = pd.Series(A[c].to_numpy()[b]).map(st[c]).fillna(0.0).to_numpy()
        st = stats(np.arange(len(A)))
        oth = [pd.DataFrame({f"{c}_rte": pd.Series(o[c].to_numpy()).map(st[c]).fillna(0.0).to_numpy() for c in cols}) for o in others]
        return pd.DataFrame(tr_out), oth

    def te_hier_enc(A, yA, others, pairs, m=20.0, mp=50.0, inner=5, seed=SEED):
        def stats(idx, ch_, pa_):
            gm = yA[idx].mean()
            kp, kc, yy = A[pa_].to_numpy()[idx], A[ch_].to_numpy()[idx], yA[idx]
            sp = pd.DataFrame({"k": kp, "y": yy}).groupby("k")["y"].agg(["sum", "count"])
            par = (sp["sum"] + mp * gm) / (sp["count"] + mp)
            pte = pd.Series(kp).map(par).to_numpy()
            sc = pd.DataFrame({"k": kc, "y": yy, "p": pte}).groupby("k").agg(s=("y", "sum"), n=("y", "count"), p=("p", "mean"))
            return gm, par, (sc["s"] + m * sc["p"]) / (sc["n"] + m)

        def apply(df, ch_, pa_, st):
            gm, par, child = st
            p = pd.Series(df[pa_].to_numpy()).map(par).fillna(gm).to_numpy()
            c = pd.Series(df[ch_].to_numpy()).map(child).to_numpy()
            return np.where(np.isnan(c), p, c)

        tr_out = {f"{ch_}_hte": np.zeros(len(A)) for ch_, _ in pairs}
        for a, b in StratifiedKFold(inner, shuffle=True, random_state=seed).split(A, yA):
            for ch_, pa_ in pairs:
                tr_out[f"{ch_}_hte"][b] = apply(A.iloc[b], ch_, pa_, stats(a, ch_, pa_))
        oth = [dict() for _ in others]
        for ch_, pa_ in pairs:
            st = stats(np.arange(len(A)), ch_, pa_)
            for d, o in zip(oth, others):
                d[f"{ch_}_hte"] = apply(o, ch_, pa_, st)
        return pd.DataFrame(tr_out), [pd.DataFrame(d) for d in oth]

    def fold_data3(cfg, tr, va):
        Xtr, Xva, Xte, f = fold_data(list(dict.fromkeys(cfg["feats"])), list(dict.fromkeys(cfg["te"])), tr, va)
        A, Bv, T = X_all.iloc[tr].reset_index(drop=True), X_all.iloc[va].reset_index(drop=True), X_test_all.reset_index(drop=True)
        parts = []
        if cfg.get("te_resid"):
            parts.append(te_resid_enc(A, y[tr], [Bv, T], cfg["te_resid"]))
        if cfg.get("te_hier"):
            parts.append(te_hier_enc(A, y[tr], [Bv, T], cfg["te_hier"]))
        if cfg.get("sub") and "submodel_feats" in globals():
            parts.append(submodel_feats(A, y[tr], [Bv, T]))
        for t_tr, (t_va, t_te) in parts:
            Xtr, Xva, Xte = (pd.concat([p, q_], axis=1) for p, q_ in ((Xtr, t_tr), (Xva, t_va), (Xte, t_te)))
            f = f + list(t_tr.columns)
        return Xtr, Xva, Xte, f

    def run_cfg3(cfg, base, seeds=(SEED,), es=100, name=""):
        t1 = time.time()
        oof, te_pred, iters, imp = np.zeros(len(y)), np.zeros(len(X_test_all)), [], None
        for tr, va in SPLITS:
            Xtr, Xva, Xte, f = fold_data3(cfg, tr, va)
            imp = pd.Series(0.0, index=f) if imp is None else imp
            prm = {**base, **cfg["params"]}
            if cfg.get("mono"):
                prm["monotone_constraints"] = [-1 if c in MONO else 0 for c in f]
            for s in seeds:
                m = lgb.LGBMClassifier(**{**prm, "random_state": s})
                m.fit(Xtr, y[tr], eval_set=[(Xva, y[va])], eval_metric="auc",
                      callbacks=[lgb.early_stopping(es, verbose=False), lgb.log_evaluation(0)])
                oof[va] += m.predict_proba(Xva)[:, 1] / len(seeds)
                te_pred += m.predict_proba(Xte)[:, 1] / (len(seeds) * N_FOLDS)
                imp += m.booster_.feature_importance("gain") / (len(seeds) * N_FOLDS)
                iters.append(m.best_iteration_)
        scores = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
        print(f"[{name}] AUC = {np.mean(scores):.4f} ± {np.std(scores):.4f} | log-loss = {log_loss(y, np.clip(oof, 1e-6, 1 - 1e-6)):.5f} "
              f"| {len(f)} feature | cây TB = {np.mean(iters):.0f} | {time.time() - t1:.0f}s")
        return {"oof": oof, "test_pred": te_pred, "scores": scores, "imp": imp.sort_values(ascending=False), "name": name,
                "iters": iters, "cfg": cfg, "base": base, "seeds": tuple(seeds)}

    def add6(cfgx, spec):
        out = dict(cfgx)
        out["feats"] = cfgx["feats"] + spec.get("feats", [])
        out["te"] = cfgx["te"] + spec.get("te", [])
        out["te_resid"] = cfgx.get("te_resid", []) + spec.get("te_resid", [])
        out["te_hier"] = cfgx.get("te_hier", []) + spec.get("te_hier", [])
        return out

    # ================== 4) TẦNG 1: SÀNG LỌC TỪNG KHỐI ==================
    REPLAY_6F = ["te_term"]    # khối đã qua sàng lọc và xác nhận ở lần chạy trước (delta +0.0006, 4/5 fold)
    if REPLAY:
        cfg_f = base_cfg
        for _nm in REPLAY_6F:
            cfg_f = add6(cfg_f, BLK6F[_nm])
        kept_f = list(REPLAY_6F)
        print("REPLAY 6f: dùng thẳng khối", kept_f, "(không sàng lọc, không huấn luyện lại)")
    else:
        cur_f = run_cfg3(base_cfg, FAST_PARAMS, es=ES_FAST, name="mốc 6f (cấu hình tốt nhất trước đó, lr 0.05)")
        rows = [{"bước": "mốc", "AUC": np.mean(cur_f["scores"]), "giữ": "-"}]
        single = {}
        for nm, spec in BLK6F.items():
            r = run_cfg3(add6(base_cfg, spec), FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
            c = compare(r, cur_f)
            single[nm] = c["delta"] if c["verdict"] == "tốt hơn" else None
            rows.append({"bước": f"riêng: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
        short = sorted([k for k, v in single.items() if v is not None], key=lambda k: -single[k])
        print("Qua sàng lọc:", short or "không có")

        # gộp dần các khối qua sàng lọc, mỗi lần kiểm lại với cấu hình đang có
        cfg_f, cur_g, kept_f = base_cfg, cur_f, []
        for nm in short:
            cand = add6(cfg_f, BLK6F[nm])
            r = run_cfg3(cand, FAST_PARAMS, es=ES_FAST, name=f"gộp +{nm}")
            c = compare(r, cur_g)
            keep = c["verdict"] == "tốt hơn"
            rows.append({"bước": f"gộp dần: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if keep else "không"})
            if keep:
                cfg_f, cur_g, kept_f = cand, r, kept_f + [nm]
        display(pd.DataFrame(rows).set_index("bước").round(4))
        print("Khối qua tầng 1:", kept_f or "không có")


    # ================== 5) TẦNG 2: XÁC NHẬN TRÊN BỘ FOLD KHÁC ==================
    confirmed = False
    if kept_f and not REPLAY:
        SPLITS_MAIN = SPLITS
        SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 101).split(np.zeros(len(y)), y))
        try:
            a_ = run_cfg3(base_cfg, FAST_PARAMS, es=ES_FAST, name="XÁC NHẬN: mốc, fold mới")
            b_ = run_cfg3(cfg_f, FAST_PARAMS, es=ES_FAST, name="XÁC NHẬN: cấu hình mới, fold mới")
        finally:
            SPLITS = SPLITS_MAIN
        c = compare(b_, a_)
        confirmed = c["verdict"] == "tốt hơn"
        print(f"Xác nhận trên fold mới: delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}"
              f" => {'GIỮ' if confirmed else 'BỎ (không qua xác nhận, nhiều khả năng do may rủi)'}")

    # ================== 6) CHẠY LẠI CUỐI VÀ SO VỚI MÔ HÌNH ĐANG DÙNG ==================
    if confirmed:
        ref = globals().get("LGB_BEST", res_lgb)
        res_6f = run_cfg3(cfg_f, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối 6f (cuối)")
        ll = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
        c = compare(res_6f, ref)
        print(f"So với LightGBM tốt nhất trước 6f ({ref['name']}): delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) "
              f"-> {c['verdict']} | log-loss {ll(res_6f):.5f} vs {ll(ref):.5f}")
        if c["verdict"] == "tốt hơn" and ll(res_6f) <= ll(ref) + 1e-5:
            LGB_BEST = res_6f
            cand_final = res_6f
            if "res_catn" in globals():
                lg = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
                ob = expit(0.7 * lg(res_6f["oof"]) + 0.3 * lg(res_catn["oof"]))
                cand_final = {"oof": ob, "test_pred": expit(0.7 * lg(res_6f["test_pred"]) + 0.3 * lg(res_catn["test_pred"])),
                              "scores": [roc_auc_score(y[va], ob[va]) for _, va in SPLITS], "name": "Blend LGB(6f) 0.7 + CatNative 0.3"}
            c2 = compare(cand_final, FINAL)
            print(f"So với mô hình đang dùng ({FINAL['name']}, AUC {np.mean(FINAL['scores']):.4f}): delta = {c2['delta']:+.4f} "
                  f"(se {c2['delta_se']:.4f}, fold dương {c2['fold_duong']}) -> {c2['verdict']}")
            if c2["verdict"] == "tốt hơn" and ll(cand_final) <= ll(FINAL) + 1e-5:
                FINAL = cand_final
    else:
        print("REPLAY 6f: xong, cấu hình cố định đã áp dụng." if REPLAY else "Không khối nào của 6f qua cả hai tầng: giữ nguyên mô hình đang dùng.")
    print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")
else:
    print("Bỏ qua mục 6f (RUN_6F = False).")

Dựng khối xong (7s): zgrp=18, cluster_gap=3, round=13, timediff=9, bureau2=5, family=6, flags=10, te_resid=10, te_hier=3, te_term=3, te_region=3, te_bins2d=6
Cột hằng số: ['fl_recent_phone']
REPLAY 6f: dùng thẳng khối ['te_term'] (không sàng lọc, không huấn luyện lại)
REPLAY 6f: xong, cấu hình cố định đã áp dụng.

>>> Mô hình dùng cho mục 7 và 8: LightGBM mục 6 (REPLAY: không huấn luyện lại) | AUC CV = nan


## 6g. Dùng mô hình khác loại để hỗ trợ cây

Ý tưởng: cây chia theo từng trục và xấp xỉ hàm trơn bằng các bậc thang; mô hình khác có "góc nhìn" khác (trơn, cộng dồn, phi tuyến liên tục). Có ba cách đưa góc nhìn đó vào, từ nhẹ đến nặng:

| Cách | Nội dung | Ghi chú |
|---|---|---|
| **Trộn / stacking ở mức dự đoán** | LightGBM + CatBoost + từng mô hình mới, rồi hồi quy logistic trên logit (stacker chọn trên 4 fold, đo trên fold còn lại) | cách ít rủi ro nhất; thường là cách duy nhất cho lợi ích thật |
| **Meta-feature** | dự đoán OOF **lồng trong fold** của mô hình GAM-logistic đưa vào LightGBM như một cột | ver4 đã thử với logistic và LightGBM nông (M1 đến M3): delta 0.0000. Lần này dùng GAM (logistic trên bin) trơn hơn |
| **`init_score`** | LightGBM khởi động từ logit của mô hình GAM rồi chỉ học **phần dư** | cây không phải dựng lại phần cộng dồn trơn; khác meta-feature ở chỗ không cần chia theo cột |

**Các mô hình mới**
- `GAM-LR`: logistic trên one-hot của 16 bin phân vị mỗi feature số (mỗi feature thành hàm bậc thang riêng) cộng biến phân loại. Trơn, cộng dồn, rất khác cây.
- `MLP`: mạng nơ-ron 2 lớp trên feature đã chuẩn hóa phân vị (+ cờ thiếu, one-hot). Dừng sớm theo **AUC** của một phần 10% tách khỏi train fold (không dùng fold kiểm tra).
- `XGBoost`: cùng feature và mã hóa của LightGBM nhưng thuật toán khác.

Meta-feature và `init_score` chỉ làm với GAM-LR, vì lồng trong fold nên MLP và XGBoost quá tốn.

**Cách đọc.** Mô hình mới nhìn riêng lẻ gần như chắc chắn thấp hơn LightGBM; điều quan trọng là bảng tương quan logit (càng thấp càng đa dạng) và delta khi trộn. Kỳ vọng thật: +0.0003 đến +0.002 nếu MLP đủ đa dạng; meta-feature và `init_score` nhiều khả năng như nhau.

In [12]:
RUN_6G = False   # đã có kết quả: GAM/MLP/XGBoost/meta-feature đều không thắng LightGBM + CatBoost
RUN_MLP = True        # MLP sklearn: vài phút mỗi fold
RUN_XGB = True
RUN_META = True       # meta-feature và init_score từ GAM-LR (OOF lồng trong fold)

if RUN_6G:
    from scipy import sparse
    from sklearn.linear_model import LogisticRegression
    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import OneHotEncoder, QuantileTransformer

    t0 = time.time()
    lg_ = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
    ll_ = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
    cfg_base = next((dict(globals()[n]) for n in ("cfg_f", "cfg_e", "cfg6", "cfg") if n in globals() and isinstance(globals()[n], dict)),
                    dict(feats=list(FEATS), te=list(TE_USED), params={}, mono=False))
    for k in ("te_resid", "te_hier"):
        cfg_base.setdefault(k, [])
    FD = globals().get("fold_data3") or globals().get("fold_data2") or (lambda c, tr, va: fold_data(c["feats"], c["te"], tr, va))
    n_te = len(X_test_all)

    # ---------- ma trận cho GAM-LR và MLP (biến đổi không dùng nhãn, fit trên train + test) ----------
    CATLR = [c for c in ["name_contract_type", "code_gender", "name_income_type", "name_family_status", "name_housing_type",
                         "occupation_type", "organization_type", "region_rating_client", "name_education_type"] if c in X_all.columns]
    num_cols = list(dict.fromkeys(c for c in cfg_base["feats"] if c in X_all.columns and c not in CATLR))
    pool = pd.concat([X_all[num_cols], X_test_all[num_cols]], ignore_index=True)
    Bi = {}
    for c in num_cols:
        v = pool[c].to_numpy(dtype=float)
        edges = np.unique(np.nanquantile(v, np.linspace(0, 1, 17)[1:-1]))
        b = np.digitize(v, edges)
        b[np.isnan(v)] = -1
        Bi[c] = b
    catp = pd.concat([X_all[CATLR], X_test_all[CATLR]], ignore_index=True).fillna(-1).astype(int)
    S = OneHotEncoder(handle_unknown="ignore", dtype=np.float32).fit_transform(pd.concat([pd.DataFrame(Bi), catp], axis=1)).tocsr()
    miss_cols = [c for c in num_cols if pool[c].isna().mean() > 0.01][:8]
    Xn = pool.replace([np.inf, -np.inf], np.nan)
    Xn = Xn.fillna(Xn.median())
    Xq = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=200000, random_state=SEED).fit_transform(Xn.to_numpy())
    D = np.hstack([Xq, pool[miss_cols].isna().to_numpy().astype(np.float32),
                   OneHotEncoder(handle_unknown="ignore", dtype=np.float32).fit_transform(catp).toarray()]).astype(np.float32)
    print(f"Ma trận xong ({time.time() - t0:.0f}s): GAM {S.shape}, MLP {D.shape}")

    def run_other(fit_predict, name):
        t1 = time.time()
        oof, te_pred = np.zeros(n_tr), np.zeros(n_te)
        for tr, va in SPLITS:
            pv, pt = fit_predict(tr, va)
            oof[va] = pv
            te_pred += pt / N_FOLDS
        scores = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
        print(f"[{name}] AUC = {np.mean(scores):.4f} ± {np.std(scores):.4f} | log-loss = {log_loss(y, np.clip(oof, 1e-6, 1 - 1e-6)):.5f} | {time.time() - t1:.0f}s")
        return {"oof": oof, "test_pred": te_pred, "scores": scores, "name": name}

    def fp_gam(tr, va):
        m = LogisticRegression(C=0.05, max_iter=300).fit(S[tr], y[tr])
        return m.predict_proba(S[va])[:, 1], m.predict_proba(S[n_tr:])[:, 1]

    def fp_mlp(tr, va, max_epochs=40, patience=4):
        rng = np.random.RandomState(SEED)
        inner = StratifiedKFold(10, shuffle=True, random_state=SEED).split(tr, y[tr])
        a, b = next(inner)
        Xf, yf, Xi, yi = D[tr[a]], y[tr[a]], D[tr[b]], y[tr[b]]
        m = MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-3, batch_size=512, learning_rate_init=1e-3, random_state=SEED)
        best, best_w, bad = -1.0, None, 0
        for ep in range(5 if DEBUG_FAST else max_epochs):
            perm = rng.permutation(len(Xf))
            m.partial_fit(Xf[perm], yf[perm], classes=[0, 1])
            s = roc_auc_score(yi, m.predict_proba(Xi)[:, 1])
            if s > best + 1e-5:
                best, bad = s, 0
                best_w = ([w.copy() for w in m.coefs_], [w.copy() for w in m.intercepts_])
            else:
                bad += 1
                if bad >= patience:
                    break
        m.coefs_, m.intercepts_ = best_w
        return m.predict_proba(D[va])[:, 1], m.predict_proba(D[n_tr:])[:, 1]

    def fp_xgb(tr, va):
        import xgboost as xgb
        Xtr, Xva, Xte, _ = FD(cfg_base, tr, va)
        m = xgb.XGBClassifier(n_estimators=200 if DEBUG_FAST else 4000, learning_rate=0.1 if DEBUG_FAST else 0.03, max_depth=4,
                              subsample=0.8, colsample_bytree=0.4, min_child_weight=20, reg_lambda=10.0, tree_method="hist",
                              eval_metric="auc", early_stopping_rounds=30 if DEBUG_FAST else 200, n_jobs=-1, random_state=SEED)
        m.fit(Xtr, y[tr], eval_set=[(Xva, y[va])], verbose=False)
        return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]

    others = {"GAM-LR": run_other(fp_gam, "GAM-LR")}
    if RUN_MLP:
        others["MLP"] = run_other(fp_mlp, "MLP")
    if RUN_XGB:
        try:
            others["XGBoost"] = run_other(fp_xgb, "XGBoost")
        except ImportError:
            print("Không có xgboost, bỏ qua.")

    # ---------- meta-feature và init_score từ GAM-LR (OOF lồng trong fold) ----------
    def lr_dec(Sx, yx, Sys, C=0.05):
        m = LogisticRegression(C=C, max_iter=300).fit(Sx, yx)
        return [m.decision_function(s) for s in Sys]

    def run_meta(cfg, mode, base, seeds=(SEED,), es=100, name=""):
        t1 = time.time()
        oof, te_pred, iters = np.zeros(n_tr), np.zeros(n_te), []
        for tr, va in SPLITS:
            Xtr, Xva, Xte, f = FD(cfg, tr, va)
            z_tr = np.zeros(len(tr))
            for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED + 7).split(tr, y[tr]):
                z_tr[b] = lr_dec(S[tr[a]], y[tr[a]], [S[tr[b]]])[0]
            z_va, z_te = lr_dec(S[tr], y[tr], [S[va], S[n_tr:]])
            prm = {**base, **cfg["params"]}
            if mode == "feat":
                Xtr, Xva, Xte = Xtr.assign(gam_lr=z_tr), Xva.assign(gam_lr=z_va), Xte.assign(gam_lr=z_te)
            for s in seeds:
                m = lgb.LGBMClassifier(**{**prm, "random_state": s})
                if mode == "feat":
                    m.fit(Xtr, y[tr], eval_set=[(Xva, y[va])], eval_metric="auc",
                          callbacks=[lgb.early_stopping(es, verbose=False), lgb.log_evaluation(0)])
                    pv, pt = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
                else:
                    m.fit(Xtr, y[tr], init_score=z_tr, eval_set=[(Xva, y[va])], eval_init_score=[z_va], eval_metric="auc",
                          callbacks=[lgb.early_stopping(es, verbose=False), lgb.log_evaluation(0)])
                    pv = expit(m.predict(Xva, raw_score=True) + z_va)
                    pt = expit(m.predict(Xte, raw_score=True) + z_te)
                oof[va] += pv / len(seeds)
                te_pred += pt / (len(seeds) * N_FOLDS)
                iters.append(m.best_iteration_)
        scores = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
        print(f"[{name}] AUC = {np.mean(scores):.4f} ± {np.std(scores):.4f} | log-loss = {log_loss(y, np.clip(oof, 1e-6, 1 - 1e-6)):.5f} "
              f"| cây TB = {np.mean(iters):.0f} | {time.time() - t1:.0f}s")
        return {"oof": oof, "test_pred": te_pred, "scores": scores, "name": name}

    if RUN_META:
        ref_fast = (globals().get("run_cfg3") or globals().get("run_cfg2") or globals().get("run_cfg"))(
            cfg_base, FAST_PARAMS, es=ES_FAST, name="mốc 6g (lr 0.05)")
        rows = [{"cách": "mốc", "AUC": np.mean(ref_fast["scores"]), "giữ": "-"}]
        best_mode, best_res = None, None
        for mode in ("feat", "init"):
            r = run_meta(cfg_base, mode, FAST_PARAMS, es=ES_FAST, name=f"GAM-LR làm {'meta-feature' if mode == 'feat' else 'init_score'}")
            c = compare(r, ref_fast)
            win = c["verdict"] == "tốt hơn"
            rows.append({"cách": "meta-feature" if mode == "feat" else "init_score", "AUC": np.mean(r["scores"]),
                         **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if win else "không"})
            if win and (best_res is None or c["delta"] > best_res[1]):
                best_mode, best_res = mode, (r, c["delta"])
        display(pd.DataFrame(rows).set_index("cách").round(4))
        if best_mode is not None:
            ref = globals().get("LGB_BEST", res_lgb)
            res_meta = run_meta(cfg_base, best_mode, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name=f"LightGBM + GAM-LR ({best_mode}, cuối)")
            c = compare(res_meta, ref)
            print(f"So với LightGBM tốt nhất trước 6g ({ref['name']}): delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}")
            if c["verdict"] == "tốt hơn" and ll_(res_meta) <= ll_(ref) + 1e-5:
                LGB_BEST = res_meta
                print("LGB_BEST cập nhật:", res_meta["name"])
        else:
            print("Meta-feature và init_score không thắng mốc: giữ LightGBM thuần.")

    # ---------- đa dạng và trộn / stacking ----------
    LGBB = globals().get("LGB_BEST", res_lgb)
    comps = {"LightGBM": LGBB}
    if "res_catn" in globals():
        comps["CatBoost native"] = res_catn
    comps.update(others)
    names = list(comps)
    LG = np.c_[[lg_(comps[k]["oof"]) for k in names]].T
    cor = pd.DataFrame(np.corrcoef(LG.T), index=names, columns=names)
    display(cor.round(3))
    print("Tương quan logit với LightGBM càng thấp thì càng đa dạng; mô hình đơn có AUC:",
          {k: round(float(np.mean(v["scores"])), 4) for k, v in comps.items()})

    def make(oof, tp, name):
        return {"oof": oof, "test_pred": tp, "scores": [roc_auc_score(y[va], oof[va]) for _, va in SPLITS], "name": name}

    cands = {}
    for k, v in comps.items():
        if k == "LightGBM":
            continue
        cands[f"Blend LGB 0.7 + {k} 0.3"] = make(expit(0.7 * lg_(LGBB["oof"]) + 0.3 * lg_(v["oof"])),
                                                  expit(0.7 * lg_(LGBB["test_pred"]) + 0.3 * lg_(v["test_pred"])), f"Blend LGB 0.7 + {k} 0.3")
    if len(comps) >= 3:
        LT = np.c_[[lg_(comps[k]["test_pred"]) for k in names]].T
        oof_s = np.zeros(n_tr)
        for tr, va in SPLITS:
            oof_s[va] = LogisticRegression(C=1.0, max_iter=500).fit(LG[tr], y[tr]).predict_proba(LG[va])[:, 1]
        st = LogisticRegression(C=1.0, max_iter=500).fit(LG, y)
        print("Hệ số stacker (trên logit):", dict(zip(names, np.round(st.coef_[0], 3))))
        cands["Stacking LR (tất cả)"] = make(oof_s, st.predict_proba(LT)[:, 1], "Stacking LR (tất cả)")

    rows, ok = [], []
    for k, r in cands.items():
        c = compare(r, FINAL)
        good = c["verdict"] == "tốt hơn" and ll_(r) <= ll_(FINAL) + 1e-5
        rows.append({"ứng viên": k, "AUC": np.mean(r["scores"]), **{kk: c[kk] for kk in ("delta", "delta_se", "fold_duong", "verdict")}, "log-loss": ll_(r)})
        if good:
            ok.append(k)
    display(pd.DataFrame(rows).set_index("ứng viên").round(4))
    if ok:
        pick = max(ok, key=lambda k: np.mean(cands[k]["scores"]))
        FINAL = cands[pick]
        print("Chọn:", pick)
    else:
        print("Không ứng viên nào hơn mô hình đang dùng theo quy tắc delta_se: giữ", FINAL["name"])
    print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")
else:
    print("Bỏ qua mục 6g (RUN_6G = False).")

Bỏ qua mục 6g (RUN_6G = False).


In [13]:
if RUN_6D:
    # ---------- mô hình phụ OOF lồng trong fold ----------
    def _cols(c):
        return [x for x in c if x in X_all.columns]

    SLICES = {
        "demo": _cols(["age_years", "c_occupation", "c_organization", "c_income_type", "c_family", "c_housing", "name_education_type",
                       "code_gender", "cnt_children", "cnt_fam_members", "region_rating_client", "region_population_relative",
                       "name_contract_type", "years_employed"]),
        "fin": _cols(["income", "credit_amount", "annuity", "amt_goods_price", "credit_term", "goods_credit_ratio", "credit_income_ratio",
                      "annuity_income_ratio", "credit_goods_diff", "income_per_person", "total_bureau_req", "own_car_age"]),
        "ext": _cols(EXT + ML_COLS + ["ext_mean", "ext_min", "ext_max", "ext_std"]),
        "doc": _cols(["years_registration", "years_id_publish", "years_last_phone_change", "flag_work_phone", "id_issue_age",
                      "registration_age", "details_change_sum"]),
        "miss": _cols(["ext_missing_pattern", "financial_missing_count", "demographic_missing_count", "bureau_missing_count",
                       "document_missing_count", "missing_total"]),
    }
    SUB_PARAMS = dict(n_estimators=150, learning_rate=0.05, num_leaves=7, min_child_samples=200, colsample_bytree=0.8,
                      subsample=0.8, subsample_freq=1, reg_lambda=10.0, n_jobs=-1, verbose=-1)

    def submodel_feats(Xtr, ytr, others, inner=5, seed=SEED):
        tr_out, oth_out = {}, [dict() for _ in others]
        for nm, cols in SLICES.items():
            oof = np.zeros(len(Xtr))
            for a, b in StratifiedKFold(inner, shuffle=True, random_state=seed).split(Xtr, ytr):
                m = lgb.LGBMClassifier(**SUB_PARAMS, random_state=seed).fit(Xtr.iloc[a][cols], ytr[a])
                oof[b] = m.predict_proba(Xtr.iloc[b][cols])[:, 1]
            tr_out[f"oof_{nm}_risk"] = oof
            m = lgb.LGBMClassifier(**SUB_PARAMS, random_state=seed).fit(Xtr[cols], ytr)
            for d, o in zip(oth_out, others):
                d[f"oof_{nm}_risk"] = m.predict_proba(o[cols])[:, 1]

        def derived(d):
            P = np.c_[[d[k] for k in sorted(d)]].T
            q = P / P.sum(axis=1, keepdims=True)
            d["risk_model_disagreement"] = P.std(axis=1)
            d["risk_model_max"], d["risk_model_min"] = P.max(axis=1), P.min(axis=1)
            d["risk_model_entropy"] = -(q * np.log(q + 1e-12)).sum(axis=1) / np.log(P.shape[1])
            return pd.DataFrame(d)
        return derived(tr_out), [derived(d) for d in oth_out]

    def fold_data2(cfg, tr, va):
        Xtr, Xva, Xte, f = fold_data(cfg["feats"], cfg["te"], tr, va)
        if cfg.get("sub"):
            a, bb = X_all.iloc[tr].reset_index(drop=True), X_all.iloc[va].reset_index(drop=True)
            s_tr, (s_va, s_te) = submodel_feats(a, y[tr], [bb, X_test_all.reset_index(drop=True)])
            Xtr, Xva, Xte = (pd.concat([p, q], axis=1) for p, q in ((Xtr, s_tr), (Xva, s_va), (Xte, s_te)))
            f = f + list(s_tr.columns)
        return Xtr, Xva, Xte, f

    def run_cfg2(cfg, base, seeds=(SEED,), es=100, name=""):
        t0 = time.time()
        oof, te_pred, iters, imp = np.zeros(len(y)), np.zeros(len(X_test_all)), [], None
        for tr, va in SPLITS:
            Xtr, Xva, Xte, f = fold_data2(cfg, tr, va)
            imp = pd.Series(0.0, index=f) if imp is None else imp
            prm = {**base, **cfg["params"]}
            if cfg.get("mono"):
                prm["monotone_constraints"] = [-1 if c in MONO else 0 for c in f]
            for s in seeds:
                m = lgb.LGBMClassifier(**{**prm, "random_state": s})
                m.fit(Xtr, y[tr], eval_set=[(Xva, y[va])], eval_metric="auc",
                      callbacks=[lgb.early_stopping(es, verbose=False), lgb.log_evaluation(0)])
                oof[va] += m.predict_proba(Xva)[:, 1] / len(seeds)
                te_pred += m.predict_proba(Xte)[:, 1] / (len(seeds) * N_FOLDS)
                imp += m.booster_.feature_importance("gain") / (len(seeds) * N_FOLDS)
                iters.append(m.best_iteration_)
        scores = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
        print(f"[{name}] AUC = {np.mean(scores):.4f} ± {np.std(scores):.4f} | log-loss = {log_loss(y, np.clip(oof, 1e-6, 1 - 1e-6)):.5f} "
              f"| {len(f)} feature | cây TB = {np.mean(iters):.0f} | {time.time() - t0:.0f}s")
        return {"oof": oof, "test_pred": te_pred, "scores": scores, "imp": imp.sort_values(ascending=False), "name": name}

    # ---------- chọn tiến từng khối so với cấu hình tốt nhất của 6c ----------
    if "cfg" in globals() and isinstance(cfg, dict):
        cfg6 = {**cfg, "sub": False}
    else:
        cfg6 = dict(feats=list(FEATS), te=list(TE_USED), params={}, mono=False, sub=False)
    MONO = globals().get("MONO", set())
    cur6 = run_cfg2(cfg6, FAST_PARAMS, es=ES_FAST, name="mốc 6d (cấu hình tốt nhất của 6c, lr 0.05)")
    rows6 = [{"khối": "mốc", "AUC": np.mean(cur6["scores"]), "giữ": "-"}]
    order = ["peer", "incons", "miss", "mresid", "cluster", "peerdist", "pca"] + (["submodel"] if RUN_SUBMODEL else [])
    for nm in order:
        if nm == "submodel":
            cand = {**cfg6, "sub": True}
        else:
            bl = BL6[nm]
            cand = {**cfg6, "feats": cfg6["feats"] + bl["feats"], "te": cfg6["te"] + bl["te"]}
        new = run_cfg2(cand, FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
        c = compare(new, cur6)
        keep = c["verdict"] == "tốt hơn"
        rows6.append({"khối": f"+{nm}", "AUC": np.mean(new["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")},
                      "giữ": "có" if keep else "không"})
        if keep:
            cfg6, cur6 = cand, new
    display(pd.DataFrame(rows6).set_index("khối").round(4))
    kept6 = [r["khối"] for r in rows6 if r["giữ"] == "có"]
    print("Khối được giữ:", kept6 or "không có")

    # ---------- chạy lại cấu hình thắng với tham số cuối + nhiều seed, rồi trộn với CatBoost native nếu có ----------
    if kept6:
        ref = globals().get("LGB_BEST", res_lgb)
        res_6d = run_cfg2(cfg6, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối 6d (cuối)")
        ll = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
        c = compare(res_6d, ref)
        print(f"So với LightGBM tốt nhất trước 6d ({ref['name']}): delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) "
              f"-> {c['verdict']} | log-loss {ll(res_6d):.5f} vs {ll(ref):.5f}")
        if c["verdict"] == "tốt hơn" and ll(res_6d) <= ll(ref) + 1e-5:
            LGB_BEST = res_6d
            cand_final = res_6d
            if "res_catn" in globals():
                lg = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
                ob = expit(0.7 * lg(res_6d["oof"]) + 0.3 * lg(res_catn["oof"]))
                cand_final = {"oof": ob, "test_pred": expit(0.7 * lg(res_6d["test_pred"]) + 0.3 * lg(res_catn["test_pred"])),
                              "scores": [roc_auc_score(y[va], ob[va]) for _, va in SPLITS], "name": "Blend LGB(6d) 0.7 + CatNative 0.3"}
            c2 = compare(cand_final, FINAL)
            print(f"So với mô hình đang dùng ({FINAL['name']}, AUC {np.mean(FINAL['scores']):.4f}): delta = {c2['delta']:+.4f} "
                  f"(se {c2['delta_se']:.4f}, fold dương {c2['fold_duong']}) -> {c2['verdict']}")
            if c2["verdict"] == "tốt hơn" and ll(cand_final) <= ll(FINAL) + 1e-5:
                FINAL = cand_final
    print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")
else:
    print("Bỏ qua mục 6d (RUN_6D = False).")

Bỏ qua mục 6d (RUN_6D = False).


## 6h. Tăng cường CatBoost native và nhiều seed

Lý do: gain thật duy nhất từ mô hình khác (0.7647 lên 0.7652) đến từ trộn với **CatBoost native**, mà CatBoost đó đang chạy với tham số mặc định (depth 6, l2 10, 1 seed). Đây là chỗ có bằng chứng nhất để cải thiện.

**Các bước**
1. Lưới nhỏ chạy ở `lr = 0.08` cho nhanh, so với mốc depth 6 / l2 10 bằng đúng quy tắc `compare` (delta lớn hơn `delta_se` và ít nhất 3/5 fold dương): độ sâu 5/7/8, `l2_leaf_reg` 3/30, `max_ctr_complexity` 1/2 (mức kết hợp giữa các cột phân loại), `one_hot_max_size`, `bootstrap_type = Bernoulli`, `rsm`.
2. Gộp tiến các thay đổi thắng (greedy, giống 6c).
3. Chạy lại ở `lr = 0.04` với **3 seed** và lấy trung bình dự đoán (giảm phương sai, gần như không rủi ro).
4. Trộn logit với LightGBM tốt nhất (w = 0.5 / 0.6 / 0.7 / 0.8), so với `FINAL`; chỉ thay khi `compare` nói "tốt hơn" và log-loss không tệ hơn.

**Kỳ vọng.** Gain nhỏ (vài phần mười nghìn AUC). Mục này chậm vì CatBoost với nhiều biến phân loại chạy lâu hơn LightGBM nhiều lần; tắt bớt bằng `CAT_GRID`.

In [14]:
RUN_6H = True
CAT_SEEDS = (SEED, SEED + 1, SEED + 2, SEED + 3, SEED + 4)     # 5 seed (trước đây 3)
CAT_GRID = [  # (tên, thay đổi so với mốc)
    ("depth 5", dict(depth=5)), ("depth 7", dict(depth=7)), ("depth 8", dict(depth=8)),
    ("l2 3", dict(l2_leaf_reg=3)), ("l2 30", dict(l2_leaf_reg=30)),
    ("ctr_complexity 1", dict(max_ctr_complexity=1)), ("ctr_complexity 2", dict(max_ctr_complexity=2)),
    ("one_hot 10", dict(one_hot_max_size=10)),
    ("bernoulli 0.8", dict(bootstrap_type="Bernoulli", subsample=0.8)),
    ("rsm 0.5", dict(rsm=0.5)),
]

if RUN_6H:
    try:
        from catboost import CatBoostClassifier
    except ImportError:
        CatBoostClassifier = None
        print("Không có catboost, bỏ qua 6h.")

if RUN_6H and CatBoostClassifier is not None:
    lg_ = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
    ll_ = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
    cfg_c = next((globals()[n] for n in ("cfg", "cfg6") if n in globals() and isinstance(globals()[n], dict) and "feats" in globals()[n]),
                 dict(feats=list(FEATS)))
    CAT_SRC = [c for c in (TE_CORE + TE_EXTRA) if c in X_all.columns]
    feats_c = [c for c in cfg_c["feats"] if c in X_all.columns and c not in CAT_SRC] + CAT_SRC
    Xte_c = X_test_all[feats_c].copy()
    Xte_c[CAT_SRC] = Xte_c[CAT_SRC].astype("int32")
    FOLD_C = []
    for tr, va in SPLITS:
        a, b = X_all.iloc[tr][feats_c].copy(), X_all.iloc[va][feats_c].copy()
        a[CAT_SRC] = a[CAT_SRC].astype("int32")
        b[CAT_SRC] = b[CAT_SRC].astype("int32")
        FOLD_C.append((a, b))
    BASE_C = dict(depth=6, l2_leaf_reg=10)

    def run_cat(over, lr, seeds, name):
        t1 = time.time()
        prm = {**BASE_C, **over}
        oof, tp, its = np.zeros(n_tr), np.zeros(len(Xte_c)), []
        for (tr, va), (a, b) in zip(SPLITS, FOLD_C):
            for s in seeds:
                m = CatBoostClassifier(iterations=150 if DEBUG_FAST else (4000 if lr >= 0.03 else 9000), learning_rate=0.15 if DEBUG_FAST else lr,
                                       loss_function="Logloss", eval_metric="AUC", od_type="Iter",
                                       od_wait=30 if DEBUG_FAST else 150, random_seed=s, thread_count=-1, verbose=False,
                                       allow_writing_files=False, cat_features=CAT_SRC, **prm)
                m.fit(a, y[tr], eval_set=(b, y[va]), use_best_model=True)
                oof[va] += m.predict_proba(b)[:, 1] / len(seeds)
                tp += m.predict_proba(Xte_c)[:, 1] / (len(seeds) * N_FOLDS)
                its.append(m.get_best_iteration())
        sc = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
        print(f"[{name}] AUC = {np.mean(sc):.4f} ± {np.std(sc):.4f} | cây TB = {np.mean(its):.0f} | {time.time() - t1:.0f}s")
        return {"oof": oof, "test_pred": tp, "scores": sc, "name": name, "iters": its, "lr": lr, "over": dict(over), "seeds": tuple(seeds)}

    lg_ = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))   # trước đây định nghĩa ở 6g
    ll_ = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
    # Lưới đã chạy: chỉ 'depth 5' rồi 'rsm 0.5' được giữ. Dùng lại, chỉ chạy bản cuối 3 seed.
    CAT_PRESET = {"depth": 5, "rsm": 0.5}   # None = chạy lại cả lưới (mất vài giờ)
    if CAT_PRESET is not None:
        cur_over = dict(CAT_PRESET)
        print("Dùng lại tham số CatBoost có sẵn:", {**BASE_C, **cur_over})
    else:
        ref_c = run_cat({}, 0.08, (SEED,), "CatBoost mốc (lr 0.08)")
        cur_over, cur_res, rows = {}, ref_c, []
        for nm, over in CAT_GRID:
            trial = {**cur_over, **over}
            r = run_cat(trial, 0.08, (SEED,), f"CatBoost + {nm}")
            c = compare(r, cur_res)
            win = c["verdict"] == "tốt hơn"
            rows.append({"thử": nm, "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if win else "không"})
            if win:
                cur_over, cur_res = trial, r
        display(pd.DataFrame(rows).set_index("thử").round(4))
        print("Tham số CatBoost được giữ:", {**BASE_C, **cur_over})


    _sig_c = f"cat|{len(feats_c)}|{sorted({**BASE_C, **cur_over}.items())}|{len(CAT_SEEDS)}|{DEBUG_FAST}"
    res_cat_tuned = _replay_cache_load("replay5_cat_tuned_price.npz", _sig_c) if REPLAY else None
    if res_cat_tuned is None:
        res_cat_tuned = run_cat(cur_over, 0.04, CAT_SEEDS, f"CatBoost tune, {len(CAT_SEEDS)} seed")
        if REPLAY:
            _replay_cache_save("replay5_cat_tuned_price.npz", res_cat_tuned, _sig_c)
    else:
        print("REPLAY: nạp CatBoost đã huấn luyện từ replay5_cat_tuned.npz (xóa file này để huấn luyện lại)")
    res_cat_tuned.update({"seeds": tuple(CAT_SEEDS), "over": dict(cur_over), "lr": 0.04})
    if REPLAY:
        print("REPLAY 6h: CatBoost tune (depth 5, rsm 0.5, 3 seed) đã sẵn sàng; việc trộn làm ở ô 6i.")
    else:
        old_cat = globals().get("res_catn")
        if old_cat is not None:
            c = compare(res_cat_tuned, old_cat)
            print(f"So với CatBoost native cũ: delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}")

        LGBB = globals().get("LGB_BEST", res_lgb)
        cands = {}
        for w in (0.5, 0.6, 0.7, 0.8):
            o = expit(w * lg_(LGBB["oof"]) + (1 - w) * lg_(res_cat_tuned["oof"]))
            t = expit(w * lg_(LGBB["test_pred"]) + (1 - w) * lg_(res_cat_tuned["test_pred"]))
            nm = f"Blend LGB {w:.1f} + CatTune {1 - w:.1f}"
            cands[nm] = {"oof": o, "test_pred": t, "scores": [roc_auc_score(y[va], o[va]) for _, va in SPLITS], "name": nm}
        rows, ok = [], []
        for k, r in cands.items():
            c = compare(r, FINAL)
            good = c["verdict"] == "tốt hơn" and ll_(r) <= ll_(FINAL) + 1e-5
            rows.append({"ứng viên": k, "AUC": np.mean(r["scores"]), **{kk: c[kk] for kk in ("delta", "delta_se", "fold_duong", "verdict")}, "log-loss": ll_(r)})
            if good:
                ok.append(k)
        print(f"Mốc FINAL hiện tại: {FINAL['name']} | AUC = {np.mean(FINAL['scores']):.4f}")
        display(pd.DataFrame(rows).set_index("ứng viên").round(4))
        if ok:
            pick = max(ok, key=lambda k: np.mean(cands[k]["scores"]))
            FINAL = cands[pick]
            res_catn = res_cat_tuned
            print("Chọn:", pick)
        else:
            print("Không hơn FINAL theo quy tắc delta_se: giữ", FINAL["name"])
        print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")
else:
    print("Bỏ qua mục 6h.")

Dùng lại tham số CatBoost có sẵn: {'depth': 5, 'l2_leaf_reg': 10, 'rsm': 0.5}
[CatBoost tune, 5 seed] AUC = 0.7672 ± 0.0023 | cây TB = 1836 | 6834s
REPLAY 6h: CatBoost tune (depth 5, rsm 0.5, 3 seed) đã sẵn sàng; việc trộn làm ở ô 6i.


## 6i. Bám theo hướng đã thắng: kỳ hạn vay, miền tri thức, và ext_source theo ngữ cảnh kỳ hạn

**Điều kết quả thật cho thấy.** Các feature ext_source dẫn xuất (`extadv`, `extml`, `extpow`, kẹp theo nhóm) đều kém hơn hoặc như mốc; chỉ phần điền ML (`+ml`) thắng. Nguyên nhân dễ hiểu: cây bất biến với biến đổi đơn điệu, và `ext_mean_ml` đã là cột đơn mạnh nhất (AUC 0.7271), nên mọi tổ hợp chỉ lặp lại thông tin của nó. Cái thật sự thắng là **nhóm kỳ hạn vay**: `+term` và `te_term`, cùng `reg_alpha` và CatBoost `rsm 0.5`.

Vì vậy 6i không thêm tổ hợp ext thuần nữa, mà làm ba việc:

| Khối | Nội dung | Lý do thử |
|---|---|---|
| `termfam` | tuổi lúc trả hết (`age + term/12`), khoảng tới tuổi 60 và 65, `term` x ext, thu nhập khả dụng `income - k*annuity` (k = 1, 6, 12) và tỷ lệ, thu nhập quy đổi OECD | mở rộng đúng hướng đã thắng, thêm tri thức miền |
| `te_term2` | mã hóa mục tiêu cho kỳ hạn mịn hơn (làm tròn 0.5), kỳ hạn x tuổi, x học vấn, x nghề, x loại thu nhập, x quy mô khoản vay | `te_term` thắng (+0.0009, xác nhận thêm), thử các biến thể |
| `ext_term_ctx` | ext_source so với bạn cùng kỳ hạn / cùng quy mô khoản vay (z-score, hạng phần trăm), và mã hóa **phần dư** theo ext cho các nguồn kỳ hạn | đây là cách "khuếch đại" ext duy nhất mang thông tin mới: đưa ngữ cảnh kỳ hạn vào chứ không biến đổi lại chính nó |
| `term_ctx2` | kỳ hạn với **thâm niên làm việc** (số năm làm khi trả xong, tỷ lệ kỳ hạn trên thâm niên, số năm còn lại tới tuổi 65 sau khi trả xong), **nhà ở, tình trạng gia đình, sở hữu xe**; mã hóa mục tiêu cho các ô kết hợp (kỳ hạn x thâm niên, x nhà ở, x gia đình, x xe, và các ô thô hơn 5 mức kỳ hạn để đỡ thưa) | các cột chưa kết hợp với kỳ hạn; ô kết hợp ít dòng nên mã hóa nhiễu hơn, kỳ vọng thấp |
| tham số | `colsample_bytree` 0.2 và 0.45, `reg_lambda` 20, `min_child_samples` 150 | CatBoost `rsm 0.5` thắng +0.0012, nên cũng thử giảm tỷ lệ cột cho LightGBM |

**Quy trình giống 6f:** sàng lọc từng khối ở lr 0.05, gộp tiến, **xác nhận trên bộ fold khác**, chạy lại 3 seed, so với LightGBM tốt nhất rồi trộn với CatBoost đã tune (w = 0.5 và 0.6) và so với mô hình đang dùng.

**Kỳ vọng thật:** gain nhỏ (vài phần mười nghìn AUC) và có khả năng không khối nào qua xác nhận. Trần thông tin của bảng đơn vẫn quanh 0.766 đến 0.768; hiện đã ở 0.7662.

In [15]:
RUN_6I = True
if RUN_6I:
    t0 = time.time()
    cfg_i = dict(cfg_f) if "cfg_f" in globals() else dict(base_cfg)
    for k in ("te_resid", "te_hier"):
        cfg_i.setdefault(k, [])
    cfg_i["params"] = dict(cfg_i.get("params", {}))
    F_ = FULL
    inc, cr, an = F_["income"], F_["credit_amount"], F_["annuity"]
    age, ye = F_["age_years"], F_["years_employed"]
    term = F_["credit_term"]
    e_m, e2, e3 = F_["ext_mean_ml"], F_["ext_source_2_ml"], F_["ext_source_3_ml"]
    fm, ch = F_["cnt_fam_members"], F_["cnt_children"]

    def kc(*arrs):
        s = pd.Series(np.asarray(arrs[0])).astype(str)
        for a in arrs[1:]:
            s = s + "_" + pd.Series(np.asarray(a)).astype(str)
        return pd.factorize(s)[0]

    qb = lambda s, q: pd.qcut(pd.Series(np.asarray(s, dtype=float)), q, labels=False, duplicates="drop").to_numpy()
    sd_ = lambda a, b: a / b.replace(0, np.nan)

    NUM_I = {}
    mat = age + term / 12.0
    oecd = 1.0 + 0.5 * (fm - ch - 1).clip(lower=0) + 0.3 * ch
    NUM_I["termfam"] = pd.DataFrame({
        "age_at_maturity": mat, "gap_to_60": 60 - mat, "gap_to_65": 65 - mat,
        "term_x_ext": term * e_m, "term_x_ext3": term * e3,
        "disp_k1": inc - an, "disp_k6": inc - 6 * an, "disp_k12": inc - 12 * an,
        "disp_k12_ratio": sd_(inc - 12 * an, inc), "disp_k6_ratio": sd_(inc - 6 * an, inc),
        "inc_oecd": inc / oecd, "disp_k12_oecd": (inc - 12 * an) / oecd,
        "credit_oecd": cr / oecd, "annuity_oecd": an / oecd,
        "mat_over_employed": sd_(mat, age), "term_over_years_left": sd_(term / 12.0, (65 - age)),
    })

    tb10, tb5 = qb(term, 10), qb(term, 5)
    crb, ageb = qb(cr, 10), qb(age, 10)
    z_ctx = lambda v, key, nmin=100: _zin(v, key, nmin)

    def _zin(v, key, min_n):
        d = pd.DataFrame({"v": np.asarray(v, dtype=float), "k": key})
        g = d.groupby("k")["v"]
        cnt, mu, s_ = g.transform("count"), g.transform("mean"), g.transform("std")
        ok = (cnt >= min_n) & (s_ > 0)
        return np.where(ok, (d["v"] - mu) / s_, (d["v"] - d["v"].mean()) / d["v"].std())

    ctx_keys = {"term": kc(tb10), "term_cr": kc(tb5, qb(cr, 5)), "cr": kc(crb)}
    pct = lambda v, key: pd.Series(np.asarray(v, dtype=float)).groupby(key).rank(pct=True).to_numpy()
    NUM_I["ext_term_ctx"] = pd.DataFrame({
        **{f"zc_{vn}_{kn}": _zin(v, k, 100) for vn, v in (("e", e_m), ("e2", e2), ("e3", e3)) for kn, k in ctx_keys.items()},
        **{f"pc_e_{kn}": pct(e_m, k) for kn, k in ctx_keys.items()},
    })

    t2 = (term * 2).round().clip(0, 240)
    tn = term.round().clip(0, 120)
    SRC_I = pd.DataFrame({
        "i_term_half": kc(t2), "i_term_age": kc(tn, ageb), "i_term_edu": kc(tn, F_["name_education_type"]),
        "i_term_occ": kc(tn, F_["c_occupation"]), "i_term_inc": kc(tn, F_["c_income_type"]),
        "i_term_cr": kc(tn, crb), "i_term_rating": kc(tn, F_["region_rating_client"]),
    }, index=F_.index)

    has_car = F_["own_car_age"].notna().astype(float) if "own_car_age" in F_.columns else pd.Series(0.0, index=F_.index)
    ye_f = ye.clip(lower=0)
    emp_mat = ye_f + term / 12.0
    NUM_I["term_ctx2"] = pd.DataFrame({
        "emp_at_maturity": emp_mat, "term_over_emp": sd_(term / 12.0, ye_f + 1.0),
        "emp_share_at_mat": sd_(emp_mat, mat), "emp_years_left": (65 - age) - ye_f - term / 12.0,
        "term_x_has_car": term * has_car, "has_car": has_car,
        "term_x_children": term * ch, "term_per_person": sd_(term, fm),
    })
    emp_b = np.where(np.isnan(qb(ye_f.where(ye.notna()), 5)), -1, qb(ye_f.where(ye.notna()), 5))
    SRC_I["i_term_emp"] = kc(tn, emp_b)
    SRC_I["i_term_house"] = kc(tn, F_["c_housing"])
    SRC_I["i_term_family"] = kc(tn, F_["c_family"])
    SRC_I["i_term_car"] = kc(tn, has_car)
    SRC_I["i_t10_house_fam"] = kc(tb10, F_["c_housing"], F_["c_family"])
    SRC_I["i_t5_emp_house_car"] = kc(tb5, emp_b, F_["c_housing"], has_car)
    SRC_I["i_t5_emp_age"] = kc(tb5, emp_b, ageb)
    TERM2_SRC = ["i_term_emp", "i_term_house", "i_term_family", "i_term_car", "i_t10_house_fam", "i_t5_emp_house_car", "i_t5_emp_age"]

    ALL6I = pd.concat([v for v in NUM_I.values()] + [SRC_I], axis=1).replace([np.inf, -np.inf], np.nan).astype("float32")
    ALL6I.index = F_.index
    X_all = pd.concat([X_all.drop(columns=[c for c in ALL6I.columns if c in X_all.columns]), ALL6I.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in ALL6I.columns if c in X_test_all.columns]),
                            ALL6I.iloc[n_tr:].reset_index(drop=True)], axis=1)

    TERM_SRC = [c for c in SRC_I.columns if c not in TERM2_SRC]
    BLK6I = {
        "termfam": dict(feats=list(NUM_I["termfam"].columns)),
        "te_term2": dict(te=TERM_SRC),
        "term_ctx2": dict(feats=list(NUM_I["term_ctx2"].columns), te=TERM2_SRC),
        "ext_term_ctx": dict(feats=list(NUM_I["ext_term_ctx"].columns),
                             te_resid=["c_term", "c_term_ctype", "c_term_inc"] + TERM_SRC[:4]),
        "p_col02": dict(params={"colsample_bytree": 0.2}),
        "p_col045": dict(params={"colsample_bytree": 0.45}),
        "p_lambda20": dict(params={"reg_lambda": 20}),
        "p_minchild150": dict(params={"min_child_samples": 150}),
    }
    SKIP_PARAM_TRIALS = True   # 4 biến thể tham số đã thử, không biến thể nào qua sàng lọc: bỏ (tiết kiệm ~5 phút)
    if SKIP_PARAM_TRIALS:
        BLK6I = {k: v for k, v in BLK6I.items() if "params" not in v}
    print(f"Dựng khối xong ({time.time() - t0:.0f}s): " + ", ".join(f"{k}={sum(len(x) for x in v.values() if isinstance(x, list))}" for k, v in BLK6I.items()
                                                                  if "params" not in v))
    chk = ALL6I.iloc[:n_tr]
    print("Cột hằng số:", [c for c in chk.columns if chk[c].std() == 0] or "không có", "| thiếu nhiều nhất (%):",
          (chk.isna().mean() * 100).sort_values(ascending=False).head(3).round(1).to_dict())

    def add_i(cfgx, spec):
        out = dict(cfgx)
        out["feats"] = cfgx["feats"] + [c for c in spec.get("feats", []) if c not in cfgx["feats"]]
        out["te"] = cfgx["te"] + [c for c in spec.get("te", []) if c not in cfgx["te"]]
        out["te_resid"] = cfgx.get("te_resid", []) + [c for c in spec.get("te_resid", []) if c not in cfgx.get("te_resid", [])]
        out["te_hier"] = cfgx.get("te_hier", [])
        out["params"] = {**cfgx.get("params", {}), **spec.get("params", {})}
        return out

    # ---------- tầng 1: sàng lọc ----------
    REPLAY_6I = ["ext_term_ctx"]    # khối đã qua sàng lọc và xác nhận ở lần chạy trước (delta +0.0013, 5/5 fold)
    if REPLAY:
        cfg_j = cfg_i
        for _nm in REPLAY_6I:
            cfg_j = add_i(cfg_j, BLK6I[_nm])
        kept_i = list(REPLAY_6I)
        _sig_l = f"lgb6i|{len(cfg_j['feats'])}|{len(cfg_j['te'])}|{len(FINAL_SEEDS)}|{DEBUG_FAST}"
        res_6i = _replay_cache_load("replay5_lgb6i_price.npz", _sig_l)
        if res_6i is None:
            res_6i = run_cfg3(cfg_j, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LGB(6i), cấu hình cố định, huấn luyện lại 1 lần")
            _replay_cache_save("replay5_lgb6i_price.npz", res_6i, _sig_l)
        else:
            print("REPLAY: nạp LGB(6i) đã huấn luyện từ replay5_lgb6i.npz (xóa file này để huấn luyện lại)")
        res_6i.update({"cfg": cfg_j, "base": PARAMS, "seeds": tuple(FINAL_SEEDS)})
        LGB_BEST = res_6i
        W_BLEND = 0.6
        _lg = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
        _ob = expit(W_BLEND * _lg(res_6i["oof"]) + (1 - W_BLEND) * _lg(res_cat_tuned["oof"]))
        _tp = expit(W_BLEND * _lg(res_6i["test_pred"]) + (1 - W_BLEND) * _lg(res_cat_tuned["test_pred"]))
        FINAL = {"oof": _ob, "test_pred": _tp, "scores": [roc_auc_score(y[va], _ob[va]) for _, va in SPLITS],
                 "name": f"Blend LGB(6i) {W_BLEND:.1f} + CatTune {1 - W_BLEND:.1f} (REPLAY, huấn luyện lại 1 lần)"}
        print(f"REPLAY: LGB(6i) {np.mean(res_6i['scores']):.4f} | CatTune {np.mean(res_cat_tuned['scores']):.4f} | BLEND {np.mean(FINAL['scores']):.4f} (mốc đã biết: 0.7670 đến 0.7671)")
        cfg_best = cfg_j
        print("REPLAY 6i: dùng thẳng khối", kept_i, "-> LGB_BEST là mô hình đã huấn luyện,", len(cfg_j["feats"]), "feature gốc + mã hóa nhãn trong fold")
    else:
        cur_i = run_cfg3(cfg_i, FAST_PARAMS, es=ES_FAST, name="mốc 6i (cấu hình tốt nhất trước đó, lr 0.05)")
        rows, single = [{"bước": "mốc", "AUC": np.mean(cur_i["scores"]), "giữ": "-"}], {}
        for nm, spec in BLK6I.items():
            r = run_cfg3(add_i(cfg_i, spec), FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
            c = compare(r, cur_i)
            single[nm] = c["delta"] if c["verdict"] == "tốt hơn" else None
            rows.append({"bước": f"riêng: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
        short = sorted([k for k, v in single.items() if v is not None], key=lambda k: -single[k])
        print("Qua sàng lọc:", short or "không có")

        cfg_j, cur_j, kept_i = cfg_i, cur_i, []
        for nm in short:
            cand = add_i(cfg_j, BLK6I[nm])
            r = run_cfg3(cand, FAST_PARAMS, es=ES_FAST, name=f"gộp +{nm}")
            c = compare(r, cur_j)
            keep = c["verdict"] == "tốt hơn"
            rows.append({"bước": f"gộp dần: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if keep else "không"})
            if keep:
                cfg_j, cur_j, kept_i = cand, r, kept_i + [nm]
        display(pd.DataFrame(rows).set_index("bước").round(4))
        print("Khối qua tầng 1:", kept_i or "không có")


    # ---------- tầng 2: xác nhận trên bộ fold khác ----------
    confirmed_i = False
    if kept_i and not REPLAY:
        SPLITS_MAIN = SPLITS
        SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 202).split(np.zeros(len(y)), y))
        try:
            a_ = run_cfg3(cfg_i, FAST_PARAMS, es=ES_FAST, name="XÁC NHẬN: mốc, fold mới")
            b_ = run_cfg3(cfg_j, FAST_PARAMS, es=ES_FAST, name="XÁC NHẬN: cấu hình mới, fold mới")
        finally:
            SPLITS = SPLITS_MAIN
        c = compare(b_, a_)
        confirmed_i = c["verdict"] == "tốt hơn"
        print(f"Xác nhận trên fold mới: delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}"
              f" => {'GIỮ' if confirmed_i else 'BỎ (không qua xác nhận, nhiều khả năng do may rủi)'}")

    # ---------- chạy lại cuối, so và trộn ----------
    if confirmed_i:
        ref = globals().get("LGB_BEST", res_lgb)
        res_6i = run_cfg3(cfg_j, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối 6i (cuối)")
        ll_ = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
        lg_ = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
        c = compare(res_6i, ref)
        print(f"So với LightGBM tốt nhất trước 6i ({ref['name']}): delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) "
              f"-> {c['verdict']} | log-loss {ll_(res_6i):.5f} vs {ll_(ref):.5f}")
        if c["verdict"] == "tốt hơn" and ll_(res_6i) <= ll_(ref) + 1e-5:
            LGB_BEST = res_6i
            cat_ref = globals().get("res_cat_tuned", globals().get("res_catn"))
            cands = {"LightGBM 6i đơn": res_6i}
            if cat_ref is not None:
                for w in (0.5, 0.6):
                    ob = expit(w * lg_(res_6i["oof"]) + (1 - w) * lg_(cat_ref["oof"]))
                    nm = f"Blend LGB(6i) {w:.1f} + CatTune {1 - w:.1f}"
                    cands[nm] = {"oof": ob, "test_pred": expit(w * lg_(res_6i["test_pred"]) + (1 - w) * lg_(cat_ref["test_pred"])),
                                 "scores": [roc_auc_score(y[va], ob[va]) for _, va in SPLITS], "name": nm}
            ok = []
            for nm, r in cands.items():
                c2 = compare(r, FINAL)
                print(f"[{nm}] AUC = {np.mean(r['scores']):.4f} | so với đang dùng ({FINAL['name']}, {np.mean(FINAL['scores']):.4f}): "
                      f"delta = {c2['delta']:+.4f} (se {c2['delta_se']:.4f}, fold dương {c2['fold_duong']}) -> {c2['verdict']}")
                if c2["verdict"] == "tốt hơn" and ll_(r) <= ll_(FINAL) + 1e-5:
                    ok.append(nm)
            if ok:
                FINAL = cands[max(ok, key=lambda k: np.mean(cands[k]["scores"]))]
    else:
        print("REPLAY 6i: xong, cấu hình cố định đã áp dụng." if REPLAY else "Không khối nào của 6i qua cả hai tầng: giữ nguyên mô hình đang dùng.")
    print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")
else:
    print("Bỏ qua mục 6i (RUN_6I = False).")

Dựng khối xong (7s): termfam=16, te_term2=7, term_ctx2=15, ext_term_ctx=19
Cột hằng số: không có | thiếu nhiều nhất (%): {'emp_years_left': 18.0, 'emp_share_at_mat': 18.0, 'term_over_emp': 18.0}
[LGB(6i), cấu hình cố định, huấn luyện lại 1 lần] AUC = 0.7670 ± 0.0027 | log-loss = 0.24369 | 108 feature | cây TB = 1384 | 959s
REPLAY: LGB(6i) 0.7670 | CatTune 0.7672 | BLEND 0.7682 (mốc đã biết: 0.7670 đến 0.7671)
REPLAY 6i: dùng thẳng khối ['ext_term_ctx'] -> LGB_BEST là mô hình đã huấn luyện, 96 feature gốc + mã hóa nhãn trong fold
REPLAY 6i: xong, cấu hình cố định đã áp dụng.

>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6j. Mở rộng hướng "ext_source theo ngữ cảnh kỳ hạn"

**Điều 6i đã cho thấy (kết quả thật).** Khối `ext_term_ctx` là khối duy nhất thắng: sàng lọc +0.0003 (4/5 fold), xác nhận trên fold mới +0.0013 (5/5), LightGBM cuối +0.0006 so với 6f, trộn với CatBoost đã tune đạt 0.7670 (hơn 0.7662 là +0.0007, 5/5 fold). Các khối kỳ hạn thuần (`termfam`, `term_ctx2`) kém hơn mốc (−0.0010, 0/5 fold), `te_term2` chỉ ≈ như. Nghĩa là **thêm cột kỳ hạn thô không giúp; thứ giúp là đặt ext_source vào ngữ cảnh của nhóm cùng kỳ hạn**, và mã hóa phần dư (phần mà kỳ hạn giải thích thêm ngoài ext).

**6j làm bốn việc**
1. **Tách phần nào của `ext_term_ctx` thật sự có ích** (chỉ phần số hay chỉ phần mã hóa phần dư); chạy một lần để biết nên mở rộng bên nào.
2. `ctx_more`: z-score và hạng phần trăm của ext trong các nhóm **mịn hơn** (kỳ hạn x tỷ lệ trả góp trên thu nhập, kỳ hạn x tuổi, kỳ hạn x quy mô khoản vay x tỷ lệ trả góp, kỳ hạn x hạng vùng x loại hợp đồng), cùng z-score của vài biến tài chính trong nhóm kỳ hạn.
3. `rte_more`: mã hóa phần dư cho các nguồn kỳ hạn còn lại (kỳ hạn x thâm niên, nhà ở, gia đình, xe, thu nhập, quy mô khoản vay, hạng vùng).
4. `rte_e23`: phần dư so với **ext_source_2 và ext_source_3 riêng lẻ** (không chỉ trung bình), vì mỗi điểm có thể giải thích phần khác nhau của kỳ hạn.

**Kiểm soát chọn nhầm.** Mọi thứ ở đây xây trên cấu hình đã qua xác nhận ở 6i, nên 6j xác nhận trên **bộ fold thứ ba** (seed khác với 6f và 6i). Chỉ có 3 khối mới, để giảm số phép thử.

**Kỳ vọng thật:** mỗi bước thêm là vài phần mười nghìn AUC, và có thể không khối nào qua. Hướng này không phá được trần thông tin của bảng đơn (khoảng 0.766 đến 0.768 cho LightGBM đơn), mà chỉ giúp tới gần trần hơn.

In [16]:
RUN_6J = False   # đã chạy, không khối nào qua: bỏ để tiết kiệm thời gian
if RUN_6J and "ALL6I" in globals():
    t0 = time.time()
    cfg_i0 = dict(cfg_i)
    base_j = dict(cfg_j) if ("cfg_j" in globals() and "confirmed_i" in globals() and confirmed_i) else dict(cfg_i)
    for d_ in (cfg_i0, base_j):
        for k in ("te_resid", "te_hier", "te_resid2"):
            d_.setdefault(k, [])
        d_["params"] = dict(d_.get("params", {}))

    # --- gói fold_data3 để hỗ trợ phần dư theo từng ext riêng (te_resid2), chỉ gói một lần ---
    if "_fd3_orig" not in globals():
        _fd3_orig = fold_data3

    def fold_data3(cfg, tr, va):
        Xtr, Xva, Xte, f = _fd3_orig(cfg, tr, va)
        spec2 = [(e, [c for c in cols if c in X_all.columns]) for e, cols in cfg.get("te_resid2", []) if e in X_all.columns]
        if spec2:
            A, Bv, T = X_all.iloc[tr].reset_index(drop=True), X_all.iloc[va].reset_index(drop=True), X_test_all.reset_index(drop=True)
            for e, cols in spec2:
                tag = e.replace("ext_source_", "e").replace("_ml", "")
                t_tr, (t_va, t_te) = te_resid_enc(A, y[tr], [Bv, T], cols, ext_col=e)
                ren = {c: c.replace("_rte", f"_rte_{tag}") for c in t_tr.columns}
                t_tr, t_va, t_te = t_tr.rename(columns=ren), t_va.rename(columns=ren), t_te.rename(columns=ren)
                Xtr, Xva, Xte = (pd.concat([p, q_], axis=1) for p, q_ in ((Xtr, t_tr), (Xva, t_va), (Xte, t_te)))
                f = f + list(t_tr.columns)
        return Xtr, Xva, Xte, f

    # ---------- khối số mới: ngữ cảnh mịn hơn ----------
    ai = sd_(an, inc)
    ci = sd_(cr, inc)
    aib5, crb5, ageb5 = qb(ai, 5), qb(cr, 5), qb(age, 5)
    rr, ct = F_["region_rating_client"].to_numpy(), F_["name_contract_type"].to_numpy()
    keys_j = {
        "term_ai": kc(tb5, aib5), "term_age": kc(tb5, ageb5),
        "term_cr_ai": kc(tb5, crb5, aib5), "term_rr_ct": kc(tb10, rr, ct),
    }
    NUM_J = pd.DataFrame({
        **{f"zj_{vn}_{kn}": _zin(v, k, 100) for vn, v in (("e", e_m), ("e2", e2), ("e3", e3)) for kn, k in keys_j.items()},
        **{f"pj_e_{kn}": pct(e_m, k) for kn, k in keys_j.items()},
        **{f"zj_{vn}_term": _zin(v, ctx_keys["term"], 100) for vn, v in (("ai", ai), ("ci", ci), ("age", age), ("inc", np.log1p(inc)))},
    }, index=F_.index).replace([np.inf, -np.inf], np.nan).astype("float32")
    X_all = pd.concat([X_all.drop(columns=[c for c in NUM_J.columns if c in X_all.columns]), NUM_J.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in NUM_J.columns if c in X_test_all.columns]),
                            NUM_J.iloc[n_tr:].reset_index(drop=True)], axis=1)

    cols_resid_done = ["c_term", "c_term_ctype", "c_term_inc"] + TERM_SRC[:4]
    more_src = [c for c in (TERM_SRC[4:] + TERM2_SRC) if c in X_all.columns]
    BLK6J = {
        "ctx_more": dict(feats=list(NUM_J.columns)),
        "rte_more": dict(te_resid=more_src),
        "rte_e23": dict(te_resid2=[("ext_source_2_ml", cols_resid_done[:4]), ("ext_source_3_ml", cols_resid_done[:4])]),
    }
    print(f"Dựng khối xong ({time.time() - t0:.0f}s): ctx_more={NUM_J.shape[1]} cột, rte_more={len(more_src)} nguồn, rte_e23=2 x 4 nguồn")

    def add_j(cfgx, spec):
        out = add_i(cfgx, spec)
        out["te_resid2"] = list(cfgx.get("te_resid2", [])) + [x for x in spec.get("te_resid2", []) if x not in cfgx.get("te_resid2", [])]
        return out

    # ---------- 1) tách phần có ích của ext_term_ctx (chẩn đoán) ----------
    ctx_cols = list(NUM_I["ext_term_ctx"].columns)
    resid_cols = BLK6I["ext_term_ctx"]["te_resid"]
    ref_j = run_cfg3(cfg_i0, FAST_PARAMS, es=ES_FAST, name="mốc: trước ext_term_ctx")
    d_rows = []
    for nm, spec in (("chỉ phần số (z, hạng)", dict(feats=ctx_cols)), ("chỉ phần dư (mã hóa)", dict(te_resid=resid_cols)),
                     ("cả hai (cấu hình 6i)", BLK6I["ext_term_ctx"])):
        r = run_cfg3(add_j(cfg_i0, spec), FAST_PARAMS, es=ES_FAST, name=f"ext_term_ctx: {nm}")
        c = compare(r, ref_j)
        d_rows.append({"phần": nm, "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}})
    print("Tách phần có ích (so với trước ext_term_ctx; chỉ để đọc, không quyết định gì):")
    display(pd.DataFrame(d_rows).set_index("phần").round(4))

    # ---------- 2) tầng 1: sàng lọc khối mới trên cấu hình tốt nhất hiện có ----------
    cur_k = run_cfg3(base_j, FAST_PARAMS, es=ES_FAST, name="mốc 6j (cấu hình tốt nhất trước đó, lr 0.05)")
    rows, single = [{"bước": "mốc", "AUC": np.mean(cur_k["scores"]), "giữ": "-"}], {}
    for nm, spec in BLK6J.items():
        r = run_cfg3(add_j(base_j, spec), FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
        c = compare(r, cur_k)
        single[nm] = c["delta"] if c["verdict"] == "tốt hơn" else None
        rows.append({"bước": f"riêng: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
    short = sorted([k for k, v in single.items() if v is not None], key=lambda k: -single[k])
    print("Qua sàng lọc:", short or "không có")

    cfg_k, cur_kk, kept_j = base_j, cur_k, []
    for nm in short:
        cand = add_j(cfg_k, BLK6J[nm])
        r = run_cfg3(cand, FAST_PARAMS, es=ES_FAST, name=f"gộp +{nm}")
        c = compare(r, cur_kk)
        keep = c["verdict"] == "tốt hơn"
        rows.append({"bước": f"gộp dần: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if keep else "không"})
        if keep:
            cfg_k, cur_kk, kept_j = cand, r, kept_j + [nm]
    display(pd.DataFrame(rows).set_index("bước").round(4))
    print("Khối qua tầng 1:", kept_j or "không có")

    # ---------- 3) tầng 2: xác nhận trên bộ fold thứ ba ----------
    confirmed_j = False
    if kept_j:
        SPLITS_MAIN = SPLITS
        SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 303).split(np.zeros(len(y)), y))
        try:
            a_ = run_cfg3(base_j, FAST_PARAMS, es=ES_FAST, name="XÁC NHẬN: mốc, fold thứ ba")
            b_ = run_cfg3(cfg_k, FAST_PARAMS, es=ES_FAST, name="XÁC NHẬN: cấu hình mới, fold thứ ba")
        finally:
            SPLITS = SPLITS_MAIN
        c = compare(b_, a_)
        confirmed_j = c["verdict"] == "tốt hơn"
        print(f"Xác nhận trên fold thứ ba: delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}"
              f" => {'GIỮ' if confirmed_j else 'BỎ (không qua xác nhận, nhiều khả năng do may rủi)'}")

    # ---------- 4) chạy lại cuối, so và trộn ----------
    if confirmed_j:
        ref = globals().get("LGB_BEST", res_lgb)
        res_6j = run_cfg3(cfg_k, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối 6j (cuối)")
        ll_ = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
        lg_ = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
        c = compare(res_6j, ref)
        print(f"So với LightGBM tốt nhất trước 6j ({ref['name']}): delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) "
              f"-> {c['verdict']} | log-loss {ll_(res_6j):.5f} vs {ll_(ref):.5f}")
        if c["verdict"] == "tốt hơn" and ll_(res_6j) <= ll_(ref) + 1e-5:
            LGB_BEST = res_6j
            cat_ref = globals().get("res_cat_tuned", globals().get("res_catn"))
            cands = {"LightGBM 6j đơn": res_6j}
            if cat_ref is not None:
                for w in (0.5, 0.6, 0.7):
                    ob = expit(w * lg_(res_6j["oof"]) + (1 - w) * lg_(cat_ref["oof"]))
                    nm = f"Blend LGB(6j) {w:.1f} + CatTune {1 - w:.1f}"
                    cands[nm] = {"oof": ob, "test_pred": expit(w * lg_(res_6j["test_pred"]) + (1 - w) * lg_(cat_ref["test_pred"])),
                                 "scores": [roc_auc_score(y[va], ob[va]) for _, va in SPLITS], "name": nm}
            ok = []
            for nm, r in cands.items():
                c2 = compare(r, FINAL)
                print(f"[{nm}] AUC = {np.mean(r['scores']):.4f} | so với đang dùng ({FINAL['name']}, {np.mean(FINAL['scores']):.4f}): "
                      f"delta = {c2['delta']:+.4f} (se {c2['delta_se']:.4f}, fold dương {c2['fold_duong']}) -> {c2['verdict']}")
                if c2["verdict"] == "tốt hơn" and ll_(r) <= ll_(FINAL) + 1e-5:
                    ok.append(nm)
            if ok:
                FINAL = cands[max(ok, key=lambda k: np.mean(cands[k]["scores"]))]
    else:
        print("Không khối nào của 6j qua cả hai tầng: giữ nguyên mô hình đang dùng.")
    print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")
else:
    print("Bỏ qua mục 6j (RUN_6J = False hoặc chưa chạy 6i).")

Bỏ qua mục 6j (RUN_6J = False hoặc chưa chạy 6i).


## 6k, 6l, 6m. Tune lại tham số, denoising autoencoder, nhãn giả

Ba hướng được chọn thêm, tất cả chạy trên **cấu hình feature tốt nhất đã qua xác nhận** (6j nếu qua, rồi 6i, rồi cấu hình trước đó) và cùng quy tắc kiểm soát chọn nhầm: so bằng `compare`, **xác nhận trên bộ fold mới**, rồi mới chạy lại 3 seed, so với LightGBM tốt nhất và trộn với CatBoost đã tune để so với mô hình đang dùng.

| Mục | Làm gì | Kỳ vọng thật |
|---|---|---|
| **6k** tune lại | Optuna cho LightGBM trên bộ feature mới (bộ tham số cũ được tune khi còn khoảng 70 cột; giờ khoảng 100). Dữ liệu từng fold được dựng một lần rồi dùng lại cho mọi lần thử, nên nhanh hơn nhiều so với gọi lại `run_cfg3`. Thêm một vòng nhỏ cho CatBoost (`rsm`, `l2`, độ sâu) | vài phần mười nghìn; **tune trên fold chính nên lạc quan**, vì thế bắt buộc xác nhận trên fold mới |
| **6l** DAE | Autoencoder khử nhiễu (swap noise 15%) học biểu diễn **không dùng nhãn** từ cả train và test (232k dòng), bằng `MLPRegressor` của sklearn (không cần PyTorch). Dùng theo ba cách: (a) MLP phân loại trên biểu diễn DAE, trộn vào mô hình cuối; (b) thành phần PCA của lớp giữa làm feature cho LightGBM; (c) sai số tái tạo làm điểm "bất thường" | không chắc có ích; MLP ở 6g chỉ đạt 0.7457 và trộn còn làm kém đi |
| **6m** nhãn giả | Trong **mỗi fold**: mô hình phụ chỉ huấn luyện trên dòng train của fold đó dự đoán test, rồi dòng test được thêm vào với nhãn mềm (xác suất) và trọng số nhỏ (`cross_entropy`). Fold kiểm tra không bao giờ góp vào nhãn giả | thường không có ích khi tín hiệu yếu; chạy để có số đo thật |

**Vì sao nhãn giả làm trong fold.** Nếu lấy dự đoán test từ mô hình đã thấy cả train, nhãn của fold kiểm tra đã rò vào nhãn giả và AUC CV sẽ lạc quan giả. Làm trong fold tốn gấp đôi thời gian nhưng đo đúng.

Cờ: `RUN_6K`, `RUN_CAT_RETUNE`, `RUN_6L`, `RUN_6M`. Thời gian: 6k mặc định dừng sau `TUNE_MINUTES` phút.

In [17]:
RUN_6K = False   # Optuna không qua xác nhận (delta +0.0002): bỏ
RUN_CAT_RETUNE = False   # CatBoost vòng 2 không thắng: bỏ
N_TRIALS = 40          # số lần thử Optuna tối đa
TUNE_MINUTES = 120     # dừng sớm nếu quá thời gian này

def _pick_best_cfg():
    if globals().get("confirmed_j") and "cfg_k" in globals():
        c = dict(cfg_k)
    elif globals().get("confirmed_i") and "cfg_j" in globals():
        c = dict(cfg_j)
    else:
        c = dict(cfg_i) if "cfg_i" in globals() else dict(base_cfg)
    for k in ("te_resid", "te_hier", "te_resid2"):
        c.setdefault(k, [])
    c["params"] = dict(c.get("params", {}))
    return c

_ll = lambda r: log_loss(y, np.clip(r["oof"], 1e-6, 1 - 1e-6))
_lg = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))

def try_blends(lgb_res, cat_res, label):
    """Thử trộn lgb_res với cat_res; chỉ đổi FINAL khi tốt hơn rõ rệt."""
    global FINAL
    cands = {f"{label} (LightGBM đơn)": lgb_res}
    if cat_res is not None:
        for w in (0.5, 0.6, 0.7):
            ob = expit(w * _lg(lgb_res["oof"]) + (1 - w) * _lg(cat_res["oof"]))
            nm = f"Blend {label} {w:.1f} + Cat {1 - w:.1f}"
            cands[nm] = {"oof": ob, "test_pred": expit(w * _lg(lgb_res["test_pred"]) + (1 - w) * _lg(cat_res["test_pred"])),
                         "scores": [roc_auc_score(y[va], ob[va]) for _, va in SPLITS], "name": nm}
    ok = []
    for nm, r in cands.items():
        c2 = compare(r, FINAL)
        print(f"[{nm}] AUC = {np.mean(r['scores']):.4f} | so với đang dùng ({FINAL['name']}, {np.mean(FINAL['scores']):.4f}): "
              f"delta = {c2['delta']:+.4f} (se {c2['delta_se']:.4f}, fold dương {c2['fold_duong']}) -> {c2['verdict']}")
        if c2["verdict"] == "tốt hơn" and _ll(r) <= _ll(FINAL) + 1e-5:
            ok.append(nm)
    if ok:
        FINAL = cands[max(ok, key=lambda k: np.mean(cands[k]["scores"]))]
        print("FINAL cập nhật:", FINAL["name"])
    return bool(ok)

def consider_lgb(res_new, label):
    """So res_new với LightGBM tốt nhất; nếu hơn thì cập nhật LGB_BEST và thử trộn."""
    global LGB_BEST
    ref = globals().get("LGB_BEST", res_lgb)
    c = compare(res_new, ref)
    print(f"So với LightGBM tốt nhất trước đó ({ref['name']}): delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) "
          f"-> {c['verdict']} | log-loss {_ll(res_new):.5f} vs {_ll(ref):.5f}")
    if c["verdict"] == "tốt hơn" and _ll(res_new) <= _ll(ref) + 1e-5:
        LGB_BEST = res_new
        try_blends(res_new, globals().get("res_cat_tuned", globals().get("res_catn")), label)
        return True
    return False

def fresh_confirm(cfg_a, cfg_b, seed_off, run_fn=None, tag=""):
    """Chạy cfg_a (mốc) và cfg_b (mới) trên bộ fold mới; trả về (tốt hơn?, so sánh)."""
    global SPLITS
    run_fn = run_fn or (lambda cfg, name: run_cfg3(cfg, FAST_PARAMS, es=ES_FAST, name=name))
    SPLITS_MAIN = SPLITS
    SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + seed_off).split(np.zeros(len(y)), y))
    try:
        a_ = run_fn(cfg_a, f"XÁC NHẬN {tag}: mốc, fold mới")
        b_ = run_fn(cfg_b, f"XÁC NHẬN {tag}: cấu hình mới, fold mới")
    finally:
        SPLITS = SPLITS_MAIN
    c = compare(b_, a_)
    ok = c["verdict"] == "tốt hơn"
    print(f"Xác nhận trên fold mới: delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}"
          f" => {'GIỮ' if ok else 'BỎ (không qua xác nhận, nhiều khả năng do may rủi)'}")
    return ok, c

if RUN_6K:
    try:
        import optuna
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=False)
        try:
            import optuna
        except ImportError:
            optuna = None
            print("Không cài được optuna, bỏ qua tune LightGBM.")

if RUN_6K and optuna is not None:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    t0 = time.time()
    cfg_best = _pick_best_cfg()
    base_prm = {**FAST_PARAMS, **cfg_best["params"]}
    CACHE = []
    for tr, va in SPLITS:
        Xtr, Xva, _, f = fold_data3(cfg_best, tr, va)
        CACHE.append((Xtr.astype("float32"), Xva.astype("float32"), tr, va))
    print(f"Dựng dữ liệu {N_FOLDS} fold xong ({time.time() - t0:.0f}s), {len(f)} feature")

    def objective(trial):
        prm = dict(
            num_leaves=trial.suggest_int("num_leaves", 8, 48),
            max_depth=trial.suggest_int("max_depth", 3, 9),
            min_child_samples=trial.suggest_int("min_child_samples", 50, 600, log=True),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.15, 0.6),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 0.01, 30.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 0.01, 60.0, log=True),
        )
        prm["num_leaves"] = min(prm["num_leaves"], 2 ** prm["max_depth"] - 1)
        sc = []
        for i, (Xtr, Xva, tr, va) in enumerate(CACHE):
            m = lgb.LGBMClassifier(**{**base_prm, **prm, "subsample_freq": 1, "random_state": SEED})
            m.fit(Xtr, y[tr], eval_set=[(Xva, y[va])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(ES_FAST, verbose=False), lgb.log_evaluation(0)])
            sc.append(roc_auc_score(y[va], m.predict_proba(Xva)[:, 1]))
            trial.report(float(np.mean(sc)), i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(sc))

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
    cur_p = {k: base_prm[k] for k in ("num_leaves", "max_depth", "min_child_samples", "colsample_bytree", "subsample", "reg_alpha", "reg_lambda")
             if k in base_prm}
    study.enqueue_trial(cur_p)   # lần thử đầu tiên là tham số hiện tại, để so công bằng
    study.optimize(objective, n_trials=3 if DEBUG_FAST else N_TRIALS, timeout=TUNE_MINUTES * 60)
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    tab = pd.DataFrame([{**t.params, "AUC": t.value} for t in done]).sort_values("AUC", ascending=False)
    print(f"Optuna xong ({time.time() - t0:.0f}s): {len(done)} lần thử hoàn tất, {len(study.trials) - len(done)} bị cắt")
    print(f"Tham số hiện tại (lần thử 0): AUC = {study.trials[0].value:.4f}" if study.trials[0].value is not None else "")
    display(tab.head(5).round(4))

    tuned = dict(study.best_params)
    tuned["num_leaves"] = min(tuned["num_leaves"], 2 ** tuned["max_depth"] - 1)
    cfg_t = dict(cfg_best, params={**cfg_best["params"], **tuned, "subsample_freq": 1})
    print("Tham số tốt nhất:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in tuned.items()})
    ok_t, _ = fresh_confirm(cfg_best, cfg_t, 404, tag="tune")
    if ok_t:
        res_6k = run_cfg3(cfg_t, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM tune lại (cuối)")
        if consider_lgb(res_6k, "LGB(6k)"):
            cfg_best = cfg_t
    else:
        print("Tune lại không qua xác nhận: giữ tham số cũ.")
    del CACHE

if RUN_6K and RUN_CAT_RETUNE and "run_cat" in globals() and "cur_res" in globals():
    cat_rows, best_cat = [], None
    for nm, over in (("rsm 0.3", dict(rsm=0.3)), ("l2 30", dict(l2_leaf_reg=30)), ("depth 4", dict(depth=4))):
        trial_o = {**cur_over, **over}
        r = run_cat(trial_o, 0.08, (SEED,), f"CatBoost vòng 2 + {nm}")
        c = compare(r, cur_res)
        cat_rows.append({"thử": nm, "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}})
        if c["verdict"] == "tốt hơn" and (best_cat is None or c["delta"] > best_cat[2]):
            best_cat = (trial_o, r, c["delta"])
    display(pd.DataFrame(cat_rows).set_index("thử").round(4))
    if best_cat is not None:
        res_cat_new = run_cat(best_cat[0], 0.04, CAT_SEEDS, "CatBoost vòng 2, nhiều seed")
        cc = compare(res_cat_new, res_cat_tuned)
        print(f"So với CatBoost đã tune ở 6h: delta = {cc['delta']:+.4f} (se {cc['delta_se']:.4f}, fold dương {cc['fold_duong']}) -> {cc['verdict']}")
        if cc["verdict"] == "tốt hơn":
            res_cat_tuned = res_cat_new
            res_catn = res_cat_new
            try_blends(globals().get("LGB_BEST", res_lgb), res_cat_tuned, "LGB + CatVòng2")
    else:
        print("CatBoost vòng 2 không thắng ở bước sàng lọc: giữ CatBoost cũ.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")


>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


In [18]:
RUN_6L = False   # DAE không thắng: bỏ
RUN_DAE_MLP = True      # (a) MLP trên biểu diễn DAE, thử trộn
RUN_DAE_FEATS = True    # (b, c) PCA lớp giữa và sai số tái tạo làm feature cho LightGBM
DAE_EPOCHS, DAE_HIDDEN, SWAP_P = 12, (192, 96, 192), 0.15

if RUN_6L:
    from sklearn.neural_network import MLPRegressor, MLPClassifier
    from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
    from sklearn.decomposition import PCA
    t0 = time.time()
    if "cfg_best" not in globals():
        cfg_best = _pick_best_cfg()
    rng = np.random.RandomState(SEED)

    CATLR = [c for c in ["name_contract_type", "code_gender", "name_income_type", "name_family_status", "name_housing_type",
                         "occupation_type", "organization_type", "region_rating_client", "name_education_type"] if c in X_all.columns]
    num_cols = [c for c in dict.fromkeys(cfg_best["feats"]) if c in X_all.columns and c in X_test_all.columns and c not in CATLR]
    pool = pd.concat([X_all[num_cols], X_test_all[num_cols]], ignore_index=True).replace([np.inf, -np.inf], np.nan).astype("float64")
    pool = pool.loc[:, pool.std() > 0]
    miss_cols = [c for c in pool.columns if pool[c].isna().mean() > 0.01][:20]
    Xq = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=200000, random_state=SEED).fit_transform(
        pool.fillna(pool.median()).to_numpy())
    catp = pd.concat([X_all[CATLR], X_test_all[CATLR]], ignore_index=True).fillna(-1).astype(int)
    DM = np.hstack([np.clip(Xq, -5, 5), pool[miss_cols].isna().to_numpy().astype(np.float32),
                    OneHotEncoder(handle_unknown="ignore", dtype=np.float32).fit_transform(catp).toarray()]).astype(np.float32)
    N_ALL = len(DM)
    print(f"Ma trận DAE: {DM.shape} ({time.time() - t0:.0f}s)")

    def swap_noise(M, p):
        mask = rng.random_sample(M.shape) < p
        r_, c_ = np.nonzero(mask)
        out = M.copy()
        out[r_, c_] = M[rng.randint(0, len(M), len(r_)), c_]
        return out

    Dtr = DM if not DEBUG_FAST else DM[rng.choice(N_ALL, 20000, replace=False)]
    dae = MLPRegressor(hidden_layer_sizes=DAE_HIDDEN, activation="relu", batch_size=512, learning_rate_init=1e-3, alpha=1e-5,
                       random_state=SEED)
    for ep in range(2 if DEBUG_FAST else DAE_EPOCHS):
        dae.partial_fit(swap_noise(Dtr, SWAP_P), Dtr)
        if ep % 3 == 0 or ep == DAE_EPOCHS - 1:
            print(f"  DAE epoch {ep + 1}: loss = {dae.loss_:.4f} ({time.time() - t0:.0f}s)")

    def dae_forward(M, bs=50000):
        outs = []
        for i in range(0, len(M), bs):
            h = M[i:i + bs]
            hs = []
            for W, b in zip(dae.coefs_[:-1], dae.intercepts_[:-1]):
                h = np.maximum(h @ W + b, 0.0)
                hs.append(h)
            rec = h @ dae.coefs_[-1] + dae.intercepts_[-1]
            outs.append((hs[1], hs[2], ((rec - M[i:i + bs]) ** 2).mean(axis=1)))
        return (np.vstack([o[0] for o in outs]), np.vstack([o[1] for o in outs]), np.concatenate([o[2] for o in outs]))

    H2, H3, RERR = dae_forward(DM)
    print(f"Biểu diễn DAE xong ({time.time() - t0:.0f}s): lớp giữa {H2.shape[1]} chiều")

    # ---------- (a) MLP trên biểu diễn DAE ----------
    if RUN_DAE_MLP:
        _sc = lambda H: (H - H.mean(axis=0)) / (H.std(axis=0) + 1e-3)
        MX = np.nan_to_num(np.hstack([DM, _sc(H2), _sc(H3)]), nan=0.0, posinf=5.0, neginf=-5.0).astype(np.float32)

        def fp_mlp_x(tr, va, max_epochs=40, patience=4):
            r_ = np.random.RandomState(SEED)
            a, b = next(StratifiedKFold(10, shuffle=True, random_state=SEED).split(tr, y[tr]))
            Xf, yf, Xi, yi = MX[tr[a]], y[tr[a]], MX[tr[b]], y[tr[b]]
            m = MLPClassifier(hidden_layer_sizes=(256, 64), alpha=1e-3, batch_size=512, learning_rate_init=1e-3, random_state=SEED)
            best, best_w, bad = -1.0, None, 0
            for ep in range(5 if DEBUG_FAST else max_epochs):
                perm = r_.permutation(len(Xf))
                m.partial_fit(Xf[perm], yf[perm], classes=[0, 1])
                s = roc_auc_score(yi, m.predict_proba(Xi)[:, 1])
                if s > best + 1e-5:
                    best, bad = s, 0
                    best_w = ([w.copy() for w in m.coefs_], [w.copy() for w in m.intercepts_])
                else:
                    bad += 1
                    if bad >= patience:
                        break
            if best_w is None:
                raise RuntimeError(f"MLP trên DAE không có epoch hợp lệ (AUC nội bộ = {s}); kiểm tra NaN/inf: "
                                   f"MX hữu hạn = {bool(np.isfinite(MX).all())}, y trong fold = {np.unique(yf)}")
            m.coefs_, m.intercepts_ = best_w
            return m.predict_proba(MX[va])[:, 1], m.predict_proba(MX[n_tr:])[:, 1]

        t1 = time.time()
        oof_d, te_d = np.zeros(n_tr), np.zeros(len(X_test_all))
        for tr, va in SPLITS:
            pv, pt = fp_mlp_x(tr, va)
            oof_d[va] = pv
            te_d += pt / N_FOLDS
        sc_d = [roc_auc_score(y[va], oof_d[va]) for _, va in SPLITS]
        res_dae = {"oof": oof_d, "test_pred": te_d, "scores": sc_d, "name": "MLP trên biểu diễn DAE"}
        print(f"[MLP trên DAE] AUC = {np.mean(sc_d):.4f} ± {np.std(sc_d):.4f} | log-loss = {_ll(res_dae):.5f} | {time.time() - t1:.0f}s "
              f"(MLP thường ở 6g: tham chiếu 0.7457)")
        print("Tương quan logit với mô hình đang dùng:", round(float(np.corrcoef(_lg(FINAL["oof"]), _lg(oof_d))[0, 1]), 3))
        cands = {}
        for w in (0.9, 0.85, 0.8):
            ob = expit(w * _lg(FINAL["oof"]) + (1 - w) * _lg(oof_d))
            nm = f"{FINAL['name']} {w:.2f} + DAE-MLP {1 - w:.2f}"
            cands[nm] = {"oof": ob, "test_pred": expit(w * _lg(FINAL["test_pred"]) + (1 - w) * _lg(te_d)),
                         "scores": [roc_auc_score(y[va], ob[va]) for _, va in SPLITS], "name": nm}
        ok = []
        for nm, r in cands.items():
            c2 = compare(r, FINAL)
            print(f"[{nm}] AUC = {np.mean(r['scores']):.4f} | delta = {c2['delta']:+.4f} (se {c2['delta_se']:.4f}, fold dương {c2['fold_duong']}) -> {c2['verdict']}")
            if c2["verdict"] == "tốt hơn" and _ll(r) <= _ll(FINAL) + 1e-5:
                ok.append(nm)
        if ok:
            FINAL = cands[max(ok, key=lambda k: np.mean(cands[k]["scores"]))]
            print("FINAL cập nhật:", FINAL["name"])
        else:
            print("Trộn thêm DAE-MLP không hơn theo quy tắc delta_se: giữ", FINAL["name"])
        del MX

    # ---------- (b, c) feature từ DAE cho LightGBM ----------
    if RUN_DAE_FEATS:
        pcs = PCA(n_components=12, random_state=SEED).fit_transform(H2)
        DF = pd.DataFrame(pcs, columns=[f"dae_pc{i + 1}" for i in range(12)])
        DF["dae_recon_err"] = RERR
        DF = DF.astype("float32")
        X_all = pd.concat([X_all.drop(columns=[c for c in DF.columns if c in X_all.columns]), DF.iloc[:n_tr].reset_index(drop=True)], axis=1)
        X_test_all = pd.concat([X_test_all.drop(columns=[c for c in DF.columns if c in X_test_all.columns]),
                                DF.iloc[n_tr:].reset_index(drop=True)], axis=1)
        cfg_dae = dict(cfg_best, feats=cfg_best["feats"] + [c for c in DF.columns if c not in cfg_best["feats"]])
        ref_d = run_cfg3(cfg_best, FAST_PARAMS, es=ES_FAST, name="mốc 6l")
        r_all = run_cfg3(cfg_dae, FAST_PARAMS, es=ES_FAST, name="+DAE (PCA + sai số tái tạo)")
        r_err = run_cfg3(dict(cfg_best, feats=cfg_best["feats"] + ["dae_recon_err"]), FAST_PARAMS, es=ES_FAST, name="+chỉ sai số tái tạo")
        rows = []
        for nm, r in (("PCA + sai số tái tạo", r_all), ("chỉ sai số tái tạo", r_err)):
            c = compare(r, ref_d)
            rows.append({"thử": nm, "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}})
        display(pd.DataFrame(rows).set_index("thử").round(4))
        pick = max((("PCA + sai số tái tạo", r_all, cfg_dae), ("chỉ sai số tái tạo", r_err, dict(cfg_best, feats=cfg_best["feats"] + ["dae_recon_err"]))),
                   key=lambda t: np.mean(t[1]["scores"]))
        if compare(pick[1], ref_d)["verdict"] == "tốt hơn":
            ok_d, _ = fresh_confirm(cfg_best, pick[2], 505, tag="DAE")
            if ok_d:
                res_dae_f = run_cfg3(pick[2], PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name=f"LightGBM + DAE ({pick[0]}, cuối)")
                if consider_lgb(res_dae_f, "LGB(6l)"):
                    cfg_best = pick[2]
        else:
            print("Feature DAE không thắng mốc ở bước sàng lọc: không dùng.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")


>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


In [19]:
RUN_6M = False   # nhãn giả không thắng: bỏ
PSEUDO_W = (0.3, 1.0)    # trọng số của dòng test có nhãn mềm so với dòng train (=1)

if RUN_6M:
    if "cfg_best" not in globals():
        cfg_best = _pick_best_cfg()

    def run_pseudo(cfg, base, w, seeds=(SEED,), es=100, name="", n1=500):
        """Trong mỗi fold: mô hình phụ chỉ học dòng train của fold, dự đoán test; test được thêm với nhãn mềm."""
        t1 = time.time()
        oof, te_pred, iters = np.zeros(n_tr), np.zeros(len(X_test_all)), []
        for tr, va in SPLITS:
            Xtr, Xva, Xte, f = fold_data3(cfg, tr, va)
            prm = {**base, **cfg["params"]}
            m1 = lgb.LGBMClassifier(**{**prm, "n_estimators": 60 if DEBUG_FAST else n1, "random_state": SEED})
            m1.fit(Xtr, y[tr])     # không dùng fold kiểm tra, không dừng sớm
            p_t = m1.predict_proba(Xte)[:, 1]
            Xa = pd.concat([Xtr, Xte], ignore_index=True)
            ya = np.r_[y[tr].astype(float), p_t]
            wa = np.r_[np.ones(len(tr)), np.full(len(Xte), w)]
            for s in seeds:
                m = lgb.LGBMRegressor(**{**prm, "objective": "cross_entropy", "random_state": s})
                m.fit(Xa, ya, sample_weight=wa, eval_set=[(Xva, y[va].astype(float))], eval_metric="auc",
                      callbacks=[lgb.early_stopping(es, verbose=False), lgb.log_evaluation(0)])
                oof[va] += m.predict(Xva) / len(seeds)
                te_pred += m.predict(Xte) / (len(seeds) * N_FOLDS)
                iters.append(m.best_iteration_)
        scores = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
        print(f"[{name}] AUC = {np.mean(scores):.4f} ± {np.std(scores):.4f} | log-loss = {_ll({'oof': oof}):.5f} "
              f"| cây TB = {np.mean(iters):.0f} | {time.time() - t1:.0f}s")
        return {"oof": oof, "test_pred": te_pred, "scores": scores, "name": name}

    ref_m = run_cfg3(cfg_best, FAST_PARAMS, es=ES_FAST, name="mốc 6m (không nhãn giả)")
    rows, wins = [], []
    for w in PSEUDO_W:
        r = run_pseudo(cfg_best, FAST_PARAMS, w, es=ES_FAST, name=f"nhãn giả mềm, trọng số {w}")
        c = compare(r, ref_m)
        rows.append({"thử": f"trọng số {w}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}})
        if c["verdict"] == "tốt hơn":
            wins.append((c["delta"], w))
    display(pd.DataFrame(rows).set_index("thử").round(4))
    if wins:
        w_best = max(wins)[1]
        ok_m, _ = fresh_confirm(cfg_best, cfg_best, 606, tag="nhãn giả",
                                run_fn=lambda cfg, name: (run_pseudo(cfg, FAST_PARAMS, w_best, es=ES_FAST, name=name)
                                                          if "mới" in name and "mốc" not in name
                                                          else run_cfg3(cfg, FAST_PARAMS, es=ES_FAST, name=name)))
        if ok_m:
            res_m = run_pseudo(cfg_best, PARAMS, w_best, seeds=FINAL_SEEDS, es=ES_FINAL, name=f"LightGBM + nhãn giả (w = {w_best}, cuối)")
            consider_lgb(res_m, "LGB(6m)")
    else:
        print("Nhãn giả không thắng mốc: không dùng (đúng như kỳ vọng khi tín hiệu yếu).")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")


>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6n. Phần chưa đào: hồ sơ trùng người / hộ gia đình, và cột ID

**Vì sao mục này.** Sau 6d đến 6m, kết quả đã rõ: thêm biến đổi của ext_source, kỳ hạn, mô hình khác, nhãn giả, DAE đều **không** cải thiện quá nhiễu của CV (hiện 0.7670, trần thông tin ước chừng 0.766 đến 0.768 cho LightGBM đơn). Phần chưa ai đụng tới là **cấu trúc của chính bảng**, không phải giá trị từng cột:

1. **Cùng một người nộp nhiều hồ sơ** (hoặc cùng hộ gia đình). Các cột `DAYS_BIRTH`, `DAYS_REGISTRATION`, `DAYS_ID_PUBLISH`, `DAYS_EMPLOYED` được ghi theo ngày nguyên và gần như không đổi theo người, nên khớp nhiều cột cùng lúc gần như chỉ xảy ra với cùng một người hoặc cùng một hộ. Nếu có, hai thứ mới xuất hiện: số hồ sơ của họ (không dùng nhãn) và **nhãn của hồ sơ kia** nếu nó nằm trong tập huấn luyện. Tất cả nằm trong bảng đã cho, không dùng dữ liệu ngoài.
2. **Cột `SK_ID_CURR`** (số hiệu hồ sơ): thường là nhiễu, nhưng vài bộ dữ liệu có thứ tự theo thời gian nên cần đo một lần cho chắc.

**Mục này làm gì**
- In **chẩn đoán trước khi dựng gì**: tỷ lệ dòng nằm trong nhóm trùng cho từng khóa, và tỷ lệ vỡ nợ khi hồ sơ kia vỡ nợ so với khi không; AUC đơn của ID cùng tỷ lệ vỡ nợ theo thập phân vị ID. Nếu hai chẩn đoán này phẳng thì dừng ở đó, đừng mong gì.
- Bốn khối gated: `grp_size` (kích thước nhóm, không dùng nhãn), `twin_person` (nhãn của hồ sơ trùng theo khóa người), `twin_all` (mọi khóa), `id_feat`.
- Nhãn của hồ sơ trùng chỉ lấy từ **tập huấn luyện của fold**, dòng huấn luyện **loại trừ chính nó** (không bao giờ thấy nhãn của mình); dòng kiểm tra và test chỉ thấy tập huấn luyện của fold. Cùng quy trình sàng lọc, xác nhận trên fold mới (seed khác), chạy lại 3 seed, rồi trộn.

**Kỳ vọng thật:** phụ thuộc hoàn toàn vào việc nhóm trùng có đủ nhiều hay không. Nếu chỉ vài phần trăm dòng có hồ sơ trùng thì tác động lên AUC rất nhỏ dù tín hiệu mạnh. Chẩn đoán ở đầu sẽ cho biết.

In [20]:
RUN_6N = False   # đã chạy, không khối nào qua: bỏ để tiết kiệm thời gian
if RUN_6N and "consider_lgb" in globals():
    t0 = time.time()
    cfg_best = _pick_best_cfg()
    cfg_best.setdefault("twin", [])
    F_ = FULL
    n_all = len(F_)
    n_te_ = n_all - n_tr

    # ---------- khóa nhóm (cùng người / cùng hộ), chỉ dùng cột có sẵn ----------
    def gid_from(cols):
        parts = [F_[c] for c in cols]
        valid = np.ones(n_all, bool)
        for p in parts:
            valid &= p.notna().to_numpy()
        s = pd.Series(np.asarray(parts[0]), index=range(n_all)).astype(str)
        for p in parts[1:]:
            s = s + "|" + pd.Series(np.asarray(p), index=range(n_all)).astype(str)
        g = pd.factorize(s)[0].astype(np.int64)
        g[~valid] = -1 - np.arange((~valid).sum())     # dòng thiếu thành nhóm riêng, không gộp nhầm
        return g

    KEYS = {
        "person": ["age_years", "years_registration", "years_id_publish"],
        "birth_gender_emp": ["age_years", "code_gender", "years_employed"],
        "reg_id_region": ["years_registration", "years_id_publish", "region_population_relative"],
        "birth_reg": ["age_years", "years_registration"],
    }
    KEYS = {k: [c for c in v if c in F_.columns] for k, v in KEYS.items()}
    KEYS = {k: v for k, v in KEYS.items() if len(v) >= 2}
    GID = {k: gid_from(v) for k, v in KEYS.items()}
    ytr_f = y.astype(float)

    # ---------- chẩn đoán 1: nhóm trùng có nhiều không, và nhãn hồ sơ kia có thông tin không ----------
    diag = []
    SIZE = {}
    for k, g in GID.items():
        sz = pd.Series(g).map(pd.Series(g).value_counts()).to_numpy()
        SIZE[k] = sz
        gt = g[:n_tr]
        S = pd.DataFrame({"g": gt, "y": ytr_f}).groupby("g")["y"].agg(["sum", "count"])
        n_i = pd.Series(gt).map(S["count"]).to_numpy() - 1
        s_i = pd.Series(gt).map(S["sum"]).to_numpy() - ytr_f
        has = n_i > 0
        pos = has & (s_i > 0)
        neg = has & (s_i == 0)
        diag.append({"khóa": k, "% dòng trong nhóm >= 2 (train+test)": 100 * float((sz >= 2).mean()),
                     "% dòng train có hồ sơ trùng trong train": 100 * float(has.mean()),
                     "P(vỡ nợ | hồ sơ kia vỡ nợ) %": 100 * float(ytr_f[pos].mean()) if pos.any() else np.nan,
                     "P(vỡ nợ | hồ sơ kia không vỡ) %": 100 * float(ytr_f[neg].mean()) if neg.any() else np.nan,
                     "số dòng có hồ sơ kia vỡ nợ": int(pos.sum())})
    print(f"Mức nền: P(vỡ nợ) = {100 * ytr_f.mean():.2f}%")
    display(pd.DataFrame(diag).set_index("khóa").round(2))

    # ---------- chẩn đoán 2: cột ID ----------
    id_col = next((c for c in train.columns if c.lower() == "sk_id_curr"), None)
    id_all = None
    if id_col is not None:
        id_all = np.r_[train[id_col].to_numpy(dtype=float), test[id_col].to_numpy(dtype=float)] if id_col in test.columns else None
    if id_all is not None:
        a_id = roc_auc_score(y, id_all[:n_tr])
        dec = pd.qcut(pd.Series(id_all[:n_tr]), 10, labels=False)
        rate = pd.Series(ytr_f).groupby(dec.to_numpy()).mean() * 100
        print(f"ID: AUC đơn = {a_id:.4f} (0.5 = không có tín hiệu) | tỷ lệ vỡ nợ theo thập phân vị ID (%): {rate.round(2).tolist()}")
    else:
        print("Không tìm thấy cột ID trong train/test: bỏ qua chẩn đoán ID.")

    # ---------- cột mới: kích thước nhóm (không dùng nhãn) và ID ----------
    NEWC = pd.DataFrame({f"grp_size_{k}": v.astype(np.float32) for k, v in SIZE.items()})
    if id_all is not None:
        NEWC["sk_id_num"] = id_all.astype(np.float32)
        NEWC["sk_id_rank"] = pd.Series(id_all).rank(pct=True).to_numpy().astype(np.float32)
    NEWC.index = F_.index
    X_all = pd.concat([X_all.drop(columns=[c for c in NEWC.columns if c in X_all.columns]), NEWC.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in NEWC.columns if c in X_test_all.columns]),
                            NEWC.iloc[n_tr:].reset_index(drop=True)], axis=1)

    # ---------- gói fold_data3: nhãn của hồ sơ trùng, chỉ từ tập huấn luyện của fold ----------
    if "_fd3_prev_n" not in globals():
        _fd3_prev_n = fold_data3

    def fold_data3(cfg, tr, va):
        Xtr, Xva, Xte, f = _fd3_prev_n(cfg, tr, va)
        keys = [k for k in cfg.get("twin", []) if k in GID]
        if keys:
            ytr_ = ytr_f[tr]
            cols_tr, cols_va, cols_te = {}, {}, {}
            for k in keys:
                g = GID[k]
                ser = pd.DataFrame({"g": g[tr], "y": ytr_}).groupby("g")["y"].agg(["sum", "count"])

                def look(garr, own=None):
                    s_ = pd.Series(garr).map(ser["sum"]).fillna(0).to_numpy()
                    n_ = pd.Series(garr).map(ser["count"]).fillna(0).to_numpy()
                    if own is not None:
                        s_, n_ = s_ - own, n_ - 1          # loại trừ chính dòng này
                    mean = np.where(n_ > 0, s_ / np.maximum(n_, 1), np.nan)
                    return n_, mean, (s_ > 0).astype(float)

                for dst, garr, own in ((cols_tr, g[tr], ytr_), (cols_va, g[va], None), (cols_te, g[n_tr:], None)):
                    n_, m_, p_ = look(garr, own)
                    dst[f"twin_n_{k}"], dst[f"twin_mean_{k}"], dst[f"twin_pos_{k}"] = n_, m_, p_
            Xtr = pd.concat([Xtr.reset_index(drop=True), pd.DataFrame(cols_tr)], axis=1)
            Xva = pd.concat([Xva.reset_index(drop=True), pd.DataFrame(cols_va)], axis=1)
            Xte = pd.concat([Xte.reset_index(drop=True), pd.DataFrame(cols_te)], axis=1)
            f = f + list(cols_tr.keys())
        return Xtr, Xva, Xte, f

    def add_n(cfgx, spec):
        out = dict(cfgx)
        out["feats"] = cfgx["feats"] + [c for c in spec.get("feats", []) if c not in cfgx["feats"]]
        out["twin"] = list(dict.fromkeys(list(cfgx.get("twin", [])) + list(spec.get("twin", []))))
        return out

    size_cols = [c for c in NEWC.columns if c.startswith("grp_size_")]
    BLK6N = {"grp_size": dict(feats=size_cols), "twin_person": dict(twin=["person"] if "person" in GID else list(GID)[:1]),
             "twin_all": dict(twin=list(GID))}
    if id_all is not None:
        BLK6N["id_feat"] = dict(feats=["sk_id_num", "sk_id_rank"])
    print(f"Dựng xong ({time.time() - t0:.0f}s): " + ", ".join(f"{k}" for k in BLK6N))

    # ---------- tầng 1 ----------
    cur_n = run_cfg3(cfg_best, FAST_PARAMS, es=ES_FAST, name="mốc 6n")
    rows, single = [{"bước": "mốc", "AUC": np.mean(cur_n["scores"]), "giữ": "-"}], {}
    for nm, spec in BLK6N.items():
        r = run_cfg3(add_n(cfg_best, spec), FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
        c = compare(r, cur_n)
        single[nm] = c["delta"] if c["verdict"] == "tốt hơn" else None
        rows.append({"bước": f"riêng: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
    short = sorted([k for k, v in single.items() if v is not None], key=lambda k: -single[k])
    print("Qua sàng lọc:", short or "không có")
    cfg_n, cur_nn, kept_n = cfg_best, cur_n, []
    for nm in short:
        cand = add_n(cfg_n, BLK6N[nm])
        r = run_cfg3(cand, FAST_PARAMS, es=ES_FAST, name=f"gộp +{nm}")
        c = compare(r, cur_nn)
        keep = c["verdict"] == "tốt hơn"
        rows.append({"bước": f"gộp dần: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if keep else "không"})
        if keep:
            cfg_n, cur_nn, kept_n = cand, r, kept_n + [nm]
    display(pd.DataFrame(rows).set_index("bước").round(4))
    print("Khối qua tầng 1:", kept_n or "không có")

    # ---------- tầng 2 và cuối ----------
    if kept_n:
        ok_n, _ = fresh_confirm(cfg_best, cfg_n, 707, tag="6n")
        if ok_n:
            res_6n = run_cfg3(cfg_n, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối 6n (cuối)")
            if consider_lgb(res_6n, "LGB(6n)"):
                cfg_best = cfg_n
    else:
        print("Không khối nào của 6n qua sàng lọc: giữ nguyên mô hình đang dùng.")
elif RUN_6N:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6n.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")


>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6o. Bureau theo cách đã làm với ext_source: tổ hợp, rồi ghép với ngữ cảnh

**Ý tưởng bạn đề xuất**: làm với bốn cột bureau điều tương tự đã làm với ext (trung bình, tích, mũ, rồi ghép với các nhóm). Có hai điểm khác cần nói trước:

- Bốn cột bureau là **số lần tra cứu trong các cửa sổ lồng nhau** (giờ ⊂ tháng ⊂ quý ⊂ năm), không phải ba điểm liên tục ngang hàng như ext. Vì thế trung bình đơn giản đã có sẵn (`total_bureau_req`), và cây bất biến với biến đổi đơn điệu của từng cột, nên "mũ" từng cột riêng không thêm gì. Chỉ có **tổ hợp nhiều cột** mới mang thông tin cây không tự dựng được.
- Với ext, các tổ hợp mũ và tích **không** thắng được cột đơn mạnh nhất (`ext_mean_ml`); chỉ phần điền ML và ngữ cảnh kỳ hạn thắng. Nên 6o theo đúng bài học đó: tổ hợp có ý nghĩa, rồi ngữ cảnh.

**Ba khối (gated, xác nhận trên fold mới như các mục trước)**
- `bur_comb` (tổ hợp): đếm theo **khoảng rời nhau** (tháng, 1 đến 3 tháng, 3 đến 12 tháng), hai điểm "gần đây có trọng số" (trọng số giảm theo thời gian), trung bình luỹ thừa p = 0.5 và 2, trung bình hình học, tốc độ tra cứu gần so với xa (`accel`, `burst`), cờ bất thường (`qrt < mon`, `year < qrt`, `hour > mon`), cờ thiếu bureau, và tích với ext (`x ext_mean`, `x ext_3`) và tuổi.
- `bur_ctx` (ngữ cảnh, như `ext_term_ctx`): z-score và hạng phần trăm của điểm bureau trong các nhóm cùng tuổi, loại thu nhập, nghề, kỳ hạn, **cùng thập phân vị ext**, hạng vùng; cộng hạng ext x hạng bureau.
- `bur_te` (mã hóa mục tiêu): mã mẫu hình bureau (bốn số bị chặn trên rồi ghép), mẫu hình x ext, mẫu hình x tuổi, thiếu-bureau x loại thu nhập hoặc nghề, điểm bureau x ext; cả mã hóa thường lẫn mã hóa phần dư.

**Kỳ vọng thật:** thấp. Bốn cột đếm tra cứu mang ít thông tin, và `bureau2` (tỷ lệ) đã −0.0005. Mục này chủ yếu trả lời một câu hỏi: khi ghép đúng cách (rời nhau, ngữ cảnh, mẫu hình), bureau có thêm gì không.

In [21]:
RUN_6O = False   # đã chạy, không khối nào qua: bỏ để tiết kiệm thời gian
if RUN_6O and "consider_lgb" in globals():
    t0 = time.time()
    cfg_o = dict(globals().get("cfg_best") or _pick_best_cfg())
    for k in ("te_resid", "te_hier", "te_resid2", "twin"):
        cfg_o.setdefault(k, [])
    cfg_o["params"] = dict(cfg_o.get("params", {}))
    F_ = FULL
    h, m_, q_, yr = (F_[f"amt_req_credit_bureau_{k}"] for k in ("hour", "mon", "qrt", "year"))
    e_m, e3 = F_["ext_mean_ml"], F_["ext_source_3_ml"]
    age = F_["age_years"]
    term = F_["credit_term"]

    def kc_(*arrs):
        s = pd.Series(np.asarray(arrs[0])).astype(str)
        for a in arrs[1:]:
            s = s + "_" + pd.Series(np.asarray(a)).astype(str)
        return pd.factorize(s)[0]

    qb_ = lambda s, q: pd.qcut(pd.Series(np.asarray(s, dtype=float)), q, labels=False, duplicates="drop").to_numpy()
    zin_ = lambda v, key, nmin=100: _zin_o(v, key, nmin)

    def _zin_o(v, key, min_n):
        d = pd.DataFrame({"v": np.asarray(v, dtype=float), "k": key})
        g = d.groupby("k")["v"]
        cnt, mu, s_ = g.transform("count"), g.transform("mean"), g.transform("std")
        ok = (cnt >= min_n) & (s_ > 0)
        return np.where(ok, (d["v"] - mu) / s_, (d["v"] - d["v"].mean()) / d["v"].std())

    pct_ = lambda v, key: pd.Series(np.asarray(v, dtype=float)).groupby(key).rank(pct=True).to_numpy()

    # ---------- khoảng rời nhau, điểm gần đây, trung bình luỹ thừa ----------
    hh, mm, qq, yy = (x.clip(lower=0) for x in (h, m_, q_, yr))
    d1, d2, d3 = (mm - hh).clip(lower=0), (qq - mm).clip(lower=0), (yy - qq).clip(lower=0)
    dec_a = 8 * hh + 4 * d1 + 2 * d2 + 1 * d3
    dec_b = 3 * hh + 2 * d1 + 1.5 * d2 + 1 * d3
    C = pd.concat([hh, mm, qq, yy], axis=1)
    pm = lambda p: (C.pow(p).mean(axis=1)) ** (1.0 / p)
    rate_r, rate_m, rate_o = mm / 1.0, d2 / 2.0, d3 / 9.0
    NUM_O = {}
    NUM_O["bur_comb"] = pd.DataFrame({
        "bur_d_mon_hour": d1, "bur_d_qrt_mon": d2, "bur_d_year_qrt": d3,
        "bur_decay_a": dec_a, "bur_decay_b": dec_b,
        "bur_pmean_0p5": pm(0.5), "bur_pmean_2": pm(2.0), "bur_geo": np.expm1(np.log1p(C).mean(axis=1)),
        "bur_accel": (rate_r + 0.1) / (rate_o + 0.1), "bur_burst": mm / (yy + 1.0), "bur_mid_share": d2 / (yy + 1.0),
        "bur_an_q_lt_m": (q_ < m_).astype(float).where(q_.notna()), "bur_an_y_lt_q": (yr < q_).astype(float).where(yr.notna()),
        "bur_an_h_gt_m": (h > m_).astype(float).where(h.notna()),
        "bur_na": yr.isna().astype(float),
        "bur_x_ext": np.log1p(dec_a) * e_m, "bur_x_ext3": np.log1p(dec_a) * e3, "bur_x_age": np.log1p(dec_a) * age,
        "bur_total_x_ext": np.log1p(F_["total_bureau_req"]) * e_m,
    })

    # ---------- ngữ cảnh: so với nhóm cùng tuổi, thu nhập, nghề, kỳ hạn, cùng thập phân vị ext ----------
    keys_o = {"age": kc_(qb_(age, 10)), "inc": kc_(F_["c_income_type"]), "occ": kc_(F_["c_occupation"]),
              "term": kc_(qb_(term, 5)), "ext": kc_(qb_(e_m, 10)), "rating": kc_(F_["region_rating_client"])}
    r_ext = pd.Series(np.asarray(e_m, dtype=float)).rank(pct=True).to_numpy()
    r_bur = pd.Series(np.asarray(dec_a, dtype=float)).rank(pct=True).to_numpy()
    NUM_O["bur_ctx"] = pd.DataFrame({
        **{f"bz_a_{kn}": zin_(dec_a, k) for kn, k in keys_o.items()},
        **{f"bp_a_{kn}": pct_(dec_a, k) for kn, k in keys_o.items()},
        "bz_tot_ext": zin_(F_["total_bureau_req"], keys_o["ext"]),
        "bur_rank_x_ext_rank": r_bur * r_ext, "bur_rank_minus_ext_rank": r_bur - r_ext,
    })

    # ---------- mẫu hình bureau và các nguồn mã hóa mục tiêu ----------
    cap = lambda s, k: s.clip(upper=k).fillna(-1)
    pat = kc_(cap(h, 2), cap(m_, 3), cap(q_, 3), cap(yr, 6))
    SRC_O = pd.DataFrame({
        "o_pat": pat, "o_pat_ext": kc_(pat, qb_(e_m, 10)), "o_pat_age": kc_(pat, qb_(age, 10)),
        "o_na_inc": kc_(yr.isna(), F_["c_income_type"]), "o_na_occ": kc_(yr.isna(), F_["c_occupation"]),
        "o_dec_ext": kc_(qb_(dec_a, 10), qb_(e_m, 10)),
    }, index=F_.index)

    ALL6O = pd.concat([v for v in NUM_O.values()] + [SRC_O], axis=1).replace([np.inf, -np.inf], np.nan).astype("float32")
    ALL6O.index = F_.index
    X_all = pd.concat([X_all.drop(columns=[c for c in ALL6O.columns if c in X_all.columns]), ALL6O.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in ALL6O.columns if c in X_test_all.columns]),
                            ALL6O.iloc[n_tr:].reset_index(drop=True)], axis=1)

    BLK6O = {
        "bur_comb": dict(feats=list(NUM_O["bur_comb"].columns)),
        "bur_ctx": dict(feats=list(NUM_O["bur_ctx"].columns)),
        "bur_te": dict(te=list(SRC_O.columns), te_resid=["o_pat", "o_pat_age", "o_na_inc"]),
    }
    chk = ALL6O.iloc[:n_tr]
    print(f"Dựng xong ({time.time() - t0:.0f}s): " + ", ".join(f"{k}={sum(len(x) for x in v.values())}" for k, v in BLK6O.items()))
    print("Cột hằng số:", [c for c in chk.columns if chk[c].std() == 0] or "không có", "| số mẫu hình bureau:", len(np.unique(pat)),
          "| % thiếu bureau:", round(100 * float(yr.isna().mean()), 1))

    def add_o(cfgx, spec):
        out = dict(cfgx)
        out["feats"] = cfgx["feats"] + [c for c in spec.get("feats", []) if c not in cfgx["feats"]]
        out["te"] = cfgx["te"] + [c for c in spec.get("te", []) if c not in cfgx["te"]]
        out["te_resid"] = list(cfgx.get("te_resid", [])) + [c for c in spec.get("te_resid", []) if c not in cfgx.get("te_resid", [])]
        return out

    cur_o = run_cfg3(cfg_o, FAST_PARAMS, es=ES_FAST, name="mốc 6o")
    rows, single = [{"bước": "mốc", "AUC": np.mean(cur_o["scores"]), "giữ": "-"}], {}
    for nm, spec in BLK6O.items():
        r = run_cfg3(add_o(cfg_o, spec), FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
        c = compare(r, cur_o)
        single[nm] = c["delta"] if c["verdict"] == "tốt hơn" else None
        rows.append({"bước": f"riêng: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
    short = sorted([k for k, v in single.items() if v is not None], key=lambda k: -single[k])
    print("Qua sàng lọc:", short or "không có")
    cfg_p, cur_p, kept_o = cfg_o, cur_o, []
    for nm in short:
        cand = add_o(cfg_p, BLK6O[nm])
        r = run_cfg3(cand, FAST_PARAMS, es=ES_FAST, name=f"gộp +{nm}")
        c = compare(r, cur_p)
        keep = c["verdict"] == "tốt hơn"
        rows.append({"bước": f"gộp dần: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if keep else "không"})
        if keep:
            cfg_p, cur_p, kept_o = cand, r, kept_o + [nm]
    display(pd.DataFrame(rows).set_index("bước").round(4))
    print("Khối qua tầng 1:", kept_o or "không có")

    if kept_o:
        ok_o, _ = fresh_confirm(cfg_o, cfg_p, 808, tag="6o")
        if ok_o:
            res_6o = run_cfg3(cfg_p, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối 6o (cuối)")
            if consider_lgb(res_6o, "LGB(6o)"):
                cfg_best = cfg_p
    else:
        print("Không khối nào của 6o qua sàng lọc: bureau không thêm gì dù ghép đúng cách. Giữ nguyên mô hình đang dùng.")
elif RUN_6O:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6o.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")


>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6r. Xử lý missing của ext_source: đo chất lượng imputer, thử imputer giàu thông tin hơn

**Hiện trạng.** `ext_source_{1,2,3}_ml` điền chỗ thiếu bằng LightGBM (`ext_source_1` thiếu nhiều nhất), đã cho `+0.0016`. Chưa ai đo imputer đó đúng đến đâu.

**Làm gì.**
1. Che 10% giá trị đã quan sát của từng cột, điền lại, đo R² và tương quan. Đây là cận trên lạc quan, vì dòng thật sự thiếu thường thiếu cả cột khác (không thiếu ngẫu nhiên).
2. Imputer v2: thêm đầu vào (kỳ hạn vay, mã nghề/thu nhập/nhà ở/gia đình/học vấn, cờ "cột ext nào đang thiếu", số cột ext khác thiếu), mô hình lớn hơn, trung bình 2 seed.
3. **Cổng R²**: chỉ chạy các bước tốn thời gian nếu v2 tăng R² ít nhất `R2_MIN_GAIN` ở một cột. Nếu v2 không điền tốt hơn thì không có gì để mong ở AUC.
4. Nếu qua cổng: thêm cột `*_ml2` (và bản ngữ cảnh kỳ hạn tính từ chúng), qua `compare` tầng 1, xác nhận trên fold mới (seed +909), chạy 3 seed rồi `consider_lgb`.

**Cố ý không làm "hai lượt" kiểu MICE** (điền cột A bằng giá trị đã điền của cột B). Giá trị đã điền của B ở dòng huấn luyện của A đã dùng chính A để dự đoán, nên imputer sẽ học bằng thông tin mà lúc thật (A thiếu) không có: rò rỉ nhẹ trong chính imputer. v2 dùng giá trị gốc (có NaN) của các cột kia cộng cờ thiếu, như v1.

**Kỳ vọng thật.** Vài `1e-4` nếu có. Cây vốn xử lý NaN được, và `ext_source_1_ml` đã lấy phần lớn thông tin từ các cột khác.

In [22]:
RUN_6R = False   # đã chạy, không khối nào qua: bỏ để tiết kiệm thời gian
R2_MIN_GAIN = 0.005   # v2 phải tăng R² ít nhất mức này ở ≥1 cột thì mới chạy tầng 1 (mỗi lần chạy ~1-2 phút/khối)
FORCE_6R = False      # True: bỏ cổng R², luôn chạy tầng 1
if RUN_6R and "consider_lgb" in globals():
    t0 = time.time()
    cfg_r = dict(globals().get("cfg_best") or _pick_best_cfg())
    for k in ("te_resid", "te_hier", "te_resid2", "twin"):
        cfg_r.setdefault(k, [])
    cfg_r["params"] = dict(cfg_r.get("params", {}))
    F_ = FULL

    # ---------- đầu vào cho imputer ----------
    extra_in = [c for c in ("credit_term", "c_occupation", "c_income_type", "c_housing", "c_family", "name_education_type",
                            "region_rating_client", "own_car_age", "total_bureau_req") if c in F_.columns]
    base_v2 = list(dict.fromkeys(list(BASE_PRED) + extra_in))
    na_ = {c: F_[c].isna().to_numpy().astype("float32") for c in EXT}

    def X_for(c, v2):
        others = [o for o in EXT if o != c]
        cols = list(dict.fromkeys((base_v2 if v2 else list(BASE_PRED)) + others))
        Xc = F_[cols].copy()
        if v2:
            for o in others:
                Xc[f"na_{o}"] = na_[o]
            Xc["na_others_n"] = sum(na_[o] for o in others)   # chỉ đếm cột KHÁC c: không để lộ c thiếu hay không
        return Xc

    n_e1, n_e2 = (60, 80) if DEBUG_FAST else (300, 600)
    P_V1 = dict(objective="regression", n_estimators=n_e1, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.8)
    P_V2 = dict(objective="regression", n_estimators=n_e2, learning_rate=0.03, num_leaves=63, min_child_samples=50,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=5.0)

    def fit_pred(Xc, yv, tr_idx, pr_idx, prm, seed):
        m = lgb.LGBMRegressor(**prm, random_state=seed, n_jobs=-1, verbosity=-1)
        m.fit(Xc.iloc[tr_idx], yv[tr_idx])
        return np.clip(m.predict(Xc.iloc[pr_idx]), 0, 1)

    # ---------- (1) đo chất lượng imputer trên giá trị đã che ----------
    rng_d = np.random.RandomState(SEED + 909)
    drows, gains = [], []
    for c in EXT:
        yv = F_[c].to_numpy(dtype=float)
        obs = np.flatnonzero(~np.isnan(yv))
        hold = rng_d.choice(obs, size=max(1, int(0.1 * len(obs))), replace=False)
        tr_ = np.setdiff1d(obs, hold)
        res_ = {}
        for tag, v2, prm in (("v1", False, P_V1), ("v2", True, P_V2)):
            p = fit_pred(X_for(c, v2), yv, tr_, hold, prm, SEED)
            ss = float(((yv[hold] - p) ** 2).sum()) / float(((yv[hold] - yv[hold].mean()) ** 2).sum())
            res_[tag] = (1 - ss, float(np.corrcoef(yv[hold], p)[0, 1]))
        gains.append(res_["v2"][0] - res_["v1"][0])
        drows.append({"cột": c, "% thiếu": 100 * float(np.isnan(yv).mean()), "R² v1 (hiện tại)": res_["v1"][0], "R² v2": res_["v2"][0],
                      "tăng R²": gains[-1], "corr v1": res_["v1"][1], "corr v2": res_["v2"][1]})
    display(pd.DataFrame(drows).set_index("cột").round(4))
    print(f"(đo trên {len(EXT)} cột, che 10% giá trị đã có; thật sự thiếu thì khó điền hơn con số này) | {time.time() - t0:.0f}s")

    go = FORCE_6R or max(gains) >= R2_MIN_GAIN
    if not go:
        print(f"v2 chỉ tăng R² tối đa {max(gains):+.4f} < {R2_MIN_GAIN}: imputer hiện tại đã gần đủ, bỏ qua tầng 1. "
              "Muốn thử vẫn chạy thì đặt FORCE_6R = True.")
    else:
        # ---------- (2) imputer v2 trên toàn bộ dòng quan sát (trung bình 2 seed) ----------
        out = {}
        for c in EXT:
            yv = F_[c].to_numpy(dtype=float)
            obs, mis = np.flatnonzero(~np.isnan(yv)), np.flatnonzero(np.isnan(yv))
            vals = yv.copy()
            if len(mis):
                Xc = X_for(c, True)
                vals[mis] = np.mean([fit_pred(Xc, yv, obs, mis, P_V2, SEED + s_) for s_ in (0, 1)], axis=0)
            out[f"{c}_ml2"] = vals
        out["ext_mean_ml2"] = np.mean([out[f"{c}_ml2"] for c in EXT], axis=0)
        NUM_R = {"ml2": pd.DataFrame(out, index=F_.index)}
        e_m2, e2_2, e3_2 = (NUM_R["ml2"][k] for k in ("ext_mean_ml2", "ext_source_2_ml2", "ext_source_3_ml2"))
        corr_old = {c: float(np.corrcoef(F_[f"{c}_ml"], out[f"{c}_ml2"])[0, 1]) for c in EXT}
        print("Tương quan ml (cũ) với ml2 (mới):", {k: round(v, 4) for k, v in corr_old.items()})

        if all(k in globals() for k in ("ctx_keys", "_zin", "pct")):
            NUM_R["ml2_ctx"] = pd.DataFrame({
                **{f"zc2_{vn}_{kn}": _zin(v, k, 100) for vn, v in (("e", e_m2), ("e2", e2_2), ("e3", e3_2)) for kn, k in ctx_keys.items()},
                **{f"pc2_e_{kn}": pct(e_m2, k) for kn, k in ctx_keys.items()},
            })
        ALL6R = pd.concat(list(NUM_R.values()), axis=1).replace([np.inf, -np.inf], np.nan).astype("float32")
        ALL6R.index = F_.index
        X_all = pd.concat([X_all.drop(columns=[c for c in ALL6R.columns if c in X_all.columns]), ALL6R.iloc[:n_tr].reset_index(drop=True)], axis=1)
        X_test_all = pd.concat([X_test_all.drop(columns=[c for c in ALL6R.columns if c in X_test_all.columns]),
                                ALL6R.iloc[n_tr:].reset_index(drop=True)], axis=1)
        BLK6R = {"ml2": dict(feats=list(NUM_R["ml2"].columns))}
        if "ml2_ctx" in NUM_R:
            BLK6R["ml2_ctx"] = dict(feats=list(NUM_R["ml2"].columns) + list(NUM_R["ml2_ctx"].columns))
        print(f"Dựng xong ({time.time() - t0:.0f}s): " + ", ".join(f"{k}={len(v['feats'])} cột" for k, v in BLK6R.items()))

        def add_r(cfgx, spec):
            o_ = dict(cfgx)
            o_["feats"] = list(cfgx["feats"]) + [c for c in spec.get("feats", []) if c not in cfgx["feats"]]
            return o_

        # ---------- (3) tầng 1: sàng lọc ----------
        cur_r = run_cfg3(cfg_r, FAST_PARAMS, es=ES_FAST, name="mốc 6r")
        rows, single = [{"bước": "mốc", "AUC": np.mean(cur_r["scores"]), "giữ": "-"}], {}
        for nm, spec in BLK6R.items():
            r = run_cfg3(add_r(cfg_r, spec), FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
            c = compare(r, cur_r)
            single[nm] = c["delta"] if c["verdict"] == "tốt hơn" else None
            rows.append({"bước": f"riêng: {nm}", "AUC": np.mean(r["scores"]),
                         **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
        display(pd.DataFrame(rows).set_index("bước").round(4))
        short = sorted([k for k, v in single.items() if v is not None], key=lambda k: -single[k])
        print("Qua sàng lọc:", short or "không có")

        # ---------- (4) xác nhận trên fold mới, chạy cuối ----------
        if short:
            cfg_s = add_r(cfg_r, BLK6R[short[0]])     # chỉ lấy khối tốt nhất, không gộp (hai khối chứa nhau)
            ok_r, _ = fresh_confirm(cfg_r, cfg_s, 909, tag="6r")
            if ok_r:
                res_6r = run_cfg3(cfg_s, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name=f"LightGBM + khối 6r:{short[0]} (cuối)")
                if consider_lgb(res_6r, "LGB(6r)"):
                    cfg_best = cfg_s
        else:
            print("Imputer v2 không thêm gì qua sàng lọc: giữ nguyên cách xử lý missing hiện tại.")
elif RUN_6R:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6r.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")



>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6t. Ghép bộ ba feature: tổ hợp số và mã hóa nhãn của bộ ba biến phân loại

**Ý tưởng.** Cây chia theo từng trục, nên khó dựng các hướng chéo của ba biến cùng lúc, như tỉ lệ `a·b/c` hay `a/(b·c)`. Chưa ai thử có hệ thống: ext_source đã ghép ba (6e), nhưng các biến khác (thu nhập, khoản vay, annuity, tuổi, thâm niên, giấy tờ, vùng...) thì chưa.

**Vấn đề của "thử thật nhiều".** Với 14 biến số có 364 bộ ba, mỗi bộ có 4 dạng tích/tỉ lệ (trong thang log, ext trong thang logit): 1456 ứng viên số, thêm khoảng 455 bộ ba biến phân loại/phân vị. Đưa tất cả vào mô hình thì chỉ thêm nhiễu, và thử từng cái bằng CV thì chắc chắn có vài cái "thắng" do may.

**Cách làm: sàng lọc bằng phần dư rồi mới CV.**
1. Lấy phần dư OOF của mô hình tốt nhất `r = y − p`. Với mỗi ứng viên, chia thành 10 nhóm phân vị (hoặc nhóm mã, với biến phân loại) và tính χ² = Σ (tổng phần dư của nhóm)² / Σ p(1−p). Nếu ứng viên không chứa thông tin mà mô hình chưa có thì χ² ~ χ²(df).
2. Chỉ giữ ứng viên có p-value dưới `P_BONF / tổng số ứng viên` (Bonferroni cho ~1900 phép thử) **và** có z cao hơn z của thành phần mạnh nhất đứng riêng. Điều kiện thứ hai cần vì mô hình không bao giờ hiệu chỉnh hoàn hảo theo từng biến, nên mọi tổ hợp chứa ext_source_2 đều dễ "qua" chỉ vì thành phần đó còn dư chút. Trước khi đo, xác suất OOF được hiệu chỉnh lại bằng hồi quy logistic 1 biến (trung bình nhiều seed làm xác suất co lại). Nếu không có ứng viên nào qua thì kết luận ngay: không còn gì ở hướng này.
3. Ứng viên qua cổng được lọc trùng lặp (|tương quan| < 0.9) rồi đưa vào quy trình quen thuộc: sàng lọc tầng 1 bằng `compare`, xác nhận trên fold mới (seed +1010), chạy 3 seed, `consider_lgb`.

Bước 1 dùng nhãn nhưng chỉ để **chọn ứng viên**; quyết định cuối vẫn dựa trên fold mới nên không bị ảnh hưởng bởi bước chọn. `FORCE_6T = True` để chạy tầng 1 với top ứng viên dù không qua cổng.

**Kỳ vọng thật:** thấp. Các phân tích trước cho thấy tổ hợp nhiều cột của ext và bureau đều hòa. Giá trị chính của ô này là câu trả lời dứt khoát cho "ghép bộ ba còn gì không".

In [23]:
RUN_6T = False   # đã chạy: 0/1911 ứng viên qua cổng, ghép bộ ba đã cạn
P_BONF = 0.01         # mức ý nghĩa cả nhóm phép thử (chia đều cho mọi ứng viên)
TOP_NUM, TOP_CAT = 12, 6
FORCE_6T = False      # True: chạy tầng 1 với top ứng viên dù không qua cổng
if RUN_6T and "consider_lgb" in globals():
    from scipy.stats import chi2 as _chi2
    t0 = time.time()
    cfg_t = dict((globals().get("LGB_BEST") or {}).get("cfg") or globals().get("cfg_best") or _pick_best_cfg())
    for k in ("te_resid", "te_hier", "te_resid2", "twin"):
        cfg_t.setdefault(k, [])
    cfg_t["params"] = dict(cfg_t.get("params", {}))
    F_ = FULL
    n_all = len(F_)
    from sklearn.linear_model import LogisticRegression as _LR
    p_ = np.clip(np.asarray(LGB_BEST["oof"], dtype=float), 1e-6, 1 - 1e-6)
    lz_ = np.log(p_ / (1 - p_)).reshape(-1, 1)
    p_ = _LR(C=1e6).fit(lz_, y).predict_proba(lz_)[:, 1]      # hiệu chỉnh lại độ co do trung bình seed (Platt), trước khi đo phần dư
    r_ = y - p_
    w_ = p_ * (1 - p_)

    def chi_codes(codes, nmin=50):
        c = np.asarray(codes)[:n_tr].astype(np.int64)
        cnt = np.bincount(c)
        c = np.where(cnt[c] < nmin, c.max() + 1, c)           # nhóm hiếm gộp lại
        _, inv = np.unique(c, return_inverse=True)
        s_, v_ = np.bincount(inv, weights=r_), np.bincount(inv, weights=w_)
        return float(np.sum(s_ ** 2 / v_)), len(v_) - 1

    def qcodes(x, q=10):
        b = pd.qcut(pd.Series(np.asarray(x, dtype=float)).rank(method="first"), q, labels=False, duplicates="drop").to_numpy(dtype=float)
        return np.where(np.isnan(b), q, b).astype(np.int64)

    # ---------- biến số: thang log (số dương) hoặc logit (ext), chuẩn hóa z ----------
    VARS_N = [("e1", "ext_source_1_ml", "logit"), ("e2", "ext_source_2_ml", "logit"), ("e3", "ext_source_3_ml", "logit"),
              ("age", "age_years", "log"), ("emp", "years_employed", "log"), ("reg", "years_registration", "log"),
              ("idp", "years_id_publish", "log"), ("ph", "years_last_phone_change", "log"), ("inc", "income", "log"),
              ("cr", "credit_amount", "log"), ("an", "annuity", "log"), ("gp", "amt_goods_price", "log"),
              ("pop", "region_population_relative", "log"), ("fam", "cnt_fam_members", "log")]
    VARS_N = [v for v in VARS_N if v[1] in F_.columns]
    Z = {}
    for nm, col, kind in VARS_N:
        x = F_[col].to_numpy(dtype=float)
        t = np.log(np.clip(x, 1e-3, 1 - 1e-3) / (1 - np.clip(x, 1e-3, 1 - 1e-3))) if kind == "logit" else np.log1p(np.clip(x, 0, None))
        Z[nm] = ((t - np.nanmean(t)) / (np.nanstd(t) + 1e-12)).astype("float32")
    PATS = {"ppp": (1, 1, 1), "ppm": (1, 1, -1), "pmp": (1, -1, 1), "mpp": (-1, 1, 1)}
    from itertools import combinations
    cand_n = []
    for (a, b, c) in combinations([v[0] for v in VARS_N], 3):
        for pn, sg in PATS.items():
            cand_n.append((a, b, c, pn, sg))

    def make_num(cd):
        a, b, c, pn, sg = cd
        z = sg[0] * Z[a] + sg[1] * Z[b] + sg[2] * Z[c]
        return (z / (np.nanstd(z) + 1e-12)).astype("float32")

    # ---------- bộ ba biến phân loại / phân vị ----------
    fz = lambda col: (pd.factorize(F_[col])[0] + 1).astype(np.int64)
    CATV = {}
    for col in ("c_occupation", "c_organization", "name_education_type", "c_income_type", "c_family", "c_housing", "code_gender",
                "region_rating_client"):
        if col in F_.columns:
            CATV[col] = fz(col)
    for nm, col, q in (("term5", "credit_term", 5), ("ext10", "ext_mean_ml", 10), ("age10", "age_years", 10), ("inc10", "income", 10),
                       ("cr10", "credit_amount", 10), ("an10", "annuity", 10), ("emp5", "years_employed", 5)):
        if col in F_.columns:
            CATV[nm] = qcodes(F_[col], q)
    cand_c = list(combinations(list(CATV), 3))

    def make_cat(cd):
        a, b, c = (CATV[k] for k in cd)
        return pd.factorize(a * (int(b.max()) + 1) * (int(c.max()) + 1) + b * (int(c.max()) + 1) + c)[0].astype(np.int64)

    m_tot = len(cand_n) + len(cand_c)
    thr = P_BONF / m_tot
    print(f"Ứng viên: {len(cand_n)} số (14 biến → bộ ba x 4 dạng) + {len(cand_c)} bộ ba phân loại = {m_tot} | ngưỡng p = {thr:.2e}")

    # ---------- sàng lọc bằng χ² trên phần dư OOF ----------
    sc_n, sc_c = [], []
    for cd in cand_n:
        chi, df = chi_codes(qcodes(make_num(cd)))
        sc_n.append((cd, chi, df, float(_chi2.sf(chi, df))))
    for cd in cand_c:
        chi, df = chi_codes(make_cat(cd))
        sc_c.append((cd, chi, df, float(_chi2.sf(chi, df))))
    zs = lambda s: (s[1] - s[2]) / np.sqrt(2.0 * s[2])
    # mốc so sánh: z của từng thành phần đứng riêng. Mô hình không bao giờ hiệu chỉnh hoàn hảo theo từng biến, nên một bộ ba
    # chỉ đáng giữ nếu nó cho cấu trúc phần dư NHIỀU HƠN thành phần mạnh nhất của chính nó.
    z_single = {}
    for nm, col, kind in VARS_N:
        z_single[nm] = zs((None,) + chi_codes(qcodes(Z[nm])))
    for k_, v_ in CATV.items():
        z_single[k_] = zs((None,) + chi_codes(v_))
    print("z đứng riêng của các biến (phần dư còn lại sau mô hình tốt nhất):", {k: round(v, 1) for k, v in sorted(z_single.items(), key=lambda kv: -kv[1])[:8]})
    sc_n = [(cd, chi, df, p, zs((None, chi, df)) - max(z_single[cd[0]], z_single[cd[1]], z_single[cd[2]])) for cd, chi, df, p in sc_n]
    sc_c = [(cd, chi, df, p, zs((None, chi, df)) - max(z_single[k] for k in cd)) for cd, chi, df, p in sc_c]
    allp = np.array([s[3] for s in sc_n + sc_c])
    print(f"Sàng lọc xong ({time.time() - t0:.0f}s). Qua cổng Bonferroni: {int((allp < thr).sum())} / {m_tot} | "
          f"p < 0.01 (chưa hiệu chỉnh): {int((allp < 0.01).sum())} (kỳ vọng ~{0.01 * m_tot:.0f} nếu không có gì) | p < 0.05: {int((allp < 0.05).sum())} (~{0.05 * m_tot:.0f})")
    top_n = sorted(sc_n, key=lambda s: -s[4])[:8]
    top_c = sorted(sc_c, key=lambda s: -s[4])[:8]
    display(pd.DataFrame([{"loại": "số", "ứng viên": "_".join(map(str, s[0][:3])) + "_" + s[0][3], "χ²": s[1], "df": s[2], "p": s[3], "z": zs(s), "z vượt thành phần": s[4]} for s in top_n] +
                         [{"loại": "mã", "ứng viên": "×".join(s[0]), "χ²": s[1], "df": s[2], "p": s[3], "z": zs(s), "z vượt thành phần": s[4]} for s in top_c]).round(5))

    pass_n = [s for s in sorted(sc_n, key=lambda s: -s[4]) if s[3] < thr and s[4] > 0]
    pass_c = [s for s in sorted(sc_c, key=lambda s: -s[4]) if s[3] < thr and s[4] > 0]
    print(f"Qua cổng (p < ngưỡng VÀ z vượt thành phần mạnh nhất): {len(pass_n)} số, {len(pass_c)} mã")
    if not (pass_n or pass_c) and FORCE_6T:
        pass_n, pass_c = sorted(sc_n, key=lambda s: -s[4])[:TOP_NUM], sorted(sc_c, key=lambda s: -s[4])[:TOP_CAT]
        print("FORCE_6T: lấy top ứng viên dù không qua cổng.")

    if not (pass_n or pass_c):
        print("Không ứng viên nào qua cổng Bonferroni: phần dư của mô hình hiện tại không còn cấu trúc nào ở các bộ ba này. "
              "Hướng ghép bộ ba đã cạn; không cần chạy tầng 1.")
    else:
        # lọc trùng lặp (tương quan) rồi dựng cột
        rs_ix = np.random.RandomState(SEED).choice(n_tr, min(40000, n_tr), replace=False)
        sel_n, cols_n = [], []
        for s in pass_n:
            v = make_num(s[0])
            if all(abs(np.corrcoef(np.nan_to_num(v[:n_tr][rs_ix]), np.nan_to_num(u[:n_tr][rs_ix]))[0, 1]) < 0.9 for u in cols_n):
                sel_n.append(s)
                cols_n.append(v)
            if len(sel_n) >= TOP_NUM:
                break
        names_n = [f"n3_{s[0][0]}_{s[0][1]}_{s[0][2]}_{s[0][3]}" for s in sel_n]
        sel_c = pass_c[:TOP_CAT]
        names_c = ["t3_" + "_".join(s[0]) for s in sel_c]
        parts = []
        if names_n:
            parts.append(pd.DataFrame(dict(zip(names_n, cols_n)), index=F_.index))
        if names_c:
            parts.append(pd.DataFrame({nm: make_cat(s[0]).astype("float32") for nm, s in zip(names_c, sel_c)}, index=F_.index))
        ALL6T = pd.concat(parts, axis=1).replace([np.inf, -np.inf], np.nan).astype("float32")
        X_all = pd.concat([X_all.drop(columns=[c for c in ALL6T.columns if c in X_all.columns]), ALL6T.iloc[:n_tr].reset_index(drop=True)], axis=1)
        X_test_all = pd.concat([X_test_all.drop(columns=[c for c in ALL6T.columns if c in X_test_all.columns]),
                                ALL6T.iloc[n_tr:].reset_index(drop=True)], axis=1)
        BLK6T = {}
        if names_n:
            BLK6T["num3"] = dict(feats=names_n)
        if names_c:
            BLK6T["cat3"] = dict(te=names_c, te_resid=names_c)
        print("Giữ lại:", {k: len(v.get("feats", v.get("te", []))) for k, v in BLK6T.items()}, "|", names_n[:6], names_c[:3])

        def add_t(cfgx, spec):
            o_ = dict(cfgx)
            o_["feats"] = list(cfgx["feats"]) + [c for c in spec.get("feats", []) if c not in cfgx["feats"]]
            o_["te"] = list(cfgx["te"]) + [c for c in spec.get("te", []) if c not in cfgx["te"]]
            o_["te_resid"] = list(cfgx.get("te_resid", [])) + [c for c in spec.get("te_resid", []) if c not in cfgx.get("te_resid", [])]
            return o_

        cur_t = run_cfg3(cfg_t, FAST_PARAMS, es=ES_FAST, name="mốc 6t")
        rows, single = [{"bước": "mốc", "AUC": np.mean(cur_t["scores"]), "giữ": "-"}], {}
        for nm, spec in BLK6T.items():
            r = run_cfg3(add_t(cfg_t, spec), FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
            c = compare(r, cur_t)
            single[nm] = c["delta"] if c["verdict"] == "tốt hơn" else None
            rows.append({"bước": f"riêng: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
        short = sorted([k for k, v in single.items() if v is not None], key=lambda k: -single[k])
        cfg_p, cur_p, kept_t = cfg_t, cur_t, []
        for nm in short:
            cand = add_t(cfg_p, BLK6T[nm])
            r = run_cfg3(cand, FAST_PARAMS, es=ES_FAST, name=f"gộp +{nm}")
            c = compare(r, cur_p)
            keep = c["verdict"] == "tốt hơn"
            rows.append({"bước": f"gộp dần: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if keep else "không"})
            if keep:
                cfg_p, cur_p, kept_t = cand, r, kept_t + [nm]
        display(pd.DataFrame(rows).set_index("bước").round(4))
        print("Khối qua tầng 1:", kept_t or "không có")
        if kept_t:
            ok_t, _ = fresh_confirm(cfg_t, cfg_p, 1010, tag="6t")
            if ok_t:
                res_6t = run_cfg3(cfg_p, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối 6t (cuối)")
                if consider_lgb(res_6t, "LGB(6t)"):
                    cfg_best = cfg_p
        else:
            print("Ứng viên qua cổng phần dư nhưng không qua CV tầng 1: giữ nguyên mô hình đang dùng.")
elif RUN_6T:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6t.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")



>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6u. Deep Feature Synthesis theo nhóm, và sinh feature bằng tìm kiếm tiến hóa

Hai kỹ thuật khác nhau, cùng dùng hàm mục tiêu và cùng kiểm soát chọn nhầm như 6t (phần dư OOF của mô hình tốt nhất, hiệu chỉnh Platt, χ² theo nhóm phân vị).

**(a) DFS theo nhóm.** Bảng chỉ có một bảng nên DFS thu lại thành "chuẩn hóa theo khóa rồi tổng hợp": với mỗi khóa (nghề, tổ chức, học vấn, loại thu nhập, gia đình, nhà ở, hạng vùng, giới, vùng, decile tuổi, decile ext) và mỗi biến số (13 cột), tạo 5 feature: trung bình nhóm `gm`, độ lệch chuẩn nhóm `gs`, hiệu `dm = x − gm`, `dz = dm / gs`, hạng phần trăm trong nhóm `pr`. Khoảng 600 ứng viên, không dùng nhãn khi tạo.
- Qua cổng nếu: p < `P_BONF / số ứng viên` trên toàn train, z vượt thành phần mạnh nhất của chính nó, **và** tái lập ở cả hai nửa ngẫu nhiên của train (p < 0.05 mỗi nửa).

**(b) Tiến hóa.** Quần thể biểu thức độ sâu ≤ 4 trên 14 biến số (đã chuẩn hóa log/logit): cộng, trừ, nhân, chia an toàn, max, min, |x|, √|x|. Lai ghép subtree, đột biến, giữ tinh hoa, phạt kích thước. Tự cài đặt, không cần gplearn, để dùng được hàm mục tiêu mong muốn:
- **Hàm mục tiêu = z của phần dư trên nửa A, trừ z của thành phần mạnh nhất, trừ phạt kích thước.** Không dùng tương quan đơn biến với nhãn (nó chỉ tiến hóa ra các biến giống `ext_source`).
- Sau tiến hóa, 60 biểu thức tốt nhất được **kiểm lại trên nửa B** (chưa từng dùng khi chọn): giữ nếu p_B < 0.01/60 và vẫn vượt thành phần. Lọc trùng |tương quan| < 0.9.

Cả hai khối còn phải qua CV tầng 1 (`compare`), xác nhận trên fold mới (seed +1111), chạy 3 seed rồi `consider_lgb`, như mọi khối trước. Nếu không ứng viên nào qua cổng thì dừng sớm và in rõ.

**Kỳ vọng thật:** thấp. 6t đã cạn ghép bộ ba. Giá trị mới của ô này là trung bình/độ lệch **theo nhóm** (không có trong 6t) và không gian biểu thức rộng hơn.

In [24]:
RUN_6U = False   # đã có kết quả: bỏ để tiết kiệm thời gian
RUN_DFS, RUN_EVO = True, True
P_BONF = 0.01
TOP_DFS, TOP_EVO = 12, 8
EVO_POP, EVO_GEN = (60, 5) if DEBUG_FAST else (400, 25)
FORCE_6U = False       # True: chạy tầng 1 với top ứng viên dù không qua cổng
if RUN_6U and "consider_lgb" in globals():
    import copy
    from itertools import combinations
    from scipy.stats import chi2 as _chi2
    from sklearn.linear_model import LogisticRegression as _LR
    t0 = time.time()
    rs_u = np.random.RandomState(SEED + 1111)
    cfg_u = dict((globals().get("LGB_BEST") or {}).get("cfg") or globals().get("cfg_best") or _pick_best_cfg())
    for k in ("te_resid", "te_hier", "te_resid2", "twin"):
        cfg_u.setdefault(k, [])
    cfg_u["params"] = dict(cfg_u.get("params", {}))
    F_ = FULL
    n_all = len(F_)

    p_ = np.clip(np.asarray(LGB_BEST["oof"], dtype=float), 1e-6, 1 - 1e-6)
    lz_ = np.log(p_ / (1 - p_)).reshape(-1, 1)
    p_ = _LR(C=1e6).fit(lz_, y).predict_proba(lz_)[:, 1]
    r_, w_ = y - p_, p_ * (1 - p_)
    perm_ = rs_u.permutation(n_tr)
    A_rows, B_rows, T_rows = np.sort(perm_[: n_tr // 2]), np.sort(perm_[n_tr // 2:]), np.arange(n_tr)

    def qcodes(x, q=10):
        b = pd.qcut(pd.Series(np.asarray(x, dtype=float)).rank(method="first"), q, labels=False, duplicates="drop").to_numpy(dtype=float)
        return np.where(np.isnan(b), q, b).astype(np.int64)

    def z_codes(codes, rows, nmin=50):
        c = np.asarray(codes)[rows].astype(np.int64)
        cnt = np.bincount(c)
        c = np.where(cnt[c] < nmin, c.max() + 1, c)
        _, inv = np.unique(c, return_inverse=True)
        s_, v_ = np.bincount(inv, weights=r_[rows]), np.bincount(inv, weights=w_[rows])
        chi, df = float(np.sum(s_ ** 2 / v_)), len(v_) - 1
        if df < 1:
            return 0.0, 1.0
        return (chi - df) / np.sqrt(2.0 * df), float(_chi2.sf(chi, df))

    def _zrows(x, rows):
        xr = np.asarray(x, dtype=float)[rows]
        c = qcodes(xr)
        cnt = np.bincount(c)
        c = np.where(cnt[c] < 50, c.max() + 1, c)
        _, inv = np.unique(c, return_inverse=True)
        s_, v_ = np.bincount(inv, weights=r_[rows]), np.bincount(inv, weights=w_[rows])
        chi, df = float(np.sum(s_ ** 2 / v_)), len(v_) - 1
        if df < 1:
            return 0.0, 1.0
        return (chi - df) / np.sqrt(2.0 * df), float(_chi2.sf(chi, df))

    # ---------- biến số chuẩn hóa (như 6t) ----------
    VARS_N = [("e1", "ext_source_1_ml", "logit"), ("e2", "ext_source_2_ml", "logit"), ("e3", "ext_source_3_ml", "logit"),
              ("age", "age_years", "log"), ("emp", "years_employed", "log"), ("reg", "years_registration", "log"),
              ("idp", "years_id_publish", "log"), ("ph", "years_last_phone_change", "log"), ("inc", "income", "log"),
              ("cr", "credit_amount", "log"), ("an", "annuity", "log"), ("gp", "amt_goods_price", "log"),
              ("pop", "region_population_relative", "log"), ("fam", "cnt_fam_members", "log")]
    VARS_N = [v for v in VARS_N if v[1] in F_.columns]
    Z = {}
    for nm, col, kind in VARS_N:
        x = F_[col].to_numpy(dtype=float)
        xc = np.clip(x, 1e-3, 1 - 1e-3)
        t = np.log(xc / (1 - xc)) if kind == "logit" else np.log1p(np.clip(x, 0, None))
        Z[nm] = ((t - np.nanmean(t)) / (np.nanstd(t) + 1e-12)).astype("float32")
    leaf_names = [v[0] for v in VARS_N]
    nv = len(leaf_names)
    ZT = np.column_stack([Z[n] for n in leaf_names]).astype("float32")
    zsA = np.array([_zrows(ZT[:, j], A_rows)[0] for j in range(nv)])
    zsB = np.array([_zrows(ZT[:, j], B_rows)[0] for j in range(nv)])
    zsT = np.array([_zrows(ZT[:, j], T_rows)[0] for j in range(nv)])
    print("z đứng riêng của biến số (toàn train):", {n: round(float(z), 1) for n, z in zip(leaf_names, zsT)})

    def add_u_cols(parts):
        global X_all, X_test_all
        ALL = pd.concat(parts, axis=1).replace([np.inf, -np.inf], np.nan).astype("float32")
        ALL.index = F_.index
        X_all = pd.concat([X_all.drop(columns=[c for c in ALL.columns if c in X_all.columns]), ALL.iloc[:n_tr].reset_index(drop=True)], axis=1)
        X_test_all = pd.concat([X_test_all.drop(columns=[c for c in ALL.columns if c in X_test_all.columns]), ALL.iloc[n_tr:].reset_index(drop=True)], axis=1)

    def dedupe(vecs, names, keep, thr_c=0.9):
        ix = rs_u.choice(n_tr, min(40000, n_tr), replace=False)
        out_n, out_v = [], []
        for nmx, v in zip(names, vecs):
            vv = np.nan_to_num(np.asarray(v, dtype=float)[ix])
            if vv.std() < 1e-12:
                continue
            if all(abs(np.corrcoef(vv, u)[0, 1]) < thr_c for u in out_v):
                out_n.append(nmx)
                out_v.append(vv)
            if len(out_n) >= keep:
                break
        return out_n

    parts_u, BLK6U = [], {}

    # =============== (a) DFS theo nhóm ===============
    if RUN_DFS:
        fz = lambda col: (pd.factorize(F_[col])[0] + 1).astype(np.int64)
        KEYS = {c: fz(c) for c in ("c_occupation", "c_organization", "name_education_type", "c_income_type", "c_family", "c_housing",
                                   "region_rating_client", "code_gender", "region_population_relative") if c in F_.columns}
        for nmk, col in (("age10", "age_years"), ("ext10", "ext_mean_ml")):
            if col in F_.columns:
                KEYS[nmk] = qcodes(F_[col])
        VARS_D = [c for c in ("ext_source_1_ml", "ext_source_2_ml", "ext_source_3_ml", "ext_mean_ml", "age_years", "years_employed",
                              "years_registration", "years_id_publish", "years_last_phone_change", "income", "credit_amount", "annuity",
                              "amt_goods_price", "credit_term") if c in F_.columns]
        zs_key = {k: _zrows(v.astype(float), T_rows)[0] for k, v in KEYS.items()}
        zs_var = {c: _zrows(F_[c].to_numpy(dtype=float), T_rows)[0] for c in VARS_D}

        def dfs_feats(k, x):
            g = pd.DataFrame({"k": k, "x": x}).groupby("k")["x"]
            cnt = g.transform("count").to_numpy()
            gm, gs = g.transform("mean").to_numpy(), g.transform("std").to_numpy()
            pr = g.rank(pct=True).to_numpy()
            dm = x - gm
            out = {"gm": gm, "gs": gs, "dm": dm, "dz": dm / (gs + 1e-9), "pr": pr}
            return {a: np.where(cnt >= 30, v, np.nan) for a, v in out.items()}

        res_d = []
        for kn, kc in KEYS.items():
            for vn in VARS_D:
                for agg, v in dfs_feats(kc, F_[vn].to_numpy(dtype=float)).items():
                    zt, pt = _zrows(v, T_rows)
                    res_d.append((kn, vn, agg, zt, pt, zt - max(zs_key[kn], zs_var[vn])))
        m_d = len(res_d)
        thr_d = P_BONF / m_d
        # tái lập ở hai nửa chỉ cho ứng viên qua cổng toàn train
        cand_d = [r for r in res_d if r[4] < thr_d and r[5] > 0]
        keep_d = []
        for r in sorted(cand_d, key=lambda r: -r[5]):
            v = dfs_feats(KEYS[r[0]], F_[r[1]].to_numpy(dtype=float))[r[2]]
            pa, pb = _zrows(v, A_rows)[1], _zrows(v, B_rows)[1]
            if pa < 0.05 and pb < 0.05:
                keep_d.append(r)
        top_show = sorted(res_d, key=lambda r: r[4])[:10]
        print(f"[DFS] {m_d} ứng viên | ngưỡng p = {thr_d:.1e} | qua cổng toàn train + vượt thành phần: {len(cand_d)} | tái lập ở cả hai nửa: {len(keep_d)} "
              f"| p<0.01 chưa hiệu chỉnh: {sum(r[4] < 0.01 for r in res_d)} (kỳ vọng ~{0.01 * m_d:.0f})")
        display(pd.DataFrame([{"khóa": r[0], "biến": r[1], "phép": r[2], "z": r[3], "p": r[4], "z vượt thành phần": r[5]} for r in top_show]).round(5))
        if not keep_d and FORCE_6U:
            keep_d = sorted(res_d, key=lambda r: -r[5])[:TOP_DFS]
            print("[DFS] FORCE_6U: lấy top ứng viên dù không qua cổng.")
        if keep_d:
            vec_d = [dfs_feats(KEYS[r[0]], F_[r[1]].to_numpy(dtype=float))[r[2]] for r in keep_d]
            nms_d = [f"dfs_{r[2]}_{r[1]}_by_{r[0]}" for r in keep_d]
            sel = dedupe(vec_d, nms_d, TOP_DFS)
            parts_u.append(pd.DataFrame({n: v for n, v in zip(nms_d, vec_d) if n in sel}, index=F_.index))
            BLK6U["dfs"] = dict(feats=sel)
        print(f"[DFS] xong ({time.time() - t0:.0f}s)")

    # =============== (b) tiến hóa ===============
    if RUN_EVO:
        OPS2, OPS1 = ("add", "sub", "mul", "div", "max", "min"), ("abs", "sqrtabs")

        def gen(d, full=False):
            if d == 0 or (not full and rs_u.rand() < 0.3):
                return ["v", int(rs_u.randint(nv))]
            if rs_u.rand() < 0.2:
                return ["u", OPS1[rs_u.randint(2)], gen(d - 1, full)]
            return ["b", OPS2[rs_u.randint(len(OPS2))], gen(d - 1, full), gen(d - 1, full)]

        def ev(t, X):
            if t[0] == "v":
                return X[:, t[1]]
            if t[0] == "u":
                a = ev(t[2], X)
                return np.abs(a) if t[1] == "abs" else np.sqrt(np.abs(a))
            a, b, op = ev(t[2], X), ev(t[3], X), t[1]
            if op == "add": return a + b
            if op == "sub": return a - b
            if op == "mul": return a * b
            if op == "div": return a / (np.abs(b) + 0.5)
            return np.maximum(a, b) if op == "max" else np.minimum(a, b)

        def size(t): return 1 if t[0] == "v" else 1 + (size(t[2]) if t[0] == "u" else size(t[2]) + size(t[3]))
        def depth(t): return 1 if t[0] == "v" else 1 + (depth(t[2]) if t[0] == "u" else max(depth(t[2]), depth(t[3])))
        def leaves(t): return {t[1]} if t[0] == "v" else (leaves(t[2]) if t[0] == "u" else leaves(t[2]) | leaves(t[3]))

        def show(t):
            if t[0] == "v": return leaf_names[t[1]]
            if t[0] == "u": return f"{t[1]}({show(t[2])})"
            return f"{t[1]}({show(t[2])},{show(t[3])})"

        def slots(holder):
            out, st = [], [(holder, 0)]
            while st:
                c, i = st.pop()
                out.append((c, i))
                n = c[i]
                if n[0] == "u": st.append((n, 2))
                elif n[0] == "b": st += [(n, 2), (n, 3)]
            return out

        XA = ZT[A_rows]
        cache = {}

        def fitness(t):
            key = show(t)
            if key in cache:
                return cache[key][0]
            x = np.nan_to_num(ev(t, XA), nan=0.0, posinf=0.0, neginf=0.0)
            if x.std() < 1e-9 or size(t) < 3:
                cache[key] = (-9.0, t)
                return -9.0
            xa = np.zeros(n_all, dtype="float32")
            z, _ = _zrows_sub(x)
            f = z - max(zsA[j] for j in leaves(t)) - 0.05 * size(t)
            cache[key] = (f, t)
            return f

        def _zrows_sub(x):          # x đã là giá trị trên A_rows
            c = qcodes(x)
            cnt = np.bincount(c)
            c = np.where(cnt[c] < 50, c.max() + 1, c)
            _, inv = np.unique(c, return_inverse=True)
            s_, v_ = np.bincount(inv, weights=r_[A_rows]), np.bincount(inv, weights=w_[A_rows])
            chi, df = float(np.sum(s_ ** 2 / v_)), len(v_) - 1
            return ((chi - df) / np.sqrt(2.0 * df), float(_chi2.sf(chi, df))) if df >= 1 else (0.0, 1.0)

        def cross(p1, p2):
            a, b = copy.deepcopy(p1), copy.deepcopy(p2)
            ha, hb = [a], [b]
            (c1, i1), (c2, j2) = slots(ha)[rs_u.randint(len(slots(ha)))], slots(hb)[rs_u.randint(len(slots(hb)))]
            c1[i1] = c2[j2]
            return ha[0] if depth(ha[0]) <= 4 else copy.deepcopy(p1)

        def mutate(t):
            h = [copy.deepcopy(t)]
            c, i = slots(h)[rs_u.randint(len(slots(h)))]
            c[i] = gen(2)
            return h[0] if depth(h[0]) <= 4 else copy.deepcopy(t)

        pop = [gen(int(rs_u.randint(1, 4)), full=bool(rs_u.rand() < 0.5)) for _ in range(EVO_POP)]
        hist = []
        for g_ in range(EVO_GEN):
            fits = np.array([fitness(t) for t in pop])
            order = np.argsort(-fits)
            hist.append((g_, float(fits[order[0]]), float(np.mean(fits))))
            new = [copy.deepcopy(pop[j]) for j in order[:5]]
            while len(new) < EVO_POP:
                pick = lambda: pop[max(rs_u.choice(EVO_POP, 5, replace=False), key=lambda j: fits[j])]
                r0 = rs_u.rand()
                new.append(cross(pick(), pick()) if r0 < 0.6 else mutate(pick()) if r0 < 0.9 else gen(3))
            pop = new
        print(f"[Tiến hóa] {len(cache)} biểu thức đã đánh giá | fitness tốt nhất theo thế hệ: " + ", ".join(f"{h[1]:.2f}" for h in hist[:: max(1, len(hist) // 6)]) +
              f" | {time.time() - t0:.0f}s")

        hof = sorted(cache.values(), key=lambda ft: -ft[0])[:60]
        rows_e, ok_e = [], []
        for f, t in hof:
            x = np.nan_to_num(ev(t, ZT), nan=0.0, posinf=0.0, neginf=0.0)
            zb, pb = _zrows(x, B_rows)
            exc_b = zb - max(zsB[j] for j in leaves(t))
            rows_e.append({"biểu thức": show(t), "fitness (A)": f, "z_B": zb, "p_B": pb, "z_B vượt": exc_b})
            if pb < 0.01 / 60 and exc_b > 0:
                ok_e.append((t, x))
        display(pd.DataFrame(rows_e[:10]).round(5))
        print(f"[Tiến hóa] kiểm lại trên nửa B (p_B < {0.01 / 60:.1e}, vượt thành phần): {len(ok_e)} / {len(hof)} | "
              f"p_B < 0.01: {sum(r['p_B'] < 0.01 for r in rows_e)} (kỳ vọng ~{0.01 * len(rows_e):.1f} nếu chỉ nhiễu)")
        if not ok_e and FORCE_6U:
            ok_e = [(t, np.nan_to_num(ev(t, ZT), nan=0.0, posinf=0.0, neginf=0.0)) for f, t in hof[:TOP_EVO]]
            print("[Tiến hóa] FORCE_6U: lấy top ứng viên dù không qua cổng.")
        if ok_e:
            nms_e = [f"evo{k}_" + show(t)[:40].replace("(", "_").replace(")", "").replace(",", "_") for k, (t, x) in enumerate(ok_e)]
            vec_e = [x for t, x in ok_e]
            sel = dedupe(vec_e, nms_e, TOP_EVO)
            parts_u.append(pd.DataFrame({n: v for n, v in zip(nms_e, vec_e) if n in sel}, index=F_.index))
            BLK6U["evo"] = dict(feats=sel)

    # =============== tầng 1, xác nhận, chạy cuối ===============
    if not BLK6U:
        print("\nKhông ứng viên nào qua cổng ở cả hai hướng: phần dư của mô hình hiện tại không còn cấu trúc nhóm/biểu thức nào đo được. Dừng, không chạy CV.")
    else:
        add_u_cols(parts_u)
        print("Giữ lại:", {k: len(v["feats"]) for k, v in BLK6U.items()})

        def add_u(cfgx, spec):
            o_ = dict(cfgx)
            o_["feats"] = list(cfgx["feats"]) + [c for c in spec["feats"] if c not in cfgx["feats"]]
            return o_

        cur_u = run_cfg3(cfg_u, FAST_PARAMS, es=ES_FAST, name="mốc 6u")
        rows, single = [{"bước": "mốc", "AUC": np.mean(cur_u["scores"]), "giữ": "-"}], {}
        for nm, spec in BLK6U.items():
            r = run_cfg3(add_u(cfg_u, spec), FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
            c = compare(r, cur_u)
            single[nm] = c["delta"] if c["verdict"] == "tốt hơn" else None
            rows.append({"bước": f"riêng: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
        short = sorted([k for k, v in single.items() if v is not None], key=lambda k: -single[k])
        cfg_p, cur_p, kept_u = cfg_u, cur_u, []
        for nm in short:
            cand = add_u(cfg_p, BLK6U[nm])
            r = run_cfg3(cand, FAST_PARAMS, es=ES_FAST, name=f"gộp +{nm}")
            c = compare(r, cur_p)
            keep = c["verdict"] == "tốt hơn"
            rows.append({"bước": f"gộp dần: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "có" if keep else "không"})
            if keep:
                cfg_p, cur_p, kept_u = cand, r, kept_u + [nm]
        display(pd.DataFrame(rows).set_index("bước").round(4))
        print("Khối qua tầng 1:", kept_u or "không có")
        if kept_u:
            ok_u, _ = fresh_confirm(cfg_u, cfg_p, 1111, tag="6u")
            if ok_u:
                res_6u = run_cfg3(cfg_p, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối 6u (cuối)")
                if consider_lgb(res_6u, "LGB(6u)"):
                    cfg_best = cfg_p
        else:
            print("Qua cổng phần dư nhưng không qua CV tầng 1: giữ nguyên mô hình đang dùng.")
    print(f"\n6u xong sau {time.time() - t0:.0f}s")
elif RUN_6U:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6u.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")



>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6v. Dùng ML để tạo lại / mở rộng dữ liệu huấn luyện, và khối tổ chức

Bốn thử nghiệm, mỗi cái qua cùng quy trình: so với mốc ở tầng 1 (`compare`), xác nhận trên bộ fold mới (seed +1414), chạy 3 seed, `consider_lgb`.

| Khối | Làm gì | Vì sao có thể giúp / vì sao không |
|---|---|---|
| `aug` | **Swap-noise augmentation**: nhân đôi tập huấn luyện của fold, bản sao có 15% ô bị đổi bằng giá trị cùng cột của hàng ngẫu nhiên, giữ nhãn. Kiểm tra trên fold sạch | regularization; thường giúp mạng nơ-ron hơn GBDT |
| `distill` | **Self-distillation**: nhãn huấn luyện = 0.5·nhãn thật + 0.5·xác suất của một mô hình giáo viên. Giáo viên là **OOF lồng trong tập huấn luyện của fold** (không bao giờ thấy fold kiểm tra) | làm mịn nhãn nhiễu; vài `1e-4` nếu có |
| `weight` | **Lọc nhiễu nhãn**: giảm trọng số (0.3) 10% ca dương mà giáo viên cho điểm thấp nhất và 1% ca âm mà giáo viên cho điểm cao nhất | nhãn vỡ nợ vốn khó đoán nên ít tác dụng |
| `org` | Target encoding (thường + phần dư) của **loại tổ chức × (decile tuổi, thu nhập, ext; kỳ hạn; thâm niên)**. Manh mối từ 6t: `c_organization` là biến duy nhất còn z = 5 trong phần dư | 6t cho z đứng riêng 5.0, nhưng đó là tín hiệu yếu |

**Không làm:** SMOTE và sinh dữ liệu bằng CTGAN/TabDDPM. Chúng nội suy hoặc lặp lại phân phối đã thấy, không thêm thông tin, làm xác suất lệch, và với bảng 171k hàng GBDT thường không hưởng lợi.

**Lưu ý rò rỉ.** Giáo viên phải được huấn luyện **chỉ trên tập huấn luyện của fold**. Nếu dùng OOF toàn cục làm nhãn mềm thì nhãn của fold kiểm tra sẽ chảy vào nhãn huấn luyện và CV lạc quan. Ô này dùng OOF lồng trong fold (mất thêm ~5 phút mỗi lần chạy `distill`/`weight`, được lưu cache giữa hai kiểu).

**Huấn luyện lại 100% (6s):** không áp dụng cho kiểu huấn luyện đặc biệt (`aug`/`distill`/`weight`), vì cách huấn luyện lại thường không tái tạo chúng; khi đó giữ `test_pred` từ CV.

In [25]:
RUN_6V = False   # đã có kết quả: bỏ để tiết kiệm thời gian
RUN_ORG, RUN_AUG, RUN_DISTILL, RUN_WEIGHT = True, True, True, True
P_SWAP, ALPHA_DIST, FRAC_POS, FRAC_NEG, W_LOW = 0.15, 0.5, 0.10, 0.01, 0.3
if RUN_6V and "consider_lgb" in globals():
    t0 = time.time()
    rs_v = np.random.RandomState(SEED + 1414)
    cfg_v = dict((globals().get("LGB_BEST") or {}).get("cfg") or globals().get("cfg_best") or _pick_best_cfg())
    for k in ("te_resid", "te_hier", "te_resid2", "twin"):
        cfg_v.setdefault(k, [])
    cfg_v["params"] = dict(cfg_v.get("params", {}))
    F_ = FULL
    TEACH = {}

    def teacher(Xtr, ytr, key):
        """OOF lồng trong tập huấn luyện của fold: giáo viên không bao giờ thấy fold kiểm tra."""
        if key in TEACH:
            return TEACH[key]
        p = np.zeros(len(ytr))
        for a, b in StratifiedKFold(4, shuffle=True, random_state=SEED + 1313).split(np.zeros(len(ytr)), ytr):
            m = lgb.LGBMClassifier(**{**FAST_PARAMS, "learning_rate": 0.1, "n_estimators": 100 if DEBUG_FAST else 600, "random_state": SEED})
            m.fit(Xtr.iloc[a], ytr[a], eval_set=[(Xtr.iloc[b], ytr[b])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(0)])
            p[b] = m.predict_proba(Xtr.iloc[b])[:, 1]
        TEACH[key] = p
        return p

    def run_var(cfg, base, mode, seeds=(SEED,), es=100, name=""):
        t1 = time.time()
        oof, te_pred, iters = np.zeros(len(y)), np.zeros(len(X_test_all)), []
        for tr, va in SPLITS:
            Xtr, Xva, Xte, f = fold_data3(cfg, tr, va)
            prm = {**base, **cfg["params"]}
            ytr, sw, soft = y[tr], None, None
            if mode in ("distill", "weight"):
                p_in = teacher(Xtr, ytr, (hash(tr.tobytes()), tuple(cfg["feats"]), tuple(cfg["te"])))
                if mode == "distill":
                    soft = (1 - ALPHA_DIST) * ytr + ALPHA_DIST * p_in
                else:
                    sw = np.ones(len(tr))
                    pos, neg = np.flatnonzero(ytr == 1), np.flatnonzero(ytr == 0)
                    sw[pos[p_in[pos] <= np.quantile(p_in[pos], FRAC_POS)]] = W_LOW
                    sw[neg[p_in[neg] >= np.quantile(p_in[neg], 1 - FRAC_NEG)]] = W_LOW
            if mode == "aug":
                A = Xtr.to_numpy(dtype="float32", copy=True)
                B = A.copy()
                for j in range(A.shape[1]):
                    mk = rs_v.rand(len(A)) < P_SWAP
                    B[mk, j] = A[rs_v.randint(len(A), size=int(mk.sum())), j]
                Xfit = pd.concat([Xtr, pd.DataFrame(B, columns=Xtr.columns, index=None)], ignore_index=True)
                yfit = np.concatenate([ytr, ytr])
            else:
                Xfit, yfit = Xtr, ytr
            for s in seeds:
                if mode == "distill":
                    m = lgb.LGBMRegressor(**{**{k: v for k, v in prm.items() if k != "objective"}, "objective": "cross_entropy", "random_state": s})
                    m.fit(Xfit, soft, eval_set=[(Xva, y[va])], eval_metric="auc",
                          callbacks=[lgb.early_stopping(es, verbose=False), lgb.log_evaluation(0)])
                    pv, pt = m.predict(Xva), m.predict(Xte)
                else:
                    m = lgb.LGBMClassifier(**{**prm, "random_state": s})
                    m.fit(Xfit, yfit, sample_weight=sw, eval_set=[(Xva, y[va])], eval_metric="auc",
                          callbacks=[lgb.early_stopping(es, verbose=False), lgb.log_evaluation(0)])
                    pv, pt = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
                oof[va] += pv / len(seeds)
                te_pred += pt / (len(seeds) * N_FOLDS)
                iters.append(m.best_iteration_)
        scores = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
        print(f"[{name}] AUC = {np.mean(scores):.4f} ± {np.std(scores):.4f} | log-loss = {log_loss(y, np.clip(oof, 1e-6, 1 - 1e-6)):.5f} "
              f"| {len(f)} feature | cây TB = {np.mean(iters):.0f} | {time.time() - t1:.0f}s")
        return {"oof": oof, "test_pred": te_pred, "scores": scores, "name": name, "iters": iters, "cfg": cfg, "base": base,
                "seeds": tuple(seeds), "mode": mode, "imp": pd.Series(0.0, index=f)}

    def fresh_pair(fa, fb, seed_off, tag):
        global SPLITS
        bak = SPLITS
        SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + seed_off).split(np.zeros(len(y)), y))
        try:
            a_, b_ = fa(f"XÁC NHẬN {tag}: mốc, fold mới"), fb(f"XÁC NHẬN {tag}: mới, fold mới")
        finally:
            SPLITS = bak
        c = compare(b_, a_)
        ok = c["verdict"] == "tốt hơn"
        print(f"Xác nhận trên fold mới: delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}"
              f" => {'GIỮ' if ok else 'BỎ (không qua xác nhận, nhiều khả năng do may rủi)'}")
        return ok, c

    # ---------- khối tổ chức (feature) ----------
    org_cols = []
    if RUN_ORG and "c_organization" in F_.columns:
        def qc(x, q):
            b = pd.qcut(pd.Series(np.asarray(x, dtype=float)).rank(method="first"), q, labels=False, duplicates="drop").to_numpy(dtype=float)
            return np.where(np.isnan(b), q, b).astype(np.int64)
        o = (pd.factorize(F_["c_organization"])[0] + 1).astype(np.int64)
        parts = {}
        for nm, col, q in (("age10", "age_years", 10), ("inc10", "income", 10), ("ext10", "ext_mean_ml", 10), ("term5", "credit_term", 5), ("emp5", "years_employed", 5)):
            if col in F_.columns:
                b = qc(F_[col], q)
                parts[f"org_x_{nm}"] = pd.factorize(o * 100 + b)[0].astype("float32")
        if parts:
            ALLV = pd.DataFrame(parts, index=F_.index)
            org_cols = list(parts)
            X_all = pd.concat([X_all.drop(columns=[c for c in org_cols if c in X_all.columns]), ALLV.iloc[:n_tr].reset_index(drop=True)], axis=1)
            X_test_all = pd.concat([X_test_all.drop(columns=[c for c in org_cols if c in X_test_all.columns]), ALLV.iloc[n_tr:].reset_index(drop=True)], axis=1)

    def with_org(cfgx):
        o_ = dict(cfgx)
        o_["te"] = list(cfgx["te"]) + [c for c in org_cols if c not in cfgx["te"]]
        o_["te_resid"] = list(cfgx.get("te_resid", [])) + [c for c in org_cols if c not in cfgx.get("te_resid", [])]
        return o_

    # ---------- tầng 1 ----------
    cur_v = run_cfg3(cfg_v, FAST_PARAMS, es=ES_FAST, name="mốc 6v")
    VARS = []
    if org_cols:
        VARS.append(("org", lambda cfg_, nm_: run_cfg3(with_org(cfg_), FAST_PARAMS, es=ES_FAST, name=nm_)))
    for md, flag in (("aug", RUN_AUG), ("distill", RUN_DISTILL), ("weight", RUN_WEIGHT)):
        if flag:
            VARS.append((md, (lambda m_: (lambda cfg_, nm_: run_var(cfg_, FAST_PARAMS, m_, es=ES_FAST, name=nm_)))(md)))
    rows, win = [{"khối": "mốc", "AUC": np.mean(cur_v["scores"])}], {}
    for nm, fn in VARS:
        r = fn(cfg_v, f"+{nm}")
        c = compare(r, cur_v)
        rows.append({"khối": nm, "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}})
        if c["verdict"] == "tốt hơn":
            win[nm] = c["delta"]
    display(pd.DataFrame(rows).set_index("khối").round(4))
    print("Qua tầng 1:", sorted(win, key=lambda k: -win[k]) or "không có")

    # ---------- xác nhận và chạy cuối cho khối tốt nhất ----------
    if win:
        best = max(win, key=win.get)
        fn = dict(VARS)[best]
        ok_v, _ = fresh_pair(lambda nm_: run_cfg3(cfg_v, FAST_PARAMS, es=ES_FAST, name=nm_), lambda nm_: fn(cfg_v, nm_), 1414, f"6v:{best}")
        if ok_v:
            if best == "org":
                res_6v = run_cfg3(with_org(cfg_v), PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name="LightGBM + khối tổ chức (cuối)")
                if consider_lgb(res_6v, "LGB(6v:org)"):
                    cfg_best = with_org(cfg_v)
            else:
                res_6v = run_var(cfg_v, PARAMS, best, seeds=FINAL_SEEDS, es=ES_FINAL, name=f"LightGBM kiểu {best} (cuối)")
                consider_lgb(res_6v, f"LGB(6v:{best})")
    else:
        print("Không khối nào qua tầng 1: giữ nguyên mô hình đang dùng.")
    print(f"\n6v xong sau {time.time() - t0:.0f}s")
elif RUN_6V:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6v.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")



>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6y. Khoảng cách tới các hồ sơ đã vỡ nợ gần nhất (kNN có nhãn, tính trong từng fold)

**Ý tưởng của bạn:** với mỗi hồ sơ, đo khoảng cách tới 10 hồ sơ vỡ nợ gần nhất trong không gian đặc trưng; càng gần "cụm vỡ nợ" thì càng nguy hiểm. Ngoài ra thêm khoảng cách tới các hồ sơ không vỡ nợ để lấy tỉ lệ (mật độ tương đối), vì chỉ riêng khoảng cách tới nhóm vỡ nợ bị lẫn với việc hồ sơ đó nằm ở vùng thưa.

**Cách làm đúng để không rò rỉ nhãn.**
- Tính **trong từng fold**: hồ sơ validation và test chỉ nhìn các hồ sơ vỡ nợ của phần huấn luyện. Nhãn của fold validation không xuất hiện ở bất cứ bước nào, nên điểm CV không bị thổi phồng.
- Với chính các hồ sơ huấn luyện, khoảng cách được tính **loại trừ bản thân** (leave-one-out), nếu không khoảng cách tới chính nó bằng 0 và mô hình học được nhãn.
- Tham chiếu "không vỡ nợ" là một mẫu ngẫu nhiên có số lượng bằng số hồ sơ vỡ nợ (`KNN_NEG_MULT`), để hai khoảng cách so sánh được với nhau.
- Không gian khoảng cách: `KNN_NUM_FEATS` cột số quan trọng nhất của mô hình mốc (bỏ cột tương quan hạng > 0.85 với cột đã chọn, để nhóm ext_source và các biến dẫn xuất của nó không chi phối khoảng cách), đưa về phân phối chuẩn theo phân vị, nhân trọng số `sqrt(độ quan trọng)`. Không chọn cột theo nhãn của fold validation.

**Cần biết trước.** Ver4 đã đo một biến thể gần (tỉ lệ vỡ nợ của hàng xóm, nhãn trong fold): delta 0.0000. Đây là phiên bản khác (khoảng cách và mật độ tương đối, không phải tỉ lệ), nên vẫn đáng đo, nhưng thông tin phần lớn trùng với những gì cây đã học từ chính các cột đó, nên đừng kỳ vọng nhiều.

**Hai khối, kiểm theo đúng quy tắc các ô trước (`compare`, rồi xác nhận trên fold mới):**
- `knn_pos`: chỉ khoảng cách tới 10 hồ sơ vỡ nợ gần nhất (trung bình và nhỏ nhất), đúng ý bạn.
- `knn_all`: thêm khoảng cách tới 10 hồ sơ không vỡ nợ, log tỉ số hai khoảng cách, và tỉ lệ hồ sơ vỡ nợ trong 10 hàng xóm gần nhất của tập tham chiếu cân bằng.

Kết quả qua cổng mới tính vào mô hình chính; đặc trưng được tính lại trên 100% dữ liệu khi huấn luyện lại ở 6s.


In [26]:
RUN_6Y = False   # chưa cần: bật lại nếu muốn thử khoảng cách tới hồ sơ vỡ nợ
KNN_K = 10
KNN_NUM_FEATS = 15        # số cột số quan trọng nhất làm không gian khoảng cách
KNN_NEG_MULT = 1.0        # số hồ sơ không vỡ nợ làm tham chiếu = KNN_NEG_MULT × số hồ sơ vỡ nợ
if RUN_6Y and "consider_lgb" in globals():
    from sklearn.neighbors import NearestNeighbors
    from sklearn.preprocessing import QuantileTransformer
    t0 = time.time()
    cfg_y = dict(globals().get("cfg_best") or _pick_best_cfg())
    for k in ("te_resid", "te_hier", "te_resid2", "twin"):
        cfg_y.setdefault(k, [])
    cfg_y["params"] = dict(cfg_y.get("params", {}))
    cfg_y.pop("knn", None)

    cur_y = run_cfg3(cfg_y, FAST_PARAMS, es=ES_FAST, name="mốc 6y")
    # ---- chọn không gian khoảng cách từ độ quan trọng của mô hình mốc ----
    imp_y = cur_y["imp"]
    cand_y = [c for c in imp_y.index if c in X_all.columns and pd.api.types.is_float_dtype(X_all[c]) and X_all[c].nunique() > 20
              and X_all[c].isna().mean() < 0.5 and imp_y[c] > 0][:60]
    rho_y = X_all[cand_y].sample(min(20000, len(X_all)), random_state=SEED).corr(method="spearman").abs()
    num_ok = []                                       # chọn tham lam theo độ quan trọng, bỏ cột tương quan > 0.85 với cột đã chọn (tránh một nhóm cột trùng nhau chi phối khoảng cách)
    for c in cand_y:
        if all(rho_y.loc[c, o] < 0.85 for o in num_ok):
            num_ok.append(c)
        if len(num_ok) >= KNN_NUM_FEATS:
            break
    KNN_COLS = num_ok
    w_ = np.sqrt(imp_y[KNN_COLS].to_numpy(float))
    KNN_W = (w_ / np.sqrt((w_ ** 2).mean())).astype("float32")      # chuẩn hóa để trung bình bình phương trọng số = 1
    print(f"Không gian khoảng cách: {len(KNN_COLS)} cột: {KNN_COLS}")

    KNN_NAMES = ["knn_dpos_mean", "knn_dpos_min", "knn_dneg_mean", "knn_log_ratio", "knn_pos_share"]
    KNN_CACHE = {}

    def _knn_query(Zref, Zq, self_idx_in_ref, K):
        # khoảng cách tới K hàng xóm gần nhất trong Zref; nếu self_idx_in_ref không None thì loại trừ chính hồ sơ (leave-one-out)
        nn = NearestNeighbors(n_neighbors=K + (1 if self_idx_in_ref is not None else 0), algorithm="auto", n_jobs=-1).fit(Zref)
        d, ix = nn.kneighbors(Zq)
        if self_idx_in_ref is None:
            return d
        hit = ix == self_idx_in_ref[:, None]
        keep = ~hit
        nohit = ~hit.any(axis=1)
        keep[nohit, -1] = False                       # không có chính nó trong K+1 gần nhất: bỏ hàng xóm xa nhất
        return d[keep].reshape(len(d), K)

    def knn_feats(A, ytr, others):
        Atr = A[KNN_COLS].to_numpy(float)
        med = np.nanmedian(Atr, axis=0)
        med = np.where(np.isnan(med), 0.0, med)
        fill = lambda M: np.where(np.isnan(M), med[None, :], M)
        qt = QuantileTransformer(n_quantiles=min(1000, len(Atr)), output_distribution="normal", subsample=200_000, random_state=0).fit(fill(Atr))
        tf = lambda D: (qt.transform(fill(D[KNN_COLS].to_numpy(float))) * KNN_W[None, :]).astype("float32")
        Ztr, Zo = tf(A), [tf(o) for o in others]
        pos = np.flatnonzero(ytr == 1)
        neg_all = np.flatnonzero(ytr == 0)
        neg = np.random.RandomState(SEED).choice(neg_all, min(len(neg_all), int(KNN_NEG_MULT * len(pos))), replace=False)
        K = KNN_K

        def build(Zq, self_mode):
            # self_mode: True khi Zq chính là Ztr (cần loại trừ bản thân)
            qi = np.arange(len(Zq))
            d_p = _knn_query(Ztr[pos], Zq, None if not self_mode else _self_pos(pos, qi), K)
            d_n = _knn_query(Ztr[neg], Zq, None if not self_mode else _self_pos(neg, qi), K)
            thr = np.partition(np.hstack([d_p, d_n]), K - 1, axis=1)[:, K - 1]
            return pd.DataFrame({
                "knn_dpos_mean": d_p.mean(1), "knn_dpos_min": d_p.min(1), "knn_dneg_mean": d_n.mean(1),
                "knn_log_ratio": np.log((d_p.mean(1) + 1e-3) / (d_n.mean(1) + 1e-3)),
                "knn_pos_share": (d_p <= thr[:, None]).sum(1) / K,
            }).astype("float32")

        def _self_pos(ref, qi):
            # cho mỗi hồ sơ i: vị trí của i trong tập tham chiếu `ref` nếu i thuộc ref, ngược lại -1
            m = -np.ones(len(qi), dtype=np.int64)
            m[ref] = np.arange(len(ref))
            return m

        return build(Ztr, True), [build(Z, False) for Z in Zo]

    _fd3_base = globals().get("_fd3_orig") or fold_data3
    _fd3_orig = _fd3_base

    def fold_data3(cfg, tr, va):
        Xtr, Xva, Xte, f = _fd3_orig(cfg, tr, va)
        want = cfg.get("knn")
        if want:
            key = (hash(np.asarray(tr).tobytes()), len(va), int(va[0]) if len(va) else -1, len(X_test_all))
            if key not in KNN_CACHE:
                t1 = time.time()
                A = X_all.iloc[tr].reset_index(drop=True)
                Bv, T = X_all.iloc[va].reset_index(drop=True), X_test_all.reset_index(drop=True)
                t_tr, (t_va, t_te) = knn_feats(A, y[tr], [Bv, T])
                KNN_CACHE[key] = (t_tr, t_va, t_te)
                if len(va) > 500 and len(np.unique(y[va])) == 2:
                    au = {c: roc_auc_score(y[va], t_va[c]) for c in KNN_NAMES}
                    print(f"  [kNN] fold dựng xong {time.time() - t1:.0f}s | AUC đơn biến trên validation: " + ", ".join(f"{k.replace('knn_', '')} {max(v, 1 - v):.3f}" for k, v in au.items()))
            t_tr, t_va, t_te = KNN_CACHE[key]
            cols = [c for c in KNN_NAMES if c in want]
            Xtr, Xva, Xte = (pd.concat([p.reset_index(drop=True), q_[cols].reset_index(drop=True)], axis=1) for p, q_ in ((Xtr, t_tr), (Xva, t_va), (Xte, t_te)))
            f = f + cols
        return Xtr, Xva, Xte, f

    BLK6Y = {"knn_pos": ["knn_dpos_mean", "knn_dpos_min"], "knn_all": list(KNN_NAMES)}

    rows = [{"bước": "mốc", "AUC": np.mean(cur_y["scores"]), "giữ": "-"}]
    single, res_y = {}, {}
    for nm, cols in BLK6Y.items():
        r = run_cfg3({**cfg_y, "knn": cols}, FAST_PARAMS, es=ES_FAST, name=f"+{nm}")
        c = compare(r, cur_y)
        single[nm] = c["delta"] if c["verdict"] == "tốt hơn" else None
        res_y[nm] = r
        rows.append({"bước": f"riêng: {nm}", "AUC": np.mean(r["scores"]), **{k: c[k] for k in ("delta", "delta_se", "fold_duong", "verdict")}, "giữ": "-"})
    display(pd.DataFrame(rows).set_index("bước").round(4))
    short = sorted([k for k, v in single.items() if v is not None], key=lambda k: -single[k])
    print("Qua sàng lọc:", short or "không có")
    if short:
        nm = short[0]
        cfg_k = {**cfg_y, "knn": BLK6Y[nm]}
        ok_y, _ = fresh_confirm(cfg_y, cfg_k, 1515, tag="6y")
        if ok_y:
            res_6y = run_cfg3(cfg_k, PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name=f"LightGBM + {nm} (cuối)")
            if consider_lgb(res_6y, "LGB(6y)"):
                cfg_best = cfg_k
    else:
        print("Không khối nào qua: khoảng cách tới hồ sơ vỡ nợ không thêm thông tin so với các cột cây đã dùng. Giữ nguyên mô hình đang dùng.")
elif RUN_6Y:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6y.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")



>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6z. Thử từng mẫu không nhãn, thêm dần mẫu nào làm mô hình dự đoán tốt hơn trên tập valid thêm

**Quy trình đúng như bạn mô tả (tham lam, từng mẫu một):**
1. Tách 15% hồ sơ huấn luyện làm **tập valid thêm `V`** (phân tầng theo nhãn). `V` bị loại khỏi huấn luyện và khỏi đánh giá CV.
2. Tạo các **mẫu không nhãn** trong từng fold: hồ sơ test, và bản sao của hồ sơ huấn luyện bị **đổi 3 feature** (lấy giá trị cùng cột của hàng khác), bỏ nhãn.
3. Duyệt **từng mẫu**, thử **cả hai nhánh nhãn** (0 và 1): mô hình + mẫu đó với nhãn `l` thì mất mát (log-loss) trên `V` là bao nhiêu? Lấy nhánh tốt hơn; **nếu nhánh đó làm `V` tốt hơn trạng thái hiện tại thì nhận mẫu**, và mô hình hiện tại được cập nhật, rồi sang mẫu tiếp theo. Mẫu nào không làm `V` tốt hơn ở cả hai nhánh thì bỏ.
4. Mô hình LightGBM thật được huấn luyện lại **một lần** với tập mẫu đã nhận (cộng hồ sơ huấn luyện), đo trên các hồ sơ CV chưa từng chạm `V`.

**Vì sao "thử từng mẫu" không làm bằng cách huấn luyện lại LightGBM cho mỗi mẫu.** Mỗi lần huấn luyện lại tốn khoảng nửa phút đến vài phút; với hàng chục nghìn mẫu × 2 nhánh sẽ mất hàng tuần, và một hồ sơ trong 140 nghìn gần như không đổi AUC (nhiễu giữa hai lần huấn luyện còn lớn hơn). Thay vào đó vòng lặp dùng **mô hình thay thế cập nhật được từng mẫu một**: giữ nguyên cấu trúc các cây của LightGBM, mỗi mẫu `(x, l)` cập nhật giá trị lá đúng theo công thức của chính LightGBM:

`giá trị lá (t, lá) += lr · (l − p(x)) / (Σ hessian trong lá + reg_lambda)`  với mọi cây `t` mà `x` rơi vào lá đó.

Dự đoán trên `V` được cộng dồn đúng theo bước này, nên **log-loss trên `V` sau mỗi mẫu được tính chính xác cho mô hình thay thế**; mô hình thay thế là xấp xỉ của LightGBM huấn luyện lại (cây không tách lại), nên bước 4 (huấn luyện lại thật) mới là phép đo cuối cùng.

**Bốn cánh tay, cùng fold, cùng seed, so bằng `compare`:**
| Cánh tay | Huấn luyện bằng |
|---|---|
| `base` | hồ sơ huấn luyện của fold |
| `greedy` | + các mẫu đã nhận, với nhãn nhánh được chọn (phương pháp của bạn) |
| `random` | + **đúng các mẫu đó**, cùng số nhãn 1 và nhãn 0, nhưng nhãn **xáo trộn ngẫu nhiên** (đối chứng: nếu việc chọn nhánh có thông tin thì `greedy` phải hơn `random`) |
| `ref_V` | + chính `V` với nhãn thật (tham chiếu: dùng trực tiếp `V` để huấn luyện) |

**Hai chỗ có thể làm hỏng kết quả, ô đã xử lý:**
- **Lệch tiên nghiệm.** Nếu không kiềm chế, khoảng một nửa số mẫu sẽ được gán nhãn 1 (vì phần dư trên `V` có dấu dương và âm xấp xỉ nhau), trong khi chỉ khoảng 8% hồ sơ thật sự vỡ nợ. Ô giữ **tỉ lệ nhãn 1 trong tập đã nhận không vượt tỉ lệ dương của dữ liệu** (`PL_CAP_PRIOR`).
- **Khớp nhiễu của `V`.** Chọn nhánh theo `V` cho từng mẫu sẽ khớp cả nhiễu của `V`. Ô in log-loss `V` trước và sau chọn (lạc quan, vì chọn trên chính `V`) cạnh AUC thật trên hồ sơ CV (không chạm `V`): chênh lệch giữa hai con số cho biết mức khớp nhiễu.

**Kiểm tra trung thực có sẵn:** với các bản sao bị đổi giá trị ta biết nhãn gốc, nên ô in tỉ lệ nhãn gốc là vỡ nợ trong số mẫu được gán nhãn 1 (so với tỉ lệ nền).

**Mô hình dùng:** LightGBM tốt nhất của blend (`LGB_BEST`): cùng feature và tham số ghi đè, nhưng ở `lr = 0.05`, 1 seed, và 85% dữ liệu (không có `V`), như các ô sàng lọc khác. Ô không đưa kết quả vào `FINAL`; nếu `greedy` qua cổng (hơn `base` và `random`, rồi xác nhận trên fold mới) thì quy trình sản xuất cần thiết kế riêng.


In [27]:
RUN_6Z = False
PL_FINAL_PARAMS = True    # True: dùng PARAMS cuối (lr 0.02, ES_FINAL) giống mô hình trong blend; False: FAST_PARAMS
PL_EXACT = True           # True: dùng bản huấn luyện lại thật ở ô kế tiếp (6z-exact); False: bản xấp xỉ nhân lá ở ô này
PL_V_FRAC = 0.15         # tập valid thêm (loại khỏi huấn luyện và CV)
PL_N_PERT = 20000        # số bản sao hồ sơ huấn luyện bị đổi giá trị mỗi fold (bỏ nhãn)
PL_K_SWAP = 3            # số feature bị đổi mỗi bản sao
PL_TOPCOLS = 30          # chọn feature để đổi trong top này (theo độ quan trọng của mô hình mốc)
PL_USE_TEST = True       # dùng thêm hồ sơ test (vốn không nhãn)
PL_N_CAND = 20000        # số mẫu ứng viên được thử từng mẫu một mỗi fold
PL_N_TAKE = 8000         # dừng khi đã nhận đủ chừng này mẫu
PL_MIN_GAIN = 0.0        # mẫu phải làm log-loss tổng trên V giảm hơn ngưỡng này mới được nhận
PL_CAP_PRIOR = True      # tỉ lệ nhãn 1 trong tập đã nhận không vượt tỉ lệ dương của dữ liệu
PL_WEIGHT = 1.0          # trọng số của mẫu đã nhận khi huấn luyện lại thật
PL_BLOCK = 256           # số mẫu tính nhân lá cùng lúc (chỉ ảnh hưởng tốc độ)
if DEBUG_FAST:
    PL_N_PERT, PL_N_CAND, PL_N_TAKE, PL_BLOCK = 3000, 3000, 1500, 256
if RUN_6Z and "consider_lgb" in globals():
    import scipy.sparse as sp
    t0 = time.time()
    ref_z = globals().get("LGB_BEST") or res_lgb
    cfg_z = dict(ref_z.get("cfg") or _pick_best_cfg())
    for k in ("te_resid", "te_hier", "te_resid2", "twin"):
        cfg_z.setdefault(k, [])
    cfg_z["params"] = dict(cfg_z.get("params", {}))
    cfg_z.pop("knn", None)
    params_z = {**(PARAMS if PL_FINAL_PARAMS else FAST_PARAMS), **cfg_z["params"]}
    ES_Z = ES_FINAL if PL_FINAL_PARAMS else ES_FAST
    LR_Z, LAM_Z = float(params_z.get("learning_rate", 0.05)), float(params_z.get("reg_lambda", 0.0))
    print(f"LightGBM dùng: {ref_z['name']} | {len(cfg_z['feats'])} feature gốc + mã hóa nhãn trong fold | lr {LR_Z}, reg_lambda {LAM_Z}")

    rsV = np.random.RandomState(SEED + 31)
    V_idx = np.sort(np.concatenate([rsV.choice(np.flatnonzero(y == c), int(PL_V_FRAC * (y == c).sum()), replace=False) for c in (0, 1)]))
    isV = np.zeros(len(y), bool)
    isV[V_idx] = True
    print(f"Tập valid thêm V: {len(V_idx):,} hồ sơ, {int(y[V_idx].sum()):,} vỡ nợ ({y[V_idx].mean():.2%}); phần còn lại D = {int((~isV).sum()):,}")

    def _fit_z(Xa, ya, wa, Xe, ye, f, seed):
        prm = dict(params_z)
        if cfg_z.get("mono"):
            prm["monotone_constraints"] = [-1 if c in MONO else 0 for c in f]
        m = lgb.LGBMClassifier(**{**prm, "random_state": seed})
        m.fit(Xa, ya, sample_weight=wa, eval_set=[(Xe, ye)], eval_metric="auc",
              callbacks=[lgb.early_stopping(ES_Z, verbose=False), lgb.log_evaluation(0)])
        return m

    def _swap_copy(Xtr, rs, n, k, cols):
        # bản sao của n hồ sơ huấn luyện, mỗi bản đổi k feature (chọn ngẫu nhiên trong cols) bằng giá trị cùng cột của hàng khác
        s = rs.choice(len(Xtr), n, replace=True)
        Xp = Xtr.iloc[s].reset_index(drop=True).copy()
        pick = np.argsort(rs.rand(n, len(cols)), axis=1)[:, :k]
        M = np.zeros((n, len(cols)), bool)
        np.put_along_axis(M, pick, True, axis=1)
        for j, c in enumerate(cols):
            rows = np.flatnonzero(M[:, j])
            if len(rows):
                col = Xp[c].to_numpy().copy()
                col[rows] = Xtr[c].to_numpy()[rs.choice(len(Xtr), len(rows), replace=True)]
                Xp[c] = col
        return Xp, s

    def _greedy_select(m0, Xtr, XV, yV, Xc, p_c, rs):
        # duyệt từng mẫu, thử hai nhánh nhãn, nhận mẫu nào làm log-loss trên V giảm; cập nhật dự đoán V sau mỗi lần nhận
        nit = m0.best_iteration_
        bst = m0.booster_
        sub = rs.choice(len(Xtr), min(40000, len(Xtr)), replace=False)
        lf_tr = bst.predict(Xtr.iloc[sub], pred_leaf=True, num_iteration=nit)
        lf_v = bst.predict(XV, pred_leaf=True, num_iteration=nit)
        lf_c = bst.predict(Xc, pred_leaf=True, num_iteration=nit)
        T_ = lf_c.shape[1]
        Lmax = int(max(lf_tr.max(), lf_v.max(), lf_c.max())) + 1
        # tổng hessian của lá trên toàn bộ dữ liệu huấn luyện (ước lượng từ mẫu con rồi nhân tỉ lệ)
        p_tr = m0.predict_proba(Xtr.iloc[sub])[:, 1]
        col_tr = (np.arange(T_)[None, :] * Lmax + lf_tr).ravel()
        H = np.bincount(col_tr, weights=np.repeat(p_tr * (1 - p_tr), T_), minlength=T_ * Lmax) * (len(Xtr) / len(sub))
        w = LR_Z / (H + LAM_Z + 1e-9)                 # mức dịch giá trị lá cho mỗi đơn vị (l - p(x))

        def phi(lf, weights):
            n = lf.shape[0]
            cols = np.arange(T_)[None, :] * Lmax + lf
            data = np.ones(cols.shape) if weights is None else weights[cols]
            return sp.csr_matrix((data.ravel(), cols.ravel(), np.arange(0, n * T_ + 1, T_)), shape=(n, T_ * Lmax))

        PhiV = phi(lf_v, None).T.tocsr()
        z = m0.predict(XV, raw_score=True).astype(np.float64)
        yv = yV.astype(np.float64)
        loss = lambda zz: float(np.logaddexp(0.0, zz).sum() - (yv * zz).sum())
        loss0 = cur = loss(z)
        share = float(y.mean()) if PL_CAP_PRIOR else 1.0
        acc, lab_acc, n_pos = [], [], 0
        for b0 in range(0, len(Xc), PL_BLOCK):
            idx_b = np.arange(b0, min(b0 + PL_BLOCK, len(Xc)))
            Kb = (phi(lf_c[idx_b], w) @ PhiV).toarray()
            for r_, ci_ in enumerate(idx_b):
                K, px = Kb[r_], p_c[ci_]
                l1, l0 = loss(z + (1 - px) * K), loss(z - px * K)
                cand = [(l1, 1, (1 - px)), (l0, 0, -px)]
                cand.sort(key=lambda t: t[0])
                for lv, lb, coef in cand:
                    if lv < cur - PL_MIN_GAIN:
                        if lb == 1 and (n_pos + 1) > share * (len(acc) + 1):
                            continue                                      # giữ tiên nghiệm: thử nhánh còn lại
                        z = z + coef * K
                        cur = lv
                        acc.append(ci_); lab_acc.append(lb); n_pos += lb
                        break
            if len(acc) >= PL_N_TAKE:
                break
        return np.array(acc, dtype=int), np.array(lab_acc, dtype=int), loss0, cur, min(len(Xc), b0 + PL_BLOCK)

    def run_pl_all(name, arms=("base", "greedy", "random", "ref_V")):
        t1 = time.time()
        arms = tuple(dict.fromkeys(("base",) + tuple(arms)))
        res = {a: {"oof": np.zeros(len(y)), "scores": [], "iters": []} for a in arms}
        diag = []
        need_inf = any(a in arms for a in ("greedy", "random"))
        for fi, (tr, va) in enumerate(SPLITS):
            T, E = tr[~isV[tr]], va[~isV[va]]
            nE = len(E)
            Xtr, Xva_, Xte, f = fold_data3(cfg_z, T, np.concatenate([E, V_idx]))
            XE, XV = Xva_.iloc[:nE].reset_index(drop=True), Xva_.iloc[nE:].reset_index(drop=True)
            yT, yE, yV = y[T], y[E], y[V_idx]
            m0 = _fit_z(Xtr, yT, None, XE, yE, f, SEED)
            res["base"]["oof"][E] = m0.predict_proba(XE)[:, 1]
            res["base"]["iters"].append(m0.best_iteration_)
            Xsel, lab = None, None
            if need_inf:
                rs = np.random.RandomState(SEED + 17 * fi + 5)
                gain = pd.Series(m0.booster_.feature_importance("gain"), index=f).sort_values(ascending=False)
                cols = [c for c in gain.index if c in Xtr.columns and pd.api.types.is_float_dtype(Xtr[c])][:PL_TOPCOLS]
                n_p = min(PL_N_PERT, len(Xtr))
                Xp, s_p = _swap_copy(Xtr, rs, n_p, PL_K_SWAP, cols)
                parts, orig = [Xp], [yT[s_p].astype(float)]
                if PL_USE_TEST:
                    parts.append(Xte.reset_index(drop=True)); orig.append(np.full(len(Xte), np.nan))
                Xc_all = pd.concat(parts, ignore_index=True)
                orig = np.concatenate(orig)
                src = np.concatenate([np.zeros(n_p, int)] + ([np.ones(len(Xte), int)] if PL_USE_TEST else []))
                sel_c = rs.choice(len(Xc_all), min(PL_N_CAND, len(Xc_all)), replace=False)      # thứ tự duyệt ngẫu nhiên
                Xc, orig, src = Xc_all.iloc[sel_c].reset_index(drop=True), orig[sel_c], src[sel_c]
                p_c = m0.predict_proba(Xc)[:, 1]
                acc, lab, l0, l1, n_tried = _greedy_select(m0, Xtr, XV, yV, Xc, p_c, rs)
                Xsel = Xc.iloc[acc]
                is_p = src[acc] == 0
                lab1_p = (lab == 1) & is_p
                diag.append({"fold": fi + 1, "đã thử": n_tried, "đã nhận": len(acc), "% nhận nhãn 1": 100 * float(lab.mean()) if len(lab) else np.nan,
                             "% bản sao trong số nhận": 100 * float(is_p.mean()) if len(acc) else np.nan,
                             "log-loss V trước": l0 / len(yV), "log-loss V sau (chọn trên chính V)": l1 / len(yV),
                             "% nhãn gốc=1 trong số bản sao nhận nhãn 1": 100 * float(orig[acc][lab1_p].mean()) if lab1_p.any() else np.nan,
                             "% nhãn gốc=1 trong mọi bản sao": 100 * float(np.nanmean(orig[src == 0])) if (src == 0).any() else np.nan})
            for a in arms:
                if a == "base":
                    pe = res["base"]["oof"][E]
                else:
                    if a == "greedy":
                        Xa, ya, wa = pd.concat([Xtr, Xsel], ignore_index=True), np.concatenate([yT, lab]), np.concatenate([np.ones(len(yT)), np.full(len(lab), PL_WEIGHT)])
                    elif a == "random":
                        Xa, ya, wa = pd.concat([Xtr, Xsel], ignore_index=True), np.concatenate([yT, np.random.RandomState(SEED + fi).permutation(lab)]), np.concatenate([np.ones(len(yT)), np.full(len(lab), PL_WEIGHT)])
                    else:   # ref_V
                        Xa, ya, wa = pd.concat([Xtr, XV], ignore_index=True), np.concatenate([yT, yV]), None
                    m = _fit_z(Xa, ya, wa, XE, yE, f, SEED)
                    pe = m.predict_proba(XE)[:, 1]
                    res[a]["oof"][E] = pe
                    res[a]["iters"].append(m.best_iteration_)
                res[a]["scores"].append(roc_auc_score(yE, pe))
            print(f"  [6z {name}] fold {fi + 1}/{len(SPLITS)}: " + " | ".join(f"{a} {res[a]['scores'][-1]:.4f}" for a in arms) + f" | {time.time() - t1:.0f}s")
        for a in arms:
            res[a].update({"name": f"{name}:{a}", "test_pred": np.zeros(len(Xte)), "imp": pd.Series(dtype=float)})
        res["_diag"] = pd.DataFrame(diag) if diag else None
        return res

    if PL_EXACT:
        print("PL_EXACT = True: bỏ qua bản xấp xỉ, chạy ô 6z-exact kế tiếp.")
    else:
        R = run_pl_all("chính")
        rows = [{"cánh tay": "base", "AUC": np.mean(R["base"]["scores"]), "delta so với base": 0.0}]
        for a in ("greedy", "random", "ref_V"):
            c = compare(R[a], R["base"])
            rows.append({"cánh tay": a, "AUC": np.mean(R[a]["scores"]), "delta so với base": c["delta"], "delta_se": c["delta_se"], "fold dương": c["fold_duong"], "verdict": c["verdict"]})
        display(pd.DataFrame(rows).set_index("cánh tay").round(4))
        c_co = compare(R["greedy"], R["random"])
        print(f"greedy so với random (cùng mẫu, nhãn xáo trộn; đây mới là bằng chứng việc chọn nhánh có thông tin): delta = {c_co['delta']:+.4f} (se {c_co['delta_se']:.4f}, fold dương {c_co['fold_duong']}) -> {c_co['verdict']}")
        print("Chẩn đoán vòng tham lam (nhãn gốc của bản sao được dùng để kiểm tra):")
        display(R["_diag"].set_index("fold").round(4))
        print("Cách đọc: 'log-loss V sau' thấp hơn 'trước' là điều hiển nhiên (chọn trên chính V); cái đáng tin là AUC trên hồ sơ CV ở bảng trên. "
              "Nếu '% nhãn gốc=1 trong số bản sao nhận nhãn 1' không cao hơn '% nhãn gốc=1 trong mọi bản sao' thì nhánh '1' được chọn không có thông tin.")
        c_ch = compare(R["greedy"], R["base"])
        if c_ch["verdict"] == "tốt hơn" and c_co["verdict"] == "tốt hơn":
            cfg_a, cfg_b = {**cfg_z, "_arm": "base"}, {**cfg_z, "_arm": "greedy"}
            ok_z, _ = fresh_confirm(cfg_a, cfg_b, 1616, run_fn=lambda cfg, nm: run_pl_all(nm, arms=("base", cfg["_arm"]))[cfg["_arm"]], tag="6z")
            print("Qua cổng xác nhận:", "CÓ: cần thiết kế quy trình sản xuất riêng" if ok_z else "KHÔNG: bỏ")
        elif c_ch["verdict"] == "tốt hơn":
            print("`greedy` hơn `base` nhưng KHÔNG hơn `random` (cùng mẫu, nhãn xáo trộn): lợi ích đến từ việc thêm mẫu/đổi tiên nghiệm, không phải từ việc chọn nhánh. Không đưa vào FINAL.")
        else:
            print("`greedy` không hơn `base` ở tầng 1: chọn nhánh bằng V không thêm thông tin có ích cho mô hình. Không đưa vào FINAL.")
    print(f"6z xong sau {time.time() - t0:.0f}s")
elif RUN_6Z:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6z.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")



>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) | AUC CV = 0.7682


## 6z-exact. Thêm dần từng lô mẫu, huấn luyện lại LightGBM thật, đo AUC trên V

Mỗi fold: (1) mô hình mốc huấn luyện trên D; (2) lấy từng lô ~200 mẫu ứng viên (bản sao bị đổi giá trị và hồ sơ test, không nhãn); (3) với mỗi mẫu trong lô, thử cả hai nhánh nhãn bằng bộ ước lượng nhân lá (chỉ để **đề xuất** nhãn); (4) **huấn luyện lại LightGBM thật** trên dữ liệu đã nhận + lô này, đo **AUC trên V**; (5) lô chỉ được nhận nếu AUC trên V tăng, ngược lại bỏ. Cuối cùng đo AUC trên hồ sơ CV (không dùng để chọn) so với mô hình mốc cùng số cây, và so với nhãn bị xáo trộn.

In [28]:
RUN_6ZX = False
PL_EX_BATCH = 200          # số mẫu ứng viên mỗi lô
PL_EX_NBATCH = 30          # số lô tối đa mỗi fold (30 lô x 200 = 6,000 mẫu; huấn luyện với tham số cuối nên mỗi lô chậm hơn bản FAST)
PL_EX_MIN_GAIN = 0.0       # AUC trên V phải tăng hơn mức này thì lô mới được nhận
PL_EX_MIN_PER_FOLD = 30    # ngân sách phút mỗi fold; hết thì dừng thêm lô
if DEBUG_FAST:
    PL_EX_BATCH, PL_EX_NBATCH, PL_EX_MIN_PER_FOLD = 100, 3, 2
if RUN_6ZX and PL_EXACT and "_greedy_select" in globals():
    t0x = time.time()

    def _fit_fixed(Xa, ya, f, nit, seed=SEED):
        prm = {**params_z, "n_estimators": int(nit), "random_state": seed}
        if cfg_z.get("mono"):
            prm["monotone_constraints"] = [-1 if c in MONO else 0 for c in f]
        m = lgb.LGBMClassifier(**prm)
        m.fit(Xa, ya)
        try:
            m._best_iteration = int(nit)     # để _greedy_select dùng đúng số cây
        except Exception:
            pass
        return m

    arms_x = ("base", "exact", "random")
    _lg = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
    HAS_CAT = "res_cat_tuned" in globals() and res_cat_tuned.get("oof") is not None
    W_B = globals().get("W_BLEND", 0.6)
    resb = {a: {"oof": np.zeros(len(y)), "scores": []} for a in arms_x}
    ref_final_scores = []
    resx = {a: {"oof": np.zeros(len(y)), "scores": []} for a in arms_x}
    diagx, logx = [], []
    for fi, (tr, va) in enumerate(SPLITS):
        tf = time.time()
        T, E = tr[~isV[tr]], va[~isV[va]]
        nE = len(E)
        Xtr, Xva_, Xte, f = fold_data3(cfg_z, T, np.concatenate([E, V_idx]))
        XE, XV = Xva_.iloc[:nE].reset_index(drop=True), Xva_.iloc[nE:].reset_index(drop=True)
        yT, yE, yV = y[T], y[E], y[V_idx]
        m0 = _fit_z(Xtr, yT, None, XE, yE, f, SEED)
        nit = m0.best_iteration_
        m_cur = _fit_fixed(Xtr, yT, f, nit)
        cur = roc_auc_score(yV, m_cur.predict_proba(XV)[:, 1])
        auc_V0 = cur
        resx["base"]["oof"][E] = m_cur.predict_proba(XE)[:, 1]
        resx["base"]["scores"].append(roc_auc_score(yE, resx["base"]["oof"][E]))
        # ứng viên: bản sao đổi giá trị + hồ sơ test, thứ tự ngẫu nhiên
        rs = np.random.RandomState(SEED + 17 * fi + 99)
        gain = pd.Series(m0.booster_.feature_importance("gain"), index=f).sort_values(ascending=False)
        cols = [c for c in gain.index if c in Xtr.columns and pd.api.types.is_float_dtype(Xtr[c])][:PL_TOPCOLS]
        n_pool = PL_EX_BATCH * PL_EX_NBATCH
        Xp, _ = _swap_copy(Xtr, rs, min(n_pool, len(Xtr)), PL_K_SWAP, cols)
        parts = [Xp] + ([Xte.reset_index(drop=True)] if PL_USE_TEST else [])
        Xc_all = pd.concat(parts, ignore_index=True)
        Xc_all = Xc_all.iloc[rs.choice(len(Xc_all), min(n_pool, len(Xc_all)), replace=False)].reset_index(drop=True)
        X_acc, y_acc = Xtr.iloc[:0], np.zeros(0, int)
        n_ok = n_tried = 0
        for b in range(PL_EX_NBATCH):
            if (time.time() - tf) / 60 > PL_EX_MIN_PER_FOLD:
                break
            Xb = Xc_all.iloc[b * PL_EX_BATCH:(b + 1) * PL_EX_BATCH].reset_index(drop=True)
            if len(Xb) == 0:
                break
            n_tried += 1
            p_b = m_cur.predict_proba(Xb)[:, 1]
            X_cur = pd.concat([Xtr, X_acc], ignore_index=True)
            acc, lab, _, _, _ = _greedy_select(m_cur, X_cur, XV, yV, Xb, p_b, rs)
            if len(acc) == 0:
                logx.append({"fold": fi + 1, "lô": b + 1, "đề xuất": 0, "AUC V thử": np.nan, "AUC V hiện tại": cur, "nhận": False})
                continue
            X_try = pd.concat([X_acc, Xb.iloc[acc]], ignore_index=True)
            y_try = np.concatenate([y_acc, lab])
            m_try = _fit_fixed(pd.concat([Xtr, X_try], ignore_index=True), np.concatenate([yT, y_try]), f, nit)
            a_try = roc_auc_score(yV, m_try.predict_proba(XV)[:, 1])
            ok = a_try > cur + PL_EX_MIN_GAIN
            logx.append({"fold": fi + 1, "lô": b + 1, "đề xuất": len(acc), "AUC V thử": a_try, "AUC V hiện tại": cur, "nhận": bool(ok)})
            if ok:
                X_acc, y_acc, m_cur, cur = X_try, y_try, m_try, a_try
                n_ok += 1
        pe = m_cur.predict_proba(XE)[:, 1]
        resx["exact"]["oof"][E] = pe
        resx["exact"]["scores"].append(roc_auc_score(yE, pe))
        if len(y_acc):
            m_r = _fit_fixed(pd.concat([Xtr, X_acc], ignore_index=True), np.concatenate([yT, np.random.RandomState(SEED + fi).permutation(y_acc)]), f, nit)
            pr = m_r.predict_proba(XE)[:, 1]
        else:
            pr = resx["base"]["oof"][E]
        resx["random"]["oof"][E] = pr
        resx["random"]["scores"].append(roc_auc_score(yE, pr))
        if HAS_CAT:
            CE = res_cat_tuned["oof"][E]
            for a in arms_x:
                pb = expit(W_B * _lg(resx[a]["oof"][E]) + (1 - W_B) * _lg(CE))
                resb[a]["oof"][E] = pb
                resb[a]["scores"].append(roc_auc_score(yE, pb))
            ref_final_scores.append(roc_auc_score(yE, FINAL["oof"][E]))
        diagx.append({"fold": fi + 1, "lô thử": n_tried, "lô nhận": n_ok, "mẫu nhận": len(y_acc),
                      "% nhãn 1": 100 * float(y_acc.mean()) if len(y_acc) else np.nan,
                      "AUC V trước": auc_V0, "AUC V sau (chọn trên V)": cur,
                      "AUC E base": resx["base"]["scores"][-1], "AUC E exact": resx["exact"]["scores"][-1], "AUC E random": resx["random"]["scores"][-1],
                      **({"BLEND base": resb["base"]["scores"][-1], "BLEND exact": resb["exact"]["scores"][-1], "BLEND random": resb["random"]["scores"][-1]} if HAS_CAT else {})})
        print(f"  [6z-exact] fold {fi + 1}/{len(SPLITS)}: base {resx['base']['scores'][-1]:.4f} | exact {resx['exact']['scores'][-1]:.4f} | random {resx['random']['scores'][-1]:.4f} | lô nhận {n_ok}/{n_tried} | {time.time() - tf:.0f}s")
    for a in arms_x:
        resx[a].update({"name": f"6z-exact:{a}", "test_pred": np.zeros(len(Xte)), "imp": pd.Series(dtype=float)})
    display(pd.DataFrame(diagx).set_index("fold").round(4))
    rows = [{"cánh tay": "base (cùng số cây, không ES)", "AUC": np.mean(resx["base"]["scores"]), "delta so với base": 0.0}]
    for a in ("exact", "random"):
        c = compare(resx[a], resx["base"])
        rows.append({"cánh tay": a, "AUC": np.mean(resx[a]["scores"]), "delta so với base": c["delta"], "delta_se": c["delta_se"], "fold dương": c["fold_duong"], "verdict": c["verdict"]})
    display(pd.DataFrame(rows).set_index("cánh tay").round(4))
    if HAS_CAT:
        for a in arms_x:
            resb[a].update({"name": f"blend:{a}", "test_pred": np.zeros(len(Xte)), "imp": pd.Series(dtype=float)})
        rows_b = [{"cánh tay": "BLEND base (LGB 1 seed + CatTune)", "AUC": np.mean(resb["base"]["scores"]), "delta so với base": 0.0}]
        for a in ("exact", "random"):
            cb = compare(resb[a], resb["base"])
            rows_b.append({"cánh tay": f"BLEND {a}", "AUC": np.mean(resb[a]["scores"]), "delta so với base": cb["delta"], "delta_se": cb["delta_se"], "fold dương": cb["fold_duong"], "verdict": cb["verdict"]})
        rows_b.append({"cánh tay": "FINAL đã huấn luyện (LGB 3 seed + CatTune), chỉ để tham chiếu", "AUC": np.mean(ref_final_scores), "delta so với base": np.mean(ref_final_scores) - np.mean(resb["base"]["scores"])})
        display(pd.DataFrame(rows_b).set_index("cánh tay").round(4))
        cbx = compare(resb["exact"], resb["random"])
        print(f"BLEND: exact so với random: delta = {cbx['delta']:+.4f} (se {cbx['delta_se']:.4f}, fold dương {cbx['fold_duong']}) -> {cbx['verdict']}")
    c_xr = compare(resx["exact"], resx["random"])
    c_xb = compare(resx["exact"], resx["base"])
    print(f"exact so với random (cùng mẫu, nhãn xáo trộn): delta = {c_xr['delta']:+.4f} (se {c_xr['delta_se']:.4f}, fold dương {c_xr['fold_duong']}) -> {c_xr['verdict']}")
    print("Cách đọc: 'AUC V sau' cao hơn 'trước' là điều hiển nhiên vì chọn lô theo chính V. Cái đáng tin là 'AUC E' (hồ sơ CV không dùng để chọn).")
    if c_xb["verdict"] == "tốt hơn" and c_xr["verdict"] == "tốt hơn":
        print("exact hơn base và hơn random: có tín hiệu. Cần chạy lại với seed khác (đổi SEED+99 trong ô này) để xác nhận trước khi nghĩ tới sản xuất.")
    else:
        print("exact không hơn đồng thời base và random: việc chọn nhãn theo V không thêm thông tin. Không đưa vào FINAL.")
    pd.DataFrame(logx).to_csv("6z_exact_batches.csv", index=False)
    print(f"6z-exact xong sau {(time.time() - t0x) / 60:.1f} phút; nhật ký từng lô lưu ở 6z_exact_batches.csv")


## 6aa. Thêm mô hình khác loại vào blend

Blend nền (cố định): LGB(6i) 0.6 + CatTune 0.4, dựng lại ở ô 6i hoặc nạp từ cache. Mô hình ứng viên huấn luyện trên **đúng các fold đó** với cùng feature: XGBoost, LightGBM kiểu khác (`extra_trees`, cây nông), ExtraTrees, hồi quy logistic trên feature chuẩn hóa phân vị.

**Cách đo trung thực:** trọng số của mỗi ứng viên được chọn trên 4 fold rồi chấm trên fold còn lại (nested), nên không tự thưởng cho trọng số tối ưu trên chính OOF. Chỉ khi blend nested hơn blend nền theo quy tắc `compare` mới dùng, và test_pred lấy trọng số chọn trên toàn bộ OOF.

In [29]:
RUN_6AA = False
EXTRA_MODELS = ["xgb", "lgbx", "et", "lr"]       # bỏ bớt tên để chạy nhanh hơn
W_GRID = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4]    # trọng số logit thêm vào blend nền
if RUN_6AA and "res_cat_tuned" in globals() and "LGB_BEST" in globals() and FINAL.get("oof") is not None:
    from sklearn.ensemble import ExtraTreesClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import QuantileTransformer
    try:
        import xgboost as xgb
    except ImportError:
        xgb = None
        print("Không có xgboost: bỏ ứng viên xgb")
    t0a = time.time()
    cfg_a = dict(LGB_BEST["cfg"])
    cfg_a["params"] = dict(cfg_a.get("params", {}))
    lgp = {**PARAMS, **cfg_a["params"]}
    _lg = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))

    def _fit_xgb(Xt, yt, Xv, yv, Xte_):
        m = xgb.XGBClassifier(n_estimators=150 if DEBUG_FAST else 6000, learning_rate=0.3 if DEBUG_FAST else 0.03, max_depth=4, min_child_weight=20,
                              subsample=0.8, colsample_bytree=0.4, reg_lambda=10, reg_alpha=1, tree_method="hist", eval_metric="auc",
                              early_stopping_rounds=ES_FINAL, n_jobs=-1, random_state=SEED)
        m.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)
        return m.predict_proba(Xv)[:, 1], m.predict_proba(Xte_)[:, 1]

    def _fit_lgbx(Xt, yt, Xv, yv, Xte_):
        prm = {**lgp, "extra_trees": True, "num_leaves": 31, "max_depth": 6, "min_child_samples": 200, "colsample_bytree": 0.3,
               "learning_rate": 0.3 if DEBUG_FAST else 0.03, "random_state": SEED + 5}
        prm.pop("monotone_constraints", None)
        m = lgb.LGBMClassifier(**prm)
        m.fit(Xt, yt, eval_set=[(Xv, yv)], eval_metric="auc", callbacks=[lgb.early_stopping(ES_FINAL, verbose=False), lgb.log_evaluation(0)])
        return m.predict_proba(Xv)[:, 1], m.predict_proba(Xte_)[:, 1]

    def _prep_fill(Xt, Xv, Xte_):
        med = Xt.median(numeric_only=True)
        return [d.fillna(med).fillna(0).astype("float32") for d in (Xt, Xv, Xte_)]

    def _fit_et(Xt, yt, Xv, yv, Xte_):
        a, b_, c_ = _prep_fill(Xt, Xv, Xte_)
        m = ExtraTreesClassifier(n_estimators=60 if DEBUG_FAST else 400, min_samples_leaf=60, max_features=0.3, n_jobs=-1, random_state=SEED)
        m.fit(a, yt)
        return m.predict_proba(b_)[:, 1], m.predict_proba(c_)[:, 1]

    def _fit_lr(Xt, yt, Xv, yv, Xte_):
        a, b_, c_ = _prep_fill(Xt, Xv, Xte_)
        keep = a.columns[a.nunique() > 1]
        qt = QuantileTransformer(n_quantiles=200, output_distribution="normal", subsample=200000, random_state=SEED)
        a2, b2, c2 = qt.fit_transform(a[keep]), qt.transform(b_[keep]), qt.transform(c_[keep])
        m = LogisticRegression(C=0.02, max_iter=300)
        m.fit(a2, yt)
        return m.predict_proba(b2)[:, 1], m.predict_proba(c2)[:, 1]

    FITS = {"xgb": _fit_xgb, "lgbx": _fit_lgbx, "et": _fit_et, "lr": _fit_lr}
    if xgb is None:
        EXTRA_MODELS = [m for m in EXTRA_MODELS if m != "xgb"]
    FOLDS_A = [fold_data3(cfg_a, tr, va) for tr, va in SPLITS]      # tính 1 lần, dùng cho mọi mô hình
    EX = {}
    for nm in EXTRA_MODELS:
        sig = f"extra|{nm}|{len(FOLDS_A[0][3])}|{DEBUG_FAST}"
        r = _replay_cache_load(f"replay_extra_{nm}.npz", sig)
        if r is None:
            t1 = time.time()
            oof, tp = np.zeros(len(y)), np.zeros(len(X_test_all))
            for (tr, va), (Xt, Xv, Xte_, f) in zip(SPLITS, FOLDS_A):
                po, pt = FITS[nm](Xt, y[tr], Xv, y[va], Xte_)
                oof[va], tp = po, tp + pt / N_FOLDS
            sc = [roc_auc_score(y[va], oof[va]) for _, va in SPLITS]
            r = {"oof": oof, "test_pred": tp, "scores": sc, "name": nm}
            _replay_cache_save(f"replay_extra_{nm}.npz", r, sig)
            print(f"[{nm}] AUC = {np.mean(sc):.4f} ± {np.std(sc):.4f} | {time.time() - t1:.0f}s")
        else:
            print(f"[{nm}] nạp từ cache | AUC = {np.mean(r['scores']):.4f}")
        EX[nm] = r
    zb = _lg(FINAL["oof"])
    Z = {nm: _lg(r["oof"]) for nm, r in EX.items()}
    zt_b = _lg(FINAL["test_pred"])
    Zt = {nm: _lg(r["test_pred"]) for nm, r in EX.items()}
    folds_idx = [va for _, va in SPLITS]

    def _mix(ws, idx, zs=Z, zb_=zb):
        s = (1 - sum(ws.values())) * zb_[idx]
        for k, w in ws.items():
            s = s + w * zs[k][idx]
        return s

    def _fit_w(idx):
        # coordinate ascent trên AUC của các fold trong idx; mọi trọng số >= 0 và tổng < 1
        ws = {k: 0.0 for k in EX}
        sc = lambda w_: roc_auc_score(y[idx], _mix(w_, idx))
        best = sc(ws)
        for _ in range(2):
            for k in EX:
                for g in W_GRID:
                    t = {**ws, k: g}
                    if sum(t.values()) >= 0.95:
                        continue
                    s_ = sc(t)
                    if s_ > best + 1e-7:
                        best, ws = s_, t
        return ws

    def _nested(keys):
        out = {"oof": np.zeros(len(y)), "scores": [], "name": "nested:" + "+".join(keys)}
        for k, vk in enumerate(folds_idx):
            idx_tr = np.concatenate([v for j, v in enumerate(folds_idx) if j != k])
            Zs = {kk: Z[kk] for kk in keys}
            def mix_(ws, idx):
                s = (1 - sum(ws.values())) * zb[idx]
                for kk, w in ws.items():
                    s = s + w * Zs[kk][idx]
                return s
            ws = {kk: 0.0 for kk in keys}
            best = roc_auc_score(y[idx_tr], mix_(ws, idx_tr))
            for _ in range(2):
                for kk in keys:
                    for g in W_GRID:
                        t = {**ws, kk: g}
                        if sum(t.values()) >= 0.95:
                            continue
                        s_ = roc_auc_score(y[idx_tr], mix_(t, idx_tr))
                        if s_ > best + 1e-7:
                            best, ws = s_, t
            out["oof"][vk] = expit(mix_(ws, vk))
            out["scores"].append(roc_auc_score(y[vk], out["oof"][vk]))
        out.update({"test_pred": np.zeros(len(X_test_all)), "imp": pd.Series(dtype=float)})
        return out

    FINAL.setdefault("imp", pd.Series(dtype=float))
    rows = [{"ứng viên": "blend nền (LGB 0.6 + Cat 0.4)", "AUC OOF riêng": np.nan, "AUC blend nested": np.mean(FINAL["scores"]), "delta": 0.0}]
    for nm in EX:
        rn = _nested([nm])
        c = compare(rn, FINAL)
        w_all = _fit_w(np.arange(len(y)))
        rows.append({"ứng viên": f"+ {nm}", "AUC OOF riêng": np.mean(EX[nm]["scores"]), "AUC blend nested": np.mean(rn["scores"]), "delta": c["delta"],
                     "delta_se": c["delta_se"], "fold dương": c["fold_duong"], "verdict": c["verdict"]})
    rn_all = _nested(list(EX))
    c_all = compare(rn_all, FINAL)
    rows.append({"ứng viên": "+ tất cả (trọng số chọn nested)", "AUC OOF riêng": np.nan, "AUC blend nested": np.mean(rn_all["scores"]), "delta": c_all["delta"],
                 "delta_se": c_all["delta_se"], "fold dương": c_all["fold_duong"], "verdict": c_all["verdict"]})
    display(pd.DataFrame(rows).set_index("ứng viên").round(4))
    W_ALL = _fit_w(np.arange(len(y)))
    print("Trọng số chọn trên toàn bộ OOF (chỉ dùng nếu qua cổng):", {k: round(v, 3) for k, v in W_ALL.items()}, "| phần blend nền:", round(1 - sum(W_ALL.values()), 3))
    corr = pd.DataFrame({"nền": zb, **Z}).corr().round(3)
    print("Tương quan logit OOF với blend nền:"); display(corr["nền"])
    if c_all["verdict"] == "tốt hơn":
        tp_new = expit((1 - sum(W_ALL.values())) * zt_b + sum(W_ALL[k] * Zt[k] for k in EX))
        np.save("blend_extra_test_pred.npy", tp_new)
        print("QUA cổng: lưu blend_extra_test_pred.npy (cần xác nhận trên seed/fold khác trước khi nộp).")
    else:
        print("KHÔNG qua cổng nested: giữ blend nền, không thêm mô hình.")
    print(f"6aa xong sau {(time.time() - t0a) / 60:.1f} phút")


## 6s. Learning rate thấp hơn và huấn luyện lại trên 100% dữ liệu

**Learning rate.** LightGBM cuối đã chạy lr 0.02 (chỉ bước sàng lọc dùng 0.05), CatBoost lr 0.04. Thử LightGBM lr 0.01 và CatBoost lr 0.02, cùng 3 seed, rồi qua `compare` + `consider_lgb`/`try_blends` như mọi khối khác. Mất khoảng 1/2 giờ cho LightGBM và 2 giờ cho CatBoost.

**Huấn luyện lại 100%.** Mỗi mô hình CV chỉ thấy 80% dữ liệu và `test_pred` là trung bình 5 mô hình đó. Huấn luyện lại một mô hình trên toàn bộ train với số cây = (số cây trung bình của CV) × `REFIT_SCALE` thường tốt hơn. CV không đo được điều này (không còn dữ liệu giữ lại), nên có hai bước:
1. **Kiểm tra trên tập giữ riêng.** Giữ 20% dòng (H). Trên 80% còn lại (D): so (A) trung bình 4 mô hình CV, mỗi mô hình thấy 75% của D, với (B) một mô hình huấn luyện lại trên 100% của D, số cây × {1.0, 1.1, 1.25}. Đo AUC trên H, kèm sai số bootstrap. Tỉ lệ dữ liệu (1.33) gần với thực tế (1.25).
2. **Áp dụng** vào `FINAL["test_pred"]` chỉ khi B(`REFIT_SCALE`) không kém A hơn một sai số. Trọng số blend được dò lại từ `oof` của `FINAL` (phải khớp tới 1e-6, nếu không thì bỏ qua, không đoán). `oof` và điểm CV không đổi, chỉ `test_pred` đổi, nên điểm CV in ra vẫn là điểm của các mô hình 80%.

**Kiểm tra an toàn sau khi huấn luyện lại:** tương quan với `test_pred` cũ phải ≥ 0.98 và trung bình dự đoán lệch dưới 10%.

In [30]:
RUN_6S = True
RUN_LR_LGB, LGB_LR2 = False, 0.01   # đã chạy: lr 0.01 không khác lr 0.02
RUN_LR_CAT, CAT_LR2 = False, 0.02   # đã chạy: lr 0.02 không khác lr 0.04 (mất ~80 phút)
RUN_REFIT_CHECK = False    # đã chạy: B - A = -0.0002 (sai số 0.0006), không phản đối; đặt True để chạy lại
RUN_REFIT = True           # áp dụng huấn luyện lại 100% cho bản nộp (nếu qua kiểm tra)
REFIT_SCALE = 1.1          # số cây = trung bình CV × hệ số này (dữ liệu tăng 25%)

if RUN_6S and "consider_lgb" in globals():
    t0 = time.time()
    _lg6 = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
    _best_cfg = lambda: dict((globals().get("LGB_BEST") or {}).get("cfg") or _pick_best_cfg())

    def _norm_cfg(c):
        c = dict(c)
        for k in ("te_resid", "te_hier", "te_resid2", "twin"):
            c.setdefault(k, [])
        c["params"] = dict(c.get("params", {}))
        return c

    # ---------- (1) LightGBM, learning rate thấp hơn ----------
    if RUN_LR_LGB:
        cfg_l = _norm_cfg(_best_cfg())
        base_l = {**PARAMS, "learning_rate": LGB_LR2, "n_estimators": 30000}
        res_l = run_cfg3(cfg_l, base_l, seeds=FINAL_SEEDS, es=max(ES_FINAL, 400), name=f"LightGBM lr {LGB_LR2} (3 seed)")
        consider_lgb(res_l, f"LGB(lr{LGB_LR2})")

    # ---------- (2) CatBoost, learning rate thấp hơn ----------
    if RUN_LR_CAT and "run_cat" in globals() and "res_cat_tuned" in globals():
        res_c2 = run_cat(cur_over, CAT_LR2, CAT_SEEDS, f"CatBoost lr {CAT_LR2}, {len(CAT_SEEDS)} seed")
        cc = compare(res_c2, res_cat_tuned)
        print(f"So với CatBoost lr {res_cat_tuned.get('lr', 0.04)}: delta = {cc['delta']:+.4f} (se {cc['delta_se']:.4f}, "
              f"fold dương {cc['fold_duong']}) -> {cc['verdict']}")
        if cc["verdict"] == "tốt hơn" and _ll(res_c2) <= _ll(res_cat_tuned) + 1e-5:
            res_cat_tuned = res_c2
            res_catn = res_c2
            try_blends(globals().get("LGB_BEST", res_lgb), res_cat_tuned, "LGB + CatLR2")

    # ---------- (3) kiểm tra huấn luyện lại 100% trên tập giữ riêng ----------
    REFIT_OK = True
    if RUN_REFIT_CHECK:
        from sklearn.model_selection import StratifiedKFold as _SKF
        cfgc = _norm_cfg(_best_cfg())
        D, H = next(iter(_SKF(5, shuffle=True, random_state=SEED + 1212).split(np.zeros(n_tr), y)))
        inner = list(_SKF(4, shuffle=True, random_state=SEED + 1213).split(D, y[D]))
        prm_c = {**FAST_PARAMS, **cfgc["params"]}
        X_test_bak = X_test_all
        X_test_all = X_all.iloc[H].reset_index(drop=True)      # fold_data3 mã hóa "test" = H bằng đúng tập huấn luyện của từng mô hình
        try:
            pa, ns, singles = np.zeros(len(H)), [], []
            for tri, vai in inner:
                tr_, va_ = D[tri], D[vai]
                Xtr, Xva, XH, f = fold_data3(cfgc, tr_, va_)
                m = lgb.LGBMClassifier(**{**prm_c, "random_state": SEED})
                m.fit(Xtr, y[tr_], eval_set=[(Xva, y[va_])], eval_metric="auc",
                      callbacks=[lgb.early_stopping(ES_FAST, verbose=False), lgb.log_evaluation(0)])
                p = m.predict_proba(XH)[:, 1]
                pa += p / len(inner)
                singles.append(roc_auc_score(y[H], p))
                ns.append(m.best_iteration_)
            n0 = float(np.mean(ns))
            Xtr, _, XH, f = fold_data3(cfgc, D, D[:2000])
            pb = {}
            for sc_ in (1.0, 1.1, 1.25):
                m = lgb.LGBMClassifier(**{**prm_c, "n_estimators": max(10, int(n0 * sc_)), "random_state": SEED})
                m.fit(Xtr, y[D])
                pb[sc_] = m.predict_proba(XH)[:, 1]
        finally:
            X_test_all = X_test_bak
        yH = y[H]
        auc_a = roc_auc_score(yH, pa)
        rs = np.random.RandomState(SEED)
        pk = pb[min(pb, key=lambda k: abs(k - REFIT_SCALE))]
        dif = []
        for _ in range(200):
            ix = rs.randint(0, len(H), len(H))
            if yH[ix].min() == yH[ix].max():
                continue
            dif.append(roc_auc_score(yH[ix], pk[ix]) - roc_auc_score(yH[ix], pa[ix]))
        d_ab, d_se = roc_auc_score(yH, pk) - auc_a, float(np.std(dif))
        print(f"Tập giữ riêng H: {len(H)} dòng, {int(yH.sum())} ca dương | số cây CV trung bình {n0:.0f}")
        print(f"  mô hình đơn (75% của D): AUC {np.mean(singles):.4f} | (A) trung bình 4 mô hình: {auc_a:.4f}")
        print("  (B) huấn luyện lại 100% của D: " + ", ".join(f"cây x{k}: {roc_auc_score(yH, v):.4f}" for k, v in pb.items()))
        print(f"  B(x{REFIT_SCALE}) - A = {d_ab:+.4f} (sai số bootstrap {d_se:.4f})")
        REFIT_OK = d_ab > -d_se
        print("  =>", "huấn luyện lại không kém trung bình fold: dùng được" if REFIT_OK else "huấn luyện lại kém hơn: KHÔNG áp dụng")

    # ---------- (4) áp dụng cho bản nộp ----------
    if RUN_REFIT and REFIT_OK:
        lgb_res = globals().get("LGB_BEST")
        cat_res = globals().get("res_cat_tuned")
        w_use = None
        if lgb_res is not None and cat_res is not None:
            for w in np.round(np.arange(0.0, 1.0001, 0.05), 2):
                ob = expit(w * _lg6(lgb_res["oof"]) + (1 - w) * _lg6(cat_res["oof"]))
                if np.abs(ob - FINAL["oof"]).max() < 1e-6:
                    w_use = float(w)
                    break
        if w_use is None:
            print("Không dò được FINAL = blend của LGB_BEST và CatBoost hiện tại (oof không khớp 1e-6): giữ test_pred cũ, không đoán.")
        elif lgb_res.get("mode"):
            print(f"LightGBM tốt nhất dùng kiểu huấn luyện đặc biệt ({lgb_res['mode']}): huấn luyện lại 100% theo cách thường sẽ không tái tạo nó, giữ test_pred cũ.")
        elif "iters" not in lgb_res or ("iters" not in cat_res and w_use < 1.0):
            print("Kết quả thiếu số cây (chạy từ bản cũ): giữ test_pred cũ.")
        else:
            def refit_lgb(res):
                cfg_, base_, its_, sd_ = res["cfg"], res["base"], res["iters"], res["seeds"]
                allx = np.arange(n_tr)
                Xtr, _, Xte, f = fold_data3(cfg_, allx, allx[:2000])
                prm0 = {**base_, **cfg_["params"]}
                if cfg_.get("mono"):
                    prm0["monotone_constraints"] = [-1 if c in MONO else 0 for c in f]
                tp = np.zeros(len(Xte))
                for si, s in enumerate(sd_):
                    n = max(10, int(np.mean(its_[si::len(sd_)]) * REFIT_SCALE))
                    m = lgb.LGBMClassifier(**{**prm0, "n_estimators": n, "random_state": s})
                    m.fit(Xtr, y)
                    tp += m.predict_proba(Xte)[:, 1] / len(sd_)
                    print(f"  [refit LGB] seed {s}: {n} cây | {time.time() - t0:.0f}s")
                return tp

            def refit_cat(res):
                from catboost import CatBoostClassifier as _CB
                A_ = X_all[feats_c].copy()
                A_[CAT_SRC] = A_[CAT_SRC].astype("int32")
                tp = np.zeros(len(Xte_c))
                for si, s in enumerate(res["seeds"]):
                    n = max(10, int(np.mean(res["iters"][si::len(res["seeds"])]) * REFIT_SCALE))
                    m = _CB(iterations=n, learning_rate=0.15 if DEBUG_FAST else res["lr"], loss_function="Logloss", random_seed=s,
                            thread_count=-1, verbose=False, allow_writing_files=False, cat_features=CAT_SRC,
                            **{**BASE_C, **res["over"]})
                    m.fit(A_, y)
                    tp += m.predict_proba(Xte_c)[:, 1] / len(res["seeds"])
                    print(f"  [refit Cat] seed {s}: {n} cây | {time.time() - t0:.0f}s")
                return tp

            tp_l = refit_lgb(lgb_res)
            tp_c = refit_cat(cat_res) if w_use < 1.0 else None
            new_tp = expit(w_use * _lg6(tp_l) + (1 - w_use) * _lg6(tp_c)) if tp_c is not None else tp_l
            old_tp = FINAL["test_pred"]
            cor = float(np.corrcoef(_lg6(new_tp), _lg6(old_tp))[0, 1])
            mr = float(new_tp.mean() / old_tp.mean())
            print(f"Kiểm tra an toàn: tương quan logit với test_pred cũ = {cor:.4f} | tỉ lệ trung bình = {mr:.3f}")
            if cor >= 0.98 and abs(mr - 1) < 0.10:
                FINAL = {**FINAL, "test_pred_cv": old_tp, "test_pred": new_tp, "name": FINAL["name"] + " [test_pred: huấn luyện lại 100%]"}
                print("ĐÃ ÁP DỤNG: bản nộp dùng mô hình huấn luyện lại 100%. Điểm CV bên dưới là của mô hình 80% (không đo được bản 100%).")
            else:
                print("Không qua kiểm tra an toàn: giữ test_pred cũ.")
    print(f"\n6s xong sau {time.time() - t0:.0f}s")
elif RUN_6S:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6s.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")


  [refit LGB] seed 42: 1454 cây | 38s
  [refit LGB] seed 7: 1552 cây | 76s
  [refit LGB] seed 2024: 1438 cây | 111s
  [refit LGB] seed 11: 1549 cây | 147s
  [refit LGB] seed 99: 1618 cây | 187s
  [refit Cat] seed 42: 2036 cây | 507s
  [refit Cat] seed 43: 1842 cây | 799s
  [refit Cat] seed 44: 2091 cây | 1130s
  [refit Cat] seed 45: 1923 cây | 1433s
  [refit Cat] seed 46: 2201 cây | 1781s
Kiểm tra an toàn: tương quan logit với test_pred cũ = 0.9990 | tỉ lệ trung bình = 1.004
ĐÃ ÁP DỤNG: bản nộp dùng mô hình huấn luyện lại 100%. Điểm CV bên dưới là của mô hình 80% (không đo được bản 100%).

6s xong sau 1781s

>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) [test_pred: huấn luyện lại 100%] | AUC CV = 0.7682


## 6x. TabM mở rộng (bỏ TabPFN): chọn cấu hình bằng lưới nhỏ, nhiều seed, rồi blend

**Chạy trên GPU Kaggle** (Settings → Accelerator → GPU). Không cần Internet. TabPFN đã bỏ hẳn.

**Khác với 6w:**
| | 6w (kết quả cũ, AUC CV 0.7640) | 6x |
|---|---|---|
| Số đầu `k` | 16 | **32** |
| Bề rộng | 256 | **512** (chọn trong lưới) |
| Dropout / weight decay | 0.1 / 1e-4 | **chọn trong lưới** |
| Số seed | 2 | **5** |
| Epoch tối đa / kiên nhẫn | 40 / 6 | **60 / 8** |

**Lưới cấu hình.** `TABM_GRID` có 5 cấu hình (rộng 256/512/1024, dropout 0.1 đến 0.3, weight decay 1e-4 đến 1e-2). Mỗi cấu hình huấn luyện 1 seed trên `TABM_SELECT_FOLDS` (mặc định 2 fold đầu) và được chọn bằng **AUC của chính TabM**, không dùng điểm blend. Cấu hình thắng chạy đủ 5 fold × 5 seed.

**Cách đọc kết quả cho trung thực.**
- Hai fold dùng để chọn cấu hình hơi lạc quan (chọn trên chính chúng). Ô in thêm **AUC trên các fold không dùng để chọn** để so với LightGBM cùng các fold đó.
- Dừng sớm theo AUC trên fold validation (giống LightGBM), nên mọi mô hình trong notebook đều hơi lạc quan như nhau; so sánh tương đối vẫn công bằng.
- Blend chọn trọng số trong 5 lựa chọn trên cùng fold, nên cải thiện dưới vài `1e-4` coi là chưa chắc. Trước đó TabM 6w chỉ thêm +0.0001.

**Thời gian.** Ô đo thời gian một lần huấn luyện trong lưới và tự giảm số seed nếu ước tính vượt `TABM_MINUTES` (mặc định 150 phút). Chưa đo trên GPU thật, nên con số thời gian chỉ là ước tính của ô, không phải cam kết.

**Dữ liệu lưu.** Dự đoán lưu ở `/kaggle/working/tabm_v2_res.npz` kèm chữ ký cấu hình; đổi cấu hình, seed hoặc lưới thì tự huấn luyện lại. File cũ `tabm_res.npz` của 6w không bị nạp. Kết quả lưới lưu ở `tabm_v2_grid.json` để phiên bị ngắt không chạy lại lưới.

**Chưa chạy thử được trên torch** trong môi trường soạn thảo (không có torch); phần blend và lưu/nạp đã kiểm tra bằng thành viên giả. Nếu có lỗi, ô in lỗi và các bước sau vẫn chạy.


In [31]:
RUN_6W = False   # TabM đã có kết quả (blend +0.0001, nhiễu): bỏ để tiết kiệm thời gian
RUN_TABM = True
REUSE_SAVED = True            # nạp dự đoán đã lưu nếu chữ ký cấu hình khớp
# ---- TabM-mini mở rộng ----
TABM_K, TABM_BLOCKS = 32, 3
TABM_BINS, USE_PLE = 8, True
TABM_EPOCHS, TABM_PATIENCE, TABM_BS, TABM_LR = 60, 8, 1024, 1e-3
TABM_SEEDS = (42, 7, 2024, 11, 99)
TABM_GRID = [                  # chọn bằng AUC của chính TabM trên TABM_SELECT_FOLDS
    dict(width=512, drop=0.2, wd=1e-4),
    dict(width=512, drop=0.3, wd=1e-3),
    dict(width=256, drop=0.1, wd=1e-4),
    dict(width=1024, drop=0.3, wd=1e-3),
    dict(width=512, drop=0.2, wd=1e-2),
]
TABM_SELECT_FOLDS = (0, 1)
TABM_MINUTES = 150             # ngân sách thời gian cho bước huấn luyện chính; tự giảm số seed nếu ước tính vượt
if DEBUG_FAST:
    TABM_EPOCHS, TABM_SEEDS, TABM_GRID, TABM_SELECT_FOLDS = 2, (42,), TABM_GRID[:2], (0,)

if RUN_6W and "consider_lgb" in globals():
    import sys, subprocess, traceback
    from sklearn.linear_model import LogisticRegression as _LR
    t0 = time.time()
    cfg_w = dict((globals().get("LGB_BEST") or {}).get("cfg") or globals().get("cfg_best") or _pick_best_cfg())
    for k in ("te_resid", "te_hier", "te_resid2", "twin"):
        cfg_w.setdefault(k, [])
    cfg_w["params"] = dict(cfg_w.get("params", {}))
    members = {}

    def _save(path, res):
        np.savez(path, oof=res["oof"], test_pred=res["test_pred"], scores=np.array(res["scores"]), n_tr=n_tr, seed=SEED)

    def _load(path, name):
        if not (REUSE_SAVED and os.path.exists(path)):
            return None
        z = np.load(path)
        if int(z["n_tr"]) != n_tr or int(z["seed"]) != SEED:
            return None
        print(f"[{name}] nạp dự đoán đã lưu: {path}")
        return {"oof": z["oof"], "test_pred": z["test_pred"], "scores": list(z["scores"]), "name": name}

    # =========================== TabM-mini (mở rộng) ===========================
    tabm_info = {}
    if RUN_TABM:
        import json
        sig = (f"v2|K{TABM_K}|B{TABM_BLOCKS}|bins{TABM_BINS}|ple{int(USE_PLE)}|ep{TABM_EPOCHS}|pat{TABM_PATIENCE}|bs{TABM_BS}|lr{TABM_LR}|"
               f"seeds{TABM_SEEDS}|grid{[tuple(g.values()) for g in TABM_GRID]}|sel{TABM_SELECT_FOLDS}|cfg{sorted(cfg_w.get('params', {}).items())[:6]}")
        res_path, grid_path = f"{OUT_PATH}/tabm_v2_res.npz", f"{OUT_PATH}/tabm_v2_grid.json"
        r_ = None
        if REUSE_SAVED and os.path.exists(res_path):
            z = np.load(res_path, allow_pickle=False)
            if int(z["n_tr"]) == n_tr and int(z["seed"]) == SEED and str(z["sig"]) == sig:
                r_ = {"oof": z["oof"], "test_pred": z["test_pred"], "scores": list(z["scores"]), "name": "TabM-mini"}
                tabm_info = json.loads(str(z["info"]))
                print(f"[TabM-mini] nạp dự đoán đã lưu: {res_path} | cấu hình {tabm_info.get('best_cfg')}")
        if r_ is None:
            try:
                import torch, torch.nn as nn, torch.nn.functional as Fnn
                from sklearn.preprocessing import QuantileTransformer
                dev = "cuda" if torch.cuda.is_available() else "cpu"
                K_, EP_, SD_, GRID_ = TABM_K, TABM_EPOCHS, TABM_SEEDS, TABM_GRID
                if dev == "cpu" and not DEBUG_FAST:
                    K_, EP_, SD_, GRID_ = 8, 12, (SEED,), [dict(width=128, drop=0.1, wd=1e-4)]
                    print("Không có GPU: thu nhỏ TabM (k=8, rộng 128, 12 epoch, 1 seed, không chạy lưới). Bật GPU trong Settings để chạy đủ.")
                use_amp = dev == "cuda"

                class TabMMini(nn.Module):
                    def __init__(self, d_in, k, width, n_blocks, drop):
                        super().__init__()
                        self.k = k
                        self.r = nn.Parameter(torch.where(torch.rand(k, d_in) < 0.5, -1.0, 1.0))   # adapter nhân, khởi tạo dấu ngẫu nhiên
                        layers, d = [], d_in
                        for _ in range(n_blocks):
                            layers += [nn.Linear(d, width), nn.ReLU(), nn.Dropout(drop)]
                            d = width
                        self.body = nn.Sequential(*layers)
                        self.head = nn.Linear(width, 1)

                    def forward(self, x):                                   # x: (B, d)
                        h = self.body(x[:, None, :] * self.r[None])           # (B, k, width): k đầu dùng chung trọng số
                        return self.head(h).squeeze(-1)                       # (B, k) logit từng đầu

                def tabm_prep(Xtr, Xva, Xte):
                    M = [m.to_numpy(dtype="float32", copy=True) for m in (Xtr, Xva, Xte)]
                    for m in M:
                        m[~np.isfinite(m)] = np.nan
                    nan_cols = np.flatnonzero(np.isnan(M[0]).any(0))
                    med = np.nanmedian(M[0], axis=0)
                    med = np.where(np.isnan(med), 0.0, med).astype("float32")
                    ind = [np.isnan(m[:, nan_cols]).astype("float32") for m in M]
                    M = [np.where(np.isnan(m), med[None, :], m) for m in M]
                    qt = QuantileTransformer(n_quantiles=min(1000, len(M[0])), output_distribution="normal", subsample=200_000, random_state=0).fit(M[0])
                    out = [[qt.transform(m).astype("float32"), i] for m, i in zip(M, ind)]
                    if USE_PLE:
                        ple = [[], [], []]
                        for j in range(M[0].shape[1]):
                            col = M[0][:, j]
                            if len(np.unique(col[:5000])) <= 2:
                                continue
                            e = np.unique(np.quantile(col, np.linspace(0, 1, TABM_BINS + 1)))
                            if len(e) < 3:
                                continue
                            wd = np.diff(e)[None, :]
                            for t in range(3):
                                ple[t].append(np.clip((M[t][:, j, None] - e[None, :-1]) / wd, 0, 1).astype("float32"))
                        for t in range(3):
                            if ple[t]:
                                out[t].append(np.hstack(ple[t]))
                    return [np.hstack(o).astype("float32") for o in out]

                def tabm_predict(model, X):
                    model.eval()
                    outs = []
                    with torch.no_grad():
                        for i in range(0, len(X), 4096):
                            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                                o = model(X[i:i + 4096])
                            outs.append(torch.sigmoid(o.float()).mean(1).cpu().numpy())
                    return np.concatenate(outs)

                def tabm_fit(Atr, ytr, Ava, yva, Ate, seed, g):
                    torch.manual_seed(seed)
                    model = TabMMini(Atr.shape[1], K_, g["width"], TABM_BLOCKS, g["drop"]).to(dev)
                    Xt, yt = torch.from_numpy(Atr).to(dev), torch.from_numpy(ytr.astype("float32")).to(dev)
                    Xv, Xe = torch.from_numpy(Ava).to(dev), torch.from_numpy(Ate).to(dev)
                    opt = torch.optim.AdamW(model.parameters(), lr=g.get("lr", TABM_LR), weight_decay=g["wd"])
                    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2)
                    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
                    best, bad, best_state, best_ep = -1.0, 0, None, 0
                    for ep in range(EP_):
                        model.train()
                        perm = torch.randperm(len(Xt), device=dev)
                        for i in range(0, len(Xt), TABM_BS):
                            idx = perm[i:i + TABM_BS]
                            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                                out = model(Xt[idx])
                            loss = Fnn.binary_cross_entropy_with_logits(out.float(), yt[idx][:, None].expand(-1, model.k))
                            opt.zero_grad(set_to_none=True)
                            scaler.scale(loss).backward()
                            scaler.step(opt)
                            scaler.update()
                        auc = roc_auc_score(yva, tabm_predict(model, Xv))
                        sched.step(auc)
                        if auc > best + 1e-5:
                            best, bad, best_ep = auc, 0, ep
                            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                        else:
                            bad += 1
                            if bad >= TABM_PATIENCE:
                                break
                    model.load_state_dict(best_state)
                    pv, pe = tabm_predict(model, Xv), tabm_predict(model, Xe)
                    del model, Xt, yt, Xv, Xe, opt
                    return pv, pe, best_ep, best

                fold_cache = {}

                def fold_arrays(fi):
                    if fi not in fold_cache:
                        tr_, va_ = SPLITS[fi]
                        Xtr_, Xva_, Xte_, _f = fold_data3(cfg_w, tr_, va_)
                        fold_cache[fi] = tabm_prep(Xtr_, Xva_, Xte_)
                    return fold_cache[fi]

                # ---------- bước 1: lưới nhỏ, chọn bằng AUC của chính TabM ----------
                sel = [fi for fi in TABM_SELECT_FOLDS if fi < len(SPLITS)]
                t_fit = None
                if len(GRID_) > 1:
                    grid_res = None
                    if REUSE_SAVED and os.path.exists(grid_path):
                        gj = json.load(open(grid_path))
                        if gj.get("sig") == sig:
                            grid_res = gj["res"]
                            print(f"[TabM] nạp kết quả lưới đã lưu: {grid_path}")
                    if grid_res is None:
                        grid_res, t_g = [], time.time()
                        for gi, g in enumerate(GRID_):
                            sc_g, ts_g = [], []
                            for fi in sel:
                                tr_, va_ = SPLITS[fi]
                                A_ = fold_arrays(fi)
                                t_s = time.time()
                                _pv, _pe, be_, bauc_ = tabm_fit(A_[0], y[tr_], A_[1], y[va_], A_[2], SEED, g)
                                sc_g.append(float(bauc_)); ts_g.append(time.time() - t_s)
                            grid_res.append({"cfg": g, "auc": float(np.mean(sc_g)), "scores": sc_g, "sec_per_fit": float(np.mean(ts_g))})
                            print(f"  [TabM lưới {gi + 1}/{len(GRID_)}] {g}: AUC val = {np.mean(sc_g):.4f} | {np.mean(ts_g):.0f}s/lần | tổng {time.time() - t_g:.0f}s")
                        json.dump({"sig": sig, "res": grid_res}, open(grid_path, "w"))
                    bi = int(np.argmax([r["auc"] for r in grid_res]))
                    best_g, t_fit = GRID_[bi], grid_res[bi].get("sec_per_fit")
                    display(pd.DataFrame([{**r["cfg"], "AUC val (lưới)": r["auc"], "giây/lần": r["sec_per_fit"]} for r in grid_res]).round(4))
                    print(f"=> chọn cấu hình {best_g} (AUC lưới {grid_res[bi]['auc']:.4f})")
                else:
                    best_g = GRID_[0]

                # ---------- bước 2: huấn luyện chính, mọi fold, nhiều seed ----------
                if t_fit:
                    n_max = int(TABM_MINUTES * 60 / (t_fit * 1.3 * len(SPLITS)))
                    if n_max < len(SD_):
                        SD_ = SD_[:max(1, n_max)]
                        print(f"Giảm số seed xuống {len(SD_)} để vừa ngân sách {TABM_MINUTES} phút (ước tính {t_fit:.0f}s/lần).")
                t1 = time.time()
                oof, tp, per_seed = np.zeros(len(y)), np.zeros(len(X_test_all)), []
                for fi, (tr_, va_) in enumerate(SPLITS):
                    A_ = fold_arrays(fi)
                    for s in SD_:
                        pv, pt, be, bauc = tabm_fit(A_[0], y[tr_], A_[1], y[va_], A_[2], s, best_g)
                        oof[va_] += pv / len(SD_)
                        tp += pt / (len(SD_) * len(SPLITS))
                        per_seed.append((fi, s, float(bauc)))
                        print(f"  [TabM] fold {fi + 1}/{len(SPLITS)} seed {s}: AUC val = {bauc:.4f} | epoch tốt nhất {be} | đầu vào {A_[0].shape[1]} chiều | {time.time() - t1:.0f}s")
                    fold_cache.pop(fi, None)
                    if dev == "cuda":
                        torch.cuda.empty_cache()
                scores = [roc_auc_score(y[va_], oof[va_]) for _, va_ in SPLITS]
                r_ = {"oof": oof, "test_pred": tp, "scores": scores, "name": "TabM-mini"}
                tabm_info = {"best_cfg": best_g, "seeds": list(SD_), "sel_folds": sel, "k": K_, "per_seed": per_seed}
                np.savez(res_path, oof=oof, test_pred=tp, scores=np.array(scores), n_tr=n_tr, seed=SEED, sig=np.array(sig), info=np.array(json.dumps(tabm_info)))
                print(f"[TabM-mini] AUC CV = {np.mean(scores):.4f} ± {np.std(scores):.4f} | {time.time() - t1:.0f}s | {len(SD_)} seed")
            except Exception as e:
                print("TabM lỗi, bỏ qua:", repr(e))
                traceback.print_exc()
                r_ = None
        if r_ is not None:
            members["TabM-mini"] = r_
            # so sánh trung thực: các fold KHÔNG dùng để chọn cấu hình
            ref0 = globals().get("LGB_BEST") or res_lgb
            sel_f = set(tabm_info.get("sel_folds", []))
            clean = [i for i in range(len(SPLITS)) if i not in sel_f]
            if clean and sel_f:
                print(f"Trên {len(clean)} fold không dùng để chọn cấu hình: TabM {np.mean([r_['scores'][i] for i in clean]):.4f} | "
                      f"LightGBM tốt nhất {np.mean([ref0['scores'][i] for i in clean]):.4f}")

    # =========================== đánh giá ===========================
    lg_ = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
    if not members:
        print("Không thành viên nào chạy được: giữ nguyên mô hình đang dùng.")
    else:
        ref = globals().get("LGB_BEST") or res_lgb
        rows = [{"thành viên": "LightGBM tốt nhất", "AUC CV": np.mean(ref["scores"]), "so với LGB": "-", "tương quan logit với FINAL": np.corrcoef(lg_(ref["oof"]), lg_(FINAL["oof"]))[0, 1]}]
        for nm, r in members.items():
            c = compare(r, ref)
            rows.append({"thành viên": nm, "AUC CV": np.mean(r["scores"]), "so với LGB": f"{c['delta']:+.4f} ({c['verdict']})",
                         "tương quan logit với FINAL": np.corrcoef(lg_(r["oof"]), lg_(FINAL["oof"]))[0, 1]})
        display(pd.DataFrame(rows).set_index("thành viên").round(4))

        def calibrated(r):
            z = lg_(r["oof"]).reshape(-1, 1)
            m = _LR(C=1e6).fit(z, y)
            a, b = float(m.coef_[0, 0]), float(m.intercept_[0])
            return {"oof": expit(a * z[:, 0] + b), "test_pred": expit(a * lg_(r["test_pred"]) + b)}

        cur, used = FINAL, []
        for nm in sorted(members, key=lambda k: -np.mean(members[k]["scores"])):
            mc = calibrated(members[nm])
            best, rowsb = None, []
            for w in (0.05, 0.10, 0.15, 0.20, 0.30):
                o = expit((1 - w) * lg_(cur["oof"]) + w * lg_(mc["oof"]))
                sc = [roc_auc_score(y[va], o[va]) for _, va in SPLITS]
                c = compare({"scores": sc, "oof": o}, cur)
                rowsb.append({"trọng số": w, "AUC": np.mean(sc), "delta": c["delta"], "delta_se": c["delta_se"], "fold dương": c["fold_duong"], "verdict": c["verdict"]})
                if c["verdict"] == "tốt hơn" and (best is None or c["delta"] > best[0]):
                    best = (c["delta"], w, o, sc)
            print(f"\nBlend {nm} vào mô hình hiện tại ({cur['name']}):")
            display(pd.DataFrame(rowsb).set_index("trọng số").round(4))
            if best is not None:
                _, w, o, sc = best
                cur = {**cur, "oof": o, "scores": sc, "test_pred": expit((1 - w) * lg_(cur["test_pred"]) + w * lg_(mc["test_pred"])),
                       "name": f"{cur['name']} + {nm} {w}"}
                used.append((nm, w))
                print(f"=> GIỮ {nm} với trọng số {w}")
            else:
                print(f"=> {nm} không qua quy tắc delta_se: không thêm vào blend.")
        if used:
            FINAL = cur
            print("\nFINAL mới:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}",
                  "(trọng số chọn trong 5 lựa chọn trên cùng fold: hơi lạc quan)")
    print(f"\n6x xong sau {time.time() - t0:.0f}s")
elif RUN_6W:
    print("Cần chạy ô 6k trước (nó định nghĩa các hàm dùng chung); bỏ qua 6w.")
print("\n>>> Mô hình dùng cho mục 7 và 8:", FINAL["name"], f"| AUC CV = {np.mean(FINAL['scores']):.4f}")



>>> Mô hình dùng cho mục 7 và 8: Blend LGB(6i) 0.6 + CatTune 0.4 (REPLAY, huấn luyện lại 1 lần) [test_pred: huấn luyện lại 100%] | AUC CV = 0.7682


## 6b. Thử nghiệm: mô hình 2 giai đoạn (mặc định tắt)

Ý tưởng: giai đoạn 1 là mô hình chính; giai đoạn 2 chỉ học trên vùng khách có p1 cao (nơi tập trung gần hết ca dương) để sửa riêng vùng đó.

Thiết kế đúng cho AUC, tránh hai lỗi hay gặp:
- **Giữ nguyên thang đo.** Giai đoạn 2 là *boosting phần dư*: `init_score = logit(p1)`, rồi cộng `delta` vào logit của p1. Nếu thay hẳn bằng p2 thì thứ hạng giữa vùng được sửa và vùng không sửa bị lệch, AUC giảm.
- **Không rò rỉ.** p1 của hàng train trong fold là OOF lồng bên trong fold; p1 của hàng validation lấy từ mô hình giai đoạn 1 đã huấn luyện trên toàn bộ train của fold.

So sánh theo cặp từng fold với cùng mô hình một giai đoạn (`lr = 0.05`) bằng quy tắc `delta_se` như các mục trước. Tốn khoảng 6 lần thời gian một lượt CV nhanh. **Đừng kỳ vọng nhiều:** boosting vốn đã tập trung vào mẫu khó, và ver4 cho thấy thêm mô hình thứ hai (meta-feature, stacking) không thắng mô hình đơn đã tuning.

In [32]:
RUN_CASCADE = False        # True để chạy thử
BAND_Q = 0.45              # giai đoạn 2 chỉ học trên 55% khách có p1 cao nhất
if RUN_CASCADE:
    from sklearn.model_selection import train_test_split
    lg2 = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
    S2 = dict(objective="binary", n_estimators=2000, learning_rate=0.03, num_leaves=7, max_depth=3,
              min_child_samples=200, colsample_bytree=0.5, subsample=0.8, subsample_freq=1,
              reg_lambda=20.0, random_state=SEED, n_jobs=-1, verbose=-1)
    ref1 = run_lgb_cv(FEATS, TE_USED, FAST_PARAMS, es_rounds=ES_FAST, name="1 giai đoạn (lr 0.05)")
    oof_c, sc_c, n_band, n_trees = ref1["oof"].copy(), [], [], []
    for tr, va in SPLITS:
        Xtr, Xva, _, _ = fold_data(FEATS, TE_USED, tr, va)
        ytr, yva = y[tr], y[va]
        p1_tr = np.zeros(len(tr))
        for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED + 1).split(Xtr, ytr):
            m1 = lgb.LGBMClassifier(**{**FAST_PARAMS, "random_state": SEED})
            m1.fit(Xtr.iloc[a], ytr[a], eval_set=[(Xtr.iloc[b], ytr[b])], eval_metric="auc",
                   callbacks=[lgb.early_stopping(ES_FAST, verbose=False), lgb.log_evaluation(0)])
            p1_tr[b] = m1.predict_proba(Xtr.iloc[b])[:, 1]
        p1_va = ref1["oof"][va]                      # cùng mô hình giai đoạn 1 của ref1 (cùng seed, cùng fold)
        thr = np.quantile(p1_tr, BAND_Q)
        btr, bva = p1_tr >= thr, p1_va >= thr
        # early stopping của giai đoạn 2 dùng 15% hàng giữ lại trong train fold, KHÔNG dùng fold validation,
        # để delta không bị lạc quan thêm. (Mô hình 1 giai đoạn vẫn early stopping trên validation nên phép so sánh hơi bất lợi cho cascade.)
        fit_i, es_i = train_test_split(np.where(btr)[0], test_size=0.15, stratify=ytr[btr], random_state=SEED)
        m2 = lgb.LGBMClassifier(**S2)
        m2.fit(Xtr.iloc[fit_i], ytr[fit_i], init_score=lg2(p1_tr[fit_i]),
               eval_set=[(Xtr.iloc[es_i], ytr[es_i])], eval_init_score=[lg2(p1_tr[es_i])], eval_metric="auc",
               callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof_c[va[bva]] = expit(lg2(p1_va[bva]) + m2.predict(Xva[bva], raw_score=True))
        sc_c.append(roc_auc_score(yva, oof_c[va]))
        n_band.append(int(bva.sum())); n_trees.append(m2.best_iteration_)
    res_casc = {"oof": oof_c, "scores": sc_c, "name": "2 giai đoạn"}
    c = compare(res_casc, ref1)
    print(f"[2 giai đoạn] AUC = {np.mean(sc_c):.4f} | 1 giai đoạn = {np.mean(ref1['scores']):.4f} | "
          f"delta = {c['delta']:+.4f} (se {c['delta_se']:.4f}, fold dương {c['fold_duong']}) -> {c['verdict']}")
    print(f"Số hàng vùng giai đoạn 2 mỗi fold: {n_band} | số cây giai đoạn 2: {n_trees}")
    print("Chỉ nên đưa vào mô hình cuối nếu verdict là 'tốt hơn'; khi đó cần thêm bước dự đoán test theo từng fold.")

## 7. Ngưỡng quyết định: đánh đổi giữa bắt được ca dương và báo nhầm

**Nếu cuộc thi chấm bằng AUC ROC (đã xác nhận), ngưỡng không ảnh hưởng điểm số.** AUC chỉ phụ thuộc thứ hạng của xác suất, nên bài nộp là `submission.csv` (xác suất, mục 8). Mục này chỉ để hiểu đánh đổi khi cần một quyết định 0/1 (ví dụ báo cáo nghiệp vụ).

**Mô hình không "phán đoán sai"; đây là hệ quả của AUC ≈ 0.76 trên nhãn chỉ có 8% dương.** Bảng hiệu chuẩn bên dưới cho thấy xác suất dự đoán khớp tỷ lệ thật, tức mô hình trung thực về mức độ không chắc chắn của nó. Ở ver4: chỉ khoảng 475 khách có p ≥ 0.5 (trong đó 54% thật sự vỡ nợ), nên ngưỡng 0.5 gần như không gắn cờ ai. Muốn bắt được 40% ca dương thì phải hạ ngưỡng xuống 0.155 và chấp nhận gắn nhầm 16.8 nghìn khách tốt. **Không có ngưỡng nào cho cả hai**; ngưỡng chỉ chọn một điểm trên đường đánh đổi đó, và điểm nào đúng phụ thuộc vào chỉ số chấm bài.

Đặt `SUBMISSION_METRIC` theo chỉ số thật của cuộc thi:
- `"accuracy"`: gắn cờ khi và chỉ khi mô hình tin trên 50%. Rất ít ca dương, rất ít báo nhầm.
- `"f1"` hoặc `"balanced_accuracy"`: ngưỡng thấp, gắn cờ 13% đến 34% số khách.
- `"recall:0.6"` (hoặc số khác): ngưỡng thấp nhất vẫn đạt recall mong muốn.

In [33]:
SUBMISSION_METRIC = "f1"            # chỉ dùng cho submission_binary.csv. Không ảnh hưởng AUC. "accuracy" | "f1" | "balanced_accuracy" | "recall:0.6"
oof = FINAL["oof"]
n_pos, n_neg = int(y.sum()), int(len(y) - y.sum())


def best_acc_threshold(y_true, p):
    n, neg = len(y_true), int((y_true == 0).sum())
    o = np.argsort(-p, kind="mergesort")
    ys, ps = y_true[o], p[o]
    tp = np.cumsum(ys)
    k = np.arange(1, n + 1)
    acc = (tp + neg - (k - tp)) / n
    valid = np.r_[ps[:-1] != ps[1:], True]
    i = int(np.argmax(np.where(valid, acc, -1)))
    if neg / n >= acc[i]:
        return float(ps[0] + 1e-9), neg / n          # không k nào hơn mốc toàn "0"
    return float((ps[i] + ps[i + 1]) / 2 if i + 1 < n else ps[i]), float(acc[i])


def threshold_for(metric, p):
    if metric == "accuracy":
        return best_acc_threshold(y, p)[0]
    o = np.argsort(-p, kind="mergesort")
    ps, ys = p[o], y[o]
    tp = np.cumsum(ys)
    fp = np.arange(1, len(y) + 1) - tp
    valid = np.r_[ps[:-1] != ps[1:], True]
    rec, prec = tp / n_pos, tp / (tp + fp)
    if metric == "f1":
        score = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    elif metric == "balanced_accuracy":
        score = (rec + (n_neg - fp) / n_neg) / 2
    elif metric.startswith("recall:"):
        score = np.where(rec >= float(metric.split(":")[1]), -np.arange(len(y)), -np.inf)   # k nhỏ nhất đạt recall
    else:
        raise ValueError(metric)
    i = int(np.argmax(np.where(valid, score, -np.inf)))
    return float((ps[i] + ps[i + 1]) / 2 if i + 1 < len(y) else ps[i])


def confusion_at(p, thr):
    pred = p >= thr
    tp, fp = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum())
    fn, tn = n_pos - tp, n_neg - fp
    rec, prec = tp / n_pos, tp / max(tp + fp, 1)
    return {"ngưỡng": round(thr, 4), "gắn cờ (%)": 100 * pred.mean(), "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": prec, "recall": rec, "accuracy": (tp + tn) / len(y),
            "F1": 2 * prec * rec / max(prec + rec, 1e-12), "balanced_acc": (rec + tn / n_neg) / 2}


METRICS = ["accuracy", "f1", "balanced_accuracy", "recall:0.6"]
tbl = pd.DataFrame({m: confusion_at(oof, threshold_for(m, oof)) for m in METRICS}).T
tbl.loc["toàn '0'"] = confusion_at(oof, 2.0)
display(tbl.round(4))
print("Mỗi dòng là một ngưỡng tối ưu cho một chỉ số. Cột FP là số khách tốt bị gắn nhầm, FN là số ca dương bị bỏ sót.")

SUBMISSION_THR = threshold_for(SUBMISSION_METRIC, oof)
print(f"\nSUBMISSION_METRIC = {SUBMISSION_METRIC} -> ngưỡng nộp bài = {SUBMISSION_THR:.4f}")

# Ước lượng trung thực cho accuracy: ngưỡng chọn trên 4 fold, đo trên fold còn lại
rows = []
for tr, va in SPLITS:
    thr, _ = best_acc_threshold(y[tr], oof[tr])
    pred = (oof[va] >= thr).astype(int)
    rows.append({"acc": (pred == y[va]).mean(), "base": 1 - y[va].mean(), "flag": pred.mean()})
na = pd.DataFrame(rows)
d = na["acc"] - na["base"]
print(f"Accuracy trung thực: {na['acc'].mean():.4%} | hơn mốc toàn '0' {100 * d.mean():+.3f} điểm % "
      f"(delta_se {100 * d.std(ddof=1) / np.sqrt(len(d)):.3f}) | tỷ lệ gắn cờ {na['flag'].mean():.2%}")

edges = [0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 1.01]
cal = pd.DataFrame({"p": oof, "y": y}).groupby(pd.cut(oof, edges, right=False), observed=True).agg(
    số_dòng=("y", "size"), xác_suất_dự_đoán=("p", "mean"), tỷ_lệ_thật=("y", "mean"))
display(cal.round(4))
print("Hai cột xác suất gần nhau nghĩa là mô hình đã hiệu chuẩn (không phải mô hình bị lỗi).")

,ngưỡng,gắn cờ (%),TP,FP,FN,TN,precision,recall,accuracy,F1,balanced_acc
accuracy,0.5001,0.3090,298.0,231.0,13523.0,157150.0,0.5633,0.0216,0.9197,0.0415,0.5100
f1,0.1421,15.5863,6342.0,20342.0,7479.0,137039.0,0.2377,0.4589,0.8375,0.3131,0.6648
balanced_accuracy,0.0749,35.7875,10021.0,51248.0,3800.0,106133.0,0.1636,0.7251,0.6785,0.2669,0.6997
recall:0.6,0.1033,24.8122,8293.0,34186.0,5528.0,123195.0,0.1952,0.6000,0.7680,0.2946,0.6914
toàn '0',2.0000,0.0000,0.0,0.0,13821.0,157381.0,0.0000,0.0000,0.9193,0.0000,0.5000


Mỗi dòng là một ngưỡng tối ưu cho một chỉ số. Cột FP là số khách tốt bị gắn nhầm, FN là số ca dương bị bỏ sót.

SUBMISSION_METRIC = f1 -> ngưỡng nộp bài = 0.1421
Accuracy trung thực: 91.9580% | hơn mốc toàn '0' +0.031 điểm % (delta_se 0.009) | tỷ lệ gắn cờ 0.31%


,số_dòng,xác_suất_dự_đoán,tỷ_lệ_thật
"[0.0, 0.05)",83783,0.0268,0.0260
"[0.05, 0.1)",43216,0.0713,0.0732
"[0.1, 0.15)",19788,0.1222,0.1269
"[0.15, 0.2)",10126,0.1725,0.1714
"[0.2, 0.3)",9177,0.2415,0.2460
"[0.3, 0.4)",3373,0.3406,0.3430
"[0.4, 0.5)",1209,0.4433,0.4293
"[0.5, 1.01)",530,0.5645,0.5623


Hai cột xác suất gần nhau nghĩa là mô hình đã hiệu chuẩn (không phải mô hình bị lỗi).


## 8. File nộp bài

**Với AUC ROC, nộp `submission.csv` (xác suất).** `submission_binary.csv` chỉ để tham khảo vì nhãn 0/1 làm mất thông tin thứ hạng.

Tên cột lấy từ `sample_submission.csv` nếu có trong thư mục dữ liệu. Nếu không có, dùng `SK_ID_CURR`/`TARGET` như ver4: **hãy đối chiếu với mẫu nộp của cuộc thi** trước khi nộp.

Ô này in thêm tỷ lệ dòng bị gắn cờ trên test so với OOF. Ở ver4 tỷ lệ này là 0.37% (test) so với 0.24% (OOF), lệch khoảng 1.5 lần, nhiều khả năng do meta-feature của mô hình M1 (xác suất `p_A` của test không đi qua CV lồng như của train). Bản ver5 không dùng meta-feature, nên tỷ lệ hai bên phải gần nhau hơn; nếu vẫn lệch trên 1.5 lần thì dùng ngưỡng khớp tỷ lệ gắn cờ (in bên dưới) thay cho `SUBMISSION_THR`.

In [34]:
sample_path = f"{DATA_PATH}/sample_submission.csv"
if os.path.exists(sample_path):
    sc = pd.read_csv(sample_path, nrows=1).columns.tolist()
    ID_NAME, TGT_NAME = sc[0], sc[1]
else:
    ID_NAME, TGT_NAME = "SK_ID_CURR", "TARGET"
    print("Không thấy sample_submission.csv: dùng SK_ID_CURR/TARGET, cần đối chiếu mẫu nộp của cuộc thi.")

p_test = FINAL["test_pred"]
ids = test[ID_COL].to_numpy()
pd.DataFrame({ID_NAME: ids, TGT_NAME: p_test}).to_csv(f"{OUT_PATH}/submission.csv", index=False)
pd.DataFrame({ID_NAME: ids, TGT_NAME: (p_test >= SUBMISSION_THR).astype(int)}).to_csv(f"{OUT_PATH}/submission_binary.csv", index=False)

rate_oof, rate_test = (oof >= SUBMISSION_THR).mean(), (p_test >= SUBMISSION_THR).mean()
print(f"Đã lưu submission.csv (xác suất) và submission_binary.csv (nhãn 0/1, chỉ số {SUBMISSION_METRIC}, ngưỡng {SUBMISSION_THR:.3f}) -> {OUT_PATH}")
print(f"Tỷ lệ gắn cờ: OOF = {rate_oof:.2%} | test = {rate_test:.2%} | tỉ số = {rate_test / max(rate_oof, 1e-9):.2f}")
if rate_oof > 0 and not (1 / 1.5 <= rate_test / rate_oof <= 1.5):
    thr_match = float(np.quantile(p_test, 1 - rate_oof))
    print(f"Lệch nhiều: ngưỡng khớp tỷ lệ gắn cờ OOF trên test = {thr_match:.3f} (cân nhắc dùng thay SUBMISSION_THR)")

Không thấy sample_submission.csv: dùng SK_ID_CURR/TARGET, cần đối chiếu mẫu nộp của cuộc thi.
Đã lưu submission.csv (xác suất) và submission_binary.csv (nhãn 0/1, chỉ số f1, ngưỡng 0.142) -> /kaggle/working
Tỷ lệ gắn cờ: OOF = 15.59% | test = 19.06% | tỉ số = 1.22


## 6an. Mở rộng hướng giá vay (chạy sau khi đã có submission)

Khối `price` là tín hiệu thật đầu tiên sau nhiều vòng. Ở đây thử các biến thể, so với cấu hình **đã có price**:

| Khối | Ý nghĩa |
|---|---|
| `grid` | lãi suất ngầm định ở mọi kỳ hạn chuẩn khả thi (12 đến 60), số kỳ hạn khả thi, độ dốc lãi suất 24→60 |
| `rel` | lãi suất so với nhóm cùng kỳ hạn chuẩn (và cùng loại hợp đồng): z và thứ hạng |
| `ext` | lãi suất so với mức lẽ ra phải có theo điểm ext (phần dư), lãi suất × (1 − ext), lãi suất / ext |
| `cost` | tổng lãi phải trả, tỉ lệ lãi trên vay, lãi trên thu nhập, tiền trả cả năm trên thu nhập |

Quy trình như 6al: tầng 1 từng khối so với mốc (có price), tầng 2 xác nhận trên bộ fold khác, rồi 5 seed với tham số cuối so với LGB và blend đang dùng (đều đã có price). Nếu qua cổng thì bản nộp ở mục 8 chưa dùng các đặc trưng này: cần chạy thêm một lần refit.

In [35]:
RUN_6AN = True      # chạy SAU khi đã ghi submission: không ảnh hưởng bản nộp
if RUN_6AN and PRICE_ON and "res_cat_tuned" in globals() and "LGB_BEST" in globals() and res_cat_tuned.get("oof") is not None:
    t0l = time.time()
    F_ = FULL
    ext_m = F_["ext_mean_ml"].astype(float)
    inc = F_["income"].astype(float)
    cr_s, an_s = pd.Series(PR_C, index=F_.index), pd.Series(PR_A, index=F_.index)
    n1s = pd.Series(PR_N1, index=F_.index)
    dv = lambda a, b: a / b.replace(0, np.nan)
    BL = {"grid": {}, "rel": {}, "ext": {}, "cost": {}}
    # A: lãi suất ở mọi kỳ hạn chuẩn khả thi (không chỉ hai kỳ hạn gần nhất)
    for n in (12, 18, 24, 36, 48, 60):
        BL["grid"][f"q_rate_{n}"] = PR_rate_for(np.full(len(PR_C), float(n)))
    BL["grid"]["q_n_feasible"] = np.sum([np.isfinite(PR_rate_for(np.full(len(PR_C), float(n)))) for n in (12, 18, 24, 36, 48, 60)], axis=0).astype(float)
    BL["grid"]["q_rate_slope"] = BL["grid"]["q_rate_60"] - BL["grid"]["q_rate_24"]
    # B: lãi suất so với nhóm cùng kỳ hạn chuẩn và loại hợp đồng
    r1 = pd.Series(PR_R1, index=F_.index)
    for kn, key in (("n1", n1s.astype(str)), ("n1_ct", n1s.astype(str) + "|" + F_["name_contract_type"].astype(str) if "name_contract_type" in F_.columns else n1s.astype(str))):
        gg = r1.groupby(key.to_numpy())
        mu, sd_, cnt = gg.transform("mean"), gg.transform("std"), gg.transform("count")
        BL["rel"][f"q_rate_z_{kn}"] = ((r1 - mu) / sd_.replace(0, np.nan)).where(cnt >= 50).to_numpy()
        BL["rel"][f"q_rate_pc_{kn}"] = gg.rank(pct=True).where(cnt >= 50).to_numpy()
    # C: lãi suất so với điều lẽ ra phải có theo điểm ext (phần dư), và nhân với (1 - ext)
    eb = pd.qcut(ext_m, 10, labels=False, duplicates="drop")
    key_e = n1s.astype(str) + "|" + pd.Series(eb, index=F_.index).astype(str)
    exp_r = r1.groupby(key_e.to_numpy()).transform("mean")
    BL["ext"] = {"q_rate_resid": (r1 - exp_r).to_numpy(), "q_rate_x_lowext": (r1 * (1 - ext_m)).to_numpy(),
                 "q_rate_over_ext": dv(r1, ext_m).to_numpy()}
    # D: chi phí vay: tổng lãi, tỉ lệ lãi trên vay, trên thu nhập
    tot_int = an_s * n1s - cr_s
    BL["cost"] = {"q_total_int": tot_int.to_numpy(), "q_int_share": dv(tot_int, cr_s).to_numpy(), "q_int_over_inc": dv(tot_int, inc).to_numpy(),
                  "q_gap_rel": dv(n1s - pd.Series(PR_TERM, index=F_.index), n1s).to_numpy(), "q_pay_cycle_inc": dv(an_s * 12, inc).to_numpy()}
    ALLL = pd.DataFrame({k: np.asarray(v, dtype=float) for d_ in BL.values() for k, v in d_.items()}, index=F_.index).replace([np.inf, -np.inf], np.nan).astype("float32")
    X_all = pd.concat([X_all.drop(columns=[c for c in ALLL.columns if c in X_all.columns]), ALLL.iloc[:n_tr].reset_index(drop=True)], axis=1)
    X_test_all = pd.concat([X_test_all.drop(columns=[c for c in ALLL.columns if c in X_test_all.columns]), ALLL.iloc[n_tr:].reset_index(drop=True)], axis=1)
    BLK6AN = {k: list(d_.keys()) for k, d_ in BL.items()}
    print("Khối dựng xong:", {k: len(v) for k, v in BLK6AN.items()}, f"| tổng {ALLL.shape[1]} cột, {time.time() - t0l:.0f}s")
    print("Thiếu nhiều nhất (%):", (ALLL.iloc[:n_tr].isna().mean() * 100).sort_values(ascending=False).head(3).round(1).to_dict())
    print("Cột hằng số:", [c for c in ALLL.columns if ALLL.iloc[:n_tr][c].std() == 0] or "không có")

    cfg_ref = dict(LGB_BEST["cfg"])
    def add_l(cfgx, cols):
        o = dict(cfgx); o["feats"] = list(cfgx["feats"]) + [c for c in cols if c not in cfgx["feats"]]; return o
    base_l = run_cfg3(cfg_ref, FAST_PARAMS, es=ES_FAST, name="mốc 6an (FAST)")
    rows, passed = [{"khối": "mốc", "AUC": np.mean(base_l["scores"]), "delta": 0.0, "se": 0.0, "fold dương": "-", "verdict": "-"}], []
    for nm, cols in list(BLK6AN.items()) + [("ALL", [c for v in BLK6AN.values() for c in v])]:
        r = run_cfg3(add_l(cfg_ref, cols), FAST_PARAMS, es=ES_FAST, name=f"+{nm} ({len(cols)})")
        c = compare(r, base_l)
        rows.append({"khối": nm, "AUC": np.mean(r["scores"]), "delta": c["delta"], "se": c["delta_se"], "fold dương": c["fold_duong"], "verdict": c["verdict"]})
        if nm != "ALL" and c["verdict"] == "tốt hơn":
            passed.append(nm)
    display(pd.DataFrame(rows).set_index("khối").round(4))
    print("Qua tầng 1:", passed or "không có")
    if passed:
        cols_p = [c for nm in passed for c in BLK6AN[nm]]
        SPL_MAIN = SPLITS
        SPLITS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 303).split(np.zeros(len(y)), y))
        try:
            a_ = run_cfg3(cfg_ref, FAST_PARAMS, es=ES_FAST, name="XÁC NHẬN: mốc, fold mới")
            b_ = run_cfg3(add_l(cfg_ref, cols_p), FAST_PARAMS, es=ES_FAST, name=f"XÁC NHẬN: +{passed}, fold mới")
        finally:
            SPLITS = SPL_MAIN
        c2 = compare(b_, a_)
        print(f"Xác nhận fold mới: delta {c2['delta']:+.4f} (se {c2['delta_se']:.4f}, fold dương {c2['fold_duong']}) -> {c2['verdict']}")
        if c2["verdict"] == "tốt hơn":
            r_f = run_cfg3(add_l(cfg_ref, cols_p), PARAMS, seeds=FINAL_SEEDS, es=ES_FINAL, name=f"LGB(6i)+{passed}, tham số cuối, {len(FINAL_SEEDS)} seed")
            lg_l = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))
            W_L = globals().get("W_BLEND", 0.6)
            bl = lambda r: {"oof": (o_ := expit(W_L * lg_l(r["oof"]) + (1 - W_L) * lg_l(res_cat_tuned["oof"]))), "scores": [roc_auc_score(y[va], o_[va]) for _, va in SPLITS]}
            cl, cb = compare(r_f, LGB_BEST), compare(bl(r_f), bl(LGB_BEST))
            display(pd.DataFrame([{"mô hình": "LGB", "AUC cũ": np.mean(LGB_BEST["scores"]), "AUC mới": np.mean(r_f["scores"]), "delta": cl["delta"], "se": cl["delta_se"], "fold dương": cl["fold_duong"], "verdict": cl["verdict"]},
                                  {"mô hình": "blend", "AUC cũ": np.mean(bl(LGB_BEST)["scores"]), "AUC mới": np.mean(bl(r_f)["scores"]), "delta": cb["delta"], "se": cb["delta_se"], "fold dương": cb["fold_duong"], "verdict": cb["verdict"]}]).set_index("mô hình").round(4))
            print("QUA CỔNG" if (cl["verdict"] == "tốt hơn" and cb["verdict"] == "tốt hơn" and cb["delta"] >= 0.0003) else "KHÔNG qua cổng cuối: giữ cấu hình hiện tại.")
        else:
            print("Không xác nhận được trên fold mới: giữ cấu hình hiện tại.")
    else:
        print("Không khối nào qua tầng 1: giữ cấu hình hiện tại.")
    print(f"6an xong sau {(time.time() - t0l) / 60:.1f} phút")


Khối dựng xong: {'grid': 8, 'rel': 4, 'ext': 3, 'cost': 5} | tổng 20 cột, 7s
Thiếu nhiều nhất (%): {'q_rate_12': 88.4, 'q_rate_18': 68.4, 'q_rate_24': 32.7}
Cột hằng số: không có
[mốc 6an (FAST)] AUC = 0.7660 ± 0.0030 | log-loss = 0.24402 | 108 feature | cây TB = 509 | 85s
[+grid (8)] AUC = 0.7661 ± 0.0027 | log-loss = 0.24403 | 116 feature | cây TB = 470 | 84s
[+rel (4)] AUC = 0.7658 ± 0.0027 | log-loss = 0.24403 | 112 feature | cây TB = 481 | 86s
[+ext (3)] AUC = 0.7658 ± 0.0023 | log-loss = 0.24405 | 111 feature | cây TB = 513 | 87s
[+cost (5)] AUC = 0.7658 ± 0.0025 | log-loss = 0.24406 | 113 feature | cây TB = 520 | 88s
[+ALL (20)] AUC = 0.7665 ± 0.0027 | log-loss = 0.24389 | 128 feature | cây TB = 506 | 93s


,AUC,delta,se,fold dương,verdict
khối,,,,,
mốc,0.7660,0.0000,0.0000,-,-
grid,0.7661,0.0001,0.0002,3/5,≈ như
rel,0.7658,-0.0002,0.0002,3/5,kém hơn
ext,0.7658,-0.0002,0.0004,3/5,≈ như
cost,0.7658,-0.0002,0.0003,3/5,≈ như
ALL,0.7665,0.0004,0.0003,4/5,tốt hơn


Qua tầng 1: không có
Không khối nào qua tầng 1: giữ cấu hình hiện tại.
6an xong sau 8.8 phút


## 9. Lưu ý và giới hạn

1. **Trần của bộ dữ liệu một bảng.** Mọi cải tiến ở ver4 cộng lại chỉ đưa AUC từ 0.7516 lên 0.7631. Kỳ vọng hợp lý của ver5 là khoảng 0.763 đến 0.765, không phải bước nhảy. Muốn lên 0.78 trở lên cần bảng phụ (bureau, previous_application), điều mà dữ liệu hiện có không cho.
2. **Các con số ver4 dùng làm cơ sở** (T2 = 0.7631, D4 = 0.7598, D9 = 0.7608, Optuna = 0.7630) lấy từ output thật, không phải từ phần markdown tổng kết ở mục 14 của ver4. Phần tổng kết đó còn số cũ từ lần chạy trước (ví dụ nói 143 feature và T1 = 0.7623, trong khi output thật là 157 feature và T2 = 0.7631 là cấu hình tốt nhất). Nên cập nhật hoặc xóa phần đó nếu nộp báo cáo.
3. **Khối `te_extra` và `cnt` là giả thuyết.** Nếu mục 5 báo "không" thì chúng tự bị loại, đó là kết quả bình thường.
4. **Thay đổi `colsample_bytree` từ 0.3 lên 0.4** là điều chỉnh có lý do (số cột giảm một nửa) nhưng chưa được đo riêng. Vùng 0.3 đến 0.5 phẳng ở ver4, nên rủi ro thấp.
5. **Early stopping dùng chính fold validation**, nên AUC báo cáo hơi lạc quan như ở ver4. Điều này áp dụng đều cho mọi cấu hình nên so sánh theo cặp vẫn đáng tin.